In [1]:
import sys
import torch

print("Python:", sys.version)
print("PyTorch:", torch.__version__)
print("CUDA runtime:", torch.version.cuda)

assert torch.cuda.is_available(), "CUDA is not available"

print("GPU:", torch.cuda.get_device_name(0))
print("Compute capability:", torch.cuda.get_device_capability(0))

props = torch.cuda.get_device_properties(0)
print("VRAM:", round(props.total_memory / 1024**3, 2), "GiB")
print("SMs:", props.multi_processor_count)

Python: 3.12.11 | packaged by Anaconda, Inc. | (main, Jun  5 2025, 13:09:17) [GCC 11.2.0]
PyTorch: 2.8.0+cu128
CUDA runtime: 12.8
GPU: NVIDIA A100-SXM4-40GB
Compute capability: (8, 0)
VRAM: 39.49 GiB
SMs: 108


In [2]:
!nvidia-smi

Mon Sep 14 04:57:34 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 570.148.08             Driver Version: 570.148.08     CUDA Version: 12.8     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-40GB          Off |   00000000:06:00.0 Off |                    0 |
| N/A   37C    P0             68W /  400W |       4MiB /  40960MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

In [3]:
import triton

print("Triton:", triton.__version__)

Triton: 3.4.0


In [4]:
import triton

print("Triton:", triton.__version__)

print("TF32 matmul enabled:",
      torch.backends.cuda.matmul.allow_tf32)

print("TF32 cuDNN enabled:",
      torch.backends.cudnn.allow_tf32)

Triton: 3.4.0
TF32 matmul enabled: False
TF32 cuDNN enabled: True


In [5]:
torch.backends.cuda.matmul.allow_tf32 = False
torch.backends.cudnn.allow_tf32 = False

In [6]:
print("TF32 matmul enabled:",
      torch.backends.cuda.matmul.allow_tf32)

print("TF32 cuDNN enabled:",
      torch.backends.cudnn.allow_tf32)

TF32 matmul enabled: False
TF32 cuDNN enabled: False


In [7]:
M = 4096
N = 14336
K = 512

BM = 64
BN = 128
BK = 64

WARPS = 4
PREC = "ieee"

dtype = torch.float32

In [8]:
import torch

print("Clock / utilization baseline:")
!nvidia-smi --query-gpu=name,temperature.gpu,power.draw,power.limit,clocks.sm,clocks.mem,utilization.gpu,memory.used --format=csv

Clock / utilization baseline:
name, temperature.gpu, power.draw [W], power.limit [W], clocks.current.sm [MHz], clocks.current.memory [MHz], utilization.gpu [%], memory.used [MiB]
NVIDIA A100-SXM4-40GB, 37, 68.08 W, 400.00 W, 210 MHz, 1215 MHz, 0 %, 4 MiB


# Kernel Execution

In [9]:
import math
import statistics
import sys

import torch
import triton
import triton.language as tl

torch.set_grad_enabled(False)

DEVICE = "cuda"

# EXACT same workload as T4
M = 4096
N = 14336
K = 512

LR = 1e-3
CLIP = 1e-5

# More samples than the original notebook
WARMUP = 10
ITERS = 50

# IMPORTANT on A100.
# First experiment = strict FP32, not TF32.
torch.backends.cuda.matmul.allow_tf32 = False
torch.backends.cudnn.allow_tf32 = False

torch.manual_seed(0)

print("=" * 72)
print("ENVIRONMENT")
print("=" * 72)
print("Python:", sys.version)
print("PyTorch:", torch.__version__)
print("CUDA runtime:", torch.version.cuda)
print("Triton:", triton.__version__)
print("GPU:", torch.cuda.get_device_name(0))
print("Compute capability:", torch.cuda.get_device_capability(0))

props = torch.cuda.get_device_properties(0)

print("VRAM GiB:", props.total_memory / 1024**3)
print("SM count:", props.multi_processor_count)
print("TF32 matmul:", torch.backends.cuda.matmul.allow_tf32)
print("TF32 cuDNN:", torch.backends.cudnn.allow_tf32)

!nvidia-smi

ENVIRONMENT
Python: 3.12.11 | packaged by Anaconda, Inc. | (main, Jun  5 2025, 13:09:17) [GCC 11.2.0]
PyTorch: 2.8.0+cu128
CUDA runtime: 12.8
Triton: 3.4.0
GPU: NVIDIA A100-SXM4-40GB
Compute capability: (8, 0)
VRAM GiB: 39.4945068359375
SM count: 108
TF32 matmul: False
TF32 cuDNN: False
Mon Sep 14 04:57:51 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 570.148.08             Driver Version: 570.148.08     CUDA Version: 12.8     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-40GB        

In [10]:
gen = torch.Generator(device=DEVICE).manual_seed(0)

# Z: [seq_len, d_inter]
Z = torch.randn(
    K, N,
    device=DEVICE,
    dtype=torch.float32,
    generator=gen,
)

# V: [seq_len, d_model]
V = torch.randn(
    K, M,
    device=DEVICE,
    dtype=torch.float32,
    generator=gen,
)

# [d_model, seq_len]
VT = V.T.contiguous()

# Initial fast weights
W0 = torch.randn(
    M, N,
    device=DEVICE,
    dtype=torch.float32,
    generator=gen,
)

# Lower-amplitude case for clipped correctness testing
Z_real = (
    torch.randn(
        K, N,
        device=DEVICE,
        dtype=torch.float32,
        generator=gen,
    ) * 0.05
)

V_real = (
    torch.randn(
        K, M,
        device=DEVICE,
        dtype=torch.float32,
        generator=gen,
    ) * 0.05
)

VT_real = V_real.T.contiguous()

ONE_DW_MB = M * N * 4 / 1e6

print(f"dW shape       : {M} x {N}")
print(f"One FP32 dW    : {ONE_DW_MB:.2f} MB")
print(f"Chunk size K   : {K}")

dW shape       : 4096 x 14336
One FP32 dW    : 234.88 MB
Chunk size K   : 512


In [11]:
@triton.jit
def ttt_norm_kernel(
    v_t_ptr,
    z_ptr,
    sq_norm_ptr,

    stride_vm,
    stride_vk,
    stride_zk,
    stride_zn,

    M: tl.constexpr,
    N: tl.constexpr,
    K: tl.constexpr,

    BLOCK_M: tl.constexpr,
    BLOCK_N: tl.constexpr,
    BLOCK_K: tl.constexpr,
    PREC: tl.constexpr,
):
    pid_m = tl.program_id(0)
    pid_n = tl.program_id(1)

    offs_m = pid_m * BLOCK_M + tl.arange(0, BLOCK_M)
    offs_n = pid_n * BLOCK_N + tl.arange(0, BLOCK_N)

    acc = tl.zeros(
        (BLOCK_M, BLOCK_N),
        dtype=tl.float32,
    )

    for k in range(0, K, BLOCK_K):
        offs_k = k + tl.arange(0, BLOCK_K)

        v_ptrs = (
            v_t_ptr
            + offs_m[:, None] * stride_vm
            + offs_k[None, :] * stride_vk
        )

        z_ptrs = (
            z_ptr
            + offs_k[:, None] * stride_zk
            + offs_n[None, :] * stride_zn
        )

        v_tile = tl.load(
            v_ptrs,
            mask=(
                (offs_m[:, None] < M)
                & (offs_k[None, :] < K)
            ),
            other=0.0,
        )

        z_tile = tl.load(
            z_ptrs,
            mask=(
                (offs_k[:, None] < K)
                & (offs_n[None, :] < N)
            ),
            other=0.0,
        )

        acc = tl.dot(
            v_tile,
            z_tile,
            acc,
            input_precision=PREC,
        )

    local_sq_sum = tl.sum(acc * acc)

    tl.atomic_add(
        sq_norm_ptr,
        local_sq_sum,
    )

In [12]:
@triton.jit
def ttt_update_kernel(
    v_t_ptr,
    z_ptr,
    w_ptr,
    sq_norm_ptr,

    stride_vm,
    stride_vk,
    stride_zk,
    stride_zn,
    stride_wm,
    stride_wn,

    M: tl.constexpr,
    N: tl.constexpr,
    K: tl.constexpr,

    learning_rate,
    clip_threshold,

    BLOCK_M: tl.constexpr,
    BLOCK_N: tl.constexpr,
    BLOCK_K: tl.constexpr,
    PREC: tl.constexpr,
):
    pid_m = tl.program_id(0)
    pid_n = tl.program_id(1)

    offs_m = pid_m * BLOCK_M + tl.arange(0, BLOCK_M)
    offs_n = pid_n * BLOCK_N + tl.arange(0, BLOCK_N)

    # Read global sum-of-squares computed in pass 1.
    sq = tl.load(sq_norm_ptr)

    raw_scale = (
        clip_threshold
        / (tl.sqrt(sq) + 1e-8)
    )

    scale = tl.where(
        raw_scale < 1.0,
        raw_scale,
        1.0,
    )

    acc = tl.zeros(
        (BLOCK_M, BLOCK_N),
        dtype=tl.float32,
    )

    # Rematerialize dW tile
    for k in range(0, K, BLOCK_K):
        offs_k = k + tl.arange(0, BLOCK_K)

        v_ptrs = (
            v_t_ptr
            + offs_m[:, None] * stride_vm
            + offs_k[None, :] * stride_vk
        )

        z_ptrs = (
            z_ptr
            + offs_k[:, None] * stride_zk
            + offs_n[None, :] * stride_zn
        )

        v_tile = tl.load(
            v_ptrs,
            mask=(
                (offs_m[:, None] < M)
                & (offs_k[None, :] < K)
            ),
            other=0.0,
        )

        z_tile = tl.load(
            z_ptrs,
            mask=(
                (offs_k[:, None] < K)
                & (offs_n[None, :] < N)
            ),
            other=0.0,
        )

        acc = tl.dot(
            v_tile,
            z_tile,
            acc,
            input_precision=PREC,
        )

    w_ptrs = (
        w_ptr
        + offs_m[:, None] * stride_wm
        + offs_n[None, :] * stride_wn
    )

    mask_w = (
        (offs_m[:, None] < M)
        & (offs_n[None, :] < N)
    )

    w = tl.load(
        w_ptrs,
        mask=mask_w,
        other=0.0,
    )

    w += learning_rate * acc * scale

    tl.store(
        w_ptrs,
        w,
        mask=mask_w,
    )

In [13]:
# Same configuration used by your final T4 benchmark.
BM = 64
BN = 128
BK = 64

WARPS = 4
PREC = "ieee"

grid = (
    triton.cdiv(M, BM),
    triton.cdiv(N, BN),
)

norm_ws = torch.zeros(
    1,
    device=DEVICE,
    dtype=torch.float32,
)


def triton_step(
    w,
    vt,
    z,
    lr=LR,
    clip=CLIP,
):
    norm_ws.zero_()

    # Pass 1
    ttt_norm_kernel[grid](
        vt,
        z,
        norm_ws,

        vt.stride(0),
        vt.stride(1),
        z.stride(0),
        z.stride(1),

        M,
        N,
        K,

        BLOCK_M=BM,
        BLOCK_N=BN,
        BLOCK_K=BK,

        PREC=PREC,
        num_warps=WARPS,
    )

    # Pass 2
    ttt_update_kernel[grid](
        vt,
        z,
        w,
        norm_ws,

        vt.stride(0),
        vt.stride(1),
        z.stride(0),
        z.stride(1),
        w.stride(0),
        w.stride(1),

        M,
        N,
        K,

        lr,
        clip,

        BLOCK_M=BM,
        BLOCK_N=BN,
        BLOCK_K=BK,

        PREC=PREC,
        num_warps=WARPS,
    )

    return w

In [14]:
def pytorch_naive_step(
    w,
    vt,
    z,
    lr=LR,
    clip=CLIP,
):
    dw = vt @ z

    frob = torch.linalg.matrix_norm(dw)

    scale = torch.clamp(
        clip / (frob + 1e-8),
        max=1.0,
    )

    # Allocates SECOND dW-sized tensor.
    scaled_dw = dw * scale

    w.add_(
        scaled_dw,
        alpha=lr,
    )

    return w


def pytorch_inplace_step(
    w,
    vt,
    z,
    lr=LR,
    clip=CLIP,
):
    dw = vt @ z

    frob = torch.linalg.matrix_norm(dw)

    scale = torch.clamp(
        clip / (frob + 1e-8),
        max=1.0,
    )

    # Fairer baseline:
    # reuses original dW storage.
    dw.mul_(scale)

    w.add_(
        dw,
        alpha=lr,
    )

    return w

In [ ]:
def reference_update(
    vt,
    z,
    clip,
):
    dw = vt @ z

    frob = torch.linalg.matrix_norm(dw)

    scale = torch.clamp(
        clip / (frob + 1e-8),
        max=1.0,
    )

    return dw, frob, scale


def correctness_case(
    name,
    vt,
    z,
    clip,
):
    dw_ref, norm_ref, scale_ref = reference_update(
        vt,
        z,
        clip,
    )

    # Use zeros so tiny updates aren't hidden
    # by O(1) random FP32 weights.
    w_ref = torch.zeros_like(W0)

    w_ref.add_(
        dw_ref * scale_ref,
        alpha=LR,
    )

    w_tri = torch.zeros_like(W0)

    triton_step(
        w_tri,
        vt,
        z,
        clip=clip,
    )

    torch.cuda.synchronize()

    diff = w_tri - w_ref

    rel_l2 = (
        torch.linalg.vector_norm(diff)
        / torch.linalg.vector_norm(w_ref).clamp_min(1e-30)
    ).item()

    max_abs = diff.abs().max().item()

    # Validate norm kernel independently.
    norm_ws.zero_()

    ttt_norm_kernel[grid](
        vt,
        z,
        norm_ws,

        vt.stride(0),
        vt.stride(1),
        z.stride(0),
        z.stride(1),

        M,
        N,
        K,

        BLOCK_M=BM,
        BLOCK_N=BN,
        BLOCK_K=BK,

        PREC=PREC,
        num_warps=WARPS,
    )

    torch.cuda.synchronize()

    norm_tri = math.sqrt(norm_ws.item())

    norm_rel_err = (
        abs(norm_tri - norm_ref.item())
        / max(abs(norm_ref.item()), 1e-30)
    )

    print(
        f"{name:<22}"
        f" scale={scale_ref.item():.6e}"
        f" | weight rel-L2={rel_l2:.6e}"
        f" | max abs={max_abs:.6e}"
        f" | norm rel-err={norm_rel_err:.6e}"
    )


print("=" * 72)
print("CORRECTNESS")
print("=" * 72)

correctness_case(
    "No clipping",
    VT,
    Z,
    1e12,
)

correctness_case(
    "Clipped N(0,1)",
    VT,
    Z,
    CLIP,
)

correctness_case(
    "Clipped low-scale",
    VT_real,
    Z_real,
    CLIP,
)

In [15]:
def benchmark(
    fn,
    warmup=WARMUP,
    iterations=ITERS,
):
    for _ in range(warmup):
        fn()

    torch.cuda.synchronize()

    start = torch.cuda.Event(
        enable_timing=True
    )

    end = torch.cuda.Event(
        enable_timing=True
    )

    times = []

    for _ in range(iterations):
        start.record()

        fn()

        end.record()

        end.synchronize()

        times.append(
            start.elapsed_time(end)
        )

    ordered = sorted(times)

    def percentile(p):
        idx = int(
            (len(ordered) - 1) * p
        )
        return ordered[idx]

    return {
        "mean": statistics.mean(times),
        "median": statistics.median(times),
        "best": min(times),
        "p90": percentile(0.90),
        "p99": percentile(0.99),
    }


print("=" * 72)
print("LATENCY — STRICT FP32")
print("=" * 72)


w_naive = W0.clone()

r_naive = benchmark(
    lambda: pytorch_naive_step(
        w_naive,
        VT,
        Z,
    )
)


w_inplace = W0.clone()

r_inplace = benchmark(
    lambda: pytorch_inplace_step(
        w_inplace,
        VT,
        Z,
    )
)


w_triton = W0.clone()

r_triton = benchmark(
    lambda: triton_step(
        w_triton,
        VT,
        Z,
    )
)


print("PyTorch naive  :", r_naive)
print("PyTorch inplace:", r_inplace)
print("Triton 2-pass  :", r_triton)

LATENCY — STRICT FP32
PyTorch naive  : {'mean': 4.947087383270263, 'median': 5.221888065338135, 'best': 4.297728061676025, 'p90': 5.227519989013672, 'p99': 5.232639789581299}
PyTorch inplace: {'mean': 4.302417888641357, 'median': 4.30182409286499, 'best': 4.297728061676025, 'p90': 4.304895877838135, 'p99': 4.3130879402160645}
Triton 2-pass  : {'mean': 99.59088104248048, 'median': 99.58911895751953, 'best': 99.44166564941406, 'p90': 99.65773010253906, 'p99': 99.68844604492188}


In [16]:
def peak_extra_alloc_mb(fn):
    torch.cuda.synchronize()

    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()

    baseline = torch.cuda.memory_allocated()

    fn()

    torch.cuda.synchronize()

    peak = torch.cuda.max_memory_allocated()

    return (
        peak - baseline
    ) / 1e6


print("=" * 72)
print("PEAK TEMPORARY ALLOCATION")
print("=" * 72)


w = W0.clone()

m_naive = peak_extra_alloc_mb(
    lambda: pytorch_naive_step(
        w,
        VT,
        Z,
    )
)


w = W0.clone()

m_inplace = peak_extra_alloc_mb(
    lambda: pytorch_inplace_step(
        w,
        VT,
        Z,
    )
)


w = W0.clone()

m_triton = peak_extra_alloc_mb(
    lambda: triton_step(
        w,
        VT,
        Z,
    )
)


print(
    f"PyTorch naive   : {m_naive:.2f} MB"
)

print(
    f"PyTorch inplace : {m_inplace:.2f} MB"
)

print(
    f"Triton 2-pass   : {m_triton:.2f} MB"
)

print(
    f"One FP32 dW     : {ONE_DW_MB:.2f} MB"
)

print(
    f"Norm workspace  : {norm_ws.numel() * 4} bytes"
)

PEAK TEMPORARY ALLOCATION
PyTorch naive   : 469.76 MB
PyTorch inplace : 234.88 MB
Triton 2-pass   : 0.00 MB
One FP32 dW     : 234.88 MB
Norm workspace  : 4 bytes


In [17]:
print()
print("=" * 72)
print("ASYNC-TTT — A100 ISOLATED UPDATE BENCHMARK")
print("=" * 72)

print(f"""
Hardware
--------
GPU                  : {torch.cuda.get_device_name(0)}
Compute capability   : {torch.cuda.get_device_capability(0)}
SMs                  : {props.multi_processor_count}
VRAM                 : {props.total_memory / 1024**3:.2f} GiB

Software
--------
PyTorch              : {torch.__version__}
CUDA runtime         : {torch.version.cuda}
Triton               : {triton.__version__}

Experiment
----------
M                    : {M}
N                    : {N}
K                    : {K}
dtype                : FP32
TF32                 : disabled
Triton precision     : {PREC}

Tile M               : {BM}
Tile N               : {BN}
Tile K               : {BK}
Warps                : {WARPS}

Latency
-------
PyTorch naive
    median           : {r_naive["median"]:.3f} ms
    mean             : {r_naive["mean"]:.3f} ms
    best             : {r_naive["best"]:.3f} ms
    p90              : {r_naive["p90"]:.3f} ms
    p99              : {r_naive["p99"]:.3f} ms

PyTorch inplace
    median           : {r_inplace["median"]:.3f} ms
    mean             : {r_inplace["mean"]:.3f} ms
    best             : {r_inplace["best"]:.3f} ms
    p90              : {r_inplace["p90"]:.3f} ms
    p99              : {r_inplace["p99"]:.3f} ms

Triton 2-pass
    median           : {r_triton["median"]:.3f} ms
    mean             : {r_triton["mean"]:.3f} ms
    best             : {r_triton["best"]:.3f} ms
    p90              : {r_triton["p90"]:.3f} ms
    p99              : {r_triton["p99"]:.3f} ms

Peak temporary allocation
-------------------------
PyTorch naive        : {m_naive:.2f} MB
PyTorch inplace      : {m_inplace:.2f} MB
Triton 2-pass        : {m_triton:.2f} MB

Derived
-------
Triton / naive latency
                     : {r_triton["median"] / r_naive["median"]:.3f}x

Triton / inplace latency
                     : {r_triton["median"] / r_inplace["median"]:.3f}x

Memory removed vs inplace
                     : {m_inplace - m_triton:.2f} MB
""")


ASYNC-TTT — A100 ISOLATED UPDATE BENCHMARK

Hardware
--------
GPU                  : NVIDIA A100-SXM4-40GB
Compute capability   : (8, 0)
SMs                  : 108
VRAM                 : 39.49 GiB

Software
--------
PyTorch              : 2.8.0+cu128
CUDA runtime         : 12.8
Triton               : 3.4.0

Experiment
----------
M                    : 4096
N                    : 14336
K                    : 512
dtype                : FP32
TF32                 : disabled
Triton precision     : ieee

Tile M               : 64
Tile N               : 128
Tile K               : 64
Warps                : 4

Latency
-------
PyTorch naive
    median           : 5.222 ms
    mean             : 4.947 ms
    best             : 4.298 ms
    p90              : 5.228 ms
    p99              : 5.233 ms

PyTorch inplace
    median           : 4.302 ms
    mean             : 4.302 ms
    best             : 4.298 ms
    p90              : 4.305 ms
    p99              : 4.313 ms

Triton 2-pass
    medi

In [18]:
def norm_pass():
    norm_ws.zero_()

    ttt_norm_kernel[grid](
        VT,
        Z,
        norm_ws,

        VT.stride(0),
        VT.stride(1),
        Z.stride(0),
        Z.stride(1),

        M,
        N,
        K,

        BLOCK_M=BM,
        BLOCK_N=BN,
        BLOCK_K=BK,

        PREC=PREC,
        num_warps=WARPS,
    )


w_debug = W0.clone()


def update_pass():
    ttt_update_kernel[grid](
        VT,
        Z,
        w_debug,
        norm_ws,

        VT.stride(0),
        VT.stride(1),
        Z.stride(0),
        Z.stride(1),
        w_debug.stride(0),
        w_debug.stride(1),

        M,
        N,
        K,

        LR,
        CLIP,

        BLOCK_M=BM,
        BLOCK_N=BN,
        BLOCK_K=BK,

        PREC=PREC,
        num_warps=WARPS,
    )


# Get a valid norm first
norm_pass()
torch.cuda.synchronize()


r_norm = benchmark(norm_pass)
r_update = benchmark(update_pass)

print("=" * 72)
print("TRITON PASS BREAKDOWN")
print("=" * 72)

print("Pass 1 — norm:")
print(r_norm)

print()

print("Pass 2 — update:")
print(r_update)

print()

print(
    "Combined median:",
    r_norm["median"] + r_update["median"],
    "ms"
)

TRITON PASS BREAKDOWN
Pass 1 — norm:
{'mean': 4.471562271118164, 'median': 4.329983949661255, 'best': 4.295680046081543, 'p90': 5.230591773986816, 'p99': 5.243904113769531}

Pass 2 — update:
{'mean': 95.54169860839843, 'median': 95.54636764526367, 'best': 95.43885040283203, 'p90': 95.60883331298828, 'p99': 95.62521362304688}

Combined median: 99.87635159492493 ms


In [19]:
def raw_pytorch_gemm():
    torch.mm(
        VT,
        Z,
    )


r_gemm = benchmark(
    raw_pytorch_gemm
)

print("=" * 72)
print("RAW PYTORCH GEMM")
print("=" * 72)

print(r_gemm)

RAW PYTORCH GEMM
{'mean': 3.22471932888031, 'median': 3.2225279808044434, 'best': 3.2174079418182373, 'p90': 3.240959882736206, 'p99': 3.2419838905334473}


In [22]:
def make_update_grid(bm, bn):
    return (
        triton.cdiv(M, bm),
        triton.cdiv(N, bn),
    )


def run_update_config(
    BM_U,
    BN_U,
    BK_U=64,
    WARPS_U=4,
):
    grid_u = make_update_grid(
        BM_U,
        BN_U,
    )

    w = W0.clone()

    # norm_ws already contains a valid norm,
    # but recompute it once for cleanliness.
    norm_ws.zero_()

    ttt_norm_kernel[grid](
        VT,
        Z,
        norm_ws,

        VT.stride(0),
        VT.stride(1),
        Z.stride(0),
        Z.stride(1),

        M,
        N,
        K,

        BLOCK_M=BM,
        BLOCK_N=BN,
        BLOCK_K=BK,

        PREC=PREC,
        num_warps=WARPS,
    )

    torch.cuda.synchronize()

    def update():
        ttt_update_kernel[grid_u](
            VT,
            Z,
            w,
            norm_ws,

            VT.stride(0),
            VT.stride(1),
            Z.stride(0),
            Z.stride(1),
            w.stride(0),
            w.stride(1),

            M,
            N,
            K,

            LR,
            CLIP,

            BLOCK_M=BM_U,
            BLOCK_N=BN_U,
            BLOCK_K=BK_U,

            PREC=PREC,
            num_warps=WARPS_U,
        )

    result = benchmark(
        update,
        warmup=5,
        iterations=20,
    )

    return result

In [21]:
configs = [
    (64, 128, 64, 4),   # current
    (64, 64, 64, 4),
    (32, 128, 64, 4),
    (32, 64, 64, 4),
    (32, 32, 64, 4),
    (16, 64, 64, 4),
]

print("=" * 90)
print("PASS-2 TILE SWEEP")
print("=" * 90)

results = []

for bm_u, bn_u, bk_u, warps_u in configs:
    try:
        r = run_update_config(
            bm_u,
            bn_u,
            bk_u,
            warps_u,
        )

        results.append(
            (
                bm_u,
                bn_u,
                bk_u,
                warps_u,
                r["median"],
                r["best"],
            )
        )

        print(
            f"{bm_u:>3}x{bn_u:<3} "
            f"BK={bk_u:<3} "
            f"warps={warps_u} | "
            f"median={r['median']:8.3f} ms | "
            f"best={r['best']:8.3f} ms"
        )

    except Exception as e:
        print(
            f"{bm_u}x{bn_u} FAILED:",
            e,
        )

PASS-2 TILE SWEEP
 64x128 BK=64  warps=4 | median=  95.580 ms | best=  95.497 ms
 64x64  BK=64  warps=4 | median=   3.550 ms | best=   3.544 ms
 32x128 BK=64  warps=4 | median=   7.434 ms | best=   7.426 ms
 32x64  BK=64  warps=4 | median= 107.368 ms | best= 107.269 ms
 32x32  BK=64  warps=4 | median=   3.949 ms | best=   3.944 ms
 16x64  BK=64  warps=4 | median=   3.864 ms | best=   3.859 ms


In [23]:
configs = [
    (32, 64, 64, 4),
    (32, 64, 64, 8),

    (32, 128, 64, 4),
    (32, 128, 64, 8),

    (64, 64, 64, 4),
    (64, 64, 64, 8),
]
print("=" * 90)
print("PASS-2 TILE SWEEP")
print("=" * 90)

results = []

for bm_u, bn_u, bk_u, warps_u in configs:
    try:
        r = run_update_config(
            bm_u,
            bn_u,
            bk_u,
            warps_u,
        )

        results.append(
            (
                bm_u,
                bn_u,
                bk_u,
                warps_u,
                r["median"],
                r["best"],
            )
        )

        print(
            f"{bm_u:>3}x{bn_u:<3} "
            f"BK={bk_u:<3} "
            f"warps={warps_u} | "
            f"median={r['median']:8.3f} ms | "
            f"best={r['best']:8.3f} ms"
        )

    except Exception as e:
        print(
            f"{bm_u}x{bn_u} FAILED:",
            e,
        )

PASS-2 TILE SWEEP
 32x64  BK=64  warps=4 | median= 107.362 ms | best= 107.264 ms
 32x64  BK=64  warps=8 | median=   3.747 ms | best=   3.743 ms
 32x128 BK=64  warps=4 | median=   7.435 ms | best=   7.425 ms
 32x128 BK=64  warps=8 | median= 117.735 ms | best= 117.604 ms
 64x64  BK=64  warps=4 | median=   3.544 ms | best=   3.542 ms
 64x64  BK=64  warps=8 | median=   3.444 ms | best=   3.440 ms


In [24]:
@triton.jit
def weight_only_kernel(
    w_ptr,
    n_elements,
    alpha,

    BLOCK: tl.constexpr,
):
    pid = tl.program_id(0)

    offsets = (
        pid * BLOCK
        + tl.arange(0, BLOCK)
    )

    mask = offsets < n_elements

    w = tl.load(
        w_ptr + offsets,
        mask=mask,
    )

    w = w + alpha

    tl.store(
        w_ptr + offsets,
        w,
        mask=mask,
    )

In [25]:
W_test = W0.clone()

N_ELEMENTS = W_test.numel()

BLOCK = 1024

weight_grid = (
    triton.cdiv(
        N_ELEMENTS,
        BLOCK,
    ),
)


def weight_only():
    weight_only_kernel[weight_grid](
        W_test,
        N_ELEMENTS,
        1e-8,
        BLOCK=BLOCK,
    )


r_weight = benchmark(
    weight_only,
    warmup=10,
    iterations=50,
)

print("=" * 72)
print("WEIGHT READ + WRITE ONLY")
print("=" * 72)

print(r_weight)

WEIGHT READ + WRITE ONLY
{'mean': 0.37447679936885836, 'median': 0.37222400307655334, 'best': 0.3696640133857727, 'p90': 0.37887999415397644, 'p99': 0.3973119854927063}


In [26]:
BM_N = 64
BN_N = 128
BK_N = 64
WARPS_N = 4

BM_U = 64
BN_U = 64
BK_U = 64
WARPS_U = 8

norm_grid = (
    triton.cdiv(M, BM_N),
    triton.cdiv(N, BN_N),
)

update_grid = (
    triton.cdiv(M, BM_U),
    triton.cdiv(N, BN_U),
)

def triton_tuned_step(
    w,
    vt,
    z,
    lr=LR,
    clip=CLIP,
):
    norm_ws.zero_()

    ttt_norm_kernel[norm_grid](
        vt,
        z,
        norm_ws,

        vt.stride(0),
        vt.stride(1),
        z.stride(0),
        z.stride(1),

        M,
        N,
        K,

        BLOCK_M=BM_N,
        BLOCK_N=BN_N,
        BLOCK_K=BK_N,

        PREC=PREC,
        num_warps=WARPS_N,
    )

    ttt_update_kernel[update_grid](
        vt,
        z,
        w,
        norm_ws,

        vt.stride(0),
        vt.stride(1),
        z.stride(0),
        z.stride(1),
        w.stride(0),
        w.stride(1),

        M,
        N,
        K,

        lr,
        clip,

        BLOCK_M=BM_U,
        BLOCK_N=BN_U,
        BLOCK_K=BK_U,

        PREC=PREC,
        num_warps=WARPS_U,
    )

    return w

In [27]:
w_tuned = W0.clone()

r_tuned = benchmark(
    lambda: triton_tuned_step(
        w_tuned,
        VT,
        Z,
    ),
    warmup=10,
    iterations=50,
)

print(r_tuned)

{'mean': 7.651225557327271, 'median': 7.490560054779053, 'best': 7.476223945617676, 'p90': 7.521279811859131, 'p99': 9.61740779876709}


In [28]:
# =============================================================================
# Async-TTT — A100 Kernel Freeze Benchmark
#
# Target:
#   NVIDIA A100-SXM4-40GB / SM80
#
# Purpose:
#   1. Validate final source -> destination TTT update semantics.
#   2. Compare against naive and optimized PyTorch.
#   3. Measure numerical correctness.
#   4. Measure latency distributions.
#   5. Measure temporary allocation separately from persistent double-buffer cost.
#   6. Measure Pass-1 / Pass-2 breakdown.
#   7. Measure raw GEMM and weight read/write sanity baselines.
#
# IMPORTANT:
#   - Strict FP32 / IEEE comparison.
#   - TF32 disabled.
#   - Tuned A100 configuration:
#         Norm:   64 x 128 x 64, 4 warps
#         Update: 64 x  64 x 64, 8 warps
#
# This is intended to be run as ONE notebook cell / ONE Python script.
# =============================================================================

import math
import statistics
import subprocess
import sys
import time

import torch
import triton
import triton.language as tl


# =============================================================================
# 0. GLOBAL CONFIGURATION
# =============================================================================

DEVICE = "cuda"

# Llama-3 8B-style W_down dimensions
M = 4096
N = 14336
K = 512

LR = 1e-3
CLIP = 1e-5

# Benchmark settings
WARMUP = 20
ITERS = 100
INDEPENDENT_RUNS = 5

torch.set_grad_enabled(False)

# Strict FP32 comparison.
# A100 supports TF32, therefore explicitly disable it.
torch.backends.cuda.matmul.allow_tf32 = False
torch.backends.cudnn.allow_tf32 = False

torch.manual_seed(0)

gen = torch.Generator(
    device=DEVICE
).manual_seed(0)


# =============================================================================
# 1. ENVIRONMENT
# =============================================================================

print("\n" + "=" * 92)
print("ASYNC-TTT — A100 KERNEL FREEZE BENCHMARK")
print("=" * 92)

print("\n[ENVIRONMENT]")

print("Python              :", sys.version.replace("\n", " "))
print("PyTorch             :", torch.__version__)
print("CUDA runtime        :", torch.version.cuda)
print("Triton              :", triton.__version__)

assert torch.cuda.is_available(), "CUDA not available"

GPU_NAME = torch.cuda.get_device_name(0)
CAPABILITY = torch.cuda.get_device_capability(0)
PROPS = torch.cuda.get_device_properties(0)

print("GPU                 :", GPU_NAME)
print("Compute capability  :", CAPABILITY)
print("SM count            :", PROPS.multi_processor_count)
print(
    "VRAM                :",
    f"{PROPS.total_memory / 1024**3:.2f} GiB"
)

print(
    "TF32 matmul         :",
    torch.backends.cuda.matmul.allow_tf32
)

print(
    "TF32 cuDNN          :",
    torch.backends.cudnn.allow_tf32
)


def print_gpu_state(title):
    print(f"\n[{title}]")

    try:
        out = subprocess.check_output(
            [
                "nvidia-smi",
                "--query-gpu="
                "name,"
                "temperature.gpu,"
                "power.draw,"
                "power.limit,"
                "clocks.sm,"
                "clocks.mem,"
                "utilization.gpu,"
                "memory.used",
                "--format=csv",
            ],
            text=True,
        )

        print(out)

    except Exception as exc:
        print("nvidia-smi query failed:", exc)


print_gpu_state("GPU STATE BEFORE BENCHMARK")


# =============================================================================
# 2. INPUT TENSORS
# =============================================================================

print("\n" + "=" * 92)
print("WORKLOAD")
print("=" * 92)

Z = torch.randn(
    K,
    N,
    device=DEVICE,
    dtype=torch.float32,
    generator=gen,
)

V = torch.randn(
    K,
    M,
    device=DEVICE,
    dtype=torch.float32,
    generator=gen,
)

VT = V.T.contiguous()

W0 = torch.randn(
    M,
    N,
    device=DEVICE,
    dtype=torch.float32,
    generator=gen,
)


# Smaller-amplitude distribution used to test clipped behavior
Z_LOW = (
    torch.randn(
        K,
        N,
        device=DEVICE,
        dtype=torch.float32,
        generator=gen,
    )
    * 0.05
)

V_LOW = (
    torch.randn(
        K,
        M,
        device=DEVICE,
        dtype=torch.float32,
        generator=gen,
    )
    * 0.05
)

VT_LOW = V_LOW.T.contiguous()


ONE_DW_BYTES = M * N * 4
ONE_DW_MB = ONE_DW_BYTES / 1e6
ONE_DW_MIB = ONE_DW_BYTES / 1024**2

print(f"M                   : {M}")
print(f"N                   : {N}")
print(f"K                   : {K}")
print("dtype               : FP32")
print(f"One W/dW            : {ONE_DW_MB:.2f} MB")
print(f"One W/dW            : {ONE_DW_MIB:.2f} MiB")


# =============================================================================
# 3. TRITON PASS 1 — GLOBAL FROBENIUS NORM
# =============================================================================

@triton.jit
def ttt_norm_kernel(
    v_t_ptr,
    z_ptr,
    sq_norm_ptr,

    stride_vm,
    stride_vk,
    stride_zk,
    stride_zn,

    M: tl.constexpr,
    N: tl.constexpr,
    K: tl.constexpr,

    BLOCK_M: tl.constexpr,
    BLOCK_N: tl.constexpr,
    BLOCK_K: tl.constexpr,

    PREC: tl.constexpr,
):
    pid_m = tl.program_id(axis=0)
    pid_n = tl.program_id(axis=1)

    offs_m = (
        pid_m * BLOCK_M
        + tl.arange(0, BLOCK_M)
    )

    offs_n = (
        pid_n * BLOCK_N
        + tl.arange(0, BLOCK_N)
    )

    acc = tl.zeros(
        (BLOCK_M, BLOCK_N),
        dtype=tl.float32,
    )

    for k in range(0, K, BLOCK_K):
        offs_k = (
            k
            + tl.arange(0, BLOCK_K)
        )

        v_ptrs = (
            v_t_ptr
            + offs_m[:, None] * stride_vm
            + offs_k[None, :] * stride_vk
        )

        z_ptrs = (
            z_ptr
            + offs_k[:, None] * stride_zk
            + offs_n[None, :] * stride_zn
        )

        v_tile = tl.load(
            v_ptrs,
            mask=(
                (offs_m[:, None] < M)
                &
                (offs_k[None, :] < K)
            ),
            other=0.0,
        )

        z_tile = tl.load(
            z_ptrs,
            mask=(
                (offs_k[:, None] < K)
                &
                (offs_n[None, :] < N)
            ),
            other=0.0,
        )

        acc = tl.dot(
            v_tile,
            z_tile,
            acc,
            input_precision=PREC,
        )

    local_sq_sum = tl.sum(
        acc * acc
    )

    tl.atomic_add(
        sq_norm_ptr,
        local_sq_sum,
    )


# =============================================================================
# 4. TRITON PASS 2 — SOURCE -> DESTINATION UPDATE
#
# IMPORTANT:
#
# Old:
#
#       W <- W + update
#
# Final asynchronous-compatible design:
#
#       W_dst <- W_src + update
#
# Inference may continue reading W_src while the learning stream builds W_dst.
# =============================================================================

@triton.jit
def ttt_update_srcdst_kernel(
    v_t_ptr,
    z_ptr,

    w_src_ptr,
    w_dst_ptr,

    sq_norm_ptr,

    stride_vm,
    stride_vk,
    stride_zk,
    stride_zn,

    stride_src_m,
    stride_src_n,

    stride_dst_m,
    stride_dst_n,

    M: tl.constexpr,
    N: tl.constexpr,
    K: tl.constexpr,

    learning_rate,
    clip_threshold,

    BLOCK_M: tl.constexpr,
    BLOCK_N: tl.constexpr,
    BLOCK_K: tl.constexpr,

    PREC: tl.constexpr,
):
    pid_m = tl.program_id(axis=0)
    pid_n = tl.program_id(axis=1)

    offs_m = (
        pid_m * BLOCK_M
        + tl.arange(0, BLOCK_M)
    )

    offs_n = (
        pid_n * BLOCK_N
        + tl.arange(0, BLOCK_N)
    )

    # -------------------------------------------------------------------------
    # Read global norm
    # -------------------------------------------------------------------------

    sq_norm = tl.load(
        sq_norm_ptr
    )

    raw_scale = (
        clip_threshold
        /
        (tl.sqrt(sq_norm) + 1e-8)
    )

    scale = tl.where(
        raw_scale < 1.0,
        raw_scale,
        1.0,
    )

    # -------------------------------------------------------------------------
    # Rematerialize dW tile
    # -------------------------------------------------------------------------

    acc = tl.zeros(
        (BLOCK_M, BLOCK_N),
        dtype=tl.float32,
    )

    for k in range(0, K, BLOCK_K):
        offs_k = (
            k
            + tl.arange(0, BLOCK_K)
        )

        v_ptrs = (
            v_t_ptr
            + offs_m[:, None] * stride_vm
            + offs_k[None, :] * stride_vk
        )

        z_ptrs = (
            z_ptr
            + offs_k[:, None] * stride_zk
            + offs_n[None, :] * stride_zn
        )

        v_tile = tl.load(
            v_ptrs,
            mask=(
                (offs_m[:, None] < M)
                &
                (offs_k[None, :] < K)
            ),
            other=0.0,
        )

        z_tile = tl.load(
            z_ptrs,
            mask=(
                (offs_k[:, None] < K)
                &
                (offs_n[None, :] < N)
            ),
            other=0.0,
        )

        acc = tl.dot(
            v_tile,
            z_tile,
            acc,
            input_precision=PREC,
        )

    # -------------------------------------------------------------------------
    # Read source weight
    # -------------------------------------------------------------------------

    mask_w = (
        (offs_m[:, None] < M)
        &
        (offs_n[None, :] < N)
    )

    src_ptrs = (
        w_src_ptr
        + offs_m[:, None] * stride_src_m
        + offs_n[None, :] * stride_src_n
    )

    dst_ptrs = (
        w_dst_ptr
        + offs_m[:, None] * stride_dst_m
        + offs_n[None, :] * stride_dst_n
    )

    w_src = tl.load(
        src_ptrs,
        mask=mask_w,
        other=0.0,
    )

    # -------------------------------------------------------------------------
    # Construct next complete state
    # -------------------------------------------------------------------------

    w_next = (
        w_src
        +
        learning_rate * acc * scale
    )

    # -------------------------------------------------------------------------
    # Destination is independent from source
    # -------------------------------------------------------------------------

    tl.store(
        dst_ptrs,
        w_next,
        mask=mask_w,
    )


# =============================================================================
# 5. A100-TUNED CONFIGURATION
# =============================================================================

# Pass 1
NORM_BM = 64
NORM_BN = 128
NORM_BK = 64
NORM_WARPS = 4

# Pass 2
UPDATE_BM = 64
UPDATE_BN = 64
UPDATE_BK = 64
UPDATE_WARPS = 8

PREC = "ieee"

norm_grid = (
    triton.cdiv(M, NORM_BM),
    triton.cdiv(N, NORM_BN),
)

update_grid = (
    triton.cdiv(M, UPDATE_BM),
    triton.cdiv(N, UPDATE_BN),
)

norm_ws = torch.zeros(
    1,
    device=DEVICE,
    dtype=torch.float32,
)


print("\n[TUNED KERNEL CONFIGURATION]")

print(
    "Norm                :",
    f"{NORM_BM} x {NORM_BN} x {NORM_BK},",
    f"{NORM_WARPS} warps",
)

print(
    "Update              :",
    f"{UPDATE_BM} x {UPDATE_BN} x {UPDATE_BK},",
    f"{UPDATE_WARPS} warps",
)


# =============================================================================
# 6. TRITON FULL STEP
# =============================================================================

def triton_tuned_srcdst_step(
    w_src,
    w_dst,
    vt,
    z,
    lr=LR,
    clip=CLIP,
):
    norm_ws.zero_()

    # Pass 1
    ttt_norm_kernel[norm_grid](
        vt,
        z,
        norm_ws,

        vt.stride(0),
        vt.stride(1),
        z.stride(0),
        z.stride(1),

        M,
        N,
        K,

        BLOCK_M=NORM_BM,
        BLOCK_N=NORM_BN,
        BLOCK_K=NORM_BK,

        PREC=PREC,

        num_warps=NORM_WARPS,
    )

    # Pass 2
    ttt_update_srcdst_kernel[update_grid](
        vt,
        z,

        w_src,
        w_dst,

        norm_ws,

        vt.stride(0),
        vt.stride(1),

        z.stride(0),
        z.stride(1),

        w_src.stride(0),
        w_src.stride(1),

        w_dst.stride(0),
        w_dst.stride(1),

        M,
        N,
        K,

        lr,
        clip,

        BLOCK_M=UPDATE_BM,
        BLOCK_N=UPDATE_BN,
        BLOCK_K=UPDATE_BK,

        PREC=PREC,

        num_warps=UPDATE_WARPS,
    )

    return w_dst


# =============================================================================
# 7. PYTORCH BASELINES
# =============================================================================

def pytorch_naive_step(
    w,
    vt,
    z,
    lr=LR,
    clip=CLIP,
):
    dw = torch.matmul(
        vt,
        z,
    )

    frob = torch.linalg.matrix_norm(
        dw
    )

    scale = torch.clamp(
        clip / (frob + 1e-8),
        max=1.0,
    )

    # Second dW-sized allocation
    scaled_dw = dw * scale

    w.add_(
        scaled_dw,
        alpha=lr,
    )

    return w


def pytorch_inplace_step(
    w,
    vt,
    z,
    lr=LR,
    clip=CLIP,
):
    dw = torch.matmul(
        vt,
        z,
    )

    frob = torch.linalg.matrix_norm(
        dw
    )

    scale = torch.clamp(
        clip / (frob + 1e-8),
        max=1.0,
    )

    # Reuse dW buffer
    dw.mul_(scale)

    w.add_(
        dw,
        alpha=lr,
    )

    return w


# =============================================================================
# 8. BENCHMARK UTILITIES
# =============================================================================

def percentile(values, p):
    values = sorted(values)

    index = int(
        (len(values) - 1) * p
    )

    return values[index]


def benchmark_once(
    fn,
    warmup=WARMUP,
    iterations=ITERS,
):
    for _ in range(warmup):
        fn()

    torch.cuda.synchronize()

    start = torch.cuda.Event(
        enable_timing=True
    )

    end = torch.cuda.Event(
        enable_timing=True
    )

    values = []

    for _ in range(iterations):
        start.record()

        fn()

        end.record()

        end.synchronize()

        values.append(
            start.elapsed_time(end)
        )

    return {
        "mean": statistics.mean(values),
        "median": statistics.median(values),
        "best": min(values),
        "p90": percentile(values, 0.90),
        "p99": percentile(values, 0.99),
        "all": values,
    }


def benchmark_multi_run(
    name,
    fn_factory,
):
    run_results = []

    print(f"\n{name}")

    for run_idx in range(
        INDEPENDENT_RUNS
    ):
        fn = fn_factory()

        result = benchmark_once(
            fn
        )

        run_results.append(
            result
        )

        print(
            f"  run {run_idx + 1}: "
            f"median={result['median']:.4f} ms | "
            f"mean={result['mean']:.4f} ms | "
            f"best={result['best']:.4f} ms | "
            f"p90={result['p90']:.4f} ms | "
            f"p99={result['p99']:.4f} ms"
        )

    medians = [
        x["median"]
        for x in run_results
    ]

    means = [
        x["mean"]
        for x in run_results
    ]

    bests = [
        x["best"]
        for x in run_results
    ]

    p90s = [
        x["p90"]
        for x in run_results
    ]

    p99s = [
        x["p99"]
        for x in run_results
    ]

    combined = {
        "median": statistics.median(medians),
        "mean": statistics.mean(means),
        "best": min(bests),
        "p90": statistics.median(p90s),
        "p99": statistics.median(p99s),
    }

    print(
        "  -------------------------------------------------------------"
    )

    print(
        "  aggregate: "
        f"median={combined['median']:.4f} ms | "
        f"mean={combined['mean']:.4f} ms | "
        f"best={combined['best']:.4f} ms | "
        f"p90={combined['p90']:.4f} ms | "
        f"p99={combined['p99']:.4f} ms"
    )

    return combined


# =============================================================================
# 9. FORCE JIT COMPILATION BEFORE BENCHMARKS
# =============================================================================

print("\n" + "=" * 92)
print("JIT WARMUP")
print("=" * 92)

_tmp_src = W0.clone()
_tmp_dst = torch.empty_like(
    _tmp_src
)

triton_tuned_srcdst_step(
    _tmp_src,
    _tmp_dst,
    VT,
    Z,
)

_tmp_baseline = W0.clone()

pytorch_naive_step(
    _tmp_baseline,
    VT,
    Z,
)

_tmp_baseline = W0.clone()

pytorch_inplace_step(
    _tmp_baseline,
    VT,
    Z,
)

torch.cuda.synchronize()

del _tmp_src
del _tmp_dst
del _tmp_baseline

torch.cuda.empty_cache()

print("Warmup complete.")


# =============================================================================
# 10. CORRECTNESS
# =============================================================================

print("\n" + "=" * 92)
print("CORRECTNESS")
print("=" * 92)


def reference_update(
    vt,
    z,
    w_src,
    lr=LR,
    clip=CLIP,
):
    dw = torch.matmul(
        vt,
        z,
    )

    norm = torch.linalg.matrix_norm(
        dw
    )

    scale = torch.clamp(
        clip / (norm + 1e-8),
        max=1.0,
    )

    expected = (
        w_src
        +
        lr * dw * scale
    )

    return (
        expected,
        dw,
        norm,
        scale,
    )


def correctness_case(
    name,
    vt,
    z,
    clip,
):
    # Zero source deliberately used so tiny clipped updates
    # are not hidden by FP32 rounding of O(1) weights.
    source = torch.zeros_like(
        W0
    )

    destination = torch.empty_like(
        source
    )

    (
        expected,
        dw_ref,
        norm_ref,
        scale_ref,
    ) = reference_update(
        vt,
        z,
        source,
        clip=clip,
    )

    triton_tuned_srcdst_step(
        source,
        destination,
        vt,
        z,
        clip=clip,
    )

    torch.cuda.synchronize()

    diff = (
        destination
        -
        expected
    )

    rel_l2 = (
        torch.linalg.vector_norm(diff)
        /
        torch.linalg.vector_norm(expected).clamp_min(
            1e-30
        )
    ).item()

    max_abs = (
        diff.abs().max().item()
    )

    # Validate norm independently
    norm_ws.zero_()

    ttt_norm_kernel[norm_grid](
        vt,
        z,
        norm_ws,

        vt.stride(0),
        vt.stride(1),

        z.stride(0),
        z.stride(1),

        M,
        N,
        K,

        BLOCK_M=NORM_BM,
        BLOCK_N=NORM_BN,
        BLOCK_K=NORM_BK,

        PREC=PREC,

        num_warps=NORM_WARPS,
    )

    torch.cuda.synchronize()

    norm_triton = math.sqrt(
        norm_ws.item()
    )

    norm_relative_error = (
        abs(
            norm_triton
            -
            norm_ref.item()
        )
        /
        max(
            abs(norm_ref.item()),
            1e-30,
        )
    )

    passed = (
        rel_l2 < 1e-4
        and
        norm_relative_error < 1e-4
    )

    print(
        f"{name:<24}"
        f"scale={scale_ref.item():.6e} | "
        f"update rel-L2={rel_l2:.6e} | "
        f"max abs={max_abs:.6e} | "
        f"norm rel-err={norm_relative_error:.6e} | "
        f"{'PASS' if passed else 'FAIL'}"
    )

    return {
        "relative_l2": rel_l2,
        "max_abs": max_abs,
        "norm_relative_error": norm_relative_error,
        "pass": passed,
    }


CORRECT_NO_CLIP = correctness_case(
    "No clipping",
    VT,
    Z,
    1e12,
)

CORRECT_CLIPPED = correctness_case(
    "Clipped N(0,1)",
    VT,
    Z,
    CLIP,
)

CORRECT_LOW = correctness_case(
    "Clipped low-scale",
    VT_LOW,
    Z_LOW,
    CLIP,
)

assert (
    CORRECT_NO_CLIP["pass"]
    and CORRECT_CLIPPED["pass"]
    and CORRECT_LOW["pass"]
), "Correctness validation failed"


# =============================================================================
# 11. FULL LATENCY BENCHMARK
# =============================================================================

print("\n" + "=" * 92)
print("FULL UPDATE LATENCY — STRICT FP32")
print("=" * 92)


def make_naive():
    w = W0.clone()

    return lambda: pytorch_naive_step(
        w,
        VT,
        Z,
    )


def make_inplace():
    w = W0.clone()

    return lambda: pytorch_inplace_step(
        w,
        VT,
        Z,
    )


def make_triton():
    src = W0.clone()

    dst = torch.empty_like(
        src
    )

    # Intentionally always construct next state from src.
    # This isolates one TTT update.
    return lambda: triton_tuned_srcdst_step(
        src,
        dst,
        VT,
        Z,
    )


R_NAIVE = benchmark_multi_run(
    "PyTorch naive",
    make_naive,
)

R_INPLACE = benchmark_multi_run(
    "PyTorch inplace",
    make_inplace,
)

R_TRITON = benchmark_multi_run(
    "Triton tuned src -> dst",
    make_triton,
)


# =============================================================================
# 12. PASS BREAKDOWN
# =============================================================================

print("\n" + "=" * 92)
print("TRITON PASS BREAKDOWN")
print("=" * 92)


breakdown_src = W0.clone()
breakdown_dst = torch.empty_like(
    breakdown_src
)


def norm_pass():
    norm_ws.zero_()

    ttt_norm_kernel[norm_grid](
        VT,
        Z,
        norm_ws,

        VT.stride(0),
        VT.stride(1),

        Z.stride(0),
        Z.stride(1),

        M,
        N,
        K,

        BLOCK_M=NORM_BM,
        BLOCK_N=NORM_BN,
        BLOCK_K=NORM_BK,

        PREC=PREC,

        num_warps=NORM_WARPS,
    )


# Generate valid norm before timing update pass
norm_pass()
torch.cuda.synchronize()


def update_pass():
    ttt_update_srcdst_kernel[
        update_grid
    ](
        VT,
        Z,

        breakdown_src,
        breakdown_dst,

        norm_ws,

        VT.stride(0),
        VT.stride(1),

        Z.stride(0),
        Z.stride(1),

        breakdown_src.stride(0),
        breakdown_src.stride(1),

        breakdown_dst.stride(0),
        breakdown_dst.stride(1),

        M,
        N,
        K,

        LR,
        CLIP,

        BLOCK_M=UPDATE_BM,
        BLOCK_N=UPDATE_BN,
        BLOCK_K=UPDATE_BK,

        PREC=PREC,

        num_warps=UPDATE_WARPS,
    )


R_NORM = benchmark_once(
    norm_pass
)

R_UPDATE = benchmark_once(
    update_pass
)

print(
    "Pass 1 — norm       : "
    f"median={R_NORM['median']:.4f} ms | "
    f"best={R_NORM['best']:.4f} ms | "
    f"p90={R_NORM['p90']:.4f} ms | "
    f"p99={R_NORM['p99']:.4f} ms"
)

print(
    "Pass 2 — src->dst   : "
    f"median={R_UPDATE['median']:.4f} ms | "
    f"best={R_UPDATE['best']:.4f} ms | "
    f"p90={R_UPDATE['p90']:.4f} ms | "
    f"p99={R_UPDATE['p99']:.4f} ms"
)

print(
    "Sum of pass medians : "
    f"{R_NORM['median'] + R_UPDATE['median']:.4f} ms"
)


# =============================================================================
# 13. RAW PYTORCH GEMM BASELINE
# =============================================================================

print("\n" + "=" * 92)
print("RAW GEMM BASELINE")
print("=" * 92)

GEMM_OUT = torch.empty(
    M,
    N,
    device=DEVICE,
    dtype=torch.float32,
)


def raw_gemm():
    torch.mm(
        VT,
        Z,
        out=GEMM_OUT,
    )


R_GEMM = benchmark_once(
    raw_gemm
)

print(
    "cuBLAS/PyTorch GEMM  : "
    f"median={R_GEMM['median']:.4f} ms | "
    f"best={R_GEMM['best']:.4f} ms | "
    f"p90={R_GEMM['p90']:.4f} ms | "
    f"p99={R_GEMM['p99']:.4f} ms"
)


# =============================================================================
# 14. WEIGHT READ/WRITE BANDWIDTH SANITY CHECK
# =============================================================================

@triton.jit
def weight_only_kernel(
    w_ptr,
    n_elements,
    alpha,

    BLOCK: tl.constexpr,
):
    pid = tl.program_id(axis=0)

    offsets = (
        pid * BLOCK
        +
        tl.arange(0, BLOCK)
    )

    mask = (
        offsets
        <
        n_elements
    )

    values = tl.load(
        w_ptr + offsets,
        mask=mask,
        other=0.0,
    )

    values = (
        values
        +
        alpha
    )

    tl.store(
        w_ptr + offsets,
        values,
        mask=mask,
    )


W_BW = W0.clone()

WEIGHT_ELEMENTS = W_BW.numel()
WEIGHT_BLOCK = 1024

weight_grid = (
    triton.cdiv(
        WEIGHT_ELEMENTS,
        WEIGHT_BLOCK,
    ),
)


def weight_read_write():
    weight_only_kernel[
        weight_grid
    ](
        W_BW,
        WEIGHT_ELEMENTS,
        1e-8,
        BLOCK=WEIGHT_BLOCK,
    )


R_WEIGHT = benchmark_once(
    weight_read_write
)

weight_io_bytes = (
    2 * ONE_DW_BYTES
)

weight_bandwidth_tb_s = (
    weight_io_bytes
    /
    (
        R_WEIGHT["median"]
        / 1000
    )
    /
    1e12
)

print("\n" + "=" * 92)
print("WEIGHT READ + WRITE SANITY")
print("=" * 92)

print(
    "Latency             : "
    f"{R_WEIGHT['median']:.4f} ms"
)

print(
    "Approx bytes moved  : "
    f"{weight_io_bytes / 1e6:.2f} MB"
)

print(
    "Effective bandwidth : "
    f"{weight_bandwidth_tb_s:.3f} TB/s"
)


# =============================================================================
# 15. TEMPORARY MEMORY BENCHMARK
# =============================================================================

print("\n" + "=" * 92)
print("MEMORY")
print("=" * 92)


def peak_extra_alloc_mb(
    fn
):
    # Compile / initialize before measurement
    fn()

    torch.cuda.synchronize()

    torch.cuda.empty_cache()

    torch.cuda.reset_peak_memory_stats()

    baseline = torch.cuda.memory_allocated()

    fn()

    torch.cuda.synchronize()

    peak = torch.cuda.max_memory_allocated()

    return (
        peak
        -
        baseline
    ) / 1e6


MEM_W_NAIVE = W0.clone()

M_NAIVE = peak_extra_alloc_mb(
    lambda: pytorch_naive_step(
        MEM_W_NAIVE,
        VT,
        Z,
    )
)


MEM_W_INPLACE = W0.clone()

M_INPLACE = peak_extra_alloc_mb(
    lambda: pytorch_inplace_step(
        MEM_W_INPLACE,
        VT,
        Z,
    )
)


# IMPORTANT:
#
# Both Async-TTT buffers are allocated BEFORE resetting memory stats.
# Therefore this measures transient step allocation, not persistent state.
#
MEM_SRC = W0.clone()

MEM_DST = torch.empty_like(
    MEM_SRC
)

M_TRITON_TEMP = peak_extra_alloc_mb(
    lambda: triton_tuned_srcdst_step(
        MEM_SRC,
        MEM_DST,
        VT,
        Z,
    )
)


DOUBLE_BUFFER_EXTRA_MB = (
    MEM_DST.numel()
    *
    MEM_DST.element_size()
    /
    1e6
)


print(
    f"PyTorch naive transient        : {M_NAIVE:.2f} MB"
)

print(
    f"PyTorch inplace transient      : {M_INPLACE:.2f} MB"
)

print(
    f"Triton step transient          : {M_TRITON_TEMP:.2f} MB"
)

print(
    f"Triton persistent staging W    : {DOUBLE_BUFFER_EXTRA_MB:.2f} MB"
)

print(
    f"Norm workspace                 : {norm_ws.numel() * 4} bytes"
)

print(
    f"One full FP32 dW/W             : {ONE_DW_MB:.2f} MB"
)


# =============================================================================
# 16. DERIVED METRICS
# =============================================================================

TRITON_VS_NAIVE = (
    R_TRITON["median"]
    /
    R_NAIVE["median"]
)

TRITON_VS_INPLACE = (
    R_TRITON["median"]
    /
    R_INPLACE["median"]
)

LATENCY_OVERHEAD_VS_INPLACE = (
    (
        R_TRITON["median"]
        -
        R_INPLACE["median"]
    )
    /
    R_INPLACE["median"]
    *
    100
)

TEMP_MEMORY_REMOVED = (
    M_INPLACE
    -
    M_TRITON_TEMP
)

PASS1_SHARE = (
    R_NORM["median"]
    /
    (
        R_NORM["median"]
        +
        R_UPDATE["median"]
    )
    *
    100
)

PASS2_SHARE = (
    100
    -
    PASS1_SHARE
)


# =============================================================================
# 17. FINAL REPORT
# =============================================================================

print_gpu_state(
    "GPU STATE AFTER BENCHMARK"
)

print("\n" + "=" * 92)
print("FINAL COMBINED RESULT")
print("=" * 92)

print(f"""
HARDWARE
--------
GPU                         : {GPU_NAME}
Compute capability          : {CAPABILITY}
SM count                    : {PROPS.multi_processor_count}
VRAM                        : {PROPS.total_memory / 1024**3:.2f} GiB

SOFTWARE
--------
Python                      : {sys.version.split()[0]}
PyTorch                     : {torch.__version__}
CUDA runtime                : {torch.version.cuda}
Triton                      : {triton.__version__}

WORKLOAD
--------
M                           : {M}
N                           : {N}
K                           : {K}
dtype                       : FP32
TF32                        : disabled
Triton dot precision        : {PREC}
One full W/dW               : {ONE_DW_MB:.2f} MB

FINAL KERNEL CONFIGURATION
--------------------------
Norm tile                   : {NORM_BM} x {NORM_BN} x {NORM_BK}
Norm warps                  : {NORM_WARPS}

Update tile                 : {UPDATE_BM} x {UPDATE_BN} x {UPDATE_BK}
Update warps                : {UPDATE_WARPS}

Update semantics            : W_dst = W_src + eta * clipped(dW)

CORRECTNESS
-----------
No clipping
  relative update L2 error  : {CORRECT_NO_CLIP['relative_l2']:.6e}
  relative norm error       : {CORRECT_NO_CLIP['norm_relative_error']:.6e}

Clipped N(0,1)
  relative update L2 error  : {CORRECT_CLIPPED['relative_l2']:.6e}
  relative norm error       : {CORRECT_CLIPPED['norm_relative_error']:.6e}

Clipped low-scale
  relative update L2 error  : {CORRECT_LOW['relative_l2']:.6e}
  relative norm error       : {CORRECT_LOW['norm_relative_error']:.6e}

FULL UPDATE LATENCY
-------------------
PyTorch naive
  median                    : {R_NAIVE['median']:.4f} ms
  mean                      : {R_NAIVE['mean']:.4f} ms
  best                      : {R_NAIVE['best']:.4f} ms
  p90                       : {R_NAIVE['p90']:.4f} ms
  p99                       : {R_NAIVE['p99']:.4f} ms

PyTorch inplace
  median                    : {R_INPLACE['median']:.4f} ms
  mean                      : {R_INPLACE['mean']:.4f} ms
  best                      : {R_INPLACE['best']:.4f} ms
  p90                       : {R_INPLACE['p90']:.4f} ms
  p99                       : {R_INPLACE['p99']:.4f} ms

Triton tuned src -> dst
  median                    : {R_TRITON['median']:.4f} ms
  mean                      : {R_TRITON['mean']:.4f} ms
  best                      : {R_TRITON['best']:.4f} ms
  p90                       : {R_TRITON['p90']:.4f} ms
  p99                       : {R_TRITON['p99']:.4f} ms

TRITON BREAKDOWN
----------------
Pass 1 norm median          : {R_NORM['median']:.4f} ms
Pass 2 update median        : {R_UPDATE['median']:.4f} ms
Sum of isolated medians     : {R_NORM['median'] + R_UPDATE['median']:.4f} ms

Pass 1 share                : {PASS1_SHARE:.2f} %
Pass 2 share                : {PASS2_SHARE:.2f} %

REFERENCE MICROBENCHMARKS
-------------------------
Raw preallocated GEMM       : {R_GEMM['median']:.4f} ms
Weight read/write           : {R_WEIGHT['median']:.4f} ms
Weight R/W effective BW     : {weight_bandwidth_tb_s:.3f} TB/s

MEMORY
------
PyTorch naive transient     : {M_NAIVE:.2f} MB
PyTorch inplace transient   : {M_INPLACE:.2f} MB
Triton step transient       : {M_TRITON_TEMP:.2f} MB

Async staging buffer        : {DOUBLE_BUFFER_EXTRA_MB:.2f} MB persistent
Norm workspace              : {norm_ws.numel() * 4} bytes

DERIVED
-------
Triton / naive latency      : {TRITON_VS_NAIVE:.3f}x
Triton / inplace latency    : {TRITON_VS_INPLACE:.3f}x

Latency overhead vs inplace : {LATENCY_OVERHEAD_VS_INPLACE:.2f} %

Transient memory removed
vs optimized PyTorch        : {TEMP_MEMORY_REMOVED:.2f} MB
""")


# =============================================================================
# 18. MACHINE-PARSABLE RESULT
# =============================================================================

RESULT = {
    "hardware": {
        "gpu": GPU_NAME,
        "compute_capability": CAPABILITY,
        "sms": PROPS.multi_processor_count,
        "vram_gib": PROPS.total_memory / 1024**3,
    },

    "software": {
        "python": sys.version.split()[0],
        "pytorch": torch.__version__,
        "cuda_runtime": torch.version.cuda,
        "triton": triton.__version__,
    },

    "workload": {
        "M": M,
        "N": N,
        "K": K,
        "dtype": "FP32",
        "tf32": False,
        "precision": PREC,
    },

    "kernel": {
        "norm": {
            "BM": NORM_BM,
            "BN": NORM_BN,
            "BK": NORM_BK,
            "warps": NORM_WARPS,
        },

        "update": {
            "BM": UPDATE_BM,
            "BN": UPDATE_BN,
            "BK": UPDATE_BK,
            "warps": UPDATE_WARPS,
        },

        "semantics": "W_dst = W_src + eta * clipped(dW)",
    },

    "latency_ms": {
        "pytorch_naive": R_NAIVE,
        "pytorch_inplace": R_INPLACE,
        "triton_tuned": R_TRITON,
        "triton_norm": R_NORM,
        "triton_update": R_UPDATE,
        "raw_gemm": R_GEMM,
        "weight_rw": R_WEIGHT,
    },

    "memory_mb": {
        "pytorch_naive_transient": M_NAIVE,
        "pytorch_inplace_transient": M_INPLACE,
        "triton_transient": M_TRITON_TEMP,
        "async_staging_persistent": DOUBLE_BUFFER_EXTRA_MB,
    },

    "correctness": {
        "no_clip": CORRECT_NO_CLIP,
        "clipped": CORRECT_CLIPPED,
        "low_scale": CORRECT_LOW,
    },
}


print(
    "\nRESULT object is available as variable `RESULT`."
)

print(
    "Copy the FINAL COMBINED RESULT above back into ChatGPT."
)

print("=" * 92)


ASYNC-TTT — A100 KERNEL FREEZE BENCHMARK

[ENVIRONMENT]
Python              : 3.12.11 | packaged by Anaconda, Inc. | (main, Jun  5 2025, 13:09:17) [GCC 11.2.0]
PyTorch             : 2.8.0+cu128
CUDA runtime        : 12.8
Triton              : 3.4.0
GPU                 : NVIDIA A100-SXM4-40GB
Compute capability  : (8, 0)
SM count            : 108
VRAM                : 39.49 GiB
TF32 matmul         : False
TF32 cuDNN          : False

[GPU STATE BEFORE BENCHMARK]
name, temperature.gpu, power.draw [W], power.limit [W], clocks.current.sm [MHz], clocks.current.memory [MHz], utilization.gpu [%], memory.used [MiB]
NVIDIA A100-SXM4-40GB, 41, 75.15 W, 400.00 W, 1095 MHz, 1215 MHz, 0 %, 3241 MiB


WORKLOAD
M                   : 4096
N                   : 14336
K                   : 512
dtype               : FP32
One W/dW            : 234.88 MB
One W/dW            : 224.00 MiB

[TUNED KERNEL CONFIGURATION]
Norm                : 64 x 128 x 64, 4 warps
Update              : 64 x 64 x 64, 8 warps



In [29]:
# =============================================================================
# Async-TTT — Production Model Benchmark
#
# Model: Qwen2.5-7B-Instruct (open / ungated)
# GPU target: NVIDIA A100
#
# Measures on REAL model activations:
#   - PyTorch naive TTT update latency
#   - PyTorch optimized in-place TTT latency
#   - Tuned Triton rematerialized TTT latency
#   - PyTorch/Triton temporary memory
#   - Persistent async staging-buffer cost
#   - Numerical correctness
#   - Scaling across chunk sizes
#   - Scaling across early / middle / late transformer layers
#
# Outputs:
#   production_benchmark_results.json
#   production_benchmark_results.csv
# =============================================================================

import gc
import csv
import json
import math
import statistics
import subprocess
import sys
from pathlib import Path

import torch

try:
    import triton
    import triton.language as tl
except ImportError:
    !pip install -q triton
    import triton
    import triton.language as tl

try:
    from transformers import AutoTokenizer, AutoModelForCausalLM
except ImportError:
    !pip install -q transformers accelerate sentencepiece
    from transformers import AutoTokenizer, AutoModelForCausalLM


# =============================================================================
# CONFIG
# =============================================================================

MODEL_ID = "Qwen/Qwen2.5-7B-Instruct"

DEVICE = "cuda"

# Real production-model layers to sample
LAYERS_TO_TEST = [0, 13, 27]

# TTT chunk sizes
CHUNK_SIZES = [128, 256, 512, 1024, 2048]

LR = 1e-3
CLIP = 1e-5

WARMUP = 10
ITERS = 30

# Strict reference benchmark
torch.backends.cuda.matmul.allow_tf32 = False
torch.backends.cudnn.allow_tf32 = False

torch.set_grad_enabled(False)
torch.manual_seed(0)


# =============================================================================
# ENVIRONMENT
# =============================================================================

assert torch.cuda.is_available()

gpu_name = torch.cuda.get_device_name(0)
gpu_props = torch.cuda.get_device_properties(0)

print("=" * 110)
print("ASYNC-TTT — PRODUCTION MODEL BENCHMARK")
print("=" * 110)

print("\nENVIRONMENT")
print("-" * 110)

print("Model               :", MODEL_ID)
print("Python              :", sys.version.split()[0])
print("PyTorch             :", torch.__version__)
print("CUDA runtime        :", torch.version.cuda)
print("Triton              :", triton.__version__)
print("GPU                 :", gpu_name)
print("Compute capability  :", torch.cuda.get_device_capability(0))
print("SMs                 :", gpu_props.multi_processor_count)
print("VRAM                :", f"{gpu_props.total_memory / 1024**3:.2f} GiB")
print("TF32                :", False)

try:
    print()
    subprocess.run(["nvidia-smi"], check=False)
except Exception:
    pass


# =============================================================================
# LOAD PRODUCTION MODEL
# =============================================================================

print("\n" + "=" * 110)
print("LOADING MODEL")
print("=" * 110)

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_ID,
    trust_remote_code=True,
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.bfloat16,
    device_map={"": 0},
    trust_remote_code=True,
)

model.eval()

config = model.config

M = config.hidden_size
N = config.intermediate_size
NUM_LAYERS = config.num_hidden_layers

print("hidden_size         :", M)
print("intermediate_size   :", N)
print("layers              :", NUM_LAYERS)
print("model dtype         :", next(model.parameters()).dtype)

ONE_W_BYTES = M * N * 4
ONE_W_MB = ONE_W_BYTES / 1e6

print("FP32 W_down / dW    :", f"{ONE_W_MB:.2f} MB")

assert max(LAYERS_TO_TEST) < NUM_LAYERS


# =============================================================================
# GENERATE REAL INPUT
# =============================================================================

base_text = """
Modern large language models process long sequences using transformer blocks.
Each block contains attention and a gated feed-forward network.
Test-time training modifies selected model parameters during inference so the
model can adapt to information appearing in its current context. Efficient
execution matters because adaptation runs on the same accelerator responsible
for latency-sensitive token generation. Memory movement, kernel scheduling,
matrix multiplication throughput, and synchronization therefore directly
affect practical deployment.
"""

text = "\n".join([base_text] * 500)

tokens = tokenizer(
    text,
    return_tensors="pt",
    truncation=True,
    max_length=max(CHUNK_SIZES),
)

input_ids = tokens["input_ids"].to(DEVICE)
attention_mask = tokens["attention_mask"].to(DEVICE)

seq_len = input_ids.shape[1]

if seq_len < max(CHUNK_SIZES):
    raise RuntimeError(
        f"Tokenizer produced only {seq_len} tokens; "
        f"need {max(CHUNK_SIZES)}."
    )

print("\nInput tokens         :", seq_len)


# =============================================================================
# CAPTURE REAL MLP ACTIVATIONS
#
# V_source:
#   actual hidden state entering the MLP: [K, d_model]
#
# Z:
#   actual SwiGLU activation entering down_proj: [K, d_inter]
# =============================================================================

captured_hidden = {}
captured_z = {}

hooks = []


def make_mlp_input_hook(layer_idx):

    def hook(module, args):
        x = args[0]

        # [B, S, M]
        captured_hidden[layer_idx] = (
            x.detach()
            .clone()
        )

    return hook


def make_down_input_hook(layer_idx):

    def hook(module, args):
        z = args[0]

        # Actual input to down_proj
        # [B, S, N]
        captured_z[layer_idx] = (
            z.detach()
            .clone()
        )

    return hook


for layer_idx in LAYERS_TO_TEST:

    layer = model.model.layers[layer_idx]

    hooks.append(
        layer.mlp.register_forward_pre_hook(
            make_mlp_input_hook(layer_idx)
        )
    )

    hooks.append(
        layer.mlp.down_proj.register_forward_pre_hook(
            make_down_input_hook(layer_idx)
        )
    )


print("\nCapturing real model activations...")

with torch.inference_mode():
    _ = model(
        input_ids=input_ids,
        attention_mask=attention_mask,
        use_cache=False,
    )

torch.cuda.synchronize()

for hook in hooks:
    hook.remove()

print("Captured layers      :", sorted(captured_z.keys()))

for layer_idx in LAYERS_TO_TEST:
    print(
        f"Layer {layer_idx:02d}             : "
        f"hidden={tuple(captured_hidden[layer_idx].shape)}, "
        f"Z={tuple(captured_z[layer_idx].shape)}"
    )


# =============================================================================
# TRITON KERNELS
# =============================================================================

@triton.jit
def ttt_norm_kernel(
    vt_ptr,
    z_ptr,
    sq_norm_ptr,

    stride_vm,
    stride_vk,

    stride_zk,
    stride_zn,

    M: tl.constexpr,
    N: tl.constexpr,
    K: tl.constexpr,

    BLOCK_M: tl.constexpr,
    BLOCK_N: tl.constexpr,
    BLOCK_K: tl.constexpr,

    PREC: tl.constexpr,
):
    pid_m = tl.program_id(0)
    pid_n = tl.program_id(1)

    offs_m = (
        pid_m * BLOCK_M
        + tl.arange(0, BLOCK_M)
    )

    offs_n = (
        pid_n * BLOCK_N
        + tl.arange(0, BLOCK_N)
    )

    acc = tl.zeros(
        (BLOCK_M, BLOCK_N),
        dtype=tl.float32,
    )

    for k in range(0, K, BLOCK_K):

        offs_k = (
            k
            + tl.arange(0, BLOCK_K)
        )

        vt_ptrs = (
            vt_ptr
            + offs_m[:, None] * stride_vm
            + offs_k[None, :] * stride_vk
        )

        z_ptrs = (
            z_ptr
            + offs_k[:, None] * stride_zk
            + offs_n[None, :] * stride_zn
        )

        vt_tile = tl.load(
            vt_ptrs,
            mask=(
                (offs_m[:, None] < M)
                &
                (offs_k[None, :] < K)
            ),
            other=0.0,
        )

        z_tile = tl.load(
            z_ptrs,
            mask=(
                (offs_k[:, None] < K)
                &
                (offs_n[None, :] < N)
            ),
            other=0.0,
        )

        acc = tl.dot(
            vt_tile,
            z_tile,
            acc,
            input_precision=PREC,
        )

    local_sq_sum = tl.sum(
        acc * acc
    )

    tl.atomic_add(
        sq_norm_ptr,
        local_sq_sum,
    )


@triton.jit
def ttt_update_srcdst_kernel(
    vt_ptr,
    z_ptr,

    w_src_ptr,
    w_dst_ptr,

    sq_norm_ptr,

    stride_vm,
    stride_vk,

    stride_zk,
    stride_zn,

    stride_src_m,
    stride_src_n,

    stride_dst_m,
    stride_dst_n,

    M: tl.constexpr,
    N: tl.constexpr,
    K: tl.constexpr,

    learning_rate,
    clip_threshold,

    BLOCK_M: tl.constexpr,
    BLOCK_N: tl.constexpr,
    BLOCK_K: tl.constexpr,

    PREC: tl.constexpr,
):
    pid_m = tl.program_id(0)
    pid_n = tl.program_id(1)

    offs_m = (
        pid_m * BLOCK_M
        + tl.arange(0, BLOCK_M)
    )

    offs_n = (
        pid_n * BLOCK_N
        + tl.arange(0, BLOCK_N)
    )

    sq_norm = tl.load(
        sq_norm_ptr
    )

    raw_scale = (
        clip_threshold
        /
        (tl.sqrt(sq_norm) + 1e-8)
    )

    scale = tl.where(
        raw_scale < 1.0,
        raw_scale,
        1.0,
    )

    acc = tl.zeros(
        (BLOCK_M, BLOCK_N),
        dtype=tl.float32,
    )

    for k in range(0, K, BLOCK_K):

        offs_k = (
            k
            + tl.arange(0, BLOCK_K)
        )

        vt_ptrs = (
            vt_ptr
            + offs_m[:, None] * stride_vm
            + offs_k[None, :] * stride_vk
        )

        z_ptrs = (
            z_ptr
            + offs_k[:, None] * stride_zk
            + offs_n[None, :] * stride_zn
        )

        vt_tile = tl.load(
            vt_ptrs,
            mask=(
                (offs_m[:, None] < M)
                &
                (offs_k[None, :] < K)
            ),
            other=0.0,
        )

        z_tile = tl.load(
            z_ptrs,
            mask=(
                (offs_k[:, None] < K)
                &
                (offs_n[None, :] < N)
            ),
            other=0.0,
        )

        acc = tl.dot(
            vt_tile,
            z_tile,
            acc,
            input_precision=PREC,
        )

    mask_w = (
        (offs_m[:, None] < M)
        &
        (offs_n[None, :] < N)
    )

    src_ptrs = (
        w_src_ptr
        + offs_m[:, None] * stride_src_m
        + offs_n[None, :] * stride_src_n
    )

    dst_ptrs = (
        w_dst_ptr
        + offs_m[:, None] * stride_dst_m
        + offs_n[None, :] * stride_dst_n
    )

    w_src = tl.load(
        src_ptrs,
        mask=mask_w,
        other=0.0,
    )

    w_next = (
        w_src
        +
        learning_rate
        * acc
        * scale
    )

    tl.store(
        dst_ptrs,
        w_next,
        mask=mask_w,
    )


# =============================================================================
# A100 TUNED CONFIG
# =============================================================================

NORM_BM = 64
NORM_BN = 128
NORM_BK = 64
NORM_WARPS = 4

UPDATE_BM = 64
UPDATE_BN = 64
UPDATE_BK = 64
UPDATE_WARPS = 8

PREC = "ieee"


# =============================================================================
# PYTORCH BASELINES
# =============================================================================

def pytorch_naive_step(
    w,
    vt,
    z,
    lr=LR,
    clip=CLIP,
):
    dw = vt @ z

    frob = torch.linalg.matrix_norm(dw)

    scale = torch.clamp(
        clip / (frob + 1e-8),
        max=1.0,
    )

    scaled_dw = dw * scale

    w.add_(
        scaled_dw,
        alpha=lr,
    )

    return w


def pytorch_inplace_step(
    w,
    vt,
    z,
    lr=LR,
    clip=CLIP,
):
    dw = vt @ z

    frob = torch.linalg.matrix_norm(dw)

    scale = torch.clamp(
        clip / (frob + 1e-8),
        max=1.0,
    )

    dw.mul_(scale)

    w.add_(
        dw,
        alpha=lr,
    )

    return w


# =============================================================================
# TRITON LAUNCHER
# =============================================================================

def make_triton_step(M_, N_, K_):

    norm_grid = (
        triton.cdiv(M_, NORM_BM),
        triton.cdiv(N_, NORM_BN),
    )

    update_grid = (
        triton.cdiv(M_, UPDATE_BM),
        triton.cdiv(N_, UPDATE_BN),
    )

    norm_ws = torch.zeros(
        1,
        device=DEVICE,
        dtype=torch.float32,
    )

    def step(
        w_src,
        w_dst,
        vt,
        z,
        lr=LR,
        clip=CLIP,
    ):
        norm_ws.zero_()

        ttt_norm_kernel[norm_grid](
            vt,
            z,
            norm_ws,

            vt.stride(0),
            vt.stride(1),

            z.stride(0),
            z.stride(1),

            M_,
            N_,
            K_,

            BLOCK_M=NORM_BM,
            BLOCK_N=NORM_BN,
            BLOCK_K=NORM_BK,

            PREC=PREC,
            num_warps=NORM_WARPS,
        )

        ttt_update_srcdst_kernel[
            update_grid
        ](
            vt,
            z,

            w_src,
            w_dst,

            norm_ws,

            vt.stride(0),
            vt.stride(1),

            z.stride(0),
            z.stride(1),

            w_src.stride(0),
            w_src.stride(1),

            w_dst.stride(0),
            w_dst.stride(1),

            M_,
            N_,
            K_,

            lr,
            clip,

            BLOCK_M=UPDATE_BM,
            BLOCK_N=UPDATE_BN,
            BLOCK_K=UPDATE_BK,

            PREC=PREC,
            num_warps=UPDATE_WARPS,
        )

        return w_dst

    return step, norm_ws


# =============================================================================
# TIMING
# =============================================================================

def percentile(values, p):

    ordered = sorted(values)

    idx = int(
        (len(ordered) - 1) * p
    )

    return ordered[idx]


def benchmark(fn):

    for _ in range(WARMUP):
        fn()

    torch.cuda.synchronize()

    start = torch.cuda.Event(
        enable_timing=True
    )

    end = torch.cuda.Event(
        enable_timing=True
    )

    times = []

    for _ in range(ITERS):

        start.record()

        fn()

        end.record()

        end.synchronize()

        times.append(
            start.elapsed_time(end)
        )

    return {
        "mean_ms": statistics.mean(times),
        "median_ms": statistics.median(times),
        "best_ms": min(times),
        "p90_ms": percentile(times, 0.90),
        "p99_ms": percentile(times, 0.99),
    }


# =============================================================================
# MEMORY
# =============================================================================

def peak_extra_alloc_mb(fn):

    # warmup / JIT
    fn()

    torch.cuda.synchronize()
    torch.cuda.empty_cache()

    torch.cuda.reset_peak_memory_stats()

    base = torch.cuda.memory_allocated()

    fn()

    torch.cuda.synchronize()

    peak = torch.cuda.max_memory_allocated()

    return (
        peak - base
    ) / 1e6


# =============================================================================
# CORRECTNESS
# =============================================================================

def correctness(
    triton_step,
    vt,
    z,
    M_,
    N_,
):

    source = torch.zeros(
        M_,
        N_,
        device=DEVICE,
        dtype=torch.float32,
    )

    destination = torch.empty_like(
        source
    )

    dw = vt @ z

    norm = torch.linalg.matrix_norm(
        dw
    )

    scale = torch.clamp(
        CLIP / (norm + 1e-8),
        max=1.0,
    )

    expected = (
        source
        +
        LR * dw * scale
    )

    triton_step(
        source,
        destination,
        vt,
        z,
    )

    torch.cuda.synchronize()

    diff = (
        destination
        -
        expected
    )

    rel_l2 = (
        torch.linalg.vector_norm(diff)
        /
        torch.linalg.vector_norm(expected).clamp_min(
            1e-30
        )
    ).item()

    max_abs = (
        diff.abs().max().item()
    )

    return rel_l2, max_abs


# =============================================================================
# RESULTS
# =============================================================================

results = []


print("\n" + "=" * 110)
print("RUNNING REAL-MODEL BENCHMARKS")
print("=" * 110)


# =============================================================================
# BENCHMARK REAL MODEL LAYERS + CHUNK SIZES
# =============================================================================

for layer_idx in LAYERS_TO_TEST:

    print(
        f"\n{'=' * 110}\n"
        f"LAYER {layer_idx}\n"
        f"{'=' * 110}"
    )

    layer = model.model.layers[layer_idx]

    # Actual production down-projection weights
    model_weight = (
        layer.mlp.down_proj.weight
        .detach()
        .float()
        .contiguous()
    )

    assert model_weight.shape == (
        M,
        N,
    )

    hidden_full = (
        captured_hidden[layer_idx][0]
        .detach()
    )

    z_full = (
        captured_z[layer_idx][0]
        .detach()
    )

    for K_ in CHUNK_SIZES:

        print(
            f"\nLayer={layer_idx:02d} "
            f"K={K_:4d}"
        )

        # ---------------------------------------------------------------------
        # REAL MODEL ACTIVATIONS -> FP32 reference workload
        # ---------------------------------------------------------------------

        V_real = (
            hidden_full[:K_]
            .float()
            .contiguous()
        )

        Z_real = (
            z_full[:K_]
            .float()
            .contiguous()
        )

        VT = (
            V_real.T
            .contiguous()
        )

        assert VT.shape == (
            M,
            K_,
        )

        assert Z_real.shape == (
            K_,
            N,
        )

        triton_step, norm_ws = (
            make_triton_step(
                M,
                N,
                K_,
            )
        )

        # ---------------------------------------------------------------------
        # WARMUP COMPILATION
        # ---------------------------------------------------------------------

        tmp_src = model_weight.clone()
        tmp_dst = torch.empty_like(
            tmp_src
        )

        triton_step(
            tmp_src,
            tmp_dst,
            VT,
            Z_real,
        )

        torch.cuda.synchronize()

        # ---------------------------------------------------------------------
        # PYTORCH NAIVE
        # ---------------------------------------------------------------------

        w_naive = model_weight.clone()

        r_naive = benchmark(
            lambda: pytorch_naive_step(
                w_naive,
                VT,
                Z_real,
            )
        )

        # ---------------------------------------------------------------------
        # PYTORCH INPLACE
        # ---------------------------------------------------------------------

        w_inplace = model_weight.clone()

        r_inplace = benchmark(
            lambda: pytorch_inplace_step(
                w_inplace,
                VT,
                Z_real,
            )
        )

        # ---------------------------------------------------------------------
        # TRITON
        # ---------------------------------------------------------------------

        w_src = model_weight.clone()

        w_dst = torch.empty_like(
            w_src
        )

        r_triton = benchmark(
            lambda: triton_step(
                w_src,
                w_dst,
                VT,
                Z_real,
            )
        )

        # ---------------------------------------------------------------------
        # MEMORY
        # ---------------------------------------------------------------------

        mem_naive_w = model_weight.clone()

        mem_naive = peak_extra_alloc_mb(
            lambda: pytorch_naive_step(
                mem_naive_w,
                VT,
                Z_real,
            )
        )

        mem_inplace_w = model_weight.clone()

        mem_inplace = peak_extra_alloc_mb(
            lambda: pytorch_inplace_step(
                mem_inplace_w,
                VT,
                Z_real,
            )
        )

        mem_src = model_weight.clone()
        mem_dst = torch.empty_like(
            mem_src
        )

        # src/dst allocated BEFORE reset => persistent staging excluded
        mem_triton = peak_extra_alloc_mb(
            lambda: triton_step(
                mem_src,
                mem_dst,
                VT,
                Z_real,
            )
        )

        persistent_staging_mb = (
            mem_dst.numel()
            *
            mem_dst.element_size()
            /
            1e6
        )

        # ---------------------------------------------------------------------
        # CORRECTNESS
        # ---------------------------------------------------------------------

        rel_l2, max_abs = correctness(
            triton_step,
            VT,
            Z_real,
            M,
            N,
        )

        latency_ratio = (
            r_triton["median_ms"]
            /
            r_inplace["median_ms"]
        )

        latency_overhead_pct = (
            (
                r_triton["median_ms"]
                -
                r_inplace["median_ms"]
            )
            /
            r_inplace["median_ms"]
            *
            100
        )

        record = {
            "model": MODEL_ID,

            "layer": layer_idx,

            "M": M,
            "N": N,
            "K": K_,

            "activation_source": "real_model",

            "pytorch_naive_median_ms":
                r_naive["median_ms"],

            "pytorch_naive_p90_ms":
                r_naive["p90_ms"],

            "pytorch_naive_p99_ms":
                r_naive["p99_ms"],

            "pytorch_inplace_median_ms":
                r_inplace["median_ms"],

            "pytorch_inplace_p90_ms":
                r_inplace["p90_ms"],

            "pytorch_inplace_p99_ms":
                r_inplace["p99_ms"],

            "triton_median_ms":
                r_triton["median_ms"],

            "triton_p90_ms":
                r_triton["p90_ms"],

            "triton_p99_ms":
                r_triton["p99_ms"],

            "triton_vs_inplace_ratio":
                latency_ratio,

            "triton_overhead_pct":
                latency_overhead_pct,

            "pytorch_naive_temp_mb":
                mem_naive,

            "pytorch_inplace_temp_mb":
                mem_inplace,

            "triton_temp_mb":
                mem_triton,

            "triton_staging_mb":
                persistent_staging_mb,

            "relative_l2_error":
                rel_l2,

            "max_abs_error":
                max_abs,
        }

        results.append(record)

        print(
            f"  PyTorch naive   : "
            f"{r_naive['median_ms']:8.3f} ms "
            f"| temp={mem_naive:8.2f} MB"
        )

        print(
            f"  PyTorch inplace : "
            f"{r_inplace['median_ms']:8.3f} ms "
            f"| temp={mem_inplace:8.2f} MB"
        )

        print(
            f"  Triton          : "
            f"{r_triton['median_ms']:8.3f} ms "
            f"| temp={mem_triton:8.2f} MB "
            f"| staging={persistent_staging_mb:8.2f} MB"
        )

        print(
            f"  Ratio           : "
            f"{latency_ratio:.3f}x"
        )

        print(
            f"  Relative error  : "
            f"{rel_l2:.3e}"
        )

        # ---------------------------------------------------------------------
        # CLEAN TEMP OBJECTS
        # ---------------------------------------------------------------------

        del (
            V_real,
            Z_real,
            VT,
            tmp_src,
            tmp_dst,
            w_naive,
            w_inplace,
            w_src,
            w_dst,
            mem_naive_w,
            mem_inplace_w,
            mem_src,
            mem_dst,
        )

        gc.collect()
        torch.cuda.empty_cache()


# =============================================================================
# FINAL TABLE
# =============================================================================

print("\n" + "=" * 150)
print("FINAL PRODUCTION-MODEL RESULTS")
print("=" * 150)

header = (
    f"{'Layer':>5} "
    f"{'K':>6} "
    f"{'PT naive':>11} "
    f"{'PT inplace':>12} "
    f"{'Triton':>11} "
    f"{'Ratio':>8} "
    f"{'PT temp':>10} "
    f"{'Tri temp':>10} "
    f"{'Staging':>10} "
    f"{'RelErr':>12}"
)

print(header)
print("-" * len(header))

for r in results:

    print(
        f"{r['layer']:>5} "
        f"{r['K']:>6} "
        f"{r['pytorch_naive_median_ms']:>10.3f}ms "
        f"{r['pytorch_inplace_median_ms']:>11.3f}ms "
        f"{r['triton_median_ms']:>10.3f}ms "
        f"{r['triton_vs_inplace_ratio']:>7.3f}x "
        f"{r['pytorch_inplace_temp_mb']:>9.1f}M "
        f"{r['triton_temp_mb']:>9.1f}M "
        f"{r['triton_staging_mb']:>9.1f}M "
        f"{r['relative_l2_error']:>12.3e}"
    )


# =============================================================================
# AGGREGATE BY CHUNK SIZE
# =============================================================================

print("\n" + "=" * 110)
print("AGGREGATE BY CHUNK SIZE")
print("=" * 110)

for K_ in CHUNK_SIZES:

    subset = [
        r
        for r in results
        if r["K"] == K_
    ]

    pt = statistics.median(
        [
            r["pytorch_inplace_median_ms"]
            for r in subset
        ]
    )

    tri = statistics.median(
        [
            r["triton_median_ms"]
            for r in subset
        ]
    )

    ratio = tri / pt

    print(
        f"K={K_:4d} | "
        f"PyTorch={pt:8.3f} ms | "
        f"Triton={tri:8.3f} ms | "
        f"ratio={ratio:6.3f}x"
    )


# =============================================================================
# SAVE JSON
# =============================================================================

json_path = Path(
    "production_benchmark_results.json"
)

with open(
    json_path,
    "w",
) as f:
    json.dump(
        {
            "environment": {
                "model": MODEL_ID,
                "gpu": gpu_name,
                "compute_capability":
                    torch.cuda.get_device_capability(0),
                "sms":
                    gpu_props.multi_processor_count,
                "vram_gib":
                    gpu_props.total_memory / 1024**3,
                "python":
                    sys.version.split()[0],
                "pytorch":
                    torch.__version__,
                "cuda":
                    torch.version.cuda,
                "triton":
                    triton.__version__,
                "tf32":
                    False,
            },

            "kernel": {
                "norm": {
                    "BM": NORM_BM,
                    "BN": NORM_BN,
                    "BK": NORM_BK,
                    "warps": NORM_WARPS,
                },

                "update": {
                    "BM": UPDATE_BM,
                    "BN": UPDATE_BN,
                    "BK": UPDATE_BK,
                    "warps": UPDATE_WARPS,
                },

                "semantics":
                    "W_dst = W_src + eta * clipped(VT @ Z)",
            },

            "results": results,
        },
        f,
        indent=2,
    )


# =============================================================================
# SAVE CSV
# =============================================================================

csv_path = Path(
    "production_benchmark_results.csv"
)

with open(
    csv_path,
    "w",
    newline="",
) as f:

    writer = csv.DictWriter(
        f,
        fieldnames=results[0].keys(),
    )

    writer.writeheader()

    writer.writerows(
        results
    )


print("\n" + "=" * 110)
print("FILES")
print("=" * 110)

print("JSON :", json_path.resolve())
print("CSV  :", csv_path.resolve())

print("\nDONE")

ASYNC-TTT — PRODUCTION MODEL BENCHMARK

ENVIRONMENT
--------------------------------------------------------------------------------------------------------------
Model               : Qwen/Qwen2.5-7B-Instruct
Python              : 3.12.11
PyTorch             : 2.8.0+cu128
CUDA runtime        : 12.8
Triton              : 3.4.0
GPU                 : NVIDIA A100-SXM4-40GB
Compute capability  : (8, 0)
SMs                 : 108
VRAM                : 39.49 GiB
TF32                : False

Mon Sep 14 05:00:05 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 570.148.08             Driver Version: 570.148.08     CUDA Version: 12.8     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                  


LOADING MODEL


[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

hidden_size         : 3584
intermediate_size   : 18944
layers              : 28
model dtype         : torch.bfloat16
FP32 W_down / dW    : 271.58 MB

Input tokens         : 2048

Capturing real model activations...
Captured layers      : [0, 13, 27]
Layer 00             : hidden=(1, 2048, 3584), Z=(1, 2048, 18944)
Layer 13             : hidden=(1, 2048, 3584), Z=(1, 2048, 18944)
Layer 27             : hidden=(1, 2048, 3584), Z=(1, 2048, 18944)

RUNNING REAL-MODEL BENCHMARKS

LAYER 0

Layer=00 K= 128
  PyTorch naive   :    2.207 ms | temp=  545.26 MB
  PyTorch inplace :    2.198 ms | temp=  272.63 MB
  Triton          :    2.435 ms | temp=    0.00 MB | staging=  271.58 MB
  Ratio           : 1.108x
  Relative error  : 3.540e-07

Layer=00 K= 256
  PyTorch naive   :    3.111 ms | temp=  545.26 MB
  PyTorch inplace :    3.108 ms | temp=  272.63 MB
  Triton          :    4.509 ms | temp=    0.00 MB | staging=  271.58 MB
  Ratio           : 1.451x
  Relative error  : 1.691e-07

Layer=00 K= 5

In [30]:
# =============================================================================
# Async-TTT — A100 REAL ASYNC RUNTIME BENCHMARK
#
# Model      : Qwen/Qwen2.5-7B-Instruct
# GPU        : A100-SXM4-40GB
#
# Compares:
#   1. Full-model inference only
#   2. TTT update only — PyTorch/cuBLAS
#   3. TTT update only — Triton rematerialization
#   4. Full-model inference + async PyTorch TTT
#   5. Full-model inference + async Triton TTT
#   6. Adapted-layer projection + async TTT
#
# Reports:
#   - foreground latency
#   - foreground slowdown
#   - TTT update latency under contention
#   - concurrent makespan
#   - overlap efficiency
#   - transient memory
#   - numerical correctness
#
# Run as ONE notebook cell.
# =============================================================================

import gc
import json
import math
import statistics
import sys
import time

from pathlib import Path

import torch
import triton
import triton.language as tl

from transformers import AutoTokenizer, AutoModelForCausalLM


# =============================================================================
# CONFIG
# =============================================================================

MODEL_ID = "Qwen/Qwen2.5-7B-Instruct"

DEVICE = "cuda"

TTT_LAYER = 13
TTT_K = 512

LR = 1e-3
CLIP = 1e-5

# Full-model benchmark iterations
MODEL_WARMUP = 5
MODEL_ITERS = 20

# Layer projection benchmark
LAYER_WARMUP = 10
LAYER_ITERS = 50

torch.set_grad_enabled(False)

# Strict FP32 for TTT path
torch.backends.cuda.matmul.allow_tf32 = False
torch.backends.cudnn.allow_tf32 = False

torch.manual_seed(0)


# =============================================================================
# ENVIRONMENT
# =============================================================================

assert torch.cuda.is_available()

props = torch.cuda.get_device_properties(0)

print("=" * 110)
print("ASYNC-TTT — A100 REAL ASYNC RUNTIME BENCHMARK")
print("=" * 110)

print("Model               :", MODEL_ID)
print("Python              :", sys.version.split()[0])
print("PyTorch             :", torch.__version__)
print("CUDA runtime        :", torch.version.cuda)
print("Triton              :", triton.__version__)
print("GPU                 :", torch.cuda.get_device_name(0))
print("Compute capability  :", torch.cuda.get_device_capability(0))
print("SMs                 :", props.multi_processor_count)
print("VRAM                :", f"{props.total_memory / 1024**3:.2f} GiB")


# =============================================================================
# LOAD MODEL
# =============================================================================

print("\n" + "=" * 110)
print("LOADING MODEL")
print("=" * 110)

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_ID,
    trust_remote_code=True,
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    dtype=torch.bfloat16,
    device_map={"": 0},
    trust_remote_code=True,
)

model.eval()

M = model.config.hidden_size
N = model.config.intermediate_size

print("hidden_size         :", M)
print("intermediate_size   :", N)
print("layers              :", model.config.num_hidden_layers)
print("model dtype         :", next(model.parameters()).dtype)

W_BYTES = M * N * 4
W_MB = W_BYTES / 1e6

print("FP32 fast weight    :", f"{W_MB:.2f} MB")


# =============================================================================
# REAL INPUT
# =============================================================================

base_text = """
Test-time training enables models to continuously adapt their internal state
while processing incoming context. Efficient implementations must execute
learning concurrently with latency-sensitive inference while controlling
GPU memory traffic and synchronization overhead.
"""

text = "\n".join([base_text] * 400)

tokens = tokenizer(
    text,
    return_tensors="pt",
    truncation=True,
    max_length=max(TTT_K, 1024),
)

input_ids = tokens["input_ids"].to(DEVICE)

if input_ids.shape[1] < TTT_K:
    raise RuntimeError(
        f"Need at least {TTT_K} tokens, got {input_ids.shape[1]}"
    )


# =============================================================================
# CAPTURE REAL TTT ACTIVATIONS
# =============================================================================

captured_hidden = {}
captured_z = {}

layer = model.model.layers[TTT_LAYER]


def hidden_hook(module, args):
    captured_hidden["x"] = args[0].detach().clone()


def z_hook(module, args):
    captured_z["z"] = args[0].detach().clone()


h1 = layer.mlp.register_forward_pre_hook(hidden_hook)
h2 = layer.mlp.down_proj.register_forward_pre_hook(z_hook)

print("\nCapturing real activations...")

with torch.inference_mode():
    _ = model(
        input_ids=input_ids[:, :TTT_K],
        use_cache=False,
    )

torch.cuda.synchronize()

h1.remove()
h2.remove()

V = (
    captured_hidden["x"][0, :TTT_K]
    .float()
    .contiguous()
)

Z = (
    captured_z["z"][0, :TTT_K]
    .float()
    .contiguous()
)

VT = V.T.contiguous()

assert VT.shape == (M, TTT_K)
assert Z.shape == (TTT_K, N)

print("VT                  :", tuple(VT.shape))
print("Z                   :", tuple(Z.shape))


# =============================================================================
# FAST-WEIGHT BUFFERS
# =============================================================================

W_ACTIVE = (
    layer.mlp.down_proj.weight
    .detach()
    .float()
    .contiguous()
)

W_STAGING = torch.empty_like(W_ACTIVE)

print("W_active            :", tuple(W_ACTIVE.shape))


# =============================================================================
# TRITON PASS 1
# =============================================================================

@triton.jit
def ttt_norm_kernel(
    vt_ptr,
    z_ptr,
    sq_norm_ptr,

    stride_vm,
    stride_vk,

    stride_zk,
    stride_zn,

    M: tl.constexpr,
    N: tl.constexpr,
    K: tl.constexpr,

    BLOCK_M: tl.constexpr,
    BLOCK_N: tl.constexpr,
    BLOCK_K: tl.constexpr,

    PREC: tl.constexpr,
):
    pid_m = tl.program_id(0)
    pid_n = tl.program_id(1)

    offs_m = (
        pid_m * BLOCK_M
        + tl.arange(0, BLOCK_M)
    )

    offs_n = (
        pid_n * BLOCK_N
        + tl.arange(0, BLOCK_N)
    )

    acc = tl.zeros(
        (BLOCK_M, BLOCK_N),
        dtype=tl.float32,
    )

    for k in range(0, K, BLOCK_K):

        offs_k = (
            k
            + tl.arange(0, BLOCK_K)
        )

        vt_ptrs = (
            vt_ptr
            + offs_m[:, None] * stride_vm
            + offs_k[None, :] * stride_vk
        )

        z_ptrs = (
            z_ptr
            + offs_k[:, None] * stride_zk
            + offs_n[None, :] * stride_zn
        )

        vt_tile = tl.load(
            vt_ptrs,
            mask=(
                (offs_m[:, None] < M)
                &
                (offs_k[None, :] < K)
            ),
            other=0.0,
        )

        z_tile = tl.load(
            z_ptrs,
            mask=(
                (offs_k[:, None] < K)
                &
                (offs_n[None, :] < N)
            ),
            other=0.0,
        )

        acc = tl.dot(
            vt_tile,
            z_tile,
            acc,
            input_precision=PREC,
        )

    local_sq = tl.sum(
        acc * acc
    )

    tl.atomic_add(
        sq_norm_ptr,
        local_sq,
    )


# =============================================================================
# TRITON PASS 2
# =============================================================================

@triton.jit
def ttt_update_kernel(
    vt_ptr,
    z_ptr,

    src_ptr,
    dst_ptr,

    sq_norm_ptr,

    stride_vm,
    stride_vk,

    stride_zk,
    stride_zn,

    stride_sm,
    stride_sn,

    stride_dm,
    stride_dn,

    M: tl.constexpr,
    N: tl.constexpr,
    K: tl.constexpr,

    learning_rate,
    clip_threshold,

    BLOCK_M: tl.constexpr,
    BLOCK_N: tl.constexpr,
    BLOCK_K: tl.constexpr,

    PREC: tl.constexpr,
):
    pid_m = tl.program_id(0)
    pid_n = tl.program_id(1)

    offs_m = (
        pid_m * BLOCK_M
        + tl.arange(0, BLOCK_M)
    )

    offs_n = (
        pid_n * BLOCK_N
        + tl.arange(0, BLOCK_N)
    )

    sq_norm = tl.load(
        sq_norm_ptr
    )

    raw_scale = (
        clip_threshold
        /
        (
            tl.sqrt(sq_norm)
            +
            1e-8
        )
    )

    scale = tl.where(
        raw_scale < 1.0,
        raw_scale,
        1.0,
    )

    acc = tl.zeros(
        (BLOCK_M, BLOCK_N),
        dtype=tl.float32,
    )

    for k in range(0, K, BLOCK_K):

        offs_k = (
            k
            +
            tl.arange(0, BLOCK_K)
        )

        vt_ptrs = (
            vt_ptr
            + offs_m[:, None] * stride_vm
            + offs_k[None, :] * stride_vk
        )

        z_ptrs = (
            z_ptr
            + offs_k[:, None] * stride_zk
            + offs_n[None, :] * stride_zn
        )

        vt_tile = tl.load(
            vt_ptrs,
            mask=(
                (offs_m[:, None] < M)
                &
                (offs_k[None, :] < K)
            ),
            other=0.0,
        )

        z_tile = tl.load(
            z_ptrs,
            mask=(
                (offs_k[:, None] < K)
                &
                (offs_n[None, :] < N)
            ),
            other=0.0,
        )

        acc = tl.dot(
            vt_tile,
            z_tile,
            acc,
            input_precision=PREC,
        )

    mask = (
        (offs_m[:, None] < M)
        &
        (offs_n[None, :] < N)
    )

    src_offsets = (
        src_ptr
        + offs_m[:, None] * stride_sm
        + offs_n[None, :] * stride_sn
    )

    dst_offsets = (
        dst_ptr
        + offs_m[:, None] * stride_dm
        + offs_n[None, :] * stride_dn
    )

    w = tl.load(
        src_offsets,
        mask=mask,
        other=0.0,
    )

    out = (
        w
        +
        learning_rate
        * acc
        * scale
    )

    tl.store(
        dst_offsets,
        out,
        mask=mask,
    )


# =============================================================================
# TUNED A100 CONFIG
# =============================================================================

NORM_BM = 64
NORM_BN = 128
NORM_BK = 64
NORM_WARPS = 4

UPDATE_BM = 64
UPDATE_BN = 64
UPDATE_BK = 64
UPDATE_WARPS = 8

PREC = "ieee"

norm_grid = (
    triton.cdiv(M, NORM_BM),
    triton.cdiv(N, NORM_BN),
)

update_grid = (
    triton.cdiv(M, UPDATE_BM),
    triton.cdiv(N, UPDATE_BN),
)

norm_ws = torch.zeros(
    1,
    device=DEVICE,
    dtype=torch.float32,
)


# =============================================================================
# TRITON UPDATE
# =============================================================================

def triton_ttt(
    src=W_ACTIVE,
    dst=W_STAGING,
):
    norm_ws.zero_()

    ttt_norm_kernel[norm_grid](
        VT,
        Z,
        norm_ws,

        VT.stride(0),
        VT.stride(1),

        Z.stride(0),
        Z.stride(1),

        M,
        N,
        TTT_K,

        BLOCK_M=NORM_BM,
        BLOCK_N=NORM_BN,
        BLOCK_K=NORM_BK,

        PREC=PREC,

        num_warps=NORM_WARPS,
    )

    ttt_update_kernel[
        update_grid
    ](
        VT,
        Z,

        src,
        dst,

        norm_ws,

        VT.stride(0),
        VT.stride(1),

        Z.stride(0),
        Z.stride(1),

        src.stride(0),
        src.stride(1),

        dst.stride(0),
        dst.stride(1),

        M,
        N,
        TTT_K,

        LR,
        CLIP,

        BLOCK_M=UPDATE_BM,
        BLOCK_N=UPDATE_BN,
        BLOCK_K=UPDATE_BK,

        PREC=PREC,

        num_warps=UPDATE_WARPS,
    )


# =============================================================================
# PYTORCH/cuBLAS ASYNC-COMPATIBLE UPDATE
# =============================================================================

def pytorch_ttt(
    src=W_ACTIVE,
    dst=W_STAGING,
):
    dw = VT @ Z

    frob = torch.linalg.matrix_norm(
        dw
    )

    scale = torch.clamp(
        CLIP / (frob + 1e-8),
        max=1.0,
    )

    dw.mul_(scale)

    torch.add(
        src,
        dw,
        alpha=LR,
        out=dst,
    )


# =============================================================================
# CORRECTNESS
# =============================================================================

print("\n" + "=" * 110)
print("CORRECTNESS")
print("=" * 110)

zero_src = torch.zeros_like(
    W_ACTIVE
)

pt_dst = torch.empty_like(
    zero_src
)

tri_dst = torch.empty_like(
    zero_src
)

pytorch_ttt(
    zero_src,
    pt_dst,
)

triton_ttt(
    zero_src,
    tri_dst,
)

torch.cuda.synchronize()

diff = (
    tri_dst
    -
    pt_dst
)

rel_error = (
    torch.linalg.vector_norm(diff)
    /
    torch.linalg.vector_norm(pt_dst).clamp_min(
        1e-30
    )
).item()

max_error = (
    diff.abs().max().item()
)

print("Relative L2 error   :", f"{rel_error:.6e}")
print("Max absolute error  :", f"{max_error:.6e}")


# =============================================================================
# STREAMS
# =============================================================================

try:
    stream_inference = torch.cuda.Stream(
        priority=-1
    )
except Exception:
    stream_inference = torch.cuda.Stream()

stream_learning = torch.cuda.Stream(
    priority=0
)

torch.cuda.synchronize()


# =============================================================================
# HELPERS
# =============================================================================

def percentile(values, p):

    values = sorted(values)

    idx = int(
        (len(values) - 1) * p
    )

    return values[idx]


def summarize(values):

    return {
        "mean": statistics.mean(values),
        "median": statistics.median(values),
        "best": min(values),
        "p90": percentile(values, 0.90),
        "p99": percentile(values, 0.99),
    }


def benchmark_stream(
    fn,
    stream,
    warmup,
    iterations,
):
    for _ in range(warmup):

        with torch.cuda.stream(stream):
            fn()

    stream.synchronize()

    gpu_times = []
    host_times = []

    for _ in range(iterations):

        start = torch.cuda.Event(
            enable_timing=True
        )

        end = torch.cuda.Event(
            enable_timing=True
        )

        host_start = time.perf_counter()

        with torch.cuda.stream(stream):

            start.record()

            fn()

            end.record()

        end.synchronize()

        host_end = time.perf_counter()

        gpu_times.append(
            start.elapsed_time(end)
        )

        host_times.append(
            (host_end - host_start) * 1000
        )

    return {
        "gpu": summarize(gpu_times),
        "host": summarize(host_times),
    }


# =============================================================================
# FULL MODEL FOREGROUND WORKLOAD
#
# One-token full 7B forward.
#
# No KV reuse here: this isolates foreground model execution while TTT is
# running on the same GPU.
# =============================================================================

infer_ids = input_ids[:, :1]


def full_model_inference():

    with torch.inference_mode():

        return model(
            input_ids=infer_ids,
            use_cache=False,
        )


# =============================================================================
# LAYER-LEVEL FOREGROUND WORKLOAD
#
# Real Qwen down projection using ACTIVE fast weights.
#
# This uses PyTorch/cuBLAS for the foreground GEMM.
# =============================================================================

Z_DECODE = (
    captured_z["z"][0, -1:]
    .float()
    .contiguous()
)

LAYER_OUTPUT = torch.empty(
    1,
    M,
    device=DEVICE,
    dtype=torch.float32,
)


def layer_inference(
    active_weight=W_ACTIVE,
):

    torch.mm(
        Z_DECODE,
        active_weight.T,
        out=LAYER_OUTPUT,
    )


# =============================================================================
# JIT / CUDA WARMUP
# =============================================================================

print("\n" + "=" * 110)
print("WARMUP")
print("=" * 110)

triton_ttt()
pytorch_ttt()

with torch.cuda.stream(stream_inference):
    full_model_inference()
    layer_inference()

torch.cuda.synchronize()

print("Warmup complete.")


# =============================================================================
# ISOLATED BASELINES
# =============================================================================

print("\n" + "=" * 110)
print("ISOLATED BASELINES")
print("=" * 110)

FULL_BASE = benchmark_stream(
    full_model_inference,
    stream_inference,
    MODEL_WARMUP,
    MODEL_ITERS,
)

LAYER_BASE = benchmark_stream(
    layer_inference,
    stream_inference,
    LAYER_WARMUP,
    LAYER_ITERS,
)

PT_TTT_BASE = benchmark_stream(
    pytorch_ttt,
    stream_learning,
    LAYER_WARMUP,
    LAYER_ITERS,
)

TRITON_TTT_BASE = benchmark_stream(
    triton_ttt,
    stream_learning,
    LAYER_WARMUP,
    LAYER_ITERS,
)

print(
    f"Full model inference : "
    f"{FULL_BASE['host']['median']:.3f} ms host | "
    f"{FULL_BASE['gpu']['median']:.3f} ms GPU"
)

print(
    f"Layer projection     : "
    f"{LAYER_BASE['host']['median']:.3f} ms host | "
    f"{LAYER_BASE['gpu']['median']:.3f} ms GPU"
)

print(
    f"PyTorch TTT          : "
    f"{PT_TTT_BASE['gpu']['median']:.3f} ms"
)

print(
    f"Triton TTT           : "
    f"{TRITON_TTT_BASE['gpu']['median']:.3f} ms"
)


# =============================================================================
# MEMORY
# =============================================================================

def peak_temp_memory_mb(fn):

    fn()

    torch.cuda.synchronize()
    torch.cuda.empty_cache()

    torch.cuda.reset_peak_memory_stats()

    base = torch.cuda.memory_allocated()

    fn()

    torch.cuda.synchronize()

    peak = torch.cuda.max_memory_allocated()

    return (
        peak
        -
        base
    ) / 1e6


PT_TEMP_MB = peak_temp_memory_mb(
    pytorch_ttt
)

TRITON_TEMP_MB = peak_temp_memory_mb(
    triton_ttt
)

print("\nPyTorch TTT temp     :", f"{PT_TEMP_MB:.2f} MB")
print("Triton TTT temp     :", f"{TRITON_TEMP_MB:.2f} MB")
print("Persistent staging  :", f"{W_MB:.2f} MB")


# =============================================================================
# CONCURRENT BENCHMARK
# =============================================================================

def concurrent_benchmark(
    inference_fn,
    learning_fn,
    inference_baseline_ms,
    learning_baseline_ms,
    iterations,
    launch_order,
):

    # ---------------------------------------------------------
    # warmup
    # ---------------------------------------------------------

    for _ in range(3):

        torch.cuda.synchronize()

        if launch_order == "learning_first":

            with torch.cuda.stream(stream_learning):
                learning_fn()

            with torch.cuda.stream(stream_inference):
                inference_fn()

        else:

            with torch.cuda.stream(stream_inference):
                inference_fn()

            with torch.cuda.stream(stream_learning):
                learning_fn()

        torch.cuda.synchronize()

    # ---------------------------------------------------------
    # measured
    # ---------------------------------------------------------

    inference_gpu = []
    inference_host = []

    learning_gpu = []
    total_host = []

    for _ in range(iterations):

        torch.cuda.synchronize()

        inf_start = torch.cuda.Event(
            enable_timing=True
        )

        inf_end = torch.cuda.Event(
            enable_timing=True
        )

        learn_start = torch.cuda.Event(
            enable_timing=True
        )

        learn_end = torch.cuda.Event(
            enable_timing=True
        )

        host_start = time.perf_counter()

        if launch_order == "learning_first":

            with torch.cuda.stream(stream_learning):

                learn_start.record()

                learning_fn()

                learn_end.record()

            with torch.cuda.stream(stream_inference):

                inf_start.record()

                inference_fn()

                inf_end.record()

        else:

            with torch.cuda.stream(stream_inference):

                inf_start.record()

                inference_fn()

                inf_end.record()

            with torch.cuda.stream(stream_learning):

                learn_start.record()

                learning_fn()

                learn_end.record()

        # -----------------------------------------------------
        # user-visible foreground completion
        # -----------------------------------------------------

        inf_end.synchronize()

        foreground_done = time.perf_counter()

        # -----------------------------------------------------
        # background completion
        # -----------------------------------------------------

        learn_end.synchronize()

        all_done = time.perf_counter()

        inference_gpu.append(
            inf_start.elapsed_time(
                inf_end
            )
        )

        inference_host.append(
            (
                foreground_done
                -
                host_start
            )
            * 1000
        )

        learning_gpu.append(
            learn_start.elapsed_time(
                learn_end
            )
        )

        total_host.append(
            (
                all_done
                -
                host_start
            )
            * 1000
        )

    inf_gpu = summarize(
        inference_gpu
    )

    inf_host = summarize(
        inference_host
    )

    learn_gpu = summarize(
        learning_gpu
    )

    total = summarize(
        total_host
    )

    foreground_slowdown = (
        (
            inf_host["median"]
            -
            inference_baseline_ms
        )
        /
        inference_baseline_ms
        *
        100
    )

    serialized = (
        inference_baseline_ms
        +
        learning_baseline_ms
    )

    overlap_efficiency = (
        serialized
        -
        total["median"]
    ) / min(
        inference_baseline_ms,
        learning_baseline_ms,
    )

    hidden_fraction = (
        serialized
        -
        total["median"]
    ) / learning_baseline_ms

    return {
        "foreground_gpu_ms":
            inf_gpu,

        "foreground_host_ms":
            inf_host,

        "learning_gpu_ms":
            learn_gpu,

        "makespan_host_ms":
            total,

        "foreground_slowdown_pct":
            foreground_slowdown,

        "serialized_reference_ms":
            serialized,

        "overlap_efficiency":
            overlap_efficiency,

        "ttt_hidden_fraction":
            hidden_fraction,
    }


# =============================================================================
# FULL MODEL + PYTORCH TTT
# =============================================================================

print("\n" + "=" * 110)
print("FULL MODEL + ASYNC PYTORCH TTT")
print("=" * 110)

FULL_PT_LEARN_FIRST = concurrent_benchmark(
    full_model_inference,
    pytorch_ttt,

    FULL_BASE["host"]["median"],
    PT_TTT_BASE["gpu"]["median"],

    MODEL_ITERS,

    "learning_first",
)

FULL_PT_INF_FIRST = concurrent_benchmark(
    full_model_inference,
    pytorch_ttt,

    FULL_BASE["host"]["median"],
    PT_TTT_BASE["gpu"]["median"],

    MODEL_ITERS,

    "inference_first",
)


# =============================================================================
# FULL MODEL + TRITON TTT
# =============================================================================

print("\n" + "=" * 110)
print("FULL MODEL + ASYNC TRITON TTT")
print("=" * 110)

FULL_TRI_LEARN_FIRST = concurrent_benchmark(
    full_model_inference,
    triton_ttt,

    FULL_BASE["host"]["median"],
    TRITON_TTT_BASE["gpu"]["median"],

    MODEL_ITERS,

    "learning_first",
)

FULL_TRI_INF_FIRST = concurrent_benchmark(
    full_model_inference,
    triton_ttt,

    FULL_BASE["host"]["median"],
    TRITON_TTT_BASE["gpu"]["median"],

    MODEL_ITERS,

    "inference_first",
)


# =============================================================================
# LAYER PROJECTION + PYTORCH TTT
# =============================================================================

print("\n" + "=" * 110)
print("ACTIVE LAYER + ASYNC PYTORCH TTT")
print("=" * 110)

LAYER_PT = concurrent_benchmark(
    layer_inference,
    pytorch_ttt,

    LAYER_BASE["host"]["median"],
    PT_TTT_BASE["gpu"]["median"],

    LAYER_ITERS,

    "learning_first",
)


# =============================================================================
# LAYER PROJECTION + TRITON TTT
# =============================================================================

print("\n" + "=" * 110)
print("ACTIVE LAYER + ASYNC TRITON TTT")
print("=" * 110)

LAYER_TRI = concurrent_benchmark(
    layer_inference,
    triton_ttt,

    LAYER_BASE["host"]["median"],
    TRITON_TTT_BASE["gpu"]["median"],

    LAYER_ITERS,

    "learning_first",
)


# =============================================================================
# POINTER SWAP COST
# =============================================================================

active = W_ACTIVE
staging = W_STAGING

swap_times_ns = []

for _ in range(10000):

    t0 = time.perf_counter_ns()

    active, staging = (
        staging,
        active,
    )

    t1 = time.perf_counter_ns()

    swap_times_ns.append(
        t1 - t0
    )

POINTER_SWAP_NS = statistics.median(
    swap_times_ns
)


# =============================================================================
# RESULT PRINTING
# =============================================================================

def print_async_result(
    name,
    result,
):

    print("\n" + name)

    print(
        f"  foreground host     : "
        f"{result['foreground_host_ms']['median']:.3f} ms"
    )

    print(
        f"  foreground GPU      : "
        f"{result['foreground_gpu_ms']['median']:.3f} ms"
    )

    print(
        f"  foreground slowdown : "
        f"{result['foreground_slowdown_pct']:.2f}%"
    )

    print(
        f"  TTT under contention: "
        f"{result['learning_gpu_ms']['median']:.3f} ms"
    )

    print(
        f"  concurrent makespan : "
        f"{result['makespan_host_ms']['median']:.3f} ms"
    )

    print(
        f"  serialized reference: "
        f"{result['serialized_reference_ms']:.3f} ms"
    )

    print(
        f"  overlap efficiency  : "
        f"{result['overlap_efficiency']:.3f}"
    )

    print(
        f"  TTT hidden fraction : "
        f"{result['ttt_hidden_fraction'] * 100:.2f}%"
    )


print("\n" + "=" * 110)
print("FINAL ASYNC-TTT RESULTS")
print("=" * 110)

print("\nISOLATED")
print("-" * 110)

print(
    f"Full model baseline   : "
    f"{FULL_BASE['host']['median']:.3f} ms"
)

print(
    f"Layer baseline        : "
    f"{LAYER_BASE['host']['median']:.3f} ms"
)

print(
    f"PyTorch TTT           : "
    f"{PT_TTT_BASE['gpu']['median']:.3f} ms"
)

print(
    f"Triton TTT            : "
    f"{TRITON_TTT_BASE['gpu']['median']:.3f} ms"
)

print(
    f"PyTorch temp memory   : "
    f"{PT_TEMP_MB:.2f} MB"
)

print(
    f"Triton temp memory    : "
    f"{TRITON_TEMP_MB:.2f} MB"
)

print(
    f"Async staging buffer  : "
    f"{W_MB:.2f} MB"
)

print(
    f"Pointer swap median   : "
    f"{POINTER_SWAP_NS} ns"
)


print("\nFULL MODEL")
print("-" * 110)

print_async_result(
    "PyTorch TTT — learning launched first",
    FULL_PT_LEARN_FIRST,
)

print_async_result(
    "PyTorch TTT — inference launched first",
    FULL_PT_INF_FIRST,
)

print_async_result(
    "Triton TTT — learning launched first",
    FULL_TRI_LEARN_FIRST,
)

print_async_result(
    "Triton TTT — inference launched first",
    FULL_TRI_INF_FIRST,
)


print("\nACTIVE FAST-WEIGHT LAYER")
print("-" * 110)

print_async_result(
    "PyTorch async layer",
    LAYER_PT,
)

print_async_result(
    "Triton async layer",
    LAYER_TRI,
)


# =============================================================================
# MACHINE-READABLE RESULTS
# =============================================================================

RESULT = {
    "environment": {
        "model": MODEL_ID,
        "gpu": torch.cuda.get_device_name(0),
        "pytorch": torch.__version__,
        "cuda": torch.version.cuda,
        "triton": triton.__version__,
        "M": M,
        "N": N,
        "K": TTT_K,
        "ttt_layer": TTT_LAYER,
    },

    "correctness": {
        "relative_l2_error": rel_error,
        "max_absolute_error": max_error,
    },

    "isolated": {
        "full_model": FULL_BASE,
        "layer_projection": LAYER_BASE,
        "pytorch_ttt": PT_TTT_BASE,
        "triton_ttt": TRITON_TTT_BASE,
    },

    "memory_mb": {
        "pytorch_ttt_temp": PT_TEMP_MB,
        "triton_ttt_temp": TRITON_TEMP_MB,
        "persistent_staging": W_MB,
    },

    "full_model_async": {
        "pytorch_learning_first":
            FULL_PT_LEARN_FIRST,

        "pytorch_inference_first":
            FULL_PT_INF_FIRST,

        "triton_learning_first":
            FULL_TRI_LEARN_FIRST,

        "triton_inference_first":
            FULL_TRI_INF_FIRST,
    },

    "active_layer_async": {
        "pytorch":
            LAYER_PT,

        "triton":
            LAYER_TRI,
    },

    "pointer_swap_ns":
        POINTER_SWAP_NS,
}


OUT = Path(
    "async_ttt_a100_runtime_results.json"
)

with open(
    OUT,
    "w",
) as f:

    json.dump(
        RESULT,
        f,
        indent=2,
    )


print("\n" + "=" * 110)
print("RESULT FILE")
print("=" * 110)

print(OUT.resolve())
print("DONE")

ASYNC-TTT — A100 REAL ASYNC RUNTIME BENCHMARK
Model               : Qwen/Qwen2.5-7B-Instruct
Python              : 3.12.11
PyTorch             : 2.8.0+cu128
CUDA runtime        : 12.8
Triton              : 3.4.0
GPU                 : NVIDIA A100-SXM4-40GB
Compute capability  : (8, 0)
SMs                 : 108
VRAM                : 39.49 GiB

LOADING MODEL


Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

hidden_size         : 3584
intermediate_size   : 18944
layers              : 28
model dtype         : torch.bfloat16
FP32 fast weight    : 271.58 MB

Capturing real activations...
VT                  : (3584, 512)
Z                   : (512, 18944)
W_active            : (3584, 18944)

CORRECTNESS
Relative L2 error   : 1.425015e-06
Max absolute error  : 1.443290e-15

WARMUP
Warmup complete.

ISOLATED BASELINES
Full model inference : 21.967 ms host | 21.918 ms GPU
Layer projection     : 0.249 ms host | 0.215 ms GPU
PyTorch TTT          : 4.936 ms
Triton TTT           : 8.665 ms

PyTorch TTT temp     : 271.58 MB
Triton TTT temp     : 0.00 MB
Persistent staging  : 271.58 MB

FULL MODEL + ASYNC PYTORCH TTT

FULL MODEL + ASYNC TRITON TTT

ACTIVE LAYER + ASYNC PYTORCH TTT

ACTIVE LAYER + ASYNC TRITON TTT

FINAL ASYNC-TTT RESULTS

ISOLATED
--------------------------------------------------------------------------------------------------------------
Full model baseline   : 21.967 ms
Layer basel

In [31]:
# =============================================================================
# ASYNC-TTT — COMPLETE STANDALONE A100 BENCHMARK
#
# Runs independently in ONE notebook cell.
#
# PART A:
#   Qwen2.5-7B-Instruct
#   Real autoregressive decode with use_cache=True
#
#   Compares:
#       1. Baseline decode
#       2. Serialized Triton TTT
#       3. Async PyTorch/cuBLAS TTT
#       4. Async Triton TTT
#
#   Reports:
#       - numerical correctness
#       - TTT transient memory
#       - prefill latency
#       - ITL p50/p95/p99
#       - tokens/sec
#       - foreground slowdown
#       - async update latency
#       - update visibility lag
#       - KV-cache memory growth
#
# PART B:
#   Mock native runtime:
#
#       Rust control plane
#           -> C ABI
#           -> C++/CUDA runtime
#           -> high-priority inference stream
#           -> low-priority learning stream
#           -> CUDA events
#           -> A/B double buffering
#           -> O(1) pointer/index swap
#
# IMPORTANT FIX:
#   Every CUDA timing event is explicitly synchronized before elapsed_time().
#
# =============================================================================


# =============================================================================
# IMPORTS
# =============================================================================

import gc
import json
import math
import os
import shutil
import statistics
import subprocess
import sys
import time

from pathlib import Path

import torch
import triton
import triton.language as tl

from transformers import AutoModelForCausalLM, AutoTokenizer


# =============================================================================
# CONFIG
# =============================================================================

MODEL_ID = "Qwen/Qwen2.5-7B-Instruct"

DEVICE = "cuda"

TTT_LAYER = 13
TTT_K = 512

LR = 1e-3
CLIP = 1e-5

PROMPT_LEN = 512
DECODE_TOKENS = 64

# Trigger one update every N generated tokens.
UPDATE_EVERY = 8

# Independent repetitions of each decode mode.
DECODE_RUNS = 3

torch.set_grad_enabled(False)

torch.backends.cuda.matmul.allow_tf32 = False
torch.backends.cudnn.allow_tf32 = False

torch.manual_seed(0)


# =============================================================================
# ENVIRONMENT
# =============================================================================

assert torch.cuda.is_available(), "CUDA GPU required"

GPU_NAME = torch.cuda.get_device_name(0)
GPU_CAPABILITY = torch.cuda.get_device_capability(0)
GPU_PROPERTIES = torch.cuda.get_device_properties(0)

print("=" * 120)
print("ASYNC-TTT — COMPLETE A100 BENCHMARK")
print("=" * 120)

print("Model                  :", MODEL_ID)
print("Python                 :", sys.version.split()[0])
print("PyTorch                :", torch.__version__)
print("CUDA runtime           :", torch.version.cuda)
print("Triton                 :", triton.__version__)
print("GPU                    :", GPU_NAME)
print("Compute capability     :", GPU_CAPABILITY)
print("SMs                    :", GPU_PROPERTIES.multi_processor_count)

print(
    "VRAM                   :",
    f"{GPU_PROPERTIES.total_memory / 1024**3:.2f} GiB",
)

print("TF32                   :", False)


# =============================================================================
# LOAD MODEL
# =============================================================================

print("\n" + "=" * 120)
print("LOADING MODEL")
print("=" * 120)

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_ID,
    trust_remote_code=True,
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    dtype=torch.bfloat16,
    device_map={"": 0},
    trust_remote_code=True,
)

model.eval()

M = model.config.hidden_size
N = model.config.intermediate_size
NUM_LAYERS = model.config.num_hidden_layers

assert 0 <= TTT_LAYER < NUM_LAYERS

FAST_WEIGHT_BYTES = M * N * 4
FAST_WEIGHT_MB = FAST_WEIGHT_BYTES / 1e6

print("hidden_size            :", M)
print("intermediate_size      :", N)
print("layers                 :", NUM_LAYERS)
print("model dtype            :", next(model.parameters()).dtype)
print("FP32 fast weight       :", f"{FAST_WEIGHT_MB:.2f} MB")


# =============================================================================
# BUILD REAL INPUT
# =============================================================================

base_text = """
Test-time training allows a language model to adapt internal state while
processing incoming information. Production inference requires online learning
to coexist with latency-sensitive autoregressive generation. Efficient systems
must therefore minimize synchronization stalls, temporary memory allocation,
and GPU scheduling interference while preserving correct model execution.
"""

text = "\n".join(
    [base_text] * 700
)

MAX_INPUT_LEN = max(
    PROMPT_LEN,
    TTT_K,
)

tokenized = tokenizer(
    text,
    return_tensors="pt",
    truncation=True,
    max_length=MAX_INPUT_LEN,
)

input_ids = tokenized[
    "input_ids"
].to(
    DEVICE
)

if input_ids.shape[1] < MAX_INPUT_LEN:
    raise RuntimeError(
        f"Need {MAX_INPUT_LEN} tokens but tokenizer produced "
        f"{input_ids.shape[1]}"
    )

print("Input tokens           :", input_ids.shape[1])


# =============================================================================
# CAPTURE REAL QWEN MLP ACTIVATIONS
# =============================================================================

print("\n" + "=" * 120)
print("CAPTURING REAL QWEN ACTIVATIONS")
print("=" * 120)

target_layer = model.model.layers[
    TTT_LAYER
]

captured_hidden = {}
captured_z = {}


def capture_hidden_hook(module, args):
    captured_hidden["x"] = (
        args[0]
        .detach()
        .clone()
    )


def capture_z_hook(module, args):
    captured_z["z"] = (
        args[0]
        .detach()
        .clone()
    )


h_hidden = (
    target_layer
    .mlp
    .register_forward_pre_hook(
        capture_hidden_hook
    )
)

h_z = (
    target_layer
    .mlp
    .down_proj
    .register_forward_pre_hook(
        capture_z_hook
    )
)


with torch.inference_mode():
    capture_out = model(
        input_ids=input_ids[:, :TTT_K],
        use_cache=False,
    )

torch.cuda.synchronize()

h_hidden.remove()
h_z.remove()

del capture_out


V_REAL = (
    captured_hidden["x"][0, :TTT_K]
    .float()
    .contiguous()
)

Z_REAL = (
    captured_z["z"][0, :TTT_K]
    .float()
    .contiguous()
)

VT = (
    V_REAL.T
    .contiguous()
)

assert VT.shape == (
    M,
    TTT_K,
)

assert Z_REAL.shape == (
    TTT_K,
    N,
)

print("TTT layer              :", TTT_LAYER)
print("VT                     :", tuple(VT.shape))
print("Z                      :", tuple(Z_REAL.shape))


# =============================================================================
# DOUBLE BUFFER FAST WEIGHTS
# =============================================================================

W_ACTIVE = (
    target_layer
    .mlp
    .down_proj
    .weight
    .detach()
    .float()
    .contiguous()
)

W_STAGING = torch.empty_like(
    W_ACTIVE
)

assert W_ACTIVE.shape == (
    M,
    N,
)

print("W_active               :", tuple(W_ACTIVE.shape))
print("W_staging              :", tuple(W_STAGING.shape))


# =============================================================================
# TRITON PASS 1 — NORM
# =============================================================================

@triton.jit
def ttt_norm_kernel(
    vt_ptr,
    z_ptr,
    norm_ptr,

    stride_vm,
    stride_vk,

    stride_zk,
    stride_zn,

    M: tl.constexpr,
    N: tl.constexpr,
    K: tl.constexpr,

    BLOCK_M: tl.constexpr,
    BLOCK_N: tl.constexpr,
    BLOCK_K: tl.constexpr,

    PREC: tl.constexpr,
):

    pid_m = tl.program_id(0)
    pid_n = tl.program_id(1)

    offs_m = (
        pid_m * BLOCK_M
        +
        tl.arange(
            0,
            BLOCK_M,
        )
    )

    offs_n = (
        pid_n * BLOCK_N
        +
        tl.arange(
            0,
            BLOCK_N,
        )
    )

    acc = tl.zeros(
        (
            BLOCK_M,
            BLOCK_N,
        ),
        dtype=tl.float32,
    )

    for k in range(
        0,
        K,
        BLOCK_K,
    ):

        offs_k = (
            k
            +
            tl.arange(
                0,
                BLOCK_K,
            )
        )

        vt_ptrs = (
            vt_ptr
            +
            offs_m[:, None] * stride_vm
            +
            offs_k[None, :] * stride_vk
        )

        z_ptrs = (
            z_ptr
            +
            offs_k[:, None] * stride_zk
            +
            offs_n[None, :] * stride_zn
        )

        vt_tile = tl.load(
            vt_ptrs,
            mask=(
                (offs_m[:, None] < M)
                &
                (offs_k[None, :] < K)
            ),
            other=0.0,
        )

        z_tile = tl.load(
            z_ptrs,
            mask=(
                (offs_k[:, None] < K)
                &
                (offs_n[None, :] < N)
            ),
            other=0.0,
        )

        acc = tl.dot(
            vt_tile,
            z_tile,
            acc,
            input_precision=PREC,
        )

    local_sq = tl.sum(
        acc * acc
    )

    tl.atomic_add(
        norm_ptr,
        local_sq,
    )


# =============================================================================
# TRITON PASS 2 — SRC -> DST
# =============================================================================

@triton.jit
def ttt_update_kernel(
    vt_ptr,
    z_ptr,

    src_ptr,
    dst_ptr,

    norm_ptr,

    stride_vm,
    stride_vk,

    stride_zk,
    stride_zn,

    stride_sm,
    stride_sn,

    stride_dm,
    stride_dn,

    M: tl.constexpr,
    N: tl.constexpr,
    K: tl.constexpr,

    learning_rate,
    clip_threshold,

    BLOCK_M: tl.constexpr,
    BLOCK_N: tl.constexpr,
    BLOCK_K: tl.constexpr,

    PREC: tl.constexpr,
):

    pid_m = tl.program_id(0)
    pid_n = tl.program_id(1)

    offs_m = (
        pid_m * BLOCK_M
        +
        tl.arange(
            0,
            BLOCK_M,
        )
    )

    offs_n = (
        pid_n * BLOCK_N
        +
        tl.arange(
            0,
            BLOCK_N,
        )
    )

    sq_norm = tl.load(
        norm_ptr
    )

    raw_scale = (
        clip_threshold
        /
        (
            tl.sqrt(
                sq_norm
            )
            +
            1e-8
        )
    )

    scale = tl.where(
        raw_scale < 1.0,
        raw_scale,
        1.0,
    )

    acc = tl.zeros(
        (
            BLOCK_M,
            BLOCK_N,
        ),
        dtype=tl.float32,
    )

    for k in range(
        0,
        K,
        BLOCK_K,
    ):

        offs_k = (
            k
            +
            tl.arange(
                0,
                BLOCK_K,
            )
        )

        vt_ptrs = (
            vt_ptr
            +
            offs_m[:, None] * stride_vm
            +
            offs_k[None, :] * stride_vk
        )

        z_ptrs = (
            z_ptr
            +
            offs_k[:, None] * stride_zk
            +
            offs_n[None, :] * stride_zn
        )

        vt_tile = tl.load(
            vt_ptrs,
            mask=(
                (offs_m[:, None] < M)
                &
                (offs_k[None, :] < K)
            ),
            other=0.0,
        )

        z_tile = tl.load(
            z_ptrs,
            mask=(
                (offs_k[:, None] < K)
                &
                (offs_n[None, :] < N)
            ),
            other=0.0,
        )

        acc = tl.dot(
            vt_tile,
            z_tile,
            acc,
            input_precision=PREC,
        )

    mask = (
        (offs_m[:, None] < M)
        &
        (offs_n[None, :] < N)
    )

    src_offsets = (
        src_ptr
        +
        offs_m[:, None] * stride_sm
        +
        offs_n[None, :] * stride_sn
    )

    dst_offsets = (
        dst_ptr
        +
        offs_m[:, None] * stride_dm
        +
        offs_n[None, :] * stride_dn
    )

    w = tl.load(
        src_offsets,
        mask=mask,
        other=0.0,
    )

    out = (
        w
        +
        learning_rate
        *
        acc
        *
        scale
    )

    tl.store(
        dst_offsets,
        out,
        mask=mask,
    )


# =============================================================================
# A100-TUNED CONFIG
# =============================================================================

NORM_BM = 64
NORM_BN = 128
NORM_BK = 64
NORM_WARPS = 4

UPDATE_BM = 64
UPDATE_BN = 64
UPDATE_BK = 64
UPDATE_WARPS = 8

PREC = "ieee"

norm_grid = (
    triton.cdiv(
        M,
        NORM_BM,
    ),
    triton.cdiv(
        N,
        NORM_BN,
    ),
)

update_grid = (
    triton.cdiv(
        M,
        UPDATE_BM,
    ),
    triton.cdiv(
        N,
        UPDATE_BN,
    ),
)

norm_ws = torch.zeros(
    1,
    device=DEVICE,
    dtype=torch.float32,
)


# =============================================================================
# FINAL TRITON STEP
# =============================================================================

def triton_ttt(
    src,
    dst,
):

    norm_ws.zero_()

    ttt_norm_kernel[
        norm_grid
    ](
        VT,
        Z_REAL,
        norm_ws,

        VT.stride(0),
        VT.stride(1),

        Z_REAL.stride(0),
        Z_REAL.stride(1),

        M,
        N,
        TTT_K,

        BLOCK_M=NORM_BM,
        BLOCK_N=NORM_BN,
        BLOCK_K=NORM_BK,

        PREC=PREC,

        num_warps=NORM_WARPS,
    )

    ttt_update_kernel[
        update_grid
    ](
        VT,
        Z_REAL,

        src,
        dst,

        norm_ws,

        VT.stride(0),
        VT.stride(1),

        Z_REAL.stride(0),
        Z_REAL.stride(1),

        src.stride(0),
        src.stride(1),

        dst.stride(0),
        dst.stride(1),

        M,
        N,
        TTT_K,

        LR,
        CLIP,

        BLOCK_M=UPDATE_BM,
        BLOCK_N=UPDATE_BN,
        BLOCK_K=UPDATE_BK,

        PREC=PREC,

        num_warps=UPDATE_WARPS,
    )


# =============================================================================
# PYTORCH/cuBLAS REFERENCE
# =============================================================================

def pytorch_ttt(
    src,
    dst,
):

    dw = (
        VT
        @
        Z_REAL
    )

    frob = torch.linalg.matrix_norm(
        dw
    )

    scale = torch.clamp(
        CLIP
        /
        (
            frob
            +
            1e-8
        ),
        max=1.0,
    )

    dw.mul_(
        scale
    )

    torch.add(
        src,
        dw,
        alpha=LR,
        out=dst,
    )


# =============================================================================
# CORRECTNESS
# =============================================================================

print("\n" + "=" * 120)
print("NUMERICAL CORRECTNESS")
print("=" * 120)

zero_src = torch.zeros_like(
    W_ACTIVE
)

pt_dst = torch.empty_like(
    zero_src
)

tri_dst = torch.empty_like(
    zero_src
)

pytorch_ttt(
    zero_src,
    pt_dst,
)

triton_ttt(
    zero_src,
    tri_dst,
)

torch.cuda.synchronize()

difference = (
    tri_dst
    -
    pt_dst
)

relative_error = (
    torch.linalg.vector_norm(
        difference
    )
    /
    torch.linalg.vector_norm(
        pt_dst
    ).clamp_min(
        1e-30
    )
).item()

max_abs_error = (
    difference
    .abs()
    .max()
    .item()
)

print(
    "Relative L2 error      :",
    f"{relative_error:.6e}",
)

print(
    "Max absolute error     :",
    f"{max_abs_error:.6e}",
)


# =============================================================================
# STREAMS
# =============================================================================

# High-priority inference stream.
try:
    stream_inference = torch.cuda.Stream(
        priority=-1
    )
except Exception:
    stream_inference = torch.cuda.Stream()

# Default/lower-priority learning stream.
stream_learning = torch.cuda.Stream(
    priority=0
)

torch.cuda.synchronize()


# =============================================================================
# SAFE CUDA EVENT TIMING
# =============================================================================

def safe_elapsed_ms(
    start_event,
    end_event,
):
    """
    Robust CUDA event timing.

    Explicitly waits for the END event and device before asking CUDA
    for elapsed time.
    """

    end_event.synchronize()

    torch.cuda.synchronize()

    return start_event.elapsed_time(
        end_event
    )


# =============================================================================
# STATS HELPERS
# =============================================================================

def percentile(
    values,
    p,
):

    if not values:
        return 0.0

    ordered = sorted(
        values
    )

    position = (
        len(ordered)
        -
        1
    ) * p

    lo = int(
        math.floor(
            position
        )
    )

    hi = int(
        math.ceil(
            position
        )
    )

    if lo == hi:
        return ordered[lo]

    fraction = (
        position
        -
        lo
    )

    return (
        ordered[lo]
        *
        (
            1.0
            -
            fraction
        )
        +
        ordered[hi]
        *
        fraction
    )


def summarize(
    values,
):

    if not values:
        return None

    return {
        "mean":
            statistics.mean(
                values
            ),

        "p50":
            percentile(
                values,
                0.50,
            ),

        "p95":
            percentile(
                values,
                0.95,
            ),

        "p99":
            percentile(
                values,
                0.99,
            ),

        "best":
            min(
                values
            ),

        "worst":
            max(
                values
            ),
    }


# =============================================================================
# KV CACHE SIZE
# =============================================================================

def tensor_bytes(
    tensor,
):
    return (
        tensor.numel()
        *
        tensor.element_size()
    )


def cache_bytes(
    cache,
):

    if cache is None:
        return 0

    total = 0

    # Modern Hugging Face DynamicCache.
    if hasattr(
        cache,
        "key_cache",
    ):

        try:

            for tensor in cache.key_cache:

                if torch.is_tensor(
                    tensor
                ):
                    total += tensor_bytes(
                        tensor
                    )

            for tensor in cache.value_cache:

                if torch.is_tensor(
                    tensor
                ):
                    total += tensor_bytes(
                        tensor
                    )

            return total

        except Exception:
            pass

    # Newer DynamicCache layer abstraction.
    if hasattr(
        cache,
        "layers",
    ):

        try:

            for layer_cache in cache.layers:

                if hasattr(
                    layer_cache,
                    "keys",
                ):

                    tensor = (
                        layer_cache.keys
                    )

                    if torch.is_tensor(
                        tensor
                    ):
                        total += tensor_bytes(
                            tensor
                        )

                if hasattr(
                    layer_cache,
                    "values",
                ):

                    tensor = (
                        layer_cache.values
                    )

                    if torch.is_tensor(
                        tensor
                    ):
                        total += tensor_bytes(
                            tensor
                        )

            return total

        except Exception:
            pass

    # Legacy tuple cache.
    if isinstance(
        cache,
        (tuple, list),
    ):

        for item in cache:

            if isinstance(
                item,
                (tuple, list),
            ):

                for tensor in item:

                    if torch.is_tensor(
                        tensor
                    ):
                        total += tensor_bytes(
                            tensor
                        )

    return total


# =============================================================================
# TEMP MEMORY
# =============================================================================

def peak_temp_memory_mb(
    fn,
):

    # Warmup / compilation.
    fn()

    torch.cuda.synchronize()

    torch.cuda.empty_cache()

    torch.cuda.reset_peak_memory_stats()

    baseline = (
        torch.cuda.memory_allocated()
    )

    fn()

    torch.cuda.synchronize()

    peak = (
        torch.cuda.max_memory_allocated()
    )

    return (
        peak
        -
        baseline
    ) / 1e6


print("\n" + "=" * 120)
print("TTT MEMORY")
print("=" * 120)

pt_mem_src = W_ACTIVE.clone()
pt_mem_dst = torch.empty_like(
    pt_mem_src
)

tri_mem_src = W_ACTIVE.clone()
tri_mem_dst = torch.empty_like(
    tri_mem_src
)

PYTORCH_TEMP_MB = peak_temp_memory_mb(
    lambda:
        pytorch_ttt(
            pt_mem_src,
            pt_mem_dst,
        )
)

TRITON_TEMP_MB = peak_temp_memory_mb(
    lambda:
        triton_ttt(
            tri_mem_src,
            tri_mem_dst,
        )
)

print(
    "PyTorch transient      :",
    f"{PYTORCH_TEMP_MB:.2f} MB",
)

print(
    "Triton transient       :",
    f"{TRITON_TEMP_MB:.2f} MB",
)

print(
    "Persistent staging     :",
    f"{FAST_WEIGHT_MB:.2f} MB",
)


# =============================================================================
# REAL AUTOREGRESSIVE DECODE
# =============================================================================

def run_decode_v3(
    mode,
):

    valid_modes = {
        "baseline",
        "serialized_triton",
        "async_pytorch",
        "async_triton",
    }

    if mode not in valid_modes:
        raise ValueError(
            mode
        )

    prompt = (
        input_ids[
            :,
            :PROMPT_LEN
        ]
        .clone()
        .contiguous()
    )

    active = W_ACTIVE
    staging = W_STAGING

    update_in_flight = False

    update_start_event = None
    update_end_event = None

    update_started_token = None

    update_gpu_times = []
    visibility_lags = []

    host_token_times = []
    gpu_token_times = []

    torch.cuda.synchronize()

    # =========================================================================
    # PREFILL
    # =========================================================================

    prefill_start = torch.cuda.Event(
        enable_timing=True
    )

    prefill_end = torch.cuda.Event(
        enable_timing=True
    )

    prefill_host_start = (
        time.perf_counter()
    )

    with torch.cuda.stream(
        stream_inference
    ):

        prefill_start.record(
            stream_inference
        )

        with torch.inference_mode():

            output = model(
                input_ids=prompt,
                use_cache=True,
            )

        past_key_values = (
            output.past_key_values
        )

        next_token = (
            output
            .logits[
                :,
                -1,
                :
            ]
            .argmax(
                dim=-1,
                keepdim=True,
            )
        )

        prefill_end.record(
            stream_inference
        )

    # -------------------------------------------------------------------------
    # HARD SYNCHRONIZATION BEFORE elapsed_time()
    # -------------------------------------------------------------------------

    prefill_end.synchronize()

    torch.cuda.synchronize()

    prefill_gpu_ms = (
        prefill_start.elapsed_time(
            prefill_end
        )
    )

    # Ensure token is CPU-visible.
    _ = int(
        next_token.item()
    )

    prefill_host_end = (
        time.perf_counter()
    )

    prefill_host_ms = (
        prefill_host_end
        -
        prefill_host_start
    ) * 1000.0

    kv_prefill_mb = (
        cache_bytes(
            past_key_values
        )
        /
        1e6
    )

    # =========================================================================
    # AUTOREGRESSIVE DECODE
    # =========================================================================

    decode_wall_start = (
        time.perf_counter()
    )

    for token_index in range(
        DECODE_TOKENS
    ):

        # ---------------------------------------------------------------------
        # Commit ready background update.
        # ---------------------------------------------------------------------

        if (
            update_in_flight
            and
            update_end_event.query()
        ):

            update_end_event.synchronize()

            torch.cuda.synchronize()

            update_gpu_times.append(
                update_start_event
                .elapsed_time(
                    update_end_event
                )
            )

            visibility_lags.append(
                token_index
                -
                update_started_token
            )

            active, staging = (
                staging,
                active,
            )

            update_in_flight = False

        trigger_update = (
            token_index
            %
            UPDATE_EVERY
            ==
            0
        )

        # ---------------------------------------------------------------------
        # Launch background update.
        # ---------------------------------------------------------------------

        if (
            trigger_update
            and
            not update_in_flight
            and
            mode
            in {
                "async_pytorch",
                "async_triton",
            }
        ):

            update_start_event = (
                torch.cuda.Event(
                    enable_timing=True
                )
            )

            update_end_event = (
                torch.cuda.Event(
                    enable_timing=True
                )
            )

            update_started_token = (
                token_index
            )

            with torch.cuda.stream(
                stream_learning
            ):

                update_start_event.record(
                    stream_learning
                )

                if (
                    mode
                    ==
                    "async_pytorch"
                ):

                    pytorch_ttt(
                        active,
                        staging,
                    )

                else:

                    triton_ttt(
                        active,
                        staging,
                    )

                update_end_event.record(
                    stream_learning
                )

            update_in_flight = True

        # ---------------------------------------------------------------------
        # Foreground token.
        # ---------------------------------------------------------------------

        token_host_start = (
            time.perf_counter()
        )

        token_start = torch.cuda.Event(
            enable_timing=True
        )

        token_end = torch.cuda.Event(
            enable_timing=True
        )

        with torch.cuda.stream(
            stream_inference
        ):

            # -------------------------------------------------------------
            # Serialized comparison.
            # -------------------------------------------------------------

            if (
                trigger_update
                and
                mode
                ==
                "serialized_triton"
            ):

                triton_ttt(
                    active,
                    staging,
                )

                active, staging = (
                    staging,
                    active,
                )

            # -------------------------------------------------------------
            # Real KV-cached decode.
            # -------------------------------------------------------------

            token_start.record(
                stream_inference
            )

            with torch.inference_mode():

                output = model(
                    input_ids=
                        next_token,

                    past_key_values=
                        past_key_values,

                    use_cache=True,
                )

            past_key_values = (
                output.past_key_values
            )

            next_token = (
                output
                .logits[
                    :,
                    -1,
                    :
                ]
                .argmax(
                    dim=-1,
                    keepdim=True,
                )
            )

            token_end.record(
                stream_inference
            )

        # ---------------------------------------------------------------------
        # HARD SYNCHRONIZATION BEFORE elapsed_time()
        # ---------------------------------------------------------------------

        token_end.synchronize()

        torch.cuda.synchronize()

        token_gpu_ms = (
            token_start.elapsed_time(
                token_end
            )
        )

        # User-visible token.
        _ = int(
            next_token.item()
        )

        token_host_end = (
            time.perf_counter()
        )

        host_token_times.append(
            (
                token_host_end
                -
                token_host_start
            )
            *
            1000.0
        )

        gpu_token_times.append(
            token_gpu_ms
        )

    decode_wall_end = (
        time.perf_counter()
    )

    # =========================================================================
    # COMPLETE LAST UPDATE
    # =========================================================================

    if update_in_flight:

        update_end_event.synchronize()

        torch.cuda.synchronize()

        update_gpu_times.append(
            update_start_event
            .elapsed_time(
                update_end_event
            )
        )

        visibility_lags.append(
            DECODE_TOKENS
            -
            update_started_token
        )

        active, staging = (
            staging,
            active,
        )

        update_in_flight = False

    torch.cuda.synchronize()

    kv_final_mb = (
        cache_bytes(
            past_key_values
        )
        /
        1e6
    )

    decode_total_ms = (
        decode_wall_end
        -
        decode_wall_start
    ) * 1000.0

    result = {
        "mode":
            mode,

        "prefill_host_ms":
            prefill_host_ms,

        "prefill_gpu_ms":
            prefill_gpu_ms,

        "decode_total_ms":
            decode_total_ms,

        "tokens_per_second":
            DECODE_TOKENS
            /
            (
                decode_total_ms
                /
                1000.0
            ),

        "itl_host_ms":
            summarize(
                host_token_times
            ),

        "itl_gpu_ms":
            summarize(
                gpu_token_times
            ),

        "ttt_updates":
            len(
                update_gpu_times
            ),

        "ttt_update_gpu_ms":
            (
                summarize(
                    update_gpu_times
                )
                if update_gpu_times
                else None
            ),

        "visibility_lag_tokens":
            (
                summarize(
                    visibility_lags
                )
                if visibility_lags
                else None
            ),

        "kv_prefill_mb":
            kv_prefill_mb,

        "kv_final_mb":
            kv_final_mb,

        "kv_growth_mb":
            (
                kv_final_mb
                -
                kv_prefill_mb
            ),
    }

    del output
    del past_key_values
    del next_token
    del prompt

    gc.collect()

    torch.cuda.empty_cache()

    return result


# =============================================================================
# JIT + DECODE WARMUP
# =============================================================================

print("\n" + "=" * 120)
print("WARMUP")
print("=" * 120)

triton_ttt(
    W_ACTIVE,
    W_STAGING,
)

pytorch_ttt(
    W_ACTIVE,
    W_STAGING,
)

torch.cuda.synchronize()

print("Running baseline decode warmup...")

warmup_result = run_decode_v3(
    "baseline"
)

print(
    "Warmup baseline p50 ITL:",
    f"{warmup_result['itl_host_ms']['p50']:.3f} ms",
)

print("Warmup complete.")


# =============================================================================
# REAL BENCHMARK
# =============================================================================

MODES = [
    "baseline",
    "serialized_triton",
    "async_pytorch",
    "async_triton",
]

decode_runs = {
    mode: []
    for mode in MODES
}

print("\n" + "=" * 120)
print("REAL AUTOREGRESSIVE KV-CACHE BENCHMARK")
print("=" * 120)

print("Prompt length          :", PROMPT_LEN)
print("Decode tokens          :", DECODE_TOKENS)
print("Update every           :", UPDATE_EVERY)
print("Runs                   :", DECODE_RUNS)


for mode in MODES:

    print(
        "\n"
        +
        "-" * 120
    )

    print(
        mode.upper()
    )

    print(
        "-" * 120
    )

    for run_index in range(
        DECODE_RUNS
    ):

        result = run_decode_v3(
            mode
        )

        decode_runs[
            mode
        ].append(
            result
        )

        print(
            f"run={run_index + 1} | "
            f"p50={result['itl_host_ms']['p50']:.3f} ms | "
            f"p95={result['itl_host_ms']['p95']:.3f} ms | "
            f"p99={result['itl_host_ms']['p99']:.3f} ms | "
            f"tok/s={result['tokens_per_second']:.2f} | "
            f"KV={result['kv_final_mb']:.2f} MB"
        )


# =============================================================================
# AGGREGATION
# =============================================================================

def aggregate_mode(
    runs,
):

    result = {
        "prefill_ms":
            statistics.median(
                [
                    x[
                        "prefill_host_ms"
                    ]
                    for x in runs
                ]
            ),

        "p50_ms":
            statistics.median(
                [
                    x[
                        "itl_host_ms"
                    ][
                        "p50"
                    ]
                    for x in runs
                ]
            ),

        "p95_ms":
            statistics.median(
                [
                    x[
                        "itl_host_ms"
                    ][
                        "p95"
                    ]
                    for x in runs
                ]
            ),

        "p99_ms":
            statistics.median(
                [
                    x[
                        "itl_host_ms"
                    ][
                        "p99"
                    ]
                    for x in runs
                ]
            ),

        "tokens_per_second":
            statistics.median(
                [
                    x[
                        "tokens_per_second"
                    ]
                    for x in runs
                ]
            ),

        "kv_prefill_mb":
            statistics.median(
                [
                    x[
                        "kv_prefill_mb"
                    ]
                    for x in runs
                ]
            ),

        "kv_final_mb":
            statistics.median(
                [
                    x[
                        "kv_final_mb"
                    ]
                    for x in runs
                ]
            ),
    }

    update_values = [
        x[
            "ttt_update_gpu_ms"
        ][
            "p50"
        ]
        for x in runs
        if x[
            "ttt_update_gpu_ms"
        ]
        is not None
    ]

    lag_values = [
        x[
            "visibility_lag_tokens"
        ][
            "p50"
        ]
        for x in runs
        if x[
            "visibility_lag_tokens"
        ]
        is not None
    ]

    result[
        "ttt_update_ms"
    ] = (
        statistics.median(
            update_values
        )
        if update_values
        else None
    )

    result[
        "visibility_lag_tokens"
    ] = (
        statistics.median(
            lag_values
        )
        if lag_values
        else None
    )

    return result


aggregate = {
    mode:
        aggregate_mode(
            decode_runs[
                mode
            ]
        )
    for mode in MODES
}

baseline_p50 = (
    aggregate[
        "baseline"
    ][
        "p50_ms"
    ]
)

for mode in MODES:

    aggregate[
        mode
    ][
        "foreground_slowdown_pct"
    ] = (
        (
            aggregate[
                mode
            ][
                "p50_ms"
            ]
            -
            baseline_p50
        )
        /
        baseline_p50
        *
        100.0
    )


# =============================================================================
# FINAL TABLE
# =============================================================================

print("\n" + "=" * 145)
print("FINAL REAL-DECODE ASYNC-TTT RESULTS")
print("=" * 145)

print(
    f"{'Mode':<24}"
    f"{'p50 ITL':>13}"
    f"{'p95 ITL':>13}"
    f"{'p99 ITL':>13}"
    f"{'tok/s':>12}"
    f"{'slowdown':>13}"
    f"{'TTT ms':>12}"
    f"{'lag':>10}"
)

print(
    "-" * 145
)

for mode in MODES:

    r = aggregate[
        mode
    ]

    ttt_ms = (
        f"{r['ttt_update_ms']:.3f}"
        if r[
            "ttt_update_ms"
        ]
        is not None
        else "-"
    )

    lag = (
        f"{r['visibility_lag_tokens']:.1f}"
        if r[
            "visibility_lag_tokens"
        ]
        is not None
        else "-"
    )

    print(
        f"{mode:<24}"
        f"{r['p50_ms']:>11.3f}ms"
        f"{r['p95_ms']:>11.3f}ms"
        f"{r['p99_ms']:>11.3f}ms"
        f"{r['tokens_per_second']:>12.2f}"
        f"{r['foreground_slowdown_pct']:>12.2f}%"
        f"{ttt_ms:>12}"
        f"{lag:>10}"
    )


print("\nKV CACHE")
print("-" * 120)

print(
    f"At {PROMPT_LEN} tokens        : "
    f"{aggregate['baseline']['kv_prefill_mb']:.2f} MB"
)

print(
    f"At {PROMPT_LEN + DECODE_TOKENS} tokens        : "
    f"{aggregate['baseline']['kv_final_mb']:.2f} MB"
)

print(
    f"Growth over {DECODE_TOKENS} tokens : "
    f"{aggregate['baseline']['kv_final_mb'] - aggregate['baseline']['kv_prefill_mb']:.2f} MB"
)


# =============================================================================
# SAVE PYTHON RESULTS
# =============================================================================

PYTHON_RESULT = {
    "environment": {
        "model":
            MODEL_ID,

        "gpu":
            GPU_NAME,

        "python":
            sys.version.split()[0],

        "pytorch":
            torch.__version__,

        "cuda":
            torch.version.cuda,

        "triton":
            triton.__version__,

        "M":
            M,

        "N":
            N,

        "K":
            TTT_K,

        "layer":
            TTT_LAYER,
    },

    "correctness": {
        "relative_l2_error":
            relative_error,

        "max_absolute_error":
            max_abs_error,
    },

    "memory_mb": {
        "pytorch_transient":
            PYTORCH_TEMP_MB,

        "triton_transient":
            TRITON_TEMP_MB,

        "persistent_staging":
            FAST_WEIGHT_MB,
    },

    "config": {
        "prompt_length":
            PROMPT_LEN,

        "decode_tokens":
            DECODE_TOKENS,

        "update_every":
            UPDATE_EVERY,

        "runs":
            DECODE_RUNS,
    },

    "aggregate":
        aggregate,

    "raw":
        decode_runs,
}


PYTHON_JSON_PATH = Path(
    "async_ttt_real_decode_results.json"
)

with open(
    PYTHON_JSON_PATH,
    "w",
) as f:

    json.dump(
        PYTHON_RESULT,
        f,
        indent=2,
    )

print(
    "\nSaved:",
    PYTHON_JSON_PATH.resolve(),
)


# =============================================================================
# =============================================================================
#
# PART B — RUST + C++/CUDA MOCK RUNTIME
#
# =============================================================================
# =============================================================================

print("\n\n")
print("=" * 120)
print("BUILDING RUST + C++/CUDA MOCK ASYNC-TTT RUNTIME")
print("=" * 120)

NATIVE_ROOT = Path(
    "/tmp/async_ttt_native_runtime_v3"
)

NATIVE_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)


# =============================================================================
# CUDA/C++ SOURCE
# =============================================================================

cuda_source = r'''
#include <cuda_runtime.h>

#include <cstddef>
#include <cstdint>
#include <cstdio>
#include <cstdlib>


#define CUDA_CHECK(call)                                                   \
do {                                                                       \
    cudaError_t err__ = (call);                                            \
    if (err__ != cudaSuccess) {                                            \
        fprintf(                                                           \
            stderr,                                                        \
            "CUDA error at %s:%d: %s\n",                                   \
            __FILE__,                                                      \
            __LINE__,                                                      \
            cudaGetErrorString(err__)                                      \
        );                                                                 \
        std::abort();                                                      \
    }                                                                      \
} while (0)


__global__
void mock_inference_kernel(
    const float* weights,
    float* sink,
    std::size_t count,
    int repeats
) {
    std::size_t index =
        blockIdx.x * blockDim.x
        +
        threadIdx.x;

    std::size_t stride =
        blockDim.x
        *
        gridDim.x;

    float local =
        0.0f;

    for (
        int r = 0;
        r < repeats;
        ++r
    ) {
        for (
            std::size_t i = index;
            i < count;
            i += stride
        ) {
            local +=
                weights[i]
                *
                1e-7f;
        }
    }

    if (
        threadIdx.x
        ==
        0
    ) {
        atomicAdd(
            sink,
            local
        );
    }
}


__global__
void mock_learning_kernel(
    const float* src,
    float* dst,
    std::size_t count,
    int repeats
) {
    std::size_t index =
        blockIdx.x * blockDim.x
        +
        threadIdx.x;

    std::size_t stride =
        blockDim.x
        *
        gridDim.x;

    for (
        std::size_t i = index;
        i < count;
        i += stride
    ) {
        float value =
            src[i];

        for (
            int r = 0;
            r < repeats;
            ++r
        ) {
            value =
                value
                *
                1.0000001f
                +
                1e-8f;
        }

        dst[i] =
            value;
    }
}


struct AsyncTTTEngine {

    cudaStream_t inference_stream;
    cudaStream_t learning_stream;

    cudaEvent_t update_ready;

    cudaEvent_t buffer_last_use[2];

    cudaEvent_t inference_start;
    cudaEvent_t inference_end;

    cudaEvent_t learning_start;
    cudaEvent_t learning_end;

    float* weights[2];

    float* sink;

    std::size_t count;

    int active_index;
    int staging_index;

    bool update_in_flight;

    std::uint64_t version;

    int inference_repeats;
    int learning_repeats;
};


extern "C"
AsyncTTTEngine*
async_ttt_create(
    std::size_t count,
    int inference_repeats,
    int learning_repeats
) {
    auto* e =
        new AsyncTTTEngine();

    e->count =
        count;

    e->active_index =
        0;

    e->staging_index =
        1;

    e->update_in_flight =
        false;

    e->version =
        0;

    e->inference_repeats =
        inference_repeats;

    e->learning_repeats =
        learning_repeats;

    int least_priority =
        0;

    int greatest_priority =
        0;

    CUDA_CHECK(
        cudaDeviceGetStreamPriorityRange(
            &least_priority,
            &greatest_priority
        )
    );

    CUDA_CHECK(
        cudaStreamCreateWithPriority(
            &e->inference_stream,
            cudaStreamNonBlocking,
            greatest_priority
        )
    );

    CUDA_CHECK(
        cudaStreamCreateWithPriority(
            &e->learning_stream,
            cudaStreamNonBlocking,
            least_priority
        )
    );

    CUDA_CHECK(
        cudaEventCreateWithFlags(
            &e->update_ready,
            cudaEventDisableTiming
        )
    );

    CUDA_CHECK(
        cudaEventCreate(
            &e->inference_start
        )
    );

    CUDA_CHECK(
        cudaEventCreate(
            &e->inference_end
        )
    );

    CUDA_CHECK(
        cudaEventCreate(
            &e->learning_start
        )
    );

    CUDA_CHECK(
        cudaEventCreate(
            &e->learning_end
        )
    );

    for (
        int i = 0;
        i < 2;
        ++i
    ) {
        CUDA_CHECK(
            cudaEventCreateWithFlags(
                &e->buffer_last_use[i],
                cudaEventDisableTiming
            )
        );

        CUDA_CHECK(
            cudaMalloc(
                reinterpret_cast<void**>(
                    &e->weights[i]
                ),
                count
                *
                sizeof(float)
            )
        );

        CUDA_CHECK(
            cudaMemset(
                e->weights[i],
                0,
                count
                *
                sizeof(float)
            )
        );
    }

    CUDA_CHECK(
        cudaMalloc(
            reinterpret_cast<void**>(
                &e->sink
            ),
            sizeof(float)
        )
    );

    CUDA_CHECK(
        cudaMemset(
            e->sink,
            0,
            sizeof(float)
        )
    );

    CUDA_CHECK(
        cudaEventRecord(
            e->buffer_last_use[0],
            e->inference_stream
        )
    );

    CUDA_CHECK(
        cudaEventRecord(
            e->buffer_last_use[1],
            e->inference_stream
        )
    );

    CUDA_CHECK(
        cudaStreamSynchronize(
            e->inference_stream
        )
    );

    return e;
}


extern "C"
void
async_ttt_destroy(
    AsyncTTTEngine* e
) {
    if (
        e == nullptr
    ) {
        return;
    }

    CUDA_CHECK(
        cudaDeviceSynchronize()
    );

    for (
        int i = 0;
        i < 2;
        ++i
    ) {
        CUDA_CHECK(
            cudaFree(
                e->weights[i]
            )
        );

        CUDA_CHECK(
            cudaEventDestroy(
                e->buffer_last_use[i]
            )
        );
    }

    CUDA_CHECK(
        cudaFree(
            e->sink
        )
    );

    CUDA_CHECK(
        cudaEventDestroy(
            e->update_ready
        )
    );

    CUDA_CHECK(
        cudaEventDestroy(
            e->inference_start
        )
    );

    CUDA_CHECK(
        cudaEventDestroy(
            e->inference_end
        )
    );

    CUDA_CHECK(
        cudaEventDestroy(
            e->learning_start
        )
    );

    CUDA_CHECK(
        cudaEventDestroy(
            e->learning_end
        )
    );

    CUDA_CHECK(
        cudaStreamDestroy(
            e->inference_stream
        )
    );

    CUDA_CHECK(
        cudaStreamDestroy(
            e->learning_stream
        )
    );

    delete e;
}


extern "C"
float
async_ttt_infer_token(
    AsyncTTTEngine* e
) {
    constexpr int THREADS =
        256;

    const int BLOCKS =
        108
        *
        8;

    CUDA_CHECK(
        cudaEventRecord(
            e->inference_start,
            e->inference_stream
        )
    );

    mock_inference_kernel<<<
        BLOCKS,
        THREADS,
        0,
        e->inference_stream
    >>>(
        e->weights[
            e->active_index
        ],

        e->sink,

        e->count,

        e->inference_repeats
    );

    CUDA_CHECK(
        cudaGetLastError()
    );

    CUDA_CHECK(
        cudaEventRecord(
            e->buffer_last_use[
                e->active_index
            ],
            e->inference_stream
        )
    );

    CUDA_CHECK(
        cudaEventRecord(
            e->inference_end,
            e->inference_stream
        )
    );

    CUDA_CHECK(
        cudaEventSynchronize(
            e->inference_end
        )
    );

    float elapsed =
        0.0f;

    CUDA_CHECK(
        cudaEventElapsedTime(
            &elapsed,
            e->inference_start,
            e->inference_end
        )
    );

    return elapsed;
}


extern "C"
int
async_ttt_begin_update(
    AsyncTTTEngine* e
) {
    if (
        e->update_in_flight
    ) {
        return 0;
    }

    CUDA_CHECK(
        cudaStreamWaitEvent(
            e->learning_stream,
            e->buffer_last_use[
                e->staging_index
            ],
            0
        )
    );

    CUDA_CHECK(
        cudaEventRecord(
            e->learning_start,
            e->learning_stream
        )
    );

    constexpr int THREADS =
        256;

    const int BLOCKS =
        108
        *
        8;

    mock_learning_kernel<<<
        BLOCKS,
        THREADS,
        0,
        e->learning_stream
    >>>(
        e->weights[
            e->active_index
        ],

        e->weights[
            e->staging_index
        ],

        e->count,

        e->learning_repeats
    );

    CUDA_CHECK(
        cudaGetLastError()
    );

    CUDA_CHECK(
        cudaEventRecord(
            e->learning_end,
            e->learning_stream
        )
    );

    CUDA_CHECK(
        cudaEventRecord(
            e->update_ready,
            e->learning_stream
        )
    );

    e->update_in_flight =
        true;

    return 1;
}


extern "C"
int
async_ttt_poll_update(
    AsyncTTTEngine* e
) {
    if (
        !e->update_in_flight
    ) {
        return 0;
    }

    cudaError_t status =
        cudaEventQuery(
            e->update_ready
        );

    if (
        status
        ==
        cudaSuccess
    ) {
        return 1;
    }

    if (
        status
        ==
        cudaErrorNotReady
    ) {
        return 0;
    }

    CUDA_CHECK(
        status
    );

    return -1;
}


extern "C"
int
async_ttt_commit_update(
    AsyncTTTEngine* e
) {
    if (
        !e->update_in_flight
    ) {
        return 0;
    }

    cudaError_t status =
        cudaEventQuery(
            e->update_ready
        );

    if (
        status
        ==
        cudaErrorNotReady
    ) {
        return 0;
    }

    CUDA_CHECK(
        status
    );

    const int old_active =
        e->active_index;

    e->active_index =
        e->staging_index;

    e->staging_index =
        old_active;

    e->version++;

    e->update_in_flight =
        false;

    return 1;
}


extern "C"
float
async_ttt_last_update_ms(
    AsyncTTTEngine* e
) {
    CUDA_CHECK(
        cudaEventSynchronize(
            e->learning_end
        )
    );

    float elapsed =
        0.0f;

    CUDA_CHECK(
        cudaEventElapsedTime(
            &elapsed,
            e->learning_start,
            e->learning_end
        )
    );

    return elapsed;
}


extern "C"
std::uint64_t
async_ttt_version(
    AsyncTTTEngine* e
) {
    return e->version;
}


extern "C"
int
async_ttt_update_in_flight(
    AsyncTTTEngine* e
) {
    return
        e->update_in_flight
        ?
        1
        :
        0;
}


extern "C"
void
async_ttt_sync(
    AsyncTTTEngine* e
) {
    CUDA_CHECK(
        cudaStreamSynchronize(
            e->inference_stream
        )
    );

    CUDA_CHECK(
        cudaStreamSynchronize(
            e->learning_stream
        )
    );
}
'''


CUDA_SOURCE_PATH = (
    NATIVE_ROOT
    /
    "async_ttt_runtime.cu"
)

CUDA_SOURCE_PATH.write_text(
    cuda_source
)


# =============================================================================
# RUST SOURCE
# =============================================================================

rust_source = r'''
use std::ffi::c_void;
use std::time::Instant;


#[link(name = "asyncttt")]
unsafe extern "C" {

    fn async_ttt_create(
        count: usize,
        inference_repeats: i32,
        learning_repeats: i32,
    ) -> *mut c_void;

    fn async_ttt_destroy(
        engine: *mut c_void
    );

    fn async_ttt_infer_token(
        engine: *mut c_void
    ) -> f32;

    fn async_ttt_begin_update(
        engine: *mut c_void
    ) -> i32;

    fn async_ttt_poll_update(
        engine: *mut c_void
    ) -> i32;

    fn async_ttt_commit_update(
        engine: *mut c_void
    ) -> i32;

    fn async_ttt_last_update_ms(
        engine: *mut c_void
    ) -> f32;

    fn async_ttt_version(
        engine: *mut c_void
    ) -> u64;

    fn async_ttt_update_in_flight(
        engine: *mut c_void
    ) -> i32;

    fn async_ttt_sync(
        engine: *mut c_void
    );
}


fn percentile(
    values: &[f64],
    p: f64,
) -> f64 {

    let mut ordered =
        values.to_vec();

    ordered.sort_by(
        |a, b|
            a.partial_cmp(b)
            .unwrap()
    );

    if ordered.is_empty() {
        return 0.0;
    }

    let position =
        (
            (
                ordered.len()
                -
                1
            )
            as f64
        )
        *
        p;

    let lo =
        position.floor()
        as usize;

    let hi =
        position.ceil()
        as usize;

    if lo == hi {
        return ordered[lo];
    }

    let fraction =
        position
        -
        lo
        as f64;

    ordered[lo]
    *
    (
        1.0
        -
        fraction
    )
    +
    ordered[hi]
    *
    fraction
}


fn main() {

    const M: usize =
        3584;

    const N: usize =
        18944;

    const TOKENS: usize =
        64;

    const UPDATE_EVERY: usize =
        8;

    const INFERENCE_REPEATS: i32 =
        4;

    const LEARNING_REPEATS: i32 =
        32;

    let count =
        M
        *
        N;

    unsafe {

        let engine =
            async_ttt_create(
                count,
                INFERENCE_REPEATS,
                LEARNING_REPEATS,
            );

        if engine.is_null() {
            panic!(
                "async_ttt_create failed"
            );
        }

        println!(
            "=============================================================="
        );

        println!(
            "ASYNC-TTT — RUST + C++/CUDA MOCK RUNTIME"
        );

        println!(
            "=============================================================="
        );

        println!(
            "Fast-weight shape   : {} x {}",
            M,
            N
        );

        println!(
            "FP32 buffer         : {:.2} MB",
            (
                count
                *
                4
            )
            as f64
            /
            1e6
        );

        println!(
            "Tokens              : {}",
            TOKENS
        );

        println!(
            "Update interval     : {}",
            UPDATE_EVERY
        );

        let mut token_times:
            Vec<f64> =
            Vec::new();

        let mut update_times:
            Vec<f64> =
            Vec::new();

        let mut visibility_lags:
            Vec<f64> =
            Vec::new();

        let mut update_started:
            Option<usize> =
            None;

        let wall_start =
            Instant::now();

        for token_idx in 0..TOKENS {

            if (
                async_ttt_poll_update(
                    engine
                )
                ==
                1
            ) {

                if (
                    async_ttt_commit_update(
                        engine
                    )
                    ==
                    1
                ) {

                    let update_ms =
                        async_ttt_last_update_ms(
                            engine
                        );

                    update_times.push(
                        update_ms
                        as f64
                    );

                    if let Some(
                        start
                    ) =
                        update_started
                    {

                        visibility_lags.push(
                            (
                                token_idx
                                -
                                start
                            )
                            as f64
                        );
                    }

                    update_started =
                        None;
                }
            }

            if (
                token_idx
                %
                UPDATE_EVERY
                ==
                0
            ) {

                let launched =
                    async_ttt_begin_update(
                        engine
                    );

                if launched == 1 {

                    update_started =
                        Some(
                            token_idx
                        );
                }
            }

            let inference_ms =
                async_ttt_infer_token(
                    engine
                );

            token_times.push(
                inference_ms
                as f64
            );
        }

        async_ttt_sync(
            engine
        );

        if (
            async_ttt_update_in_flight(
                engine
            )
            ==
            1
        ) {

            if (
                async_ttt_commit_update(
                    engine
                )
                ==
                1
            ) {

                let update_ms =
                    async_ttt_last_update_ms(
                        engine
                    );

                update_times.push(
                    update_ms
                    as f64
                );

                if let Some(
                    start
                ) =
                    update_started
                {

                    visibility_lags.push(
                        (
                            TOKENS
                            -
                            start
                        )
                        as f64
                    );
                }
            }
        }

        let wall_ms =
            wall_start
            .elapsed()
            .as_secs_f64()
            *
            1000.0;

        println!();

        println!(
            "=============================================================="
        );

        println!(
            "FINAL NATIVE RUNTIME RESULT"
        );

        println!(
            "=============================================================="
        );

        println!(
            "Token p50          : {:.3} ms",
            percentile(
                &token_times,
                0.50
            )
        );

        println!(
            "Token p95          : {:.3} ms",
            percentile(
                &token_times,
                0.95
            )
        );

        println!(
            "Token p99          : {:.3} ms",
            percentile(
                &token_times,
                0.99
            )
        );

        println!(
            "Wall runtime       : {:.3} ms",
            wall_ms
        );

        println!(
            "Throughput         : {:.2} tokens/s",
            TOKENS
            as f64
            /
            (
                wall_ms
                /
                1000.0
            )
        );

        if !update_times.is_empty() {

            println!(
                "Update p50         : {:.3} ms",
                percentile(
                    &update_times,
                    0.50
                )
            );

            println!(
                "Update p95         : {:.3} ms",
                percentile(
                    &update_times,
                    0.95
                )
            );
        }

        if !visibility_lags.is_empty() {

            println!(
                "Visibility lag p50 : {:.1} tokens",
                percentile(
                    &visibility_lags,
                    0.50
                )
            );
        }

        println!(
            "Committed versions : {}",
            async_ttt_version(
                engine
            )
        );

        async_ttt_destroy(
            engine
        );
    }
}
'''


RUST_SOURCE_PATH = (
    NATIVE_ROOT
    /
    "main.rs"
)

RUST_SOURCE_PATH.write_text(
    rust_source
)


ASYNC-TTT — COMPLETE A100 BENCHMARK
Model                  : Qwen/Qwen2.5-7B-Instruct
Python                 : 3.12.11
PyTorch                : 2.8.0+cu128
CUDA runtime           : 12.8
Triton                 : 3.4.0
GPU                    : NVIDIA A100-SXM4-40GB
Compute capability     : (8, 0)
SMs                    : 108
VRAM                   : 39.49 GiB
TF32                   : False

LOADING MODEL


Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

hidden_size            : 3584
intermediate_size      : 18944
layers                 : 28
model dtype            : torch.bfloat16
FP32 fast weight       : 271.58 MB
Input tokens           : 512

CAPTURING REAL QWEN ACTIVATIONS
TTT layer              : 13
VT                     : (3584, 512)
Z                      : (512, 18944)
W_active               : (3584, 18944)
W_staging              : (3584, 18944)

NUMERICAL CORRECTNESS
Relative L2 error      : 2.209874e-07
Max absolute error     : 1.665335e-16

TTT MEMORY
PyTorch transient      : 271.58 MB
Triton transient       : 0.00 MB
Persistent staging     : 271.58 MB

WARMUP
Running baseline decode warmup...
Warmup baseline p50 ITL: 23.056 ms
Warmup complete.

REAL AUTOREGRESSIVE KV-CACHE BENCHMARK
Prompt length          : 512
Decode tokens          : 64
Update every           : 8
Runs                   : 3

------------------------------------------------------------------------------------------------------------------------
BASELINE
---

8486

In [1]:
from pathlib import Path

CPP_SOURCE_PATH = Path(
    "/tmp/async_ttt_driver_runtime/async_ttt_driver.cpp"
)

src = CPP_SOURCE_PATH.read_text()

old_create = """
    CU_CHECK(
        cuCtxCreate(
            &e->context,
            0,
            e->device
        )
    );
"""

new_create = """
    CU_CHECK(
        cuDevicePrimaryCtxRetain(
            &e->context,
            e->device
        )
    );

    CU_CHECK(
        cuCtxSetCurrent(
            e->context
        )
    );
"""

old_destroy = """
    CU_CHECK(
        cuCtxDestroy(
            e->context
        )
    );
"""

new_destroy = """
    CU_CHECK(
        cuDevicePrimaryCtxRelease(
            e->device
        )
    );
"""

if old_create not in src:
    raise RuntimeError("Could not find old cuCtxCreate block")

src = src.replace(
    old_create,
    new_create,
)

if old_destroy not in src:
    raise RuntimeError("Could not find old cuCtxDestroy block")

src = src.replace(
    old_destroy,
    new_destroy,
)

CPP_SOURCE_PATH.write_text(src)

print("patched successfully")

FileNotFoundError: [Errno 2] No such file or directory: '/tmp/async_ttt_driver_runtime/async_ttt_driver.cpp'

In [2]:
import subprocess
from pathlib import Path

CPP_SOURCE_PATH = Path(
    "/tmp/async_ttt_driver_runtime/async_ttt_driver.cpp"
)

LIB_PATH = Path(
    "/tmp/async_ttt_driver_runtime/libasyncttt.so"
)

cmd = [
    "/usr/bin/g++",

    "-O3",

    "-std=c++17",

    "-fPIC",

    "-shared",

    str(CPP_SOURCE_PATH),

    "-I",
    "/usr/local/cuda/include",

    "-L",
    "/usr/lib/x86_64-linux-gnu",

    "-lcuda",

    "-Wl,-rpath,/usr/lib/x86_64-linux-gnu",

    "-o",
    str(LIB_PATH),
]

print(" ".join(cmd))

result = subprocess.run(
    cmd,
    text=True,
    capture_output=True,
)

print("STDOUT:")
print(result.stdout)

print("STDERR:")
print(result.stderr)

print("return code:", result.returncode)
print("library exists:", LIB_PATH.exists())

/usr/bin/g++ -O3 -std=c++17 -fPIC -shared /tmp/async_ttt_driver_runtime/async_ttt_driver.cpp -I /usr/local/cuda/include -L /usr/lib/x86_64-linux-gnu -lcuda -Wl,-rpath,/usr/lib/x86_64-linux-gnu -o /tmp/async_ttt_driver_runtime/libasyncttt.so
STDOUT:

STDERR:
cc1plus: fatal error: /tmp/async_ttt_driver_runtime/async_ttt_driver.cpp: No such file or directory
compilation terminated.

return code: 1
library exists: False


In [3]:
from pathlib import Path

NATIVE_ROOT = Path(
    "/tmp/async_ttt_driver_runtime"
)

RUST_SOURCE_PATH = (
    NATIVE_ROOT
    /
    "main.rs"
)

rust_source = r'''
use std::ffi::c_void;
use std::time::Instant;

#[link(name = "asyncttt")]
unsafe extern "C" {
    fn async_ttt_create(bytes: usize) -> *mut c_void;
    fn async_ttt_destroy(engine: *mut c_void);

    fn async_ttt_infer_token(
        engine: *mut c_void
    ) -> f32;

    fn async_ttt_begin_update(
        engine: *mut c_void
    ) -> i32;

    fn async_ttt_poll_update(
        engine: *mut c_void
    ) -> i32;

    fn async_ttt_commit_update(
        engine: *mut c_void
    ) -> i32;

    fn async_ttt_last_update_ms(
        engine: *mut c_void
    ) -> f32;

    fn async_ttt_version(
        engine: *mut c_void
    ) -> u64;

    fn async_ttt_update_in_flight(
        engine: *mut c_void
    ) -> i32;

    fn async_ttt_sync(
        engine: *mut c_void
    );
}

fn percentile(
    values: &[f64],
    p: f64,
) -> f64 {
    let mut ordered = values.to_vec();

    ordered.sort_by(
        |a, b|
            a.partial_cmp(b).unwrap()
    );

    if ordered.is_empty() {
        return 0.0;
    }

    let position =
        (ordered.len() - 1) as f64
        *
        p;

    let lo =
        position.floor() as usize;

    let hi =
        position.ceil() as usize;

    if lo == hi {
        return ordered[lo];
    }

    let frac =
        position
        -
        lo as f64;

    ordered[lo]
    *
    (1.0 - frac)
    +
    ordered[hi]
    *
    frac
}

fn main() {

    const M: usize = 3584;
    const N: usize = 18944;

    const BYTES: usize =
        M * N * 4;

    const TOKENS: usize = 64;
    const UPDATE_EVERY: usize = 8;

    unsafe {

        let engine =
            async_ttt_create(
                BYTES
            );

        if engine.is_null() {
            panic!(
                "async_ttt_create failed"
            );
        }

        println!(
            "==============================================="
        );

        println!(
            "ASYNC-TTT RUST CONTROL PLANE"
        );

        println!(
            "==============================================="
        );

        println!(
            "State size: {:.2} MB",
            BYTES as f64 / 1e6
        );

        let mut token_times =
            Vec::<f64>::new();

        let mut update_times =
            Vec::<f64>::new();

        let mut visibility_lags =
            Vec::<f64>::new();

        let mut update_started:
            Option<usize> =
            None;

        let wall_start =
            Instant::now();

        for token_idx in 0..TOKENS {

            if async_ttt_poll_update(
                engine
            ) == 1 {

                if async_ttt_commit_update(
                    engine
                ) == 1 {

                    let update_ms =
                        async_ttt_last_update_ms(
                            engine
                        );

                    update_times.push(
                        update_ms as f64
                    );

                    if let Some(start) =
                        update_started
                    {
                        visibility_lags.push(
                            (token_idx - start)
                            as f64
                        );
                    }

                    update_started = None;
                }
            }

            if token_idx
                %
                UPDATE_EVERY
                ==
                0
            {
                if async_ttt_begin_update(
                    engine
                ) == 1
                {
                    update_started =
                        Some(
                            token_idx
                        );
                }
            }

            let token_ms =
                async_ttt_infer_token(
                    engine
                );

            token_times.push(
                token_ms as f64
            );
        }

        async_ttt_sync(
            engine
        );

        if async_ttt_update_in_flight(
            engine
        ) == 1
        {
            if async_ttt_commit_update(
                engine
            ) == 1
            {
                let update_ms =
                    async_ttt_last_update_ms(
                        engine
                    );

                update_times.push(
                    update_ms as f64
                );

                if let Some(start) =
                    update_started
                {
                    visibility_lags.push(
                        (TOKENS - start)
                        as f64
                    );
                }
            }
        }

        let wall_ms =
            wall_start
            .elapsed()
            .as_secs_f64()
            *
            1000.0;

        println!();

        println!(
            "Token p50: {:.4} ms",
            percentile(
                &token_times,
                0.50
            )
        );

        println!(
            "Token p95: {:.4} ms",
            percentile(
                &token_times,
                0.95
            )
        );

        println!(
            "Token p99: {:.4} ms",
            percentile(
                &token_times,
                0.99
            )
        );

        println!(
            "Wall time: {:.3} ms",
            wall_ms
        );

        println!(
            "Throughput: {:.2} tok/s",
            TOKENS as f64
            /
            (wall_ms / 1000.0)
        );

        if !update_times.is_empty() {

            println!(
                "Update p50: {:.4} ms",
                percentile(
                    &update_times,
                    0.50
                )
            );

            println!(
                "Update p95: {:.4} ms",
                percentile(
                    &update_times,
                    0.95
                )
            );
        }

        if !visibility_lags.is_empty() {

            println!(
                "Visibility lag p50: {:.1} tokens",
                percentile(
                    &visibility_lags,
                    0.50
                )
            );
        }

        println!(
            "Committed versions: {}",
            async_ttt_version(
                engine
            )
        );

        async_ttt_destroy(
            engine
        );
    }
}
'''

RUST_SOURCE_PATH.write_text(
    rust_source
)

print(
    RUST_SOURCE_PATH
)

FileNotFoundError: [Errno 2] No such file or directory: '/tmp/async_ttt_driver_runtime/main.rs'

In [4]:
import shutil
import subprocess
from pathlib import Path

NATIVE_ROOT = Path(
    "/tmp/async_ttt_driver_runtime"
)

RUST_SOURCE_PATH = (
    NATIVE_ROOT
    /
    "main.rs"
)

RUST_BINARY = (
    NATIVE_ROOT
    /
    "async_ttt_rust"
)

rust_candidates = [
    shutil.which("rustc"),
    "/teamspace/studios/this_studio/.cargo/bin/rustc",
    "/home/zeus/.cargo/bin/rustc",
]

RUSTC = None

for candidate in rust_candidates:
    if (
        candidate
        and
        Path(candidate).exists()
    ):
        RUSTC = candidate
        break

if RUSTC is None:
    raise RuntimeError(
        "rustc not found"
    )

print(
    subprocess.check_output(
        [
            RUSTC,
            "--version",
        ],
        text=True,
    ).strip()
)

result = subprocess.run(
    [
        RUSTC,

        "-O",

        "-Awarnings",

        str(
            RUST_SOURCE_PATH
        ),

        "-L",
        f"native={NATIVE_ROOT}",

        "-l",
        "dylib=asyncttt",

        "-C",
        f"link-arg=-Wl,-rpath,{NATIVE_ROOT}",

        "-o",
        str(
            RUST_BINARY
        ),
    ],

    text=True,
    capture_output=True,
)

print(
    result.stdout
)

print(
    result.stderr
)

print(
    "return code:",
    result.returncode
)

print(
    "binary exists:",
    RUST_BINARY.exists()
)

rustc 1.98.1 (48a229cea 2026-09-01)

error: couldn't read `/tmp/async_ttt_driver_runtime/main.rs`: No such file or directory (os error 2)

error: aborting due to 1 previous error


return code: 1
binary exists: False


In [5]:
import os
import subprocess
from pathlib import Path

NATIVE_ROOT = Path(
    "/tmp/async_ttt_driver_runtime"
)

RUST_BINARY = (
    NATIVE_ROOT
    /
    "async_ttt_rust"
)

runtime_env = os.environ.copy()

runtime_env[
    "LD_LIBRARY_PATH"
] = ":".join(
    [
        str(NATIVE_ROOT),
        "/usr/lib/x86_64-linux-gnu",
        runtime_env.get(
            "LD_LIBRARY_PATH",
            "",
        ),
    ]
)

result = subprocess.run(
    [
        str(
            RUST_BINARY
        )
    ],
    cwd=str(
        NATIVE_ROOT
    ),
    env=runtime_env,
    text=True,
    capture_output=True,
)

print("STDOUT:")
print(result.stdout)

print("STDERR:")
print(result.stderr)

print(
    "return code:",
    result.returncode
)

FileNotFoundError: [Errno 2] No such file or directory: '/tmp/async_ttt_driver_runtime'

In [6]:
import gc
import sys
import torch
import triton
import triton.language as tl

from transformers import AutoModelForCausalLM, AutoTokenizer


MODEL_ID = "mistralai/Mistral-Nemo-Instruct-2407"

DEVICE = "cuda"

TTT_LAYER = 20

K_MAX = 2048

LR = 1e-3
CLIP = 1e-5

torch.set_grad_enabled(False)

torch.backends.cuda.matmul.allow_tf32 = False
torch.backends.cudnn.allow_tf32 = False


print("=" * 100)
print("ASYNC-TTT — MISTRAL NEMO 12B")
print("=" * 100)

print("PyTorch :", torch.__version__)
print("CUDA    :", torch.version.cuda)
print("Triton  :", triton.__version__)
print("GPU     :", torch.cuda.get_device_name(0))


tokenizer = AutoTokenizer.from_pretrained(
    MODEL_ID,
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.bfloat16,
    device_map={"": 0},
    low_cpu_mem_usage=True,
)

model.eval()


M = model.config.hidden_size
N = model.config.intermediate_size
NUM_LAYERS = model.config.num_hidden_layers


print()
print("hidden_size       :", M)
print("intermediate_size :", N)
print("layers            :", NUM_LAYERS)
print("TTT layer         :", TTT_LAYER)
print(
    "FP32 W_down       :",
    f"{M * N * 4 / 1e6:.2f} MB",
)


assert TTT_LAYER < NUM_LAYERS


# -------------------------------------------------------------------------
# Build >= 2048 real tokens
# -------------------------------------------------------------------------

base_text = """
Online test-time training allows a model to update an internal learned state
while processing a sequence. A production serving system must perform these
updates concurrently with latency-sensitive autoregressive inference while
controlling memory allocation, synchronization and GPU contention.
"""

text = "\n".join(
    [base_text] * 2000
)


tokens = tokenizer(
    text,
    return_tensors="pt",
    truncation=True,
    max_length=K_MAX,
)


input_ids = tokens[
    "input_ids"
].to(
    DEVICE
)


print(
    "captured token count:",
    input_ids.shape[1],
)


if input_ids.shape[1] < K_MAX:
    raise RuntimeError(
        f"Need {K_MAX} tokens; got {input_ids.shape[1]}"
    )


# -------------------------------------------------------------------------
# Capture real middle-layer MLP inputs
# -------------------------------------------------------------------------

target_layer = model.model.layers[
    TTT_LAYER
]

captured = {}


def hidden_hook(
    module,
    args,
):
    captured["v"] = (
        args[0]
        .detach()
        .clone()
    )


def down_hook(
    module,
    args,
):
    captured["z"] = (
        args[0]
        .detach()
        .clone()
    )


h1 = target_layer.mlp.register_forward_pre_hook(
    hidden_hook
)

h2 = target_layer.mlp.down_proj.register_forward_pre_hook(
    down_hook
)


with torch.inference_mode():

    _ = model(
        input_ids=input_ids[:, :K_MAX],
        use_cache=False,
    )


torch.cuda.synchronize()

h1.remove()
h2.remove()


V_ALL = (
    captured["v"][0]
    .float()
    .contiguous()
)

Z_ALL = (
    captured["z"][0]
    .float()
    .contiguous()
)


W_BASE = (
    target_layer
    .mlp
    .down_proj
    .weight
    .detach()
    .float()
    .contiguous()
)


print()
print("V_ALL  :", tuple(V_ALL.shape))
print("Z_ALL  :", tuple(Z_ALL.shape))
print("W_down :", tuple(W_BASE.shape))


assert V_ALL.shape == (
    K_MAX,
    M,
)

assert Z_ALL.shape == (
    K_MAX,
    N,
)

assert W_BASE.shape == (
    M,
    N,
)


print("\n12B real activations captured successfully.")

ASYNC-TTT — MISTRAL NEMO 12B
PyTorch : 2.8.0+cu128
CUDA    : 12.8
Triton  : 3.4.0
GPU     : NVIDIA A100-SXM4-40GB


[transformers] The tokenizer you are loading from 'mistralai/Mistral-Nemo-Instruct-2407' with an incorrect regex pattern: https://huggingface.co/mistralai/Mistral-Small-3.1-24B-Instruct-2503/discussions/84#69121093e8b480e709447d5e. This will lead to incorrect tokenization. You should set the `fix_mistral_regex=True` flag when loading this tokenizer to fix this issue.
[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/363 [00:00<?, ?it/s]


hidden_size       : 5120
intermediate_size : 14336
layers            : 40
TTT layer         : 20
FP32 W_down       : 293.60 MB
captured token count: 2048

V_ALL  : (2048, 5120)
Z_ALL  : (2048, 14336)
W_down : (5120, 14336)

12B real activations captured successfully.


In [7]:
@triton.jit
def ttt_norm_kernel_12b(
    vt_ptr,
    z_ptr,
    norm_ptr,

    stride_vm,
    stride_vk,

    stride_zk,
    stride_zn,

    M: tl.constexpr,
    N: tl.constexpr,
    K: tl.constexpr,

    BLOCK_M: tl.constexpr,
    BLOCK_N: tl.constexpr,
    BLOCK_K: tl.constexpr,
):

    pid_m = tl.program_id(0)
    pid_n = tl.program_id(1)

    offs_m = (
        pid_m * BLOCK_M
        +
        tl.arange(0, BLOCK_M)
    )

    offs_n = (
        pid_n * BLOCK_N
        +
        tl.arange(0, BLOCK_N)
    )

    acc = tl.zeros(
        (BLOCK_M, BLOCK_N),
        dtype=tl.float32,
    )

    for k in range(
        0,
        K,
        BLOCK_K,
    ):

        offs_k = (
            k
            +
            tl.arange(0, BLOCK_K)
        )

        v = tl.load(
            vt_ptr
            +
            offs_m[:, None] * stride_vm
            +
            offs_k[None, :] * stride_vk,

            mask=(
                (offs_m[:, None] < M)
                &
                (offs_k[None, :] < K)
            ),

            other=0.0,
        )

        z = tl.load(
            z_ptr
            +
            offs_k[:, None] * stride_zk
            +
            offs_n[None, :] * stride_zn,

            mask=(
                (offs_k[:, None] < K)
                &
                (offs_n[None, :] < N)
            ),

            other=0.0,
        )

        acc = tl.dot(
            v,
            z,
            acc,
            input_precision="ieee",
        )

    local_sq = tl.sum(
        acc * acc
    )

    tl.atomic_add(
        norm_ptr,
        local_sq,
    )


@triton.jit
def ttt_update_kernel_12b(
    vt_ptr,
    z_ptr,

    src_ptr,
    dst_ptr,

    norm_ptr,

    stride_vm,
    stride_vk,

    stride_zk,
    stride_zn,

    stride_sm,
    stride_sn,

    stride_dm,
    stride_dn,

    M: tl.constexpr,
    N: tl.constexpr,
    K: tl.constexpr,

    LR: tl.constexpr,
    CLIP: tl.constexpr,

    BLOCK_M: tl.constexpr,
    BLOCK_N: tl.constexpr,
    BLOCK_K: tl.constexpr,
):

    pid_m = tl.program_id(0)
    pid_n = tl.program_id(1)

    offs_m = (
        pid_m * BLOCK_M
        +
        tl.arange(0, BLOCK_M)
    )

    offs_n = (
        pid_n * BLOCK_N
        +
        tl.arange(0, BLOCK_N)
    )

    norm_sq = tl.load(
        norm_ptr
    )

    scale = (
        CLIP
        /
        (
            tl.sqrt(norm_sq)
            +
            1e-8
        )
    )

    scale = tl.minimum(
        scale,
        1.0,
    )

    acc = tl.zeros(
        (BLOCK_M, BLOCK_N),
        dtype=tl.float32,
    )

    for k in range(
        0,
        K,
        BLOCK_K,
    ):

        offs_k = (
            k
            +
            tl.arange(0, BLOCK_K)
        )

        v = tl.load(
            vt_ptr
            +
            offs_m[:, None] * stride_vm
            +
            offs_k[None, :] * stride_vk,

            mask=(
                (offs_m[:, None] < M)
                &
                (offs_k[None, :] < K)
            ),

            other=0.0,
        )

        z = tl.load(
            z_ptr
            +
            offs_k[:, None] * stride_zk
            +
            offs_n[None, :] * stride_zn,

            mask=(
                (offs_k[:, None] < K)
                &
                (offs_n[None, :] < N)
            ),

            other=0.0,
        )

        acc = tl.dot(
            v,
            z,
            acc,
            input_precision="ieee",
        )

    mask = (
        (offs_m[:, None] < M)
        &
        (offs_n[None, :] < N)
    )

    src = tl.load(
        src_ptr
        +
        offs_m[:, None] * stride_sm
        +
        offs_n[None, :] * stride_sn,

        mask=mask,
        other=0.0,
    )

    result = (
        src
        +
        LR
        *
        acc
        *
        scale
    )

    tl.store(
        dst_ptr
        +
        offs_m[:, None] * stride_dm
        +
        offs_n[None, :] * stride_dn,

        result,

        mask=mask,
    )


NORM_BM = 64
NORM_BN = 128
NORM_BK = 64

UPDATE_BM = 64
UPDATE_BN = 64
UPDATE_BK = 64

NORM_WARPS = 4
UPDATE_WARPS = 8


norm_ws_12b = torch.zeros(
    1,
    device="cuda",
    dtype=torch.float32,
)


def triton_ttt_12b(
    vt,
    z,
    src,
    dst,
    K,
):

    norm_ws_12b.zero_()

    norm_grid = (
        triton.cdiv(
            M,
            NORM_BM,
        ),
        triton.cdiv(
            N,
            NORM_BN,
        ),
    )

    update_grid = (
        triton.cdiv(
            M,
            UPDATE_BM,
        ),
        triton.cdiv(
            N,
            UPDATE_BN,
        ),
    )

    ttt_norm_kernel_12b[
        norm_grid
    ](
        vt,
        z,
        norm_ws_12b,

        vt.stride(0),
        vt.stride(1),

        z.stride(0),
        z.stride(1),

        M=M,
        N=N,
        K=K,

        BLOCK_M=NORM_BM,
        BLOCK_N=NORM_BN,
        BLOCK_K=NORM_BK,

        num_warps=NORM_WARPS,
    )

    ttt_update_kernel_12b[
        update_grid
    ](
        vt,
        z,

        src,
        dst,

        norm_ws_12b,

        vt.stride(0),
        vt.stride(1),

        z.stride(0),
        z.stride(1),

        src.stride(0),
        src.stride(1),

        dst.stride(0),
        dst.stride(1),

        M=M,
        N=N,
        K=K,

        LR=LR,
        CLIP=CLIP,

        BLOCK_M=UPDATE_BM,
        BLOCK_N=UPDATE_BN,
        BLOCK_K=UPDATE_BK,

        num_warps=UPDATE_WARPS,
    )


def pytorch_ttt_12b(
    vt,
    z,
    src,
    dst,
):

    dw = (
        vt
        @
        z
    )

    norm = torch.linalg.vector_norm(
        dw
    )

    scale = torch.clamp(
        CLIP
        /
        (
            norm
            +
            1e-8
        ),
        max=1.0,
    )

    dw.mul_(
        scale
    )

    torch.add(
        src,
        dw,
        alpha=LR,
        out=dst,
    )


print("12B kernels ready.")

12B kernels ready.


In [3]:
import math
import statistics
import time


K_VALUES = [
    128,
    256,
    512,
    1024,
    2048,
]


def gpu_time_ms(
    fn,
    warmup=3,
    repeats=10,
):

    for _ in range(
        warmup
    ):
        fn()

    torch.cuda.synchronize()

    times = []

    for _ in range(
        repeats
    ):

        start = torch.cuda.Event(
            enable_timing=True
        )

        end = torch.cuda.Event(
            enable_timing=True
        )

        start.record()

        fn()

        end.record()

        end.synchronize()

        times.append(
            start.elapsed_time(
                end
            )
        )

    return statistics.median(
        times
    )


def transient_memory_mb(
    fn,
):

    gc.collect()
    torch.cuda.empty_cache()
    torch.cuda.synchronize()

    torch.cuda.reset_peak_memory_stats()

    baseline = (
        torch.cuda.memory_allocated()
    )

    fn()

    torch.cuda.synchronize()

    peak = (
        torch.cuda.max_memory_allocated()
    )

    return (
        peak
        -
        baseline
    ) / 1e6


results_12b = []


print()
print("=" * 125)
print("MISTRAL-NEMO 12B — REAL ACTIVATION TTT BENCHMARK")
print("=" * 125)


print(
    f"{'K':>6}"
    f"{'PyTorch ms':>15}"
    f"{'Triton ms':>15}"
    f"{'ratio':>12}"
    f"{'rel L2':>16}"
    f"{'PT temp MB':>15}"
    f"{'Tri temp MB':>15}"
)


print(
    "-" * 125
)


for K in K_VALUES:

    V = (
        V_ALL[
            :K
        ]
        .contiguous()
    )

    VT_K = (
        V.T
        .contiguous()
    )

    Z_K = (
        Z_ALL[
            :K
        ]
        .contiguous()
    )


    src = W_BASE

    pt_dst = torch.empty_like(
        src
    )

    tri_dst = torch.empty_like(
        src
    )


    # ---------------------------------------------------------------------
    # compile/warm Triton
    # ---------------------------------------------------------------------

    triton_ttt_12b(
        VT_K,
        Z_K,
        src,
        tri_dst,
        K,
    )

    torch.cuda.synchronize()


    # ---------------------------------------------------------------------
    # correctness
    # ---------------------------------------------------------------------

    zero_src = torch.zeros_like(
        src
    )

    pt_check = torch.empty_like(
        src
    )

    tri_check = torch.empty_like(
        src
    )


    pytorch_ttt_12b(
        VT_K,
        Z_K,
        zero_src,
        pt_check,
    )


    triton_ttt_12b(
        VT_K,
        Z_K,
        zero_src,
        tri_check,
        K,
    )


    torch.cuda.synchronize()


    diff = (
        tri_check
        -
        pt_check
    )


    rel_l2 = (
        torch.linalg.vector_norm(
            diff
        )
        /
        torch.linalg.vector_norm(
            pt_check
        ).clamp_min(
            1e-30
        )
    ).item()


    # ---------------------------------------------------------------------
    # timings
    # ---------------------------------------------------------------------

    pt_ms = gpu_time_ms(
        lambda:
            pytorch_ttt_12b(
                VT_K,
                Z_K,
                src,
                pt_dst,
            )
    )


    tri_ms = gpu_time_ms(
        lambda:
            triton_ttt_12b(
                VT_K,
                Z_K,
                src,
                tri_dst,
                K,
            )
    )


    # ---------------------------------------------------------------------
    # memory
    # ---------------------------------------------------------------------

    pt_temp = transient_memory_mb(
        lambda:
            pytorch_ttt_12b(
                VT_K,
                Z_K,
                src,
                pt_dst,
            )
    )


    tri_temp = transient_memory_mb(
        lambda:
            triton_ttt_12b(
                VT_K,
                Z_K,
                src,
                tri_dst,
                K,
            )
    )


    ratio = (
        tri_ms
        /
        pt_ms
    )


    result = {
        "K": K,

        "pytorch_ms": pt_ms,

        "triton_ms": tri_ms,

        "ratio": ratio,

        "relative_l2": rel_l2,

        "pytorch_temp_mb": pt_temp,

        "triton_temp_mb": tri_temp,

        "persistent_staging_mb":
            M * N * 4 / 1e6,
    }


    results_12b.append(
        result
    )


    print(
        f"{K:>6}"
        f"{pt_ms:>15.3f}"
        f"{tri_ms:>15.3f}"
        f"{ratio:>12.3f}"
        f"{rel_l2:>16.3e}"
        f"{pt_temp:>15.2f}"
        f"{tri_temp:>15.2f}"
    )


    del (
        V,
        VT_K,
        Z_K,
        pt_dst,
        tri_dst,
        zero_src,
        pt_check,
        tri_check,
        diff,
    )

    gc.collect()

    torch.cuda.empty_cache()


print()
print(
    "Persistent A/B staging buffer:",
    f"{M * N * 4 / 1e6:.2f} MB",
)

print(
    "Model:",
    MODEL_ID,
)

print(
    "Layer:",
    TTT_LAYER,
)

print(
    "Shape:",
    f"{M} x {N}",
)


MISTRAL-NEMO 12B — REAL ACTIVATION TTT BENCHMARK
     K     PyTorch ms      Triton ms       ratio          rel L2     PT temp MB    Tri temp MB
-----------------------------------------------------------------------------------------------------------------------------
   128          2.676          3.387       1.266       5.697e-07         293.60           0.00
   256          3.951          6.263       1.585       5.482e-07         293.60           0.00
   512          6.496          9.394       1.446       4.577e-07         293.60           0.00
  1024         11.580         18.348       1.584       4.811e-07         293.60           0.00
  2048         17.088         36.227       2.120       2.542e-07         293.60           0.00

Persistent A/B staging buffer: 293.60 MB
Model: mistralai/Mistral-Nemo-Instruct-2407
Layer: 20
Shape: 5120 x 14336


In [4]:
import gc
import math
import statistics
import time
import torch


PROMPT_LEN_12B = 512
DECODE_TOKENS_12B = 64

UPDATE_EVERY_12B = 8
TTT_K_DECODE_12B = 512


# -------------------------------------------------------------------------
# Real captured activations for K=512
# -------------------------------------------------------------------------

VT_DECODE_12B = (
    V_ALL[
        :TTT_K_DECODE_12B
    ]
    .T
    .contiguous()
)

Z_DECODE_12B = (
    Z_ALL[
        :TTT_K_DECODE_12B
    ]
    .contiguous()
)


# -------------------------------------------------------------------------
# A/B fast-weight buffers
# -------------------------------------------------------------------------

W_ACTIVE_12B = (
    W_BASE
    .clone()
    .contiguous()
)

W_STAGING_12B = (
    torch.empty_like(
        W_ACTIVE_12B
    )
)


# -------------------------------------------------------------------------
# Streams
# -------------------------------------------------------------------------

try:
    inference_stream_12b = torch.cuda.Stream(
        priority=-1
    )

except Exception:
    inference_stream_12b = torch.cuda.Stream()


learning_stream_12b = torch.cuda.Stream(
    priority=0
)


torch.cuda.synchronize()


# -------------------------------------------------------------------------
# Stats
# -------------------------------------------------------------------------

def percentile_12b(
    values,
    p,
):

    values = sorted(
        values
    )

    if not values:
        return 0.0

    position = (
        len(values) - 1
    ) * p

    lo = math.floor(
        position
    )

    hi = math.ceil(
        position
    )

    if lo == hi:
        return values[lo]

    fraction = (
        position
        -
        lo
    )

    return (
        values[lo]
        *
        (1.0 - fraction)
        +
        values[hi]
        *
        fraction
    )


def summary_12b(
    values,
):

    if not values:
        return None

    return {
        "mean":
            statistics.mean(values),

        "p50":
            percentile_12b(
                values,
                0.50,
            ),

        "p95":
            percentile_12b(
                values,
                0.95,
            ),

        "p99":
            percentile_12b(
                values,
                0.99,
            ),

        "min":
            min(values),

        "max":
            max(values),
    }


# -------------------------------------------------------------------------
# KV-cache memory
# -------------------------------------------------------------------------

def cache_bytes_12b(
    cache,
):

    if cache is None:
        return 0

    total = 0


    if hasattr(
        cache,
        "key_cache",
    ):

        try:

            for x in cache.key_cache:

                if torch.is_tensor(x):
                    total += (
                        x.numel()
                        *
                        x.element_size()
                    )

            for x in cache.value_cache:

                if torch.is_tensor(x):
                    total += (
                        x.numel()
                        *
                        x.element_size()
                    )

            return total

        except Exception:
            pass


    if hasattr(
        cache,
        "layers",
    ):

        try:

            for layer in cache.layers:

                for attr in (
                    "keys",
                    "values",
                ):

                    if hasattr(
                        layer,
                        attr,
                    ):

                        x = getattr(
                            layer,
                            attr,
                        )

                        if torch.is_tensor(x):

                            total += (
                                x.numel()
                                *
                                x.element_size()
                            )

            return total

        except Exception:
            pass


    if isinstance(
        cache,
        (tuple, list),
    ):

        for layer in cache:

            if isinstance(
                layer,
                (tuple, list),
            ):

                for x in layer:

                    if torch.is_tensor(x):

                        total += (
                            x.numel()
                            *
                            x.element_size()
                        )


    return total


print("12B serving benchmark ready.")
print(
    "TTT K:",
    TTT_K_DECODE_12B,
)

print(
    "A/B buffer:",
    f"{W_ACTIVE_12B.numel() * 4 / 1e6:.2f} MB each",
)

12B serving benchmark ready.
TTT K: 512
A/B buffer: 293.60 MB each


In [5]:
def run_12b_decode(
    mode,
):

    assert mode in {
        "baseline",
        "serialized_triton",
        "async_pytorch",
        "async_triton",
    }


    prompt = (
        input_ids[
            :,
            :PROMPT_LEN_12B
        ]
        .contiguous()
    )


    active = W_ACTIVE_12B
    staging = W_STAGING_12B


    update_in_flight = False

    update_start = None
    update_end = None

    update_started_token = None


    update_times = []
    visibility_lags = []

    token_host_times = []
    token_gpu_times = []


    torch.cuda.synchronize()


    # =====================================================================
    # PREFILL
    # =====================================================================

    prefill_start = torch.cuda.Event(
        enable_timing=True
    )

    prefill_end = torch.cuda.Event(
        enable_timing=True
    )


    host_prefill_start = (
        time.perf_counter()
    )


    with torch.cuda.stream(
        inference_stream_12b
    ):

        prefill_start.record(
            inference_stream_12b
        )


        with torch.inference_mode():

            out = model(
                input_ids=prompt,
                use_cache=True,
            )


        past_key_values = (
            out.past_key_values
        )


        next_token = (
            out.logits[
                :,
                -1,
                :
            ]
            .argmax(
                dim=-1,
                keepdim=True,
            )
        )


        prefill_end.record(
            inference_stream_12b
        )


    # Wait ONLY for inference event.
    # Do NOT synchronize the whole GPU here.
    prefill_end.synchronize()


    prefill_gpu_ms = (
        prefill_start.elapsed_time(
            prefill_end
        )
    )


    _ = int(
        next_token.item()
    )


    prefill_host_ms = (
        time.perf_counter()
        -
        host_prefill_start
    ) * 1000.0


    kv_start_mb = (
        cache_bytes_12b(
            past_key_values
        )
        /
        1e6
    )


    # =====================================================================
    # DECODE
    # =====================================================================

    decode_wall_start = (
        time.perf_counter()
    )


    for token_idx in range(
        DECODE_TOKENS_12B
    ):

        # -----------------------------------------------------------------
        # Commit completed background update
        # -----------------------------------------------------------------

        if (
            update_in_flight
            and
            update_end.query()
        ):

            update_end.synchronize()


            update_times.append(
                update_start.elapsed_time(
                    update_end
                )
            )


            visibility_lags.append(
                token_idx
                -
                update_started_token
            )


            active, staging = (
                staging,
                active,
            )


            update_in_flight = False


        trigger_update = (
            token_idx
            %
            UPDATE_EVERY_12B
            ==
            0
        )


        # -----------------------------------------------------------------
        # Launch async update FIRST
        # -----------------------------------------------------------------

        if (
            trigger_update
            and
            not update_in_flight
            and
            mode
            in {
                "async_pytorch",
                "async_triton",
            }
        ):

            update_start = torch.cuda.Event(
                enable_timing=True
            )

            update_end = torch.cuda.Event(
                enable_timing=True
            )


            update_started_token = (
                token_idx
            )


            with torch.cuda.stream(
                learning_stream_12b
            ):

                update_start.record(
                    learning_stream_12b
                )


                if mode == "async_pytorch":

                    pytorch_ttt_12b(
                        VT_DECODE_12B,
                        Z_DECODE_12B,
                        active,
                        staging,
                    )

                else:

                    triton_ttt_12b(
                        VT_DECODE_12B,
                        Z_DECODE_12B,
                        active,
                        staging,
                        TTT_K_DECODE_12B,
                    )


                update_end.record(
                    learning_stream_12b
                )


            update_in_flight = True


        # -----------------------------------------------------------------
        # Foreground token latency starts here
        # -----------------------------------------------------------------

        host_token_start = (
            time.perf_counter()
        )


        token_start = torch.cuda.Event(
            enable_timing=True
        )

        token_end = torch.cuda.Event(
            enable_timing=True
        )


        with torch.cuda.stream(
            inference_stream_12b
        ):

            # -------------------------------------------------------------
            # Serialized baseline
            # -------------------------------------------------------------

            if (
                mode
                ==
                "serialized_triton"
                and
                trigger_update
            ):

                triton_ttt_12b(
                    VT_DECODE_12B,
                    Z_DECODE_12B,
                    active,
                    staging,
                    TTT_K_DECODE_12B,
                )


                active, staging = (
                    staging,
                    active,
                )


            # -------------------------------------------------------------
            # Real autoregressive Mistral token
            # -------------------------------------------------------------

            token_start.record(
                inference_stream_12b
            )


            with torch.inference_mode():

                out = model(
                    input_ids=next_token,
                    past_key_values=
                        past_key_values,
                    use_cache=True,
                )


            past_key_values = (
                out.past_key_values
            )


            next_token = (
                out.logits[
                    :,
                    -1,
                    :
                ]
                .argmax(
                    dim=-1,
                    keepdim=True,
                )
            )


            token_end.record(
                inference_stream_12b
            )


        # IMPORTANT:
        #
        # Wait for inference stream only.
        #
        # We intentionally do NOT call:
        #
        #     torch.cuda.synchronize()
        #
        # because that would also wait for the TTT stream
        # and destroy the async experiment.

        token_end.synchronize()


        token_gpu_ms = (
            token_start.elapsed_time(
                token_end
            )
        )


        _ = int(
            next_token.item()
        )


        token_host_ms = (
            time.perf_counter()
            -
            host_token_start
        ) * 1000.0


        token_host_times.append(
            token_host_ms
        )


        token_gpu_times.append(
            token_gpu_ms
        )


    # ---------------------------------------------------------------------
    # User-visible foreground decode ends here
    # ---------------------------------------------------------------------

    foreground_end = (
        time.perf_counter()
    )


    foreground_ms = (
        foreground_end
        -
        decode_wall_start
    ) * 1000.0


    # ---------------------------------------------------------------------
    # Now finish any remaining background update
    # ---------------------------------------------------------------------

    if update_in_flight:

        update_end.synchronize()


        update_times.append(
            update_start.elapsed_time(
                update_end
            )
        )


        visibility_lags.append(
            DECODE_TOKENS_12B
            -
            update_started_token
        )


        active, staging = (
            staging,
            active,
        )


        update_in_flight = False


    # Everything complete now.
    torch.cuda.synchronize()


    makespan_ms = (
        time.perf_counter()
        -
        decode_wall_start
    ) * 1000.0


    kv_end_mb = (
        cache_bytes_12b(
            past_key_values
        )
        /
        1e6
    )


    result = {

        "mode":
            mode,

        "prefill_host_ms":
            prefill_host_ms,

        "prefill_gpu_ms":
            prefill_gpu_ms,

        "foreground_decode_ms":
            foreground_ms,

        "makespan_ms":
            makespan_ms,

        "tokens_per_second":
            DECODE_TOKENS_12B
            /
            (
                foreground_ms
                /
                1000.0
            ),

        "host_itl":
            summary_12b(
                token_host_times
            ),

        "gpu_itl":
            summary_12b(
                token_gpu_times
            ),

        "ttt_update":
            summary_12b(
                update_times
            ),

        "visibility_lag":
            summary_12b(
                visibility_lags
            ),

        "kv_start_mb":
            kv_start_mb,

        "kv_end_mb":
            kv_end_mb,

        "kv_growth_mb":
            kv_end_mb
            -
            kv_start_mb,
    }


    del out
    del past_key_values
    del next_token


    gc.collect()
    torch.cuda.empty_cache()


    return result


print("12B decode function ready.")

12B decode function ready.


In [6]:
print("Running 12B baseline warmup...")

warmup_12b = run_12b_decode(
    "baseline"
)

print(
    "p50:",
    f"{warmup_12b['host_itl']['p50']:.3f} ms",
)

print(
    "p95:",
    f"{warmup_12b['host_itl']['p95']:.3f} ms",
)

print(
    "KV:",
    f"{warmup_12b['kv_start_mb']:.2f}",
    "->",
    f"{warmup_12b['kv_end_mb']:.2f} MB",
)

print("Warmup passed.")

Running 12B baseline warmup...
p50: 30.977 ms
p95: 31.496 ms
KV: 83.89 -> 94.37 MB
Warmup passed.


In [7]:
import statistics


MODES_12B = [
    "baseline",
    "serialized_triton",
    "async_pytorch",
    "async_triton",
]

RUNS_12B = 3


all_runs_12b = {
    mode: []
    for mode in MODES_12B
}


print("=" * 140)
print("MISTRAL-NEMO 12B — REAL AUTOREGRESSIVE ASYNC-TTT BENCHMARK")
print("=" * 140)


for mode in MODES_12B:

    print()
    print("-" * 100)
    print(mode.upper())
    print("-" * 100)

    for run_idx in range(
        RUNS_12B
    ):

        result = run_12b_decode(
            mode
        )

        all_runs_12b[
            mode
        ].append(
            result
        )

        update_p50 = (
            result[
                "ttt_update"
            ][
                "p50"
            ]
            if result[
                "ttt_update"
            ]
            is not None
            else None
        )

        lag_p50 = (
            result[
                "visibility_lag"
            ][
                "p50"
            ]
            if result[
                "visibility_lag"
            ]
            is not None
            else None
        )

        print(
            f"run={run_idx + 1} | "
            f"p50={result['host_itl']['p50']:.3f} ms | "
            f"p95={result['host_itl']['p95']:.3f} ms | "
            f"p99={result['host_itl']['p99']:.3f} ms | "
            f"tok/s={result['tokens_per_second']:.2f} | "
            f"TTT={update_p50 if update_p50 is not None else '-'} | "
            f"lag={lag_p50 if lag_p50 is not None else '-'}"
        )


def aggregate_12b_mode(
    runs
):

    result = {

        "p50_ms":
            statistics.median(
                [
                    r[
                        "host_itl"
                    ][
                        "p50"
                    ]
                    for r in runs
                ]
            ),

        "p95_ms":
            statistics.median(
                [
                    r[
                        "host_itl"
                    ][
                        "p95"
                    ]
                    for r in runs
                ]
            ),

        "p99_ms":
            statistics.median(
                [
                    r[
                        "host_itl"
                    ][
                        "p99"
                    ]
                    for r in runs
                ]
            ),

        "tok_s":
            statistics.median(
                [
                    r[
                        "tokens_per_second"
                    ]
                    for r in runs
                ]
            ),

        "foreground_ms":
            statistics.median(
                [
                    r[
                        "foreground_decode_ms"
                    ]
                    for r in runs
                ]
            ),

        "makespan_ms":
            statistics.median(
                [
                    r[
                        "makespan_ms"
                    ]
                    for r in runs
                ]
            ),

        "kv_start_mb":
            statistics.median(
                [
                    r[
                        "kv_start_mb"
                    ]
                    for r in runs
                ]
            ),

        "kv_end_mb":
            statistics.median(
                [
                    r[
                        "kv_end_mb"
                    ]
                    for r in runs
                ]
            ),
    }


    updates = [
        r[
            "ttt_update"
        ][
            "p50"
        ]
        for r in runs
        if r[
            "ttt_update"
        ]
        is not None
    ]


    lags = [
        r[
            "visibility_lag"
        ][
            "p50"
        ]
        for r in runs
        if r[
            "visibility_lag"
        ]
        is not None
    ]


    result[
        "ttt_ms"
    ] = (
        statistics.median(
            updates
        )
        if updates
        else None
    )


    result[
        "lag_tokens"
    ] = (
        statistics.median(
            lags
        )
        if lags
        else None
    )


    return result


aggregate_12b = {

    mode:
        aggregate_12b_mode(
            all_runs_12b[
                mode
            ]
        )

    for mode in MODES_12B
}


baseline_p50_12b = (
    aggregate_12b[
        "baseline"
    ][
        "p50_ms"
    ]
)


for mode in MODES_12B:

    aggregate_12b[
        mode
    ][
        "slowdown_pct"
    ] = (
        (
            aggregate_12b[
                mode
            ][
                "p50_ms"
            ]
            -
            baseline_p50_12b
        )
        /
        baseline_p50_12b
        *
        100.0
    )


print()
print("=" * 150)
print("FINAL 12B RESULTS")
print("=" * 150)


print(
    f"{'Mode':<24}"
    f"{'p50 ITL':>13}"
    f"{'p95 ITL':>13}"
    f"{'p99 ITL':>13}"
    f"{'tok/s':>12}"
    f"{'slowdown':>13}"
    f"{'TTT ms':>13}"
    f"{'lag':>10}"
)


print("-" * 150)


for mode in MODES_12B:

    r = aggregate_12b[
        mode
    ]


    ttt_text = (
        f"{r['ttt_ms']:.3f}"
        if r[
            "ttt_ms"
        ]
        is not None
        else "-"
    )


    lag_text = (
        f"{r['lag_tokens']:.1f}"
        if r[
            "lag_tokens"
        ]
        is not None
        else "-"
    )


    print(
        f"{mode:<24}"
        f"{r['p50_ms']:>11.3f}ms"
        f"{r['p95_ms']:>11.3f}ms"
        f"{r['p99_ms']:>11.3f}ms"
        f"{r['tok_s']:>12.2f}"
        f"{r['slowdown_pct']:>12.2f}%"
        f"{ttt_text:>13}"
        f"{lag_text:>10}"
    )


print()
print("KV CACHE")
print("-" * 80)

print(
    f"512 tokens : "
    f"{aggregate_12b['baseline']['kv_start_mb']:.2f} MB"
)

print(
    f"576 tokens : "
    f"{aggregate_12b['baseline']['kv_end_mb']:.2f} MB"
)

print(
    f"growth     : "
    f"{aggregate_12b['baseline']['kv_end_mb'] - aggregate_12b['baseline']['kv_start_mb']:.2f} MB"
)

MISTRAL-NEMO 12B — REAL AUTOREGRESSIVE ASYNC-TTT BENCHMARK

----------------------------------------------------------------------------------------------------
BASELINE
----------------------------------------------------------------------------------------------------
run=1 | p50=31.128 ms | p95=31.908 ms | p99=32.269 ms | tok/s=32.05 | TTT=- | lag=-
run=2 | p50=31.389 ms | p95=31.946 ms | p99=32.624 ms | tok/s=31.79 | TTT=- | lag=-
run=3 | p50=31.243 ms | p95=31.850 ms | p99=32.085 ms | tok/s=31.97 | TTT=- | lag=-

----------------------------------------------------------------------------------------------------
SERIALIZED_TRITON
----------------------------------------------------------------------------------------------------
run=1 | p50=31.282 ms | p95=34.627 ms | p99=34.651 ms | tok/s=31.67 | TTT=- | lag=-
run=2 | p50=31.383 ms | p95=34.643 ms | p99=34.670 ms | tok/s=31.44 | TTT=- | lag=-
run=3 | p50=31.434 ms | p95=34.629 ms | p99=34.704 ms | tok/s=31.43 | TTT=- | lag=-

---

In [8]:
# =============================================================================
# ASYNC-TTT — QWEN2.5-14B COMPLETE A100 BENCHMARK
#
# One standalone cell after kernel restart.
#
# Tests:
#   1. Real Qwen2.5-14B weights
#   2. Real middle-layer MLP activations
#   3. K = 128 / 256 / 512 / 1024 / 2048 kernel scaling
#   4. PyTorch vs two-pass Triton
#   5. Numerical correctness
#   6. Transient memory
#   7. Real KV-cached autoregressive inference
#   8. Baseline
#   9. Serialized Triton
#  10. Async PyTorch
#  11. Async Triton
#  12. p50 / p95 / p99 ITL
#  13. tokens/sec
#  14. update latency
#  15. visibility lag
#  16. KV-cache growth
#
# NOTE:
#   The TTT update uses real model weights and real activations.
#   The A/B FP32 fast-weight buffers are NOT yet wired into Qwen's actual
#   down_proj forward. This experiment measures real inference contention
#   with the real TTT workload. Semantic fast-weight consumption comes next.
# =============================================================================


# =============================================================================
# IMPORTS
# =============================================================================

import gc
import json
import math
import statistics
import sys
import time
from pathlib import Path

import torch
import triton
import triton.language as tl

from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
)


# =============================================================================
# FLUSH PREVIOUS MODEL / CUDA STATE
# =============================================================================

print("=" * 120)
print("CLEARING PREVIOUS CUDA STATE")
print("=" * 120)


# If this is being run without restarting the kernel,
# explicitly remove common objects from the previous 7B/12B experiments.

_cleanup_names = [
    "model",
    "tokenizer",
    "target_layer",
    "V_ALL",
    "Z_ALL",
    "W_BASE",
    "W_ACTIVE",
    "W_STAGING",
    "W_ACTIVE_12B",
    "W_STAGING_12B",
    "VT",
    "VT_DECODE_12B",
    "Z_REAL",
    "Z_DECODE_12B",
    "captured",
    "all_runs_12b",
    "aggregate_12b",
    "results_12b",
]

for _name in _cleanup_names:
    if _name in globals():
        try:
            del globals()[_name]
        except Exception:
            pass


gc.collect()

if torch.cuda.is_available():
    torch.cuda.synchronize()
    torch.cuda.empty_cache()

    try:
        torch.cuda.ipc_collect()
    except Exception:
        pass


if torch.cuda.is_available():

    free_bytes, total_bytes = (
        torch.cuda.mem_get_info()
    )

    print(
        "Free VRAM :",
        f"{free_bytes / 1024**3:.2f} GiB",
    )

    print(
        "Total VRAM:",
        f"{total_bytes / 1024**3:.2f} GiB",
    )


# =============================================================================
# CONFIG
# =============================================================================

MODEL_ID = (
    "Qwen/Qwen2.5-14B-Instruct"
)

DEVICE = "cuda"


# Real activation capture.
K_MAX = 2048


# Serving experiment.
PROMPT_LEN = 512
DECODE_TOKENS = 64

TTT_K_SERVING = 512
UPDATE_EVERY = 8

DECODE_RUNS = 3


# TTT hyperparameters.
LR = 1e-3
CLIP = 1e-5


# Kernel sweep.
K_VALUES = [
    128,
    256,
    512,
    1024,
    2048,
]


torch.set_grad_enabled(
    False
)

torch.manual_seed(
    0
)

torch.backends.cuda.matmul.allow_tf32 = False
torch.backends.cudnn.allow_tf32 = False


# =============================================================================
# ENVIRONMENT
# =============================================================================

assert torch.cuda.is_available(), (
    "CUDA GPU required"
)


GPU_NAME = (
    torch.cuda.get_device_name(0)
)

GPU_PROPERTIES = (
    torch.cuda.get_device_properties(0)
)


print("\n" + "=" * 120)
print("ASYNC-TTT — QWEN2.5-14B")
print("=" * 120)

print(
    "Model                  :",
    MODEL_ID,
)

print(
    "Python                 :",
    sys.version.split()[0],
)

print(
    "PyTorch                :",
    torch.__version__,
)

print(
    "CUDA runtime           :",
    torch.version.cuda,
)

print(
    "Triton                 :",
    triton.__version__,
)

print(
    "GPU                    :",
    GPU_NAME,
)

print(
    "SMs                    :",
    GPU_PROPERTIES.multi_processor_count,
)

print(
    "VRAM                   :",
    f"{GPU_PROPERTIES.total_memory / 1024**3:.2f} GiB",
)

print(
    "TF32                   :",
    False,
)


# =============================================================================
# LOAD QWEN 14B
# =============================================================================

print("\n" + "=" * 120)
print("LOADING QWEN2.5-14B")
print("=" * 120)


tokenizer = (
    AutoTokenizer.from_pretrained(
        MODEL_ID,
        trust_remote_code=True,
    )
)


model = (
    AutoModelForCausalLM.from_pretrained(
        MODEL_ID,

        torch_dtype=
            torch.bfloat16,

        device_map={
            "": 0
        },

        low_cpu_mem_usage=True,

        trust_remote_code=True,

        attn_implementation=
            "sdpa",
    )
)


model.eval()


M = int(
    model.config.hidden_size
)

N = int(
    model.config.intermediate_size
)

NUM_LAYERS = int(
    model.config.num_hidden_layers
)


# Pick actual middle layer automatically.
TTT_LAYER = (
    NUM_LAYERS
    //
    2
)


FAST_WEIGHT_BYTES = (
    M
    *
    N
    *
    4
)

FAST_WEIGHT_MB = (
    FAST_WEIGHT_BYTES
    /
    1e6
)


print()
print(
    "hidden_size            :",
    M,
)

print(
    "intermediate_size      :",
    N,
)

print(
    "layers                 :",
    NUM_LAYERS,
)

print(
    "TTT layer              :",
    TTT_LAYER,
)

print(
    "model dtype            :",
    next(
        model.parameters()
    ).dtype,
)

print(
    "FP32 W_down            :",
    f"{FAST_WEIGHT_MB:.2f} MB",
)


free_bytes, total_bytes = (
    torch.cuda.mem_get_info()
)

print(
    "Free VRAM after load   :",
    f"{free_bytes / 1024**3:.2f} GiB",
)


# =============================================================================
# BUILD REAL 2048-TOKEN INPUT
# =============================================================================

base_text = """
Test-time training allows a language model to update a learned internal state
while processing incoming information. A production serving system must execute
online learning concurrently with latency-sensitive autoregressive inference.
The runtime must minimize synchronization, temporary memory allocation and GPU
contention while preserving numerical correctness and bounded update staleness.
"""


text = "\n".join(
    [base_text] * 2500
)


tokenized = (
    tokenizer(
        text,

        return_tensors="pt",

        truncation=True,

        max_length=K_MAX,
    )
)


input_ids = (
    tokenized[
        "input_ids"
    ]
    .to(
        DEVICE
    )
)


if (
    input_ids.shape[1]
    <
    K_MAX
):

    raise RuntimeError(
        f"Need {K_MAX} tokens; "
        f"got {input_ids.shape[1]}"
    )


print(
    "Input tokens           :",
    input_ids.shape[1],
)


del tokenized
del text

gc.collect()


# =============================================================================
# CAPTURE REAL QWEN MLP ACTIVATIONS
#
# IMPORTANT:
#   We call model.model(), NOT the LM head.
#
#   This avoids allocating logits for:
#
#       2048 × ~152k vocabulary
#
#   which saves a large amount of VRAM during activation capture.
# =============================================================================

print("\n" + "=" * 120)
print("CAPTURING REAL 14B MLP ACTIVATIONS")
print("=" * 120)


target_layer = (
    model.model.layers[
        TTT_LAYER
    ]
)


captured = {}


def capture_hidden_hook(
    module,
    args,
):

    captured[
        "v"
    ] = (
        args[0]
        .detach()
        .clone()
    )


def capture_down_hook(
    module,
    args,
):

    captured[
        "z"
    ] = (
        args[0]
        .detach()
        .clone()
    )


hook_hidden = (
    target_layer
    .mlp
    .register_forward_pre_hook(
        capture_hidden_hook
    )
)


hook_down = (
    target_layer
    .mlp
    .down_proj
    .register_forward_pre_hook(
        capture_down_hook
    )
)


try:

    with torch.inference_mode():

        capture_output = (
            model.model(
                input_ids=
                    input_ids[
                        :,
                        :K_MAX
                    ],

                use_cache=False,
            )
        )

    torch.cuda.synchronize()

finally:

    hook_hidden.remove()
    hook_down.remove()


# Convert once to strict FP32 TTT tensors.

V_ALL = (
    captured[
        "v"
    ][
        0,
        :K_MAX
    ]
    .float()
    .contiguous()
)


Z_ALL = (
    captured[
        "z"
    ][
        0,
        :K_MAX
    ]
    .float()
    .contiguous()
)


# Real Qwen down projection.

W_BASE = (
    target_layer
    .mlp
    .down_proj
    .weight
    .detach()
    .float()
    .contiguous()
)


assert V_ALL.shape == (
    K_MAX,
    M,
)


assert Z_ALL.shape == (
    K_MAX,
    N,
)


assert W_BASE.shape == (
    M,
    N,
)


print(
    "V_ALL                  :",
    tuple(
        V_ALL.shape
    ),
)

print(
    "Z_ALL                  :",
    tuple(
        Z_ALL.shape
    ),
)

print(
    "W_down                 :",
    tuple(
        W_BASE.shape
    ),
)


# Release BF16 hook clones + capture output.

del capture_output
del captured

gc.collect()
torch.cuda.empty_cache()


free_bytes, _ = (
    torch.cuda.mem_get_info()
)

print(
    "Free VRAM after capture:",
    f"{free_bytes / 1024**3:.2f} GiB",
)


# =============================================================================
# TRITON PASS 1
#
# Compute:
#
#       dW_tile = V^T Z
#
# accumulate:
#
#       ||dW||_F²
#
# discard dW.
# =============================================================================

@triton.jit
def ttt_norm_kernel_14b(
    vt_ptr,
    z_ptr,
    norm_ptr,

    stride_vm,
    stride_vk,

    stride_zk,
    stride_zn,

    M: tl.constexpr,
    N: tl.constexpr,
    K: tl.constexpr,

    BLOCK_M: tl.constexpr,
    BLOCK_N: tl.constexpr,
    BLOCK_K: tl.constexpr,
):

    pid_m = (
        tl.program_id(0)
    )

    pid_n = (
        tl.program_id(1)
    )


    offs_m = (
        pid_m
        *
        BLOCK_M
        +
        tl.arange(
            0,
            BLOCK_M,
        )
    )


    offs_n = (
        pid_n
        *
        BLOCK_N
        +
        tl.arange(
            0,
            BLOCK_N,
        )
    )


    acc = tl.zeros(
        (
            BLOCK_M,
            BLOCK_N,
        ),
        dtype=
            tl.float32,
    )


    for k in range(
        0,
        K,
        BLOCK_K,
    ):

        offs_k = (
            k
            +
            tl.arange(
                0,
                BLOCK_K,
            )
        )


        v = tl.load(
            vt_ptr
            +
            offs_m[:, None]
            *
            stride_vm
            +
            offs_k[None, :]
            *
            stride_vk,

            mask=(
                (
                    offs_m[:, None]
                    <
                    M
                )
                &
                (
                    offs_k[None, :]
                    <
                    K
                )
            ),

            other=0.0,
        )


        z = tl.load(
            z_ptr
            +
            offs_k[:, None]
            *
            stride_zk
            +
            offs_n[None, :]
            *
            stride_zn,

            mask=(
                (
                    offs_k[:, None]
                    <
                    K
                )
                &
                (
                    offs_n[None, :]
                    <
                    N
                )
            ),

            other=0.0,
        )


        acc = tl.dot(
            v,
            z,
            acc,
            input_precision=
                "ieee",
        )


    local_sq = (
        tl.sum(
            acc
            *
            acc
        )
    )


    tl.atomic_add(
        norm_ptr,
        local_sq,
    )


# =============================================================================
# TRITON PASS 2
#
# Rematerialize dW and:
#
#       W_dst =
#           W_src
#           +
#           LR * clipped(dW)
#
# =============================================================================

@triton.jit
def ttt_update_kernel_14b(
    vt_ptr,
    z_ptr,

    src_ptr,
    dst_ptr,

    norm_ptr,

    stride_vm,
    stride_vk,

    stride_zk,
    stride_zn,

    stride_sm,
    stride_sn,

    stride_dm,
    stride_dn,

    M: tl.constexpr,
    N: tl.constexpr,
    K: tl.constexpr,

    LR: tl.constexpr,
    CLIP: tl.constexpr,

    BLOCK_M: tl.constexpr,
    BLOCK_N: tl.constexpr,
    BLOCK_K: tl.constexpr,
):

    pid_m = (
        tl.program_id(0)
    )

    pid_n = (
        tl.program_id(1)
    )


    offs_m = (
        pid_m
        *
        BLOCK_M
        +
        tl.arange(
            0,
            BLOCK_M,
        )
    )


    offs_n = (
        pid_n
        *
        BLOCK_N
        +
        tl.arange(
            0,
            BLOCK_N,
        )
    )


    norm_sq = tl.load(
        norm_ptr
    )


    scale = (
        CLIP
        /
        (
            tl.sqrt(
                norm_sq
            )
            +
            1e-8
        )
    )


    scale = tl.minimum(
        scale,
        1.0,
    )


    acc = tl.zeros(
        (
            BLOCK_M,
            BLOCK_N,
        ),
        dtype=
            tl.float32,
    )


    for k in range(
        0,
        K,
        BLOCK_K,
    ):

        offs_k = (
            k
            +
            tl.arange(
                0,
                BLOCK_K,
            )
        )


        v = tl.load(
            vt_ptr
            +
            offs_m[:, None]
            *
            stride_vm
            +
            offs_k[None, :]
            *
            stride_vk,

            mask=(
                (
                    offs_m[:, None]
                    <
                    M
                )
                &
                (
                    offs_k[None, :]
                    <
                    K
                )
            ),

            other=0.0,
        )


        z = tl.load(
            z_ptr
            +
            offs_k[:, None]
            *
            stride_zk
            +
            offs_n[None, :]
            *
            stride_zn,

            mask=(
                (
                    offs_k[:, None]
                    <
                    K
                )
                &
                (
                    offs_n[None, :]
                    <
                    N
                )
            ),

            other=0.0,
        )


        acc = tl.dot(
            v,
            z,
            acc,
            input_precision=
                "ieee",
        )


    mask = (
        (
            offs_m[:, None]
            <
            M
        )
        &
        (
            offs_n[None, :]
            <
            N
        )
    )


    src = tl.load(
        src_ptr
        +
        offs_m[:, None]
        *
        stride_sm
        +
        offs_n[None, :]
        *
        stride_sn,

        mask=
            mask,

        other=
            0.0,
    )


    updated = (
        src
        +
        LR
        *
        acc
        *
        scale
    )


    tl.store(
        dst_ptr
        +
        offs_m[:, None]
        *
        stride_dm
        +
        offs_n[None, :]
        *
        stride_dn,

        updated,

        mask=
            mask,
    )


# =============================================================================
# KERNEL CONFIGURATION
# =============================================================================

NORM_BM = 64
NORM_BN = 128
NORM_BK = 64

UPDATE_BM = 64
UPDATE_BN = 64
UPDATE_BK = 64

NORM_WARPS = 4
UPDATE_WARPS = 8


norm_ws_14b = torch.zeros(
    1,
    device=
        DEVICE,
    dtype=
        torch.float32,
)


# =============================================================================
# TRITON WRAPPER
# =============================================================================

def triton_ttt_14b(
    vt,
    z,
    src,
    dst,
    K,
):

    norm_ws_14b.zero_()


    norm_grid = (
        triton.cdiv(
            M,
            NORM_BM,
        ),

        triton.cdiv(
            N,
            NORM_BN,
        ),
    )


    update_grid = (
        triton.cdiv(
            M,
            UPDATE_BM,
        ),

        triton.cdiv(
            N,
            UPDATE_BN,
        ),
    )


    ttt_norm_kernel_14b[
        norm_grid
    ](
        vt,
        z,
        norm_ws_14b,

        vt.stride(0),
        vt.stride(1),

        z.stride(0),
        z.stride(1),

        M=M,
        N=N,
        K=K,

        BLOCK_M=
            NORM_BM,

        BLOCK_N=
            NORM_BN,

        BLOCK_K=
            NORM_BK,

        num_warps=
            NORM_WARPS,
    )


    ttt_update_kernel_14b[
        update_grid
    ](
        vt,
        z,

        src,
        dst,

        norm_ws_14b,

        vt.stride(0),
        vt.stride(1),

        z.stride(0),
        z.stride(1),

        src.stride(0),
        src.stride(1),

        dst.stride(0),
        dst.stride(1),

        M=M,
        N=N,
        K=K,

        LR=LR,
        CLIP=CLIP,

        BLOCK_M=
            UPDATE_BM,

        BLOCK_N=
            UPDATE_BN,

        BLOCK_K=
            UPDATE_BK,

        num_warps=
            UPDATE_WARPS,
    )


# =============================================================================
# PYTORCH / cuBLAS REFERENCE
# =============================================================================

def pytorch_ttt_14b(
    vt,
    z,
    src,
    dst,
):

    dw = (
        vt
        @
        z
    )


    frobenius_norm = (
        torch.linalg.vector_norm(
            dw
        )
    )


    scale = torch.clamp(
        CLIP
        /
        (
            frobenius_norm
            +
            1e-8
        ),

        max=
            1.0,
    )


    dw.mul_(
        scale
    )


    torch.add(
        src,
        dw,

        alpha=
            LR,

        out=
            dst,
    )


print("\n14B TTT kernels ready.")


# =============================================================================
# TIMING HELPER
# =============================================================================

def gpu_time_ms(
    fn,
    warmup=3,
    repeats=10,
):

    for _ in range(
        warmup
    ):
        fn()


    torch.cuda.synchronize()


    measurements = []


    for _ in range(
        repeats
    ):

        start = torch.cuda.Event(
            enable_timing=True
        )

        end = torch.cuda.Event(
            enable_timing=True
        )


        start.record()

        fn()

        end.record()


        end.synchronize()


        measurements.append(
            start.elapsed_time(
                end
            )
        )


    return statistics.median(
        measurements
    )


# =============================================================================
# TRANSIENT ALLOCATOR MEMORY
# =============================================================================

def transient_memory_mb(
    fn,
):

    gc.collect()

    torch.cuda.empty_cache()

    torch.cuda.synchronize()

    torch.cuda.reset_peak_memory_stats()


    baseline = (
        torch.cuda.memory_allocated()
    )


    fn()


    torch.cuda.synchronize()


    peak = (
        torch.cuda.max_memory_allocated()
    )


    return (
        peak
        -
        baseline
    ) / 1e6


# =============================================================================
# REAL ACTIVATION K SWEEP
# =============================================================================

print("\n" + "=" * 135)
print("QWEN2.5-14B — REAL ACTIVATION TTT KERNEL BENCHMARK")
print("=" * 135)


print(
    f"{'K':>6}"
    f"{'PyTorch ms':>15}"
    f"{'Triton ms':>15}"
    f"{'ratio':>12}"
    f"{'rel L2':>16}"
    f"{'max abs':>16}"
    f"{'PT temp MB':>15}"
    f"{'Tri temp MB':>15}"
)


print(
    "-" * 135
)


kernel_results_14b = []


for K in K_VALUES:

    V_K = (
        V_ALL[
            :K
        ]
        .contiguous()
    )


    VT_K = (
        V_K
        .T
        .contiguous()
    )


    Z_K = (
        Z_ALL[
            :K
        ]
        .contiguous()
    )


    # -------------------------------------------------------------------------
    # JIT compile first.
    # -------------------------------------------------------------------------

    compile_dst = (
        torch.empty_like(
            W_BASE
        )
    )


    triton_ttt_14b(
        VT_K,
        Z_K,
        W_BASE,
        compile_dst,
        K,
    )


    torch.cuda.synchronize()

    del compile_dst


    # -------------------------------------------------------------------------
    # Correctness against the UPDATE itself.
    #
    # zero_src avoids the large base weight dominating relative error.
    # -------------------------------------------------------------------------

    zero_src = (
        torch.zeros_like(
            W_BASE
        )
    )


    pt_check = (
        torch.empty_like(
            W_BASE
        )
    )


    tri_check = (
        torch.empty_like(
            W_BASE
        )
    )


    pytorch_ttt_14b(
        VT_K,
        Z_K,
        zero_src,
        pt_check,
    )


    triton_ttt_14b(
        VT_K,
        Z_K,
        zero_src,
        tri_check,
        K,
    )


    torch.cuda.synchronize()


    diff = (
        tri_check
        -
        pt_check
    )


    relative_l2 = (
        torch.linalg.vector_norm(
            diff
        )
        /
        torch.linalg.vector_norm(
            pt_check
        ).clamp_min(
            1e-30
        )
    ).item()


    max_abs = (
        diff
        .abs()
        .max()
        .item()
    )


    del diff
    del zero_src
    del pt_check
    del tri_check

    gc.collect()
    torch.cuda.empty_cache()


    # -------------------------------------------------------------------------
    # Preallocated destinations.
    # -------------------------------------------------------------------------

    pt_dst = (
        torch.empty_like(
            W_BASE
        )
    )


    tri_dst = (
        torch.empty_like(
            W_BASE
        )
    )


    # -------------------------------------------------------------------------
    # Timing.
    # -------------------------------------------------------------------------

    pt_ms = gpu_time_ms(
        lambda:
            pytorch_ttt_14b(
                VT_K,
                Z_K,
                W_BASE,
                pt_dst,
            )
    )


    tri_ms = gpu_time_ms(
        lambda:
            triton_ttt_14b(
                VT_K,
                Z_K,
                W_BASE,
                tri_dst,
                K,
            )
    )


    # -------------------------------------------------------------------------
    # Temporary allocation.
    # -------------------------------------------------------------------------

    pt_temp_mb = transient_memory_mb(
        lambda:
            pytorch_ttt_14b(
                VT_K,
                Z_K,
                W_BASE,
                pt_dst,
            )
    )


    tri_temp_mb = transient_memory_mb(
        lambda:
            triton_ttt_14b(
                VT_K,
                Z_K,
                W_BASE,
                tri_dst,
                K,
            )
    )


    ratio = (
        tri_ms
        /
        pt_ms
    )


    result = {

        "K":
            K,

        "pytorch_ms":
            pt_ms,

        "triton_ms":
            tri_ms,

        "triton_over_pytorch":
            ratio,

        "relative_l2":
            relative_l2,

        "max_absolute_error":
            max_abs,

        "pytorch_transient_mb":
            pt_temp_mb,

        "triton_transient_mb":
            tri_temp_mb,

        "persistent_staging_mb":
            FAST_WEIGHT_MB,
    }


    kernel_results_14b.append(
        result
    )


    print(
        f"{K:>6}"
        f"{pt_ms:>15.3f}"
        f"{tri_ms:>15.3f}"
        f"{ratio:>12.3f}"
        f"{relative_l2:>16.3e}"
        f"{max_abs:>16.3e}"
        f"{pt_temp_mb:>15.2f}"
        f"{tri_temp_mb:>15.2f}"
    )


    del V_K
    del VT_K
    del Z_K
    del pt_dst
    del tri_dst

    gc.collect()
    torch.cuda.empty_cache()


print()
print(
    "Persistent staging    :",
    f"{FAST_WEIGHT_MB:.2f} MB",
)

print(
    "Layer                 :",
    TTT_LAYER,
)

print(
    "W_down shape          :",
    f"{M} x {N}",
)


# =============================================================================
# CLEAN UP BEFORE SERVING TEST
# =============================================================================

gc.collect()
torch.cuda.empty_cache()
torch.cuda.synchronize()


free_bytes, _ = (
    torch.cuda.mem_get_info()
)

print(
    "Free VRAM before decode:",
    f"{free_bytes / 1024**3:.2f} GiB",
)


# =============================================================================
# SERVING TTT ACTIVATIONS — K=512
# =============================================================================

VT_SERVING = (
    V_ALL[
        :TTT_K_SERVING
    ]
    .T
    .contiguous()
)


Z_SERVING = (
    Z_ALL[
        :TTT_K_SERVING
    ]
    .contiguous()
)


# Reuse W_BASE as active buffer.
# Only allocate one additional persistent A/B buffer.

W_STAGING_14B = (
    torch.empty_like(
        W_BASE
    )
)


# =============================================================================
# STREAMS
# =============================================================================

try:

    inference_stream = (
        torch.cuda.Stream(
            priority=-1
        )
    )

except Exception:

    inference_stream = (
        torch.cuda.Stream()
    )


learning_stream = (
    torch.cuda.Stream(
        priority=0
    )
)


torch.cuda.synchronize()


# =============================================================================
# STAT HELPERS
# =============================================================================

def percentile(
    values,
    p,
):

    if not values:
        return 0.0


    ordered = sorted(
        values
    )


    position = (
        len(ordered)
        -
        1
    ) * p


    lo = int(
        math.floor(
            position
        )
    )


    hi = int(
        math.ceil(
            position
        )
    )


    if lo == hi:
        return ordered[lo]


    fraction = (
        position
        -
        lo
    )


    return (
        ordered[lo]
        *
        (
            1.0
            -
            fraction
        )
        +
        ordered[hi]
        *
        fraction
    )


def summarize(
    values,
):

    if not values:
        return None


    return {

        "mean":
            statistics.mean(
                values
            ),

        "p50":
            percentile(
                values,
                0.50,
            ),

        "p95":
            percentile(
                values,
                0.95,
            ),

        "p99":
            percentile(
                values,
                0.99,
            ),

        "min":
            min(
                values
            ),

        "max":
            max(
                values
            ),
    }


# =============================================================================
# KV CACHE SIZE
# =============================================================================

def tensor_nbytes(
    tensor,
):

    return (
        tensor.numel()
        *
        tensor.element_size()
    )


def cache_bytes(
    cache,
):

    if cache is None:
        return 0


    total = 0


    # -------------------------------------------------------------------------
    # HF DynamicCache — older representation.
    # -------------------------------------------------------------------------

    if hasattr(
        cache,
        "key_cache",
    ):

        try:

            for tensor in (
                cache.key_cache
            ):

                if torch.is_tensor(
                    tensor
                ):

                    total += (
                        tensor_nbytes(
                            tensor
                        )
                    )


            for tensor in (
                cache.value_cache
            ):

                if torch.is_tensor(
                    tensor
                ):

                    total += (
                        tensor_nbytes(
                            tensor
                        )
                    )


            if total > 0:
                return total

        except Exception:
            pass


    # -------------------------------------------------------------------------
    # HF DynamicCache — newer layer representation.
    # -------------------------------------------------------------------------

    if hasattr(
        cache,
        "layers",
    ):

        try:

            for layer in cache.layers:

                for attr in (
                    "keys",
                    "values",
                ):

                    if hasattr(
                        layer,
                        attr,
                    ):

                        tensor = getattr(
                            layer,
                            attr,
                        )


                        if torch.is_tensor(
                            tensor
                        ):

                            total += (
                                tensor_nbytes(
                                    tensor
                                )
                            )


            if total > 0:
                return total

        except Exception:
            pass


    # -------------------------------------------------------------------------
    # Legacy tuple representation.
    # -------------------------------------------------------------------------

    if isinstance(
        cache,
        (
            tuple,
            list,
        ),
    ):

        for layer in cache:

            if isinstance(
                layer,
                (
                    tuple,
                    list,
                ),
            ):

                for tensor in layer:

                    if torch.is_tensor(
                        tensor
                    ):

                        total += (
                            tensor_nbytes(
                                tensor
                            )
                        )


    return total


# =============================================================================
# REAL 14B AUTOREGRESSIVE DECODE
# =============================================================================

def run_decode_14b(
    mode,
):

    valid_modes = {
        "baseline",
        "serialized_triton",
        "async_pytorch",
        "async_triton",
    }


    if mode not in valid_modes:

        raise ValueError(
            mode
        )


    prompt = (
        input_ids[
            :,
            :PROMPT_LEN
        ]
        .contiguous()
    )


    active = W_BASE
    staging = W_STAGING_14B


    update_in_flight = False

    update_start = None
    update_end = None

    update_started_token = None


    update_times = []
    visibility_lags = []

    host_token_times = []
    gpu_token_times = []


    torch.cuda.synchronize()


    # =========================================================================
    # PREFILL
    # =========================================================================

    prefill_start = (
        torch.cuda.Event(
            enable_timing=True
        )
    )


    prefill_end = (
        torch.cuda.Event(
            enable_timing=True
        )
    )


    prefill_host_start = (
        time.perf_counter()
    )


    with torch.cuda.stream(
        inference_stream
    ):

        prefill_start.record(
            inference_stream
        )


        with torch.inference_mode():

            output = model(
                input_ids=
                    prompt,

                use_cache=
                    True,
            )


        past_key_values = (
            output.past_key_values
        )


        next_token = (
            output
            .logits[
                :,
                -1,
                :
            ]
            .argmax(
                dim=-1,

                keepdim=
                    True,
            )
        )


        prefill_end.record(
            inference_stream
        )


    # Only wait for inference.
    # Do NOT synchronize the whole GPU.

    prefill_end.synchronize()


    prefill_gpu_ms = (
        prefill_start.elapsed_time(
            prefill_end
        )
    )


    _ = int(
        next_token.item()
    )


    prefill_host_ms = (
        time.perf_counter()
        -
        prefill_host_start
    ) * 1000.0


    kv_start_mb = (
        cache_bytes(
            past_key_values
        )
        /
        1e6
    )


    # =========================================================================
    # DECODE
    # =========================================================================

    decode_wall_start = (
        time.perf_counter()
    )


    for token_idx in range(
        DECODE_TOKENS
    ):

        # ---------------------------------------------------------------------
        # Commit an async update only when it has completed.
        # ---------------------------------------------------------------------

        if (
            update_in_flight
            and
            update_end.query()
        ):

            update_end.synchronize()


            update_times.append(
                update_start.elapsed_time(
                    update_end
                )
            )


            visibility_lags.append(
                token_idx
                -
                update_started_token
            )


            active, staging = (
                staging,
                active,
            )


            update_in_flight = (
                False
            )


        trigger_update = (
            token_idx
            %
            UPDATE_EVERY
            ==
            0
        )


        # ---------------------------------------------------------------------
        # Launch learning FIRST so scheduler gets an opportunity to overlap it
        # with subsequent inference work.
        # ---------------------------------------------------------------------

        if (
            trigger_update
            and
            not update_in_flight
            and
            mode
            in {
                "async_pytorch",
                "async_triton",
            }
        ):

            update_start = (
                torch.cuda.Event(
                    enable_timing=True
                )
            )


            update_end = (
                torch.cuda.Event(
                    enable_timing=True
                )
            )


            update_started_token = (
                token_idx
            )


            with torch.cuda.stream(
                learning_stream
            ):

                update_start.record(
                    learning_stream
                )


                if (
                    mode
                    ==
                    "async_pytorch"
                ):

                    pytorch_ttt_14b(
                        VT_SERVING,
                        Z_SERVING,
                        active,
                        staging,
                    )


                else:

                    triton_ttt_14b(
                        VT_SERVING,
                        Z_SERVING,
                        active,
                        staging,
                        TTT_K_SERVING,
                    )


                update_end.record(
                    learning_stream
                )


            update_in_flight = (
                True
            )


        # ---------------------------------------------------------------------
        # User-visible token starts here.
        # ---------------------------------------------------------------------

        host_token_start = (
            time.perf_counter()
        )


        token_start = (
            torch.cuda.Event(
                enable_timing=True
            )
        )


        token_end = (
            torch.cuda.Event(
                enable_timing=True
            )
        )


        with torch.cuda.stream(
            inference_stream
        ):

            # -----------------------------------------------------------------
            # Blocking / serialized TTT comparison.
            # -----------------------------------------------------------------

            if (
                mode
                ==
                "serialized_triton"
                and
                trigger_update
            ):

                serial_start = (
                    torch.cuda.Event(
                        enable_timing=True
                    )
                )


                serial_end = (
                    torch.cuda.Event(
                        enable_timing=True
                    )
                )


                serial_start.record(
                    inference_stream
                )


                triton_ttt_14b(
                    VT_SERVING,
                    Z_SERVING,
                    active,
                    staging,
                    TTT_K_SERVING,
                )


                serial_end.record(
                    inference_stream
                )


                active, staging = (
                    staging,
                    active,
                )


            # -----------------------------------------------------------------
            # REAL QWEN 14B KV-CACHED TOKEN.
            # -----------------------------------------------------------------

            token_start.record(
                inference_stream
            )


            with torch.inference_mode():

                output = model(
                    input_ids=
                        next_token,

                    past_key_values=
                        past_key_values,

                    use_cache=
                        True,
                )


            past_key_values = (
                output.past_key_values
            )


            next_token = (
                output
                .logits[
                    :,
                    -1,
                    :
                ]
                .argmax(
                    dim=-1,

                    keepdim=
                        True,
                )
            )


            token_end.record(
                inference_stream
            )


        # ---------------------------------------------------------------------
        # Wait ONLY for inference stream.
        #
        # Do NOT call torch.cuda.synchronize() here.
        #
        # That would wait for the learning stream and invalidate the async test.
        # ---------------------------------------------------------------------

        token_end.synchronize()


        token_gpu_ms = (
            token_start.elapsed_time(
                token_end
            )
        )


        if (
            mode
            ==
            "serialized_triton"
            and
            trigger_update
        ):

            serial_end.synchronize()


            update_times.append(
                serial_start.elapsed_time(
                    serial_end
                )
            )


        # Make generated token CPU-visible.
        _ = int(
            next_token.item()
        )


        token_host_ms = (
            time.perf_counter()
            -
            host_token_start
        ) * 1000.0


        host_token_times.append(
            token_host_ms
        )


        gpu_token_times.append(
            token_gpu_ms
        )


    # =========================================================================
    # FOREGROUND COMPLETES HERE
    # =========================================================================

    foreground_end = (
        time.perf_counter()
    )


    foreground_ms = (
        foreground_end
        -
        decode_wall_start
    ) * 1000.0


    # =========================================================================
    # FINISH FINAL BACKGROUND UPDATE
    # =========================================================================

    if update_in_flight:

        update_end.synchronize()


        update_times.append(
            update_start.elapsed_time(
                update_end
            )
        )


        visibility_lags.append(
            DECODE_TOKENS
            -
            update_started_token
        )


        active, staging = (
            staging,
            active,
        )


        update_in_flight = (
            False
        )


    torch.cuda.synchronize()


    makespan_ms = (
        time.perf_counter()
        -
        decode_wall_start
    ) * 1000.0


    kv_end_mb = (
        cache_bytes(
            past_key_values
        )
        /
        1e6
    )


    result = {

        "mode":
            mode,

        "prefill_host_ms":
            prefill_host_ms,

        "prefill_gpu_ms":
            prefill_gpu_ms,

        "foreground_decode_ms":
            foreground_ms,

        "makespan_ms":
            makespan_ms,

        "tokens_per_second":
            DECODE_TOKENS
            /
            (
                foreground_ms
                /
                1000.0
            ),

        "host_itl":
            summarize(
                host_token_times
            ),

        "gpu_itl":
            summarize(
                gpu_token_times
            ),

        "ttt_update":
            summarize(
                update_times
            ),

        "visibility_lag":
            summarize(
                visibility_lags
            ),

        "kv_start_mb":
            kv_start_mb,

        "kv_end_mb":
            kv_end_mb,

        "kv_growth_mb":
            kv_end_mb
            -
            kv_start_mb,
    }


    del output
    del past_key_values
    del next_token


    gc.collect()
    torch.cuda.empty_cache()


    return result


# =============================================================================
# WARMUP
# =============================================================================

print("\n" + "=" * 120)
print("14B AUTOREGRESSIVE WARMUP")
print("=" * 120)


warmup = (
    run_decode_14b(
        "baseline"
    )
)


print(
    "p50                   :",
    f"{warmup['host_itl']['p50']:.3f} ms",
)

print(
    "p95                   :",
    f"{warmup['host_itl']['p95']:.3f} ms",
)

print(
    "KV                    :",
    f"{warmup['kv_start_mb']:.2f}",
    "->",
    f"{warmup['kv_end_mb']:.2f} MB",
)

print(
    "Warmup passed."
)


# =============================================================================
# REAL FOUR-MODE BENCHMARK
# =============================================================================

MODES = [
    "baseline",
    "serialized_triton",
    "async_pytorch",
    "async_triton",
]


all_runs = {
    mode: []
    for mode in MODES
}


print("\n" + "=" * 145)
print("QWEN2.5-14B — REAL AUTOREGRESSIVE ASYNC-TTT BENCHMARK")
print("=" * 145)


for mode in MODES:

    print()
    print("-" * 105)
    print(mode.upper())
    print("-" * 105)


    for run_index in range(
        DECODE_RUNS
    ):

        result = (
            run_decode_14b(
                mode
            )
        )


        all_runs[
            mode
        ].append(
            result
        )


        update_text = "-"

        if (
            result[
                "ttt_update"
            ]
            is not None
        ):

            update_text = (
                f"{result['ttt_update']['p50']:.3f}"
            )


        lag_text = "-"

        if (
            result[
                "visibility_lag"
            ]
            is not None
        ):

            lag_text = (
                f"{result['visibility_lag']['p50']:.1f}"
            )


        print(
            f"run={run_index + 1} | "
            f"p50={result['host_itl']['p50']:.3f} ms | "
            f"p95={result['host_itl']['p95']:.3f} ms | "
            f"p99={result['host_itl']['p99']:.3f} ms | "
            f"tok/s={result['tokens_per_second']:.2f} | "
            f"TTT={update_text} ms | "
            f"lag={lag_text}"
        )


# =============================================================================
# AGGREGATE
# =============================================================================

def aggregate_mode(
    runs,
):

    result = {

        "p50_ms":
            statistics.median(
                [
                    r[
                        "host_itl"
                    ][
                        "p50"
                    ]
                    for r in runs
                ]
            ),

        "p95_ms":
            statistics.median(
                [
                    r[
                        "host_itl"
                    ][
                        "p95"
                    ]
                    for r in runs
                ]
            ),

        "p99_ms":
            statistics.median(
                [
                    r[
                        "host_itl"
                    ][
                        "p99"
                    ]
                    for r in runs
                ]
            ),

        "tokens_per_second":
            statistics.median(
                [
                    r[
                        "tokens_per_second"
                    ]
                    for r in runs
                ]
            ),

        "foreground_ms":
            statistics.median(
                [
                    r[
                        "foreground_decode_ms"
                    ]
                    for r in runs
                ]
            ),

        "makespan_ms":
            statistics.median(
                [
                    r[
                        "makespan_ms"
                    ]
                    for r in runs
                ]
            ),

        "kv_start_mb":
            statistics.median(
                [
                    r[
                        "kv_start_mb"
                    ]
                    for r in runs
                ]
            ),

        "kv_end_mb":
            statistics.median(
                [
                    r[
                        "kv_end_mb"
                    ]
                    for r in runs
                ]
            ),
    }


    updates = [
        r[
            "ttt_update"
        ][
            "p50"
        ]

        for r in runs

        if (
            r[
                "ttt_update"
            ]
            is not None
        )
    ]


    lags = [
        r[
            "visibility_lag"
        ][
            "p50"
        ]

        for r in runs

        if (
            r[
                "visibility_lag"
            ]
            is not None
        )
    ]


    result[
        "ttt_ms"
    ] = (
        statistics.median(
            updates
        )
        if updates
        else None
    )


    result[
        "lag_tokens"
    ] = (
        statistics.median(
            lags
        )
        if lags
        else None
    )


    return result


aggregate = {

    mode:
        aggregate_mode(
            all_runs[
                mode
            ]
        )

    for mode in MODES
}


baseline_p50 = (
    aggregate[
        "baseline"
    ][
        "p50_ms"
    ]
)


for mode in MODES:

    aggregate[
        mode
    ][
        "slowdown_pct"
    ] = (
        (
            aggregate[
                mode
            ][
                "p50_ms"
            ]
            -
            baseline_p50
        )
        /
        baseline_p50
        *
        100.0
    )


# =============================================================================
# FINAL 14B TABLE
# =============================================================================

print("\n" + "=" * 155)
print("FINAL QWEN2.5-14B RESULTS")
print("=" * 155)


print(
    f"{'Mode':<24}"
    f"{'p50 ITL':>13}"
    f"{'p95 ITL':>13}"
    f"{'p99 ITL':>13}"
    f"{'tok/s':>12}"
    f"{'slowdown':>13}"
    f"{'TTT ms':>13}"
    f"{'lag':>10}"
)


print(
    "-" * 155
)


for mode in MODES:

    result = (
        aggregate[
            mode
        ]
    )


    ttt_text = (
        f"{result['ttt_ms']:.3f}"
        if result[
            "ttt_ms"
        ]
        is not None
        else "-"
    )


    lag_text = (
        f"{result['lag_tokens']:.1f}"
        if result[
            "lag_tokens"
        ]
        is not None
        else "-"
    )


    print(
        f"{mode:<24}"
        f"{result['p50_ms']:>11.3f}ms"
        f"{result['p95_ms']:>11.3f}ms"
        f"{result['p99_ms']:>11.3f}ms"
        f"{result['tokens_per_second']:>12.2f}"
        f"{result['slowdown_pct']:>12.2f}%"
        f"{ttt_text:>13}"
        f"{lag_text:>10}"
    )


# =============================================================================
# KV RESULTS
# =============================================================================

print()
print("KV CACHE")
print("-" * 100)


print(
    f"{PROMPT_LEN} tokens : "
    f"{aggregate['baseline']['kv_start_mb']:.2f} MB"
)


print(
    f"{PROMPT_LEN + DECODE_TOKENS} tokens : "
    f"{aggregate['baseline']['kv_end_mb']:.2f} MB"
)


print(
    f"growth     : "
    f"{aggregate['baseline']['kv_end_mb'] - aggregate['baseline']['kv_start_mb']:.2f} MB"
)


# =============================================================================
# SAVE EVERYTHING
# =============================================================================

output = {

    "model":
        MODEL_ID,

    "gpu":
        GPU_NAME,

    "shape": {

        "M":
            M,

        "N":
            N,

        "layers":
            NUM_LAYERS,

        "ttt_layer":
            TTT_LAYER,

        "fast_weight_mb":
            FAST_WEIGHT_MB,
    },

    "kernel_sweep":
        kernel_results_14b,

    "serving_config": {

        "prompt_length":
            PROMPT_LEN,

        "decode_tokens":
            DECODE_TOKENS,

        "ttt_k":
            TTT_K_SERVING,

        "update_every":
            UPDATE_EVERY,

        "runs":
            DECODE_RUNS,
    },

    "serving":
        aggregate,

    "raw_serving_runs":
        all_runs,
}


OUTPUT_PATH = Path(
    "async_ttt_qwen2_5_14b_results.json"
)


with open(
    OUTPUT_PATH,
    "w",
) as file:

    json.dump(
        output,
        file,
        indent=2,
    )


print("\n" + "=" * 120)
print("DONE")
print("=" * 120)


print(
    "Saved:",
    OUTPUT_PATH.resolve(),
)


free_bytes, total_bytes = (
    torch.cuda.mem_get_info()
)


print(
    "Final free VRAM:",
    f"{free_bytes / 1024**3:.2f} GiB",
)

CLEARING PREVIOUS CUDA STATE
Free VRAM : 38.14 GiB
Total VRAM: 39.49 GiB

ASYNC-TTT — QWEN2.5-14B
Model                  : Qwen/Qwen2.5-14B-Instruct
Python                 : 3.12.11
PyTorch                : 2.8.0+cu128
CUDA runtime           : 12.8
Triton                 : 3.4.0
GPU                    : NVIDIA A100-SXM4-40GB
SMs                    : 108
VRAM                   : 39.49 GiB
TF32                   : False

LOADING QWEN2.5-14B


config.json:   0%|          | 0.00/663 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

...

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 8 files:   0%|          | 0/8 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/579 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]


hidden_size            : 5120
intermediate_size      : 13824
layers                 : 48
TTT layer              : 24
model dtype            : torch.bfloat16
FP32 W_down            : 283.12 MB
Free VRAM after load   : 10.61 GiB
Input tokens           : 2048

CAPTURING REAL 14B MLP ACTIVATIONS
V_ALL                  : (2048, 5120)
Z_ALL                  : (2048, 13824)
W_down                 : (5120, 13824)
Free VRAM after capture: 10.20 GiB

14B TTT kernels ready.

QWEN2.5-14B — REAL ACTIVATION TTT KERNEL BENCHMARK
     K     PyTorch ms      Triton ms       ratio          rel L2         max abs     PT temp MB    Tri temp MB
---------------------------------------------------------------------------------------------------------------------------------------
   128          2.627          3.286       1.251       2.072e-07       5.551e-17         283.12           0.00
   256          3.858          6.060       1.571       8.291e-07       3.331e-16         283.12           0.00
   512    

In [9]:
# =============================================================================
# QWEN2.5-14B — CORRECTED SERVING-ONLY ASYNC-TTT BENCHMARK
#
# Reuses the already-loaded 14B model + kernels.
#
# Critical correction:
#
# SERIALIZED mode measures:
#
#       [ TTT update -> HARD completion -> model decode ]
#
# as ONE user-visible token latency.
#
# This guarantees the blocking TTT cost is actually exposed.
#
# Also reports separately:
#
#       update-token ITL
#       normal-token ITL
#
# so the serialized tail stall is directly visible.
# =============================================================================

import gc
import json
import math
import statistics
import time
from pathlib import Path

import torch


# =============================================================================
# REQUIRE EXISTING 14B STATE
# =============================================================================

required = [
    "model",
    "target_layer",
    "input_ids",
    "V_ALL",
    "Z_ALL",
    "W_BASE",
    "W_STAGING_14B",
    "triton_ttt_14b",
    "pytorch_ttt_14b",
    "M",
    "N",
]

missing = [
    x
    for x in required
    if x not in globals()
]

if missing:
    raise RuntimeError(
        "Missing objects: "
        +
        ", ".join(missing)
        +
        "\nRun the 14B setup/kernel cell first."
    )


# =============================================================================
# CONFIG
# =============================================================================

PROMPT_LEN = 512
DECODE_TOKENS = 64

TTT_K = 512
UPDATE_EVERY = 8

RUNS = 5


MODES = [
    "baseline",
    "serialized_triton",
    "async_pytorch",
    "async_triton",
]


# =============================================================================
# REAL K=512 ACTIVATIONS
# =============================================================================

VT_SERVING_FIXED = (
    V_ALL[
        :TTT_K
    ]
    .T
    .contiguous()
)


Z_SERVING_FIXED = (
    Z_ALL[
        :TTT_K
    ]
    .contiguous()
)


# =============================================================================
# REUSE EXISTING A/B BUFFERS
# =============================================================================

FAST_A = W_BASE
FAST_B = W_STAGING_14B


# The actual model weight is still pristine BF16.
MODEL_DOWN_WEIGHT = (
    target_layer
    .mlp
    .down_proj
    .weight
    .detach()
)


def reset_fast_weights():

    # Restore A from the real model weight.
    FAST_A.copy_(
        MODEL_DOWN_WEIGHT
    )

    # Start B identical to A.
    FAST_B.copy_(
        FAST_A
    )

    torch.cuda.synchronize()


reset_fast_weights()


# =============================================================================
# STREAMS
# =============================================================================

try:
    inference_stream_fixed = (
        torch.cuda.Stream(
            priority=-1
        )
    )
except Exception:
    inference_stream_fixed = (
        torch.cuda.Stream()
    )


learning_stream_fixed = (
    torch.cuda.Stream(
        priority=0
    )
)


torch.cuda.synchronize()


# =============================================================================
# HELPERS
# =============================================================================

def percentile_fixed(
    values,
    p,
):

    if not values:
        return 0.0

    values = sorted(
        values
    )

    x = (
        len(values) - 1
    ) * p

    lo = int(
        math.floor(x)
    )

    hi = int(
        math.ceil(x)
    )

    if lo == hi:
        return values[lo]

    f = x - lo

    return (
        values[lo]
        *
        (1.0 - f)
        +
        values[hi]
        *
        f
    )


def summary_fixed(
    values,
):

    if not values:
        return None

    return {
        "mean":
            statistics.mean(
                values
            ),

        "p50":
            percentile_fixed(
                values,
                0.50,
            ),

        "p95":
            percentile_fixed(
                values,
                0.95,
            ),

        "p99":
            percentile_fixed(
                values,
                0.99,
            ),

        "min":
            min(values),

        "max":
            max(values),
    }


def tensor_bytes_fixed(
    x,
):
    return (
        x.numel()
        *
        x.element_size()
    )


def cache_bytes_fixed(
    cache,
):

    if cache is None:
        return 0

    total = 0


    if hasattr(
        cache,
        "key_cache",
    ):
        try:

            for x in cache.key_cache:
                if torch.is_tensor(x):
                    total += tensor_bytes_fixed(
                        x
                    )

            for x in cache.value_cache:
                if torch.is_tensor(x):
                    total += tensor_bytes_fixed(
                        x
                    )

            if total:
                return total

        except Exception:
            pass


    if hasattr(
        cache,
        "layers",
    ):
        try:

            for layer in cache.layers:

                for attr in (
                    "keys",
                    "values",
                ):

                    if hasattr(
                        layer,
                        attr,
                    ):

                        x = getattr(
                            layer,
                            attr,
                        )

                        if torch.is_tensor(x):
                            total += tensor_bytes_fixed(
                                x
                            )

            if total:
                return total

        except Exception:
            pass


    if isinstance(
        cache,
        (tuple, list),
    ):

        for layer in cache:

            if isinstance(
                layer,
                (tuple, list),
            ):

                for x in layer:

                    if torch.is_tensor(x):
                        total += tensor_bytes_fixed(
                            x
                        )

    return total


# =============================================================================
# CORRECTED DECODE
# =============================================================================

def run_fixed_decode(
    mode,
):

    if mode not in MODES:
        raise ValueError(
            mode
        )


    # Reset the A/B learned state before every benchmark run.
    reset_fast_weights()


    active = FAST_A
    staging = FAST_B


    prompt = (
        input_ids[
            :,
            :PROMPT_LEN
        ]
        .contiguous()
    )


    # -------------------------------------------------------------------------
    # Measurements
    # -------------------------------------------------------------------------

    all_itl = []

    update_token_itl = []
    normal_token_itl = []

    gpu_decode_itl = []

    ttt_times = []
    visibility_lags = []


    update_in_flight = False

    async_start_event = None
    async_end_event = None

    update_started_token = None


    torch.cuda.synchronize()


    # =========================================================================
    # PREFILL
    # =========================================================================

    prefill_start = (
        torch.cuda.Event(
            enable_timing=True
        )
    )

    prefill_end = (
        torch.cuda.Event(
            enable_timing=True
        )
    )


    prefill_host_start = (
        time.perf_counter()
    )


    with torch.cuda.stream(
        inference_stream_fixed
    ):

        prefill_start.record(
            inference_stream_fixed
        )


        with torch.inference_mode():

            output = model(
                input_ids=prompt,
                use_cache=True,
            )


        past_key_values = (
            output.past_key_values
        )


        next_token = (
            output
            .logits[
                :,
                -1,
                :
            ]
            .argmax(
                dim=-1,
                keepdim=True,
            )
        )


        prefill_end.record(
            inference_stream_fixed
        )


    prefill_end.synchronize()


    prefill_gpu_ms = (
        prefill_start.elapsed_time(
            prefill_end
        )
    )


    _ = int(
        next_token.item()
    )


    prefill_host_ms = (
        time.perf_counter()
        -
        prefill_host_start
    ) * 1000.0


    kv_start_mb = (
        cache_bytes_fixed(
            past_key_values
        )
        /
        1e6
    )


    # =========================================================================
    # DECODE
    # =========================================================================

    foreground_start = (
        time.perf_counter()
    )


    for token_idx in range(
        DECODE_TOKENS
    ):

        # ---------------------------------------------------------------------
        # Commit finished background update
        # ---------------------------------------------------------------------

        if (
            update_in_flight
            and
            async_end_event.query()
        ):

            async_end_event.synchronize()


            ttt_times.append(
                async_start_event.elapsed_time(
                    async_end_event
                )
            )


            visibility_lags.append(
                token_idx
                -
                update_started_token
            )


            active, staging = (
                staging,
                active,
            )


            update_in_flight = False


        trigger_update = (
            token_idx
            %
            UPDATE_EVERY
            ==
            0
        )


        # =====================================================================
        # USER-VISIBLE TOKEN TIMER STARTS BEFORE EVERYTHING
        # =====================================================================

        token_wall_start = (
            time.perf_counter()
        )


        # =====================================================================
        # SERIALIZED TTT
        #
        # Explicitly complete the update BEFORE model inference starts.
        #
        # This is intentionally blocking.
        # =====================================================================

        if (
            mode
            ==
            "serialized_triton"
            and
            trigger_update
        ):

            serial_wall_start = (
                time.perf_counter()
            )


            serial_start_event = (
                torch.cuda.Event(
                    enable_timing=True
                )
            )

            serial_end_event = (
                torch.cuda.Event(
                    enable_timing=True
                )
            )


            with torch.cuda.stream(
                inference_stream_fixed
            ):

                serial_start_event.record(
                    inference_stream_fixed
                )


                triton_ttt_14b(
                    VT_SERVING_FIXED,
                    Z_SERVING_FIXED,

                    active,
                    staging,

                    TTT_K,
                )


                serial_end_event.record(
                    inference_stream_fixed
                )


            # -------------------------------------------------------------
            # CRITICAL FIX
            #
            # Guarantee the TTT update has actually completed.
            # -------------------------------------------------------------

            serial_end_event.synchronize()

            # Absolute safety:
            # no hidden update can remain on another CUDA stream.
            torch.cuda.synchronize()


            serial_wall_ms = (
                time.perf_counter()
                -
                serial_wall_start
            ) * 1000.0


            serial_gpu_ms = (
                serial_start_event.elapsed_time(
                    serial_end_event
                )
            )


            # Use host blocking time for the serialized cost.
            # This is the actual exposed serving stall.
            ttt_times.append(
                serial_wall_ms
            )


            active, staging = (
                staging,
                active,
            )


        # =====================================================================
        # ASYNC UPDATE
        # =====================================================================

        if (
            mode
            in {
                "async_pytorch",
                "async_triton",
            }
            and
            trigger_update
            and
            not update_in_flight
        ):

            async_start_event = (
                torch.cuda.Event(
                    enable_timing=True
                )
            )

            async_end_event = (
                torch.cuda.Event(
                    enable_timing=True
                )
            )


            update_started_token = (
                token_idx
            )


            with torch.cuda.stream(
                learning_stream_fixed
            ):

                async_start_event.record(
                    learning_stream_fixed
                )


                if (
                    mode
                    ==
                    "async_pytorch"
                ):

                    pytorch_ttt_14b(
                        VT_SERVING_FIXED,
                        Z_SERVING_FIXED,

                        active,
                        staging,
                    )

                else:

                    triton_ttt_14b(
                        VT_SERVING_FIXED,
                        Z_SERVING_FIXED,

                        active,
                        staging,

                        TTT_K,
                    )


                async_end_event.record(
                    learning_stream_fixed
                )


            update_in_flight = True


        # =====================================================================
        # REAL QWEN TOKEN DECODE
        # =====================================================================

        decode_start_event = (
            torch.cuda.Event(
                enable_timing=True
            )
        )

        decode_end_event = (
            torch.cuda.Event(
                enable_timing=True
            )
        )


        with torch.cuda.stream(
            inference_stream_fixed
        ):

            decode_start_event.record(
                inference_stream_fixed
            )


            with torch.inference_mode():

                output = model(
                    input_ids=
                        next_token,

                    past_key_values=
                        past_key_values,

                    use_cache=
                        True,
                )


            past_key_values = (
                output.past_key_values
            )


            next_token = (
                output
                .logits[
                    :,
                    -1,
                    :
                ]
                .argmax(
                    dim=-1,
                    keepdim=True,
                )
            )


            decode_end_event.record(
                inference_stream_fixed
            )


        # ---------------------------------------------------------------------
        # Wait ONLY for foreground inference.
        #
        # Async modes do NOT synchronize the learning stream here.
        # ---------------------------------------------------------------------

        decode_end_event.synchronize()


        decode_gpu_ms = (
            decode_start_event.elapsed_time(
                decode_end_event
            )
        )


        _ = int(
            next_token.item()
        )


        token_wall_ms = (
            time.perf_counter()
            -
            token_wall_start
        ) * 1000.0


        all_itl.append(
            token_wall_ms
        )


        gpu_decode_itl.append(
            decode_gpu_ms
        )


        if trigger_update:

            update_token_itl.append(
                token_wall_ms
            )

        else:

            normal_token_itl.append(
                token_wall_ms
            )


    # =========================================================================
    # FOREGROUND END
    # =========================================================================

    foreground_ms = (
        time.perf_counter()
        -
        foreground_start
    ) * 1000.0


    # =========================================================================
    # FINAL BACKGROUND UPDATE
    # =========================================================================

    if update_in_flight:

        async_end_event.synchronize()


        ttt_times.append(
            async_start_event.elapsed_time(
                async_end_event
            )
        )


        visibility_lags.append(
            DECODE_TOKENS
            -
            update_started_token
        )


        active, staging = (
            staging,
            active,
        )


        update_in_flight = False


    torch.cuda.synchronize()


    kv_end_mb = (
        cache_bytes_fixed(
            past_key_values
        )
        /
        1e6
    )


    result = {

        "mode":
            mode,

        "prefill_host_ms":
            prefill_host_ms,

        "prefill_gpu_ms":
            prefill_gpu_ms,

        "foreground_ms":
            foreground_ms,

        "tok_s":
            DECODE_TOKENS
            /
            (
                foreground_ms
                /
                1000.0
            ),

        "all_itl":
            summary_fixed(
                all_itl
            ),

        "update_token_itl":
            summary_fixed(
                update_token_itl
            ),

        "normal_token_itl":
            summary_fixed(
                normal_token_itl
            ),

        "decode_gpu_itl":
            summary_fixed(
                gpu_decode_itl
            ),

        "ttt":
            summary_fixed(
                ttt_times
            ),

        "visibility_lag":
            summary_fixed(
                visibility_lags
            ),

        "kv_start_mb":
            kv_start_mb,

        "kv_end_mb":
            kv_end_mb,

        "raw_itl":
            all_itl,

        "raw_update_itl":
            update_token_itl,

        "raw_normal_itl":
            normal_token_itl,
    }


    del output
    del past_key_values
    del next_token


    gc.collect()
    torch.cuda.empty_cache()


    return result


# =============================================================================
# QUICK WARMUP
# =============================================================================

print(
    "=" * 140
)

print(
    "CORRECTED 14B SERVING BENCHMARK"
)

print(
    "=" * 140
)


print(
    "Warmup..."
)


_ = run_fixed_decode(
    "baseline"
)


print(
    "Warmup passed."
)


# =============================================================================
# RUN
# =============================================================================

runs_fixed = {
    mode: []
    for mode in MODES
}


for mode in MODES:

    print()
    print("-" * 110)
    print(
        mode.upper()
    )
    print("-" * 110)


    for run_idx in range(
        RUNS
    ):

        r = run_fixed_decode(
            mode
        )


        runs_fixed[
            mode
        ].append(
            r
        )


        ttt_text = (
            "-"
            if r["ttt"] is None
            else
            f"{r['ttt']['p50']:.3f}"
        )


        lag_text = (
            "-"
            if r["visibility_lag"] is None
            else
            f"{r['visibility_lag']['p50']:.1f}"
        )


        print(
            f"run={run_idx + 1} | "
            f"all p50={r['all_itl']['p50']:.3f} | "
            f"p95={r['all_itl']['p95']:.3f} | "
            f"p99={r['all_itl']['p99']:.3f} | "
            f"UPDATE-token p50={r['update_token_itl']['p50']:.3f} | "
            f"normal-token p50={r['normal_token_itl']['p50']:.3f} | "
            f"TTT={ttt_text} | "
            f"lag={lag_text}"
        )


# =============================================================================
# AGGREGATION
# =============================================================================

def median_metric(
    mode,
    group,
    metric,
):

    return statistics.median(
        [
            run[
                group
            ][
                metric
            ]
            for run in runs_fixed[
                mode
            ]
            if run[
                group
            ]
            is not None
        ]
    )


aggregate_fixed = {}


for mode in MODES:

    aggregate_fixed[
        mode
    ] = {

        "p50":
            median_metric(
                mode,
                "all_itl",
                "p50",
            ),

        "p95":
            median_metric(
                mode,
                "all_itl",
                "p95",
            ),

        "p99":
            median_metric(
                mode,
                "all_itl",
                "p99",
            ),

        "update_token_p50":
            median_metric(
                mode,
                "update_token_itl",
                "p50",
            ),

        "update_token_p95":
            median_metric(
                mode,
                "update_token_itl",
                "p95",
            ),

        "normal_token_p50":
            median_metric(
                mode,
                "normal_token_itl",
                "p50",
            ),

        "tok_s":
            statistics.median(
                [
                    x[
                        "tok_s"
                    ]
                    for x in runs_fixed[
                        mode
                    ]
                ]
            ),
    }


    ttt_values = [
        x[
            "ttt"
        ][
            "p50"
        ]
        for x in runs_fixed[
            mode
        ]
        if x[
            "ttt"
        ]
        is not None
    ]


    aggregate_fixed[
        mode
    ][
        "ttt_ms"
    ] = (
        statistics.median(
            ttt_values
        )
        if ttt_values
        else None
    )


    lag_values = [
        x[
            "visibility_lag"
        ][
            "p50"
        ]
        for x in runs_fixed[
            mode
        ]
        if x[
            "visibility_lag"
        ]
        is not None
    ]


    aggregate_fixed[
        mode
    ][
        "lag"
    ] = (
        statistics.median(
            lag_values
        )
        if lag_values
        else None
    )


baseline_p50 = (
    aggregate_fixed[
        "baseline"
    ][
        "p50"
    ]
)


for mode in MODES:

    aggregate_fixed[
        mode
    ][
        "slowdown_pct"
    ] = (
        (
            aggregate_fixed[
                mode
            ][
                "p50"
            ]
            -
            baseline_p50
        )
        /
        baseline_p50
        *
        100.0
    )


# =============================================================================
# FINAL TABLE
# =============================================================================

print()
print("=" * 170)
print("FINAL CORRECTED 14B SERVING RESULTS")
print("=" * 170)


print(
    f"{'Mode':<23}"
    f"{'all p50':>11}"
    f"{'all p95':>11}"
    f"{'all p99':>11}"
    f"{'update p50':>14}"
    f"{'update p95':>14}"
    f"{'normal p50':>14}"
    f"{'TTT ms':>11}"
    f"{'lag':>8}"
    f"{'tok/s':>11}"
)


print(
    "-" * 170
)


for mode in MODES:

    r = aggregate_fixed[
        mode
    ]


    ttt = (
        "-"
        if r["ttt_ms"] is None
        else
        f"{r['ttt_ms']:.3f}"
    )


    lag = (
        "-"
        if r["lag"] is None
        else
        f"{r['lag']:.1f}"
    )


    print(
        f"{mode:<23}"
        f"{r['p50']:>9.3f}ms"
        f"{r['p95']:>9.3f}ms"
        f"{r['p99']:>9.3f}ms"
        f"{r['update_token_p50']:>12.3f}ms"
        f"{r['update_token_p95']:>12.3f}ms"
        f"{r['normal_token_p50']:>12.3f}ms"
        f"{ttt:>11}"
        f"{lag:>8}"
        f"{r['tok_s']:>11.2f}"
    )


# =============================================================================
# DIRECT SERIALIZATION CHECK
# =============================================================================

serialized = (
    aggregate_fixed[
        "serialized_triton"
    ]
)


exposed_stall = (
    serialized[
        "update_token_p50"
    ]
    -
    serialized[
        "normal_token_p50"
    ]
)


print()
print("=" * 100)
print("SERIALIZATION SANITY CHECK")
print("=" * 100)


print(
    "Serialized normal-token p50 :",
    f"{serialized['normal_token_p50']:.3f} ms",
)


print(
    "Serialized update-token p50 :",
    f"{serialized['update_token_p50']:.3f} ms",
)


print(
    "Direct exposed stall        :",
    f"{exposed_stall:.3f} ms",
)


print(
    "Measured serialized TTT     :",
    f"{serialized['ttt_ms']:.3f} ms",
)


print()
print(
    "If the benchmark is correct, exposed stall should now be "
    "approximately the serialized TTT cost."
)


# =============================================================================
# ASYNC HIDING CHECK
# =============================================================================

async_tri = (
    aggregate_fixed[
        "async_triton"
    ]
)


async_exposed = (
    async_tri[
        "update_token_p50"
    ]
    -
    async_tri[
        "normal_token_p50"
    ]
)


print()
print("=" * 100)
print("ASYNC TRITON HIDING CHECK")
print("=" * 100)


print(
    "Async normal-token p50 :",
    f"{async_tri['normal_token_p50']:.3f} ms",
)


print(
    "Async update-token p50 :",
    f"{async_tri['update_token_p50']:.3f} ms",
)


print(
    "Exposed async overhead  :",
    f"{async_exposed:.3f} ms",
)


print(
    "Background TTT latency  :",
    f"{async_tri['ttt_ms']:.3f} ms",
)


if async_tri["ttt_ms"] > 0:

    hidden_fraction = (
        1.0
        -
        max(
            0.0,
            async_exposed,
        )
        /
        async_tri[
            "ttt_ms"
        ]
    )


    print(
        "Approx hidden fraction :",
        f"{hidden_fraction * 100.0:.2f}%",
    )


# =============================================================================
# SAVE
# =============================================================================

RESULT_PATH = Path(
    "async_ttt_qwen14b_corrected_serving.json"
)


with open(
    RESULT_PATH,
    "w",
) as f:

    json.dump(
        {
            "aggregate":
                aggregate_fixed,

            "raw":
                runs_fixed,

            "config": {
                "prompt_length":
                    PROMPT_LEN,

                "decode_tokens":
                    DECODE_TOKENS,

                "ttt_k":
                    TTT_K,

                "update_every":
                    UPDATE_EVERY,

                "runs":
                    RUNS,
            },
        },
        f,
        indent=2,
    )


print()
print(
    "Saved:",
    RESULT_PATH.resolve(),
)

CORRECTED 14B SERVING BENCHMARK
Warmup...
Warmup passed.

--------------------------------------------------------------------------------------------------------------
BASELINE
--------------------------------------------------------------------------------------------------------------
run=1 | all p50=37.913 | p95=38.258 | p99=38.575 | UPDATE-token p50=37.873 | normal-token p50=37.919 | TTT=- | lag=-
run=2 | all p50=38.161 | p95=39.111 | p99=39.398 | UPDATE-token p50=38.179 | normal-token p50=38.161 | TTT=- | lag=-
run=3 | all p50=37.965 | p95=38.918 | p99=39.364 | UPDATE-token p50=37.970 | normal-token p50=37.965 | TTT=- | lag=-
run=4 | all p50=37.862 | p95=38.240 | p99=38.454 | UPDATE-token p50=37.847 | normal-token p50=37.866 | TTT=- | lag=-
run=5 | all p50=37.448 | p95=37.919 | p99=38.089 | UPDATE-token p50=37.437 | normal-token p50=37.448 | TTT=- | lag=-

--------------------------------------------------------------------------------------------------------------
SERIALIZED_TRI

In [10]:
# =============================================================================
# ASYNC-TTT — QWEN2.5-7B REAL FAST-WEIGHT CONSUMPTION
#
# PURPOSE
# -------
# This closes the major semantic gap in the previous benchmarks.
#
# Previously:
#
#     Triton:
#         W_active -> W_staging
#
#     but Qwen inference still read:
#
#         model.down_proj.weight
#
# This experiment changes that.
#
# Qwen's target down_proj now directly reads:
#
#         FastWeightState.active
#
# and an async commit performs:
#
#         active, staging = staging, active
#
# Therefore:
#
#     token t     -> version N
#     TTT update  -> writes version N+1
#     commit      -> pointer swap
#     token t+1   -> ACTUALLY reads version N+1
#
#
# ALSO TESTS
# ----------
# 1. Existing paper LR/clip numerical visibility.
# 2. Explicit semantic/wiring update with a visible scale.
# 3. Weight pointer identity used by each token.
# 4. Logit difference after switching A -> B.
# 5. Baseline fast-weight inference.
# 6. Serialized Triton adaptation.
# 7. Async Triton adaptation.
# 8. p50/p95/p99 ITL.
# 9. Update-token vs normal-token latency.
# 10. Visibility lag.
#
#
# MEMORY HIERARCHY
# ----------------
#
# HBM / DRAM:
#     V, Z, W_active, W_staging
#     norm_ws (4 bytes)
#
# On-chip hierarchy:
#     tiles loaded via tl.load
#     tl.dot operands
#     FP32 accumulators
#     temporary Delta-W tiles
#
# Delta-W is NEVER stored as a full HBM tensor by the Triton implementation.
#
# NOTE:
# Triton chooses the precise register/shared-memory placement.
# We have NOT yet proven explicit SRAM/shared-memory residency with Nsight.
#
# =============================================================================


# =============================================================================
# IMPORTS
# =============================================================================

import gc
import json
import math
import statistics
import sys
import time

from pathlib import Path

import torch
import torch.nn.functional as F

import triton
import triton.language as tl

from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
)


# =============================================================================
# AGGRESSIVE CLEANUP OF PREVIOUS 12B / 14B EXPERIMENT
# =============================================================================

print("=" * 120)
print("FLUSHING PREVIOUS GPU STATE")
print("=" * 120)


OLD_GPU_OBJECTS = [

    # models
    "model",
    "tokenizer",
    "target_layer",

    # activations
    "V_ALL",
    "Z_ALL",
    "VT",
    "VT_K",
    "VT_SERVING",
    "VT_SERVING_FIXED",
    "Z_SERVING",
    "Z_SERVING_FIXED",

    # weights
    "W_BASE",
    "W_ACTIVE",
    "W_STAGING",
    "W_ACTIVE_12B",
    "W_STAGING_12B",
    "W_STAGING_14B",
    "FAST_A",
    "FAST_B",
    "MODEL_DOWN_WEIGHT",

    # capture data
    "captured",
    "captured_hidden",
    "captured_z",

    # outputs
    "output",
    "past_key_values",
    "next_token",

    # previous runtime state
    "fast_state",

]


for name in OLD_GPU_OBJECTS:

    if name in globals():

        try:
            del globals()[name]

        except Exception:
            pass


gc.collect()


if torch.cuda.is_available():

    torch.cuda.synchronize()

    torch.cuda.empty_cache()

    try:
        torch.cuda.ipc_collect()
    except Exception:
        pass

    gc.collect()

    torch.cuda.empty_cache()


free_bytes, total_bytes = (
    torch.cuda.mem_get_info()
)


print(
    "Free VRAM :",
    f"{free_bytes / 1024**3:.2f} GiB",
)

print(
    "Total VRAM:",
    f"{total_bytes / 1024**3:.2f} GiB",
)


# Qwen 7B easily fits if cleanup succeeded.
if (
    free_bytes
    /
    1024**3
    <
    25
):

    print()
    print(
        "WARNING: Less than 25 GiB free."
    )

    print(
        "If the previous 14B model is still referenced somewhere, "
        "restart the kernel before continuing."
    )


# =============================================================================
# CONFIG
# =============================================================================

MODEL_ID = (
    "Qwen/Qwen2.5-7B-Instruct"
)


DEVICE = "cuda"


TTT_LAYER = 13

TTT_K = 512

PROMPT_LEN = 512

DECODE_TOKENS = 64

UPDATE_EVERY = 8

RUNS = 3


# -----------------------------------------------------------------------------
# Existing performance/paper configuration.
# -----------------------------------------------------------------------------

PERF_LR = 1e-3

PERF_CLIP = 1e-5


# -----------------------------------------------------------------------------
# Semantic visibility configuration.
#
# This is deliberately separated from the performance configuration.
#
# Why?
#
# PERF_LR * PERF_CLIP gives a globally clipped update norm on the order of 1e-8.
#
# Spread across ~68 million W_down elements, that may be beneath the ULP of
# existing FP32 weight values and therefore may not actually mutate many
# parameters.
#
# We test that below rather than silently assuming otherwise.
#
# The semantic configuration exists ONLY to prove:
#
#       model reads W0
#          ->
#       update
#          ->
#       pointer swap
#          ->
#       model reads W1
#
# Do not report SEMANTIC_* as the paper's TTT hyperparameters.
# -----------------------------------------------------------------------------

SEMANTIC_LR = 1.0

SEMANTIC_CLIP = 1e-2


torch.set_grad_enabled(
    False
)


torch.manual_seed(
    0
)


torch.backends.cuda.matmul.allow_tf32 = False

torch.backends.cudnn.allow_tf32 = False


# =============================================================================
# ENVIRONMENT
# =============================================================================

assert torch.cuda.is_available()


props = (
    torch.cuda.get_device_properties(
        0
    )
)


print()
print("=" * 120)
print("QWEN2.5-7B — REAL FAST-WEIGHT CONSUMPTION")
print("=" * 120)


print(
    "Model                :",
    MODEL_ID,
)

print(
    "Python               :",
    sys.version.split()[0],
)

print(
    "PyTorch              :",
    torch.__version__,
)

print(
    "CUDA                 :",
    torch.version.cuda,
)

print(
    "Triton               :",
    triton.__version__,
)

print(
    "GPU                  :",
    torch.cuda.get_device_name(0),
)

print(
    "SMs                  :",
    props.multi_processor_count,
)

print(
    "VRAM                 :",
    f"{props.total_memory / 1024**3:.2f} GiB",
)


# =============================================================================
# LOAD QWEN 7B
# =============================================================================

print()
print("=" * 120)
print("LOADING QWEN2.5-7B")
print("=" * 120)


tokenizer = (
    AutoTokenizer.from_pretrained(
        MODEL_ID,
        trust_remote_code=True,
    )
)


model = (
    AutoModelForCausalLM.from_pretrained(
        MODEL_ID,

        dtype=
            torch.bfloat16,

        device_map={
            "": 0
        },

        trust_remote_code=True,

        low_cpu_mem_usage=True,

        attn_implementation=
            "sdpa",
    )
)


model.eval()


M = int(
    model.config.hidden_size
)

N = int(
    model.config.intermediate_size
)


print(
    "hidden_size          :",
    M,
)

print(
    "intermediate_size    :",
    N,
)

print(
    "layers               :",
    model.config.num_hidden_layers,
)

print(
    "TTT layer            :",
    TTT_LAYER,
)

print(
    "model dtype          :",
    next(
        model.parameters()
    ).dtype,
)

print(
    "FP32 fast weight     :",
    f"{M * N * 4 / 1e6:.2f} MB",
)


free_bytes, _ = (
    torch.cuda.mem_get_info()
)


print(
    "Free after load      :",
    f"{free_bytes / 1024**3:.2f} GiB",
)


# =============================================================================
# REAL INPUT
# =============================================================================

base_text = """
Test-time training allows a language model to continuously adapt a learned
internal state while processing incoming context. A production implementation
must allow online learning to execute concurrently with latency-sensitive
autoregressive inference while controlling synchronization, GPU memory traffic,
and update visibility.
"""


text = "\n".join(
    [base_text] * 1000
)


tokenized = (
    tokenizer(
        text,

        return_tensors=
            "pt",

        truncation=
            True,

        max_length=
            PROMPT_LEN,
    )
)


input_ids = (
    tokenized[
        "input_ids"
    ]
    .to(
        DEVICE
    )
)


if (
    input_ids.shape[1]
    <
    PROMPT_LEN
):

    raise RuntimeError(
        f"Need {PROMPT_LEN} tokens, "
        f"got {input_ids.shape[1]}"
    )


del tokenized
del text


print(
    "Input tokens         :",
    input_ids.shape[1],
)


# =============================================================================
# TARGET LAYER
# =============================================================================

target_layer = (
    model.model.layers[
        TTT_LAYER
    ]
)


original_down_forward = (
    target_layer
    .mlp
    .down_proj
    .forward
)


# =============================================================================
# CAPTURE REAL ACTIVATIONS
# =============================================================================

print()
print("=" * 120)
print("CAPTURING REAL QWEN ACTIVATIONS")
print("=" * 120)


captured = {}


def capture_v_hook(
    module,
    args,
):

    captured[
        "v"
    ] = (
        args[0]
        .detach()
        .clone()
    )


def capture_z_hook(
    module,
    args,
):

    captured[
        "z"
    ] = (
        args[0]
        .detach()
        .clone()
    )


hook_v = (
    target_layer
    .mlp
    .register_forward_pre_hook(
        capture_v_hook
    )
)


hook_z = (
    target_layer
    .mlp
    .down_proj
    .register_forward_pre_hook(
        capture_z_hook
    )
)


try:

    with torch.inference_mode():

        # model.model avoids allocating the giant LM-head logits tensor.
        capture_output = (
            model.model(
                input_ids=
                    input_ids[
                        :,
                        :TTT_K
                    ],

                use_cache=
                    False,
            )
        )


    torch.cuda.synchronize()


finally:

    hook_v.remove()

    hook_z.remove()


V_ALL = (
    captured[
        "v"
    ][
        0,
        :TTT_K
    ]
    .float()
    .contiguous()
)


Z_ALL = (
    captured[
        "z"
    ][
        0,
        :TTT_K
    ]
    .float()
    .contiguous()
)


VT = (
    V_ALL
    .T
    .contiguous()
)


W_MASTER = (
    target_layer
    .mlp
    .down_proj
    .weight
    .detach()
    .float()
    .contiguous()
)


assert VT.shape == (
    M,
    TTT_K,
)


assert Z_ALL.shape == (
    TTT_K,
    N,
)


assert W_MASTER.shape == (
    M,
    N,
)


print(
    "VT                   :",
    tuple(
        VT.shape
    ),
)

print(
    "Z                    :",
    tuple(
        Z_ALL.shape
    ),
)

print(
    "W_down               :",
    tuple(
        W_MASTER.shape
    ),
)


del capture_output
del captured


gc.collect()

torch.cuda.empty_cache()


# =============================================================================
# A / B FAST WEIGHTS
# =============================================================================

W_A = (
    W_MASTER
    .clone()
    .contiguous()
)


W_B = (
    W_MASTER
    .clone()
    .contiguous()
)


# =============================================================================
# TWO-PASS ON-CHIP REMATERIALIZATION KERNEL
# =============================================================================

@triton.jit
def ttt_norm_kernel(
    vt_ptr,
    z_ptr,

    norm_ptr,

    stride_vm,
    stride_vk,

    stride_zk,
    stride_zn,

    M: tl.constexpr,
    N: tl.constexpr,
    K: tl.constexpr,

    BLOCK_M: tl.constexpr,
    BLOCK_N: tl.constexpr,
    BLOCK_K: tl.constexpr,
):

    pid_m = (
        tl.program_id(0)
    )

    pid_n = (
        tl.program_id(1)
    )


    offs_m = (
        pid_m * BLOCK_M
        +
        tl.arange(
            0,
            BLOCK_M,
        )
    )


    offs_n = (
        pid_n * BLOCK_N
        +
        tl.arange(
            0,
            BLOCK_N,
        )
    )


    # This accumulator is NOT a global HBM Delta-W tensor.
    # Triton keeps the tile in the on-chip execution hierarchy.
    acc = tl.zeros(
        (
            BLOCK_M,
            BLOCK_N,
        ),
        dtype=
            tl.float32,
    )


    for k in range(
        0,
        K,
        BLOCK_K,
    ):

        offs_k = (
            k
            +
            tl.arange(
                0,
                BLOCK_K,
            )
        )


        # HBM -> on-chip tile
        v = tl.load(
            vt_ptr
            +
            offs_m[:, None]
            *
            stride_vm
            +
            offs_k[None, :]
            *
            stride_vk,

            mask=(
                (
                    offs_m[:, None]
                    <
                    M
                )
                &
                (
                    offs_k[None, :]
                    <
                    K
                )
            ),

            other=
                0.0,
        )


        # HBM -> on-chip tile
        z = tl.load(
            z_ptr
            +
            offs_k[:, None]
            *
            stride_zk
            +
            offs_n[None, :]
            *
            stride_zn,

            mask=(
                (
                    offs_k[:, None]
                    <
                    K
                )
                &
                (
                    offs_n[None, :]
                    <
                    N
                )
            ),

            other=
                0.0,
        )


        acc = tl.dot(
            v,
            z,
            acc,

            input_precision=
                "ieee",
        )


    local_sq = (
        tl.sum(
            acc
            *
            acc
        )
    )


    # Only the scalar reduction hits global HBM.
    tl.atomic_add(
        norm_ptr,
        local_sq,
    )


@triton.jit
def ttt_update_kernel(
    vt_ptr,
    z_ptr,

    src_ptr,
    dst_ptr,

    norm_ptr,

    stride_vm,
    stride_vk,

    stride_zk,
    stride_zn,

    stride_sm,
    stride_sn,

    stride_dm,
    stride_dn,

    learning_rate,
    clip_threshold,

    M: tl.constexpr,
    N: tl.constexpr,
    K: tl.constexpr,

    BLOCK_M: tl.constexpr,
    BLOCK_N: tl.constexpr,
    BLOCK_K: tl.constexpr,
):

    pid_m = (
        tl.program_id(0)
    )

    pid_n = (
        tl.program_id(1)
    )


    offs_m = (
        pid_m * BLOCK_M
        +
        tl.arange(
            0,
            BLOCK_M,
        )
    )


    offs_n = (
        pid_n * BLOCK_N
        +
        tl.arange(
            0,
            BLOCK_N,
        )
    )


    norm_sq = (
        tl.load(
            norm_ptr
        )
    )


    scale = (
        clip_threshold
        /
        (
            tl.sqrt(
                norm_sq
            )
            +
            1e-8
        )
    )


    scale = (
        tl.minimum(
            scale,
            1.0,
        )
    )


    # Delta-W tile is REMATERIALIZED here rather than loaded from HBM.
    acc = tl.zeros(
        (
            BLOCK_M,
            BLOCK_N,
        ),
        dtype=
            tl.float32,
    )


    for k in range(
        0,
        K,
        BLOCK_K,
    ):

        offs_k = (
            k
            +
            tl.arange(
                0,
                BLOCK_K,
            )
        )


        v = tl.load(
            vt_ptr
            +
            offs_m[:, None]
            *
            stride_vm
            +
            offs_k[None, :]
            *
            stride_vk,

            mask=(
                (
                    offs_m[:, None]
                    <
                    M
                )
                &
                (
                    offs_k[None, :]
                    <
                    K
                )
            ),

            other=
                0.0,
        )


        z = tl.load(
            z_ptr
            +
            offs_k[:, None]
            *
            stride_zk
            +
            offs_n[None, :]
            *
            stride_zn,

            mask=(
                (
                    offs_k[:, None]
                    <
                    K
                )
                &
                (
                    offs_n[None, :]
                    <
                    N
                )
            ),

            other=
                0.0,
        )


        acc = tl.dot(
            v,
            z,
            acc,

            input_precision=
                "ieee",
        )


    mask = (
        (
            offs_m[:, None]
            <
            M
        )
        &
        (
            offs_n[None, :]
            <
            N
        )
    )


    w = tl.load(
        src_ptr
        +
        offs_m[:, None]
        *
        stride_sm
        +
        offs_n[None, :]
        *
        stride_sn,

        mask=
            mask,

        other=
            0.0,
    )


    updated = (
        w
        +
        learning_rate
        *
        scale
        *
        acc
    )


    # Only the final adapted weight is written to HBM.
    tl.store(
        dst_ptr
        +
        offs_m[:, None]
        *
        stride_dm
        +
        offs_n[None, :]
        *
        stride_dn,

        updated,

        mask=
            mask,
    )


# =============================================================================
# A100 CONFIG
# =============================================================================

NORM_BM = 64
NORM_BN = 128
NORM_BK = 64
NORM_WARPS = 4


UPDATE_BM = 64
UPDATE_BN = 64
UPDATE_BK = 64
UPDATE_WARPS = 8


norm_ws = torch.zeros(
    1,

    device=
        DEVICE,

    dtype=
        torch.float32,
)


norm_grid = (

    triton.cdiv(
        M,
        NORM_BM,
    ),

    triton.cdiv(
        N,
        NORM_BN,
    ),
)


update_grid = (

    triton.cdiv(
        M,
        UPDATE_BM,
    ),

    triton.cdiv(
        N,
        UPDATE_BN,
    ),
)


# =============================================================================
# TRITON UPDATE
# =============================================================================

def triton_update(
    src,
    dst,

    learning_rate,
    clip_threshold,
):

    norm_ws.zero_()


    ttt_norm_kernel[
        norm_grid
    ](
        VT,
        Z_ALL,

        norm_ws,

        VT.stride(0),
        VT.stride(1),

        Z_ALL.stride(0),
        Z_ALL.stride(1),

        M=M,
        N=N,
        K=TTT_K,

        BLOCK_M=
            NORM_BM,

        BLOCK_N=
            NORM_BN,

        BLOCK_K=
            NORM_BK,

        num_warps=
            NORM_WARPS,
    )


    ttt_update_kernel[
        update_grid
    ](
        VT,
        Z_ALL,

        src,
        dst,

        norm_ws,

        VT.stride(0),
        VT.stride(1),

        Z_ALL.stride(0),
        Z_ALL.stride(1),

        src.stride(0),
        src.stride(1),

        dst.stride(0),
        dst.stride(1),

        learning_rate,
        clip_threshold,

        M=M,
        N=N,
        K=TTT_K,

        BLOCK_M=
            UPDATE_BM,

        BLOCK_N=
            UPDATE_BN,

        BLOCK_K=
            UPDATE_BK,

        num_warps=
            UPDATE_WARPS,
    )


# =============================================================================
# JIT
# =============================================================================

triton_update(
    W_A,
    W_B,

    PERF_LR,
    PERF_CLIP,
)


torch.cuda.synchronize()


# =============================================================================
# TEST NUMERICAL PRECISION FLOOR OF CURRENT PAPER CONFIG
# =============================================================================

print()
print("=" * 120)
print("PAPER-CONFIG WEIGHT-VISIBILITY TEST")
print("=" * 120)


W_A.copy_(
    W_MASTER
)

W_B.copy_(
    W_MASTER
)


torch.cuda.synchronize()


triton_update(
    W_A,
    W_B,

    PERF_LR,
    PERF_CLIP,
)


torch.cuda.synchronize()


paper_changed = (
    W_A
    !=
    W_B
)


paper_changed_count = (
    paper_changed
    .sum()
    .item()
)


paper_changed_fraction = (
    paper_changed_count
    /
    W_A.numel()
)


paper_weight_delta = (
    torch.linalg.vector_norm(
        W_B
        -
        W_A
    )
    .item()
)


paper_max_delta = (
    (
        W_B
        -
        W_A
    )
    .abs()
    .max()
    .item()
)


print(
    "LR                    :",
    PERF_LR,
)

print(
    "clip                  :",
    PERF_CLIP,
)

print(
    "changed elements      :",
    f"{paper_changed_count:,}",
    "/",
    f"{W_A.numel():,}",
)

print(
    "changed fraction      :",
    f"{paper_changed_fraction * 100:.8f}%",
)

print(
    "actual ||W1-W0||      :",
    f"{paper_weight_delta:.8e}",
)

print(
    "actual max |W1-W0|    :",
    f"{paper_max_delta:.8e}",
)


if paper_changed_fraction < 0.01:

    print()
    print(
        "IMPORTANT: the current LR/clip is near/below the FP32 "
        "weight precision floor."
    )

    print(
        "This is a hyperparameter/numerical issue, NOT a kernel "
        "correctness issue."
    )


del paper_changed


# =============================================================================
# FAST-WEIGHT STATE
# =============================================================================

class FastWeightState:

    def __init__(
        self,
        active,
        staging,
    ):

        self.active = active

        self.staging = staging

        self.version = 0

        self.last_used_version = None

        self.last_used_ptr = None


fast_state = FastWeightState(
    W_A,
    W_B,
)


def reset_fast_state():

    W_A.copy_(
        W_MASTER
    )

    W_B.copy_(
        W_MASTER
    )


    torch.cuda.synchronize()


    fast_state.active = W_A

    fast_state.staging = W_B

    fast_state.version = 0

    fast_state.last_used_version = None

    fast_state.last_used_ptr = None


def commit_fast_weight():

    fast_state.active, fast_state.staging = (
        fast_state.staging,
        fast_state.active,
    )


    fast_state.version += 1


# =============================================================================
# PATCH QWEN down_proj
#
# IMPORTANT:
#
# model inference now ACTUALLY reads:
#
#       fast_state.active
#
# not:
#
#       down_proj.weight
#
# =============================================================================

def fast_weight_down_forward(
    x,
):

    fast_state.last_used_version = (
        fast_state.version
    )


    fast_state.last_used_ptr = (
        fast_state.active.data_ptr()
    )


    # Master fast weight is FP32.
    #
    # Compute this one projection in FP32 and then return BF16 to the
    # remainder of the Qwen layer.
    y = F.linear(
        x.float(),
        fast_state.active,
        bias=None,
    )


    return y.to(
        dtype=
            x.dtype
    )


target_layer.mlp.down_proj.forward = (
    fast_weight_down_forward
)


print()
print("=" * 120)
print("FAST-WEIGHT PROJECTION INSTALLED")
print("=" * 120)


print(
    "W_A pointer           :",
    W_A.data_ptr(),
)

print(
    "W_B pointer           :",
    W_B.data_ptr(),
)

print(
    "model target layer    :",
    TTT_LAYER,
)


# =============================================================================
# STATIC FAST-WEIGHT BASELINE
#
# Compare original BF16 Qwen down_proj vs the patched FP32-master path.
# =============================================================================

probe_ids = (
    input_ids[
        :,
        :64
    ]
    .contiguous()
)


# Original path
target_layer.mlp.down_proj.forward = (
    original_down_forward
)


with torch.inference_mode():

    original_logits = (
        model(
            input_ids=
                probe_ids,

            use_cache=
                False,
        )
        .logits[
            :,
            -1,
            :
        ]
        .float()
        .clone()
    )


torch.cuda.synchronize()


# Reinstall fast path
target_layer.mlp.down_proj.forward = (
    fast_weight_down_forward
)


reset_fast_state()


with torch.inference_mode():

    fast_v0_logits = (
        model(
            input_ids=
                probe_ids,

            use_cache=
                False,
        )
        .logits[
            :,
            -1,
            :
        ]
        .float()
        .clone()
    )


torch.cuda.synchronize()


baseline_path_rel = (
    torch.linalg.vector_norm(
        fast_v0_logits
        -
        original_logits
    )
    /
    torch.linalg.vector_norm(
        original_logits
    ).clamp_min(
        1e-30
    )
).item()


baseline_path_max = (
    (
        fast_v0_logits
        -
        original_logits
    )
    .abs()
    .max()
    .item()
)


print()
print("Patched static-path difference vs original BF16 Qwen")

print(
    "relative L2           :",
    f"{baseline_path_rel:.6e}",
)

print(
    "max abs               :",
    f"{baseline_path_max:.6e}",
)


# =============================================================================
# SEMANTIC POINTER-SWAP PROOF
# =============================================================================

print()
print("=" * 120)
print("SEMANTIC FAST-WEIGHT CONSUMPTION PROOF")
print("=" * 120)


reset_fast_state()


# W0 logits
with torch.inference_mode():

    logits_v0 = (
        model(
            input_ids=
                probe_ids,

            use_cache=
                False,
        )
        .logits[
            :,
            -1,
            :
        ]
        .float()
        .clone()
    )


torch.cuda.synchronize()


v0_ptr = (
    fast_state.last_used_ptr
)


# Real Triton update:
#
# W_A -> W_B
#
triton_update(
    fast_state.active,
    fast_state.staging,

    SEMANTIC_LR,
    SEMANTIC_CLIP,
)


torch.cuda.synchronize()


semantic_changed = (
    fast_state.active
    !=
    fast_state.staging
)


semantic_changed_count = (
    semantic_changed
    .sum()
    .item()
)


semantic_weight_delta = (
    torch.linalg.vector_norm(
        fast_state.staging
        -
        fast_state.active
    )
    .item()
)


semantic_max_weight_delta = (
    (
        fast_state.staging
        -
        fast_state.active
    )
    .abs()
    .max()
    .item()
)


# O(1) Python pointer/reference swap.
commit_fast_weight()


v1_expected_ptr = (
    fast_state.active.data_ptr()
)


# Same input, but model now reads W1.
with torch.inference_mode():

    logits_v1 = (
        model(
            input_ids=
                probe_ids,

            use_cache=
                False,
        )
        .logits[
            :,
            -1,
            :
        ]
        .float()
        .clone()
    )


torch.cuda.synchronize()


v1_actual_ptr = (
    fast_state.last_used_ptr
)


logit_delta = (
    logits_v1
    -
    logits_v0
)


logit_rel = (
    torch.linalg.vector_norm(
        logit_delta
    )
    /
    torch.linalg.vector_norm(
        logits_v0
    ).clamp_min(
        1e-30
    )
).item()


logit_max = (
    logit_delta
    .abs()
    .max()
    .item()
)


top0 = int(
    logits_v0.argmax(
        dim=-1
    ).item()
)


top1 = int(
    logits_v1.argmax(
        dim=-1
    ).item()
)


print(
    "semantic LR           :",
    SEMANTIC_LR,
)

print(
    "semantic clip         :",
    SEMANTIC_CLIP,
)

print(
    "changed weights       :",
    f"{semantic_changed_count:,}",
)

print(
    "||W1-W0||             :",
    f"{semantic_weight_delta:.6e}",
)

print(
    "max |W1-W0|           :",
    f"{semantic_max_weight_delta:.6e}",
)

print()
print(
    "W0 pointer used       :",
    v0_ptr,
)

print(
    "W1 expected pointer   :",
    v1_expected_ptr,
)

print(
    "W1 actual pointer     :",
    v1_actual_ptr,
)

print(
    "pointer switch valid  :",
    v1_expected_ptr
    ==
    v1_actual_ptr
    and
    v0_ptr
    !=
    v1_actual_ptr,
)

print()
print(
    "logit relative L2     :",
    f"{logit_rel:.6e}",
)

print(
    "logit max abs         :",
    f"{logit_max:.6e}",
)

print(
    "top-1 token W0        :",
    top0,
)

print(
    "top-1 token W1        :",
    top1,
)

print(
    "top-1 changed         :",
    top0
    !=
    top1,
)


del semantic_changed
del logit_delta


# =============================================================================
# STREAMS
# =============================================================================

try:

    stream_inference = (
        torch.cuda.Stream(
            priority=-1
        )
    )

except Exception:

    stream_inference = (
        torch.cuda.Stream()
    )


stream_learning = (
    torch.cuda.Stream(
        priority=0
    )
)


torch.cuda.synchronize()


# =============================================================================
# STATS
# =============================================================================

def percentile(
    values,
    p,
):

    if not values:
        return 0.0


    ordered = sorted(
        values
    )


    x = (
        len(ordered)
        -
        1
    ) * p


    lo = int(
        math.floor(
            x
        )
    )


    hi = int(
        math.ceil(
            x
        )
    )


    if lo == hi:

        return ordered[
            lo
        ]


    f = (
        x
        -
        lo
    )


    return (
        ordered[
            lo
        ]
        *
        (
            1.0
            -
            f
        )
        +
        ordered[
            hi
        ]
        *
        f
    )


def summarize(
    values,
):

    if not values:
        return None


    return {

        "mean":
            statistics.mean(
                values
            ),

        "p50":
            percentile(
                values,
                0.50,
            ),

        "p95":
            percentile(
                values,
                0.95,
            ),

        "p99":
            percentile(
                values,
                0.99,
            ),

        "min":
            min(
                values
            ),

        "max":
            max(
                values
            ),
    }


# =============================================================================
# ACTUAL ADAPTIVE DECODE
# =============================================================================

MODES = [

    "baseline_fastweight",

    "serialized_triton",

    "async_triton",

]


def run_adaptive_decode(
    mode,
):

    if mode not in MODES:

        raise ValueError(
            mode
        )


    reset_fast_state()


    prompt = (
        input_ids[
            :,
            :PROMPT_LEN
        ]
        .contiguous()
    )


    token_itl = []

    update_token_itl = []

    normal_token_itl = []

    update_times = []

    visibility_lags = []

    generated_tokens = []

    used_versions = []

    used_pointers = []

    commits = []


    update_in_flight = False

    update_start = None

    update_end = None

    update_started_token = None


    torch.cuda.synchronize()


    # =========================================================================
    # PREFILL
    # =========================================================================

    with torch.cuda.stream(
        stream_inference
    ):

        with torch.inference_mode():

            out = model(
                input_ids=
                    prompt,

                use_cache=
                    True,
            )


        past_key_values = (
            out.past_key_values
        )


        next_token = (
            out.logits[
                :,
                -1,
                :
            ]
            .argmax(
                dim=-1,

                keepdim=
                    True,
            )
        )


    stream_inference.synchronize()


    # =========================================================================
    # TOKEN LOOP
    # =========================================================================

    wall_start = (
        time.perf_counter()
    )


    for token_idx in range(
        DECODE_TOKENS
    ):

        # ---------------------------------------------------------------------
        # Commit an async update at a token boundary.
        # ---------------------------------------------------------------------

        if (
            update_in_flight
            and
            update_end.query()
        ):

            update_end.synchronize()


            update_times.append(
                update_start.elapsed_time(
                    update_end
                )
            )


            lag = (
                token_idx
                -
                update_started_token
            )


            visibility_lags.append(
                lag
            )


            commit_fast_weight()


            commits.append(
                {
                    "visible_from_token":
                        token_idx,

                    "version":
                        fast_state.version,

                    "pointer":
                        fast_state.active.data_ptr(),

                    "lag_tokens":
                        lag,
                }
            )


            update_in_flight = False


        trigger_update = (
            token_idx
            %
            UPDATE_EVERY
            ==
            0
        )


        token_wall_start = (
            time.perf_counter()
        )


        # =====================================================================
        # SERIALIZED
        #
        # Update -> completion -> commit -> token
        # =====================================================================

        if (
            mode
            ==
            "serialized_triton"
            and
            trigger_update
        ):

            serial_start = (
                torch.cuda.Event(
                    enable_timing=True
                )
            )


            serial_end = (
                torch.cuda.Event(
                    enable_timing=True
                )
            )


            with torch.cuda.stream(
                stream_inference
            ):

                serial_start.record(
                    stream_inference
                )


                triton_update(
                    fast_state.active,
                    fast_state.staging,

                    SEMANTIC_LR,
                    SEMANTIC_CLIP,
                )


                serial_end.record(
                    stream_inference
                )


            serial_end.synchronize()


            update_times.append(
                serial_start.elapsed_time(
                    serial_end
                )
            )


            commit_fast_weight()


            commits.append(
                {
                    "visible_from_token":
                        token_idx,

                    "version":
                        fast_state.version,

                    "pointer":
                        fast_state.active.data_ptr(),

                    "lag_tokens":
                        0,
                }
            )


        # =====================================================================
        # ASYNC
        #
        # launch update against:
        #
        #       active -> staging
        #
        # token still reads active.
        # =====================================================================

        if (
            mode
            ==
            "async_triton"
            and
            trigger_update
            and
            not update_in_flight
        ):

            update_start = (
                torch.cuda.Event(
                    enable_timing=True
                )
            )


            update_end = (
                torch.cuda.Event(
                    enable_timing=True
                )
            )


            update_started_token = (
                token_idx
            )


            with torch.cuda.stream(
                stream_learning
            ):

                update_start.record(
                    stream_learning
                )


                triton_update(
                    fast_state.active,
                    fast_state.staging,

                    SEMANTIC_LR,
                    SEMANTIC_CLIP,
                )


                update_end.record(
                    stream_learning
                )


            update_in_flight = (
                True
            )


        # =====================================================================
        # REAL QWEN TOKEN
        #
        # patched down_proj reads fast_state.active.
        # =====================================================================

        fast_state.last_used_version = None

        fast_state.last_used_ptr = None


        with torch.cuda.stream(
            stream_inference
        ):

            with torch.inference_mode():

                out = model(
                    input_ids=
                        next_token,

                    past_key_values=
                        past_key_values,

                    use_cache=
                        True,
                )


            past_key_values = (
                out.past_key_values
            )


            next_token = (
                out.logits[
                    :,
                    -1,
                    :
                ]
                .argmax(
                    dim=-1,

                    keepdim=
                        True,
                )
            )


        # Wait only for foreground inference.
        stream_inference.synchronize()


        token_ms = (
            time.perf_counter()
            -
            token_wall_start
        ) * 1000.0


        token_id = int(
            next_token.item()
        )


        generated_tokens.append(
            token_id
        )


        token_itl.append(
            token_ms
        )


        used_versions.append(
            int(
                fast_state.last_used_version
            )
        )


        used_pointers.append(
            int(
                fast_state.last_used_ptr
            )
        )


        if trigger_update:

            update_token_itl.append(
                token_ms
            )

        else:

            normal_token_itl.append(
                token_ms
            )


    foreground_ms = (
        time.perf_counter()
        -
        wall_start
    ) * 1000.0


    # =========================================================================
    # FINAL UPDATE
    # =========================================================================

    if update_in_flight:

        update_end.synchronize()


        update_times.append(
            update_start.elapsed_time(
                update_end
            )
        )


        lag = (
            DECODE_TOKENS
            -
            update_started_token
        )


        visibility_lags.append(
            lag
        )


        commit_fast_weight()


        commits.append(
            {
                "visible_from_token":
                    DECODE_TOKENS,

                "version":
                    fast_state.version,

                "pointer":
                    fast_state.active.data_ptr(),

                "lag_tokens":
                    lag,
            }
        )


    torch.cuda.synchronize()


    return {

        "mode":
            mode,

        "itl":
            summarize(
                token_itl
            ),

        "update_token_itl":
            summarize(
                update_token_itl
            ),

        "normal_token_itl":
            summarize(
                normal_token_itl
            ),

        "ttt":
            summarize(
                update_times
            ),

        "visibility_lag":
            summarize(
                visibility_lags
            ),

        "foreground_ms":
            foreground_ms,

        "tok_s":
            DECODE_TOKENS
            /
            (
                foreground_ms
                /
                1000.0
            ),

        "generated_tokens":
            generated_tokens,

        "used_versions":
            used_versions,

        "used_pointers":
            used_pointers,

        "commits":
            commits,
    }


# =============================================================================
# WARMUP
# =============================================================================

print()
print("=" * 120)
print("ADAPTIVE QWEN WARMUP")
print("=" * 120)


warmup = (
    run_adaptive_decode(
        "baseline_fastweight"
    )
)


print(
    "Baseline p50         :",
    f"{warmup['itl']['p50']:.3f} ms",
)

print(
    "Warmup passed."
)


# =============================================================================
# BENCHMARK
# =============================================================================

print()
print("=" * 150)
print("REAL FAST-WEIGHT QWEN2.5-7B BENCHMARK")
print("=" * 150)


runs = {

    mode: []

    for mode in MODES
}


for mode in MODES:

    print()
    print("-" * 110)
    print(
        mode.upper()
    )
    print("-" * 110)


    for run_idx in range(
        RUNS
    ):

        result = (
            run_adaptive_decode(
                mode
            )
        )


        runs[
            mode
        ].append(
            result
        )


        ttt_text = (
            "-"
            if result[
                "ttt"
            ]
            is None
            else
            f"{result['ttt']['p50']:.3f}"
        )


        lag_text = (
            "-"
            if result[
                "visibility_lag"
            ]
            is None
            else
            f"{result['visibility_lag']['p50']:.1f}"
        )


        print(
            f"run={run_idx + 1} | "
            f"p50={result['itl']['p50']:.3f} | "
            f"p95={result['itl']['p95']:.3f} | "
            f"p99={result['itl']['p99']:.3f} | "
            f"update-token={result['update_token_itl']['p50']:.3f} | "
            f"normal={result['normal_token_itl']['p50']:.3f} | "
            f"TTT={ttt_text} | "
            f"lag={lag_text}"
        )


# =============================================================================
# AGGREGATE
# =============================================================================

def median_from_runs(
    mode,
    group,
    field,
):

    return statistics.median(
        [
            run[
                group
            ][
                field
            ]

            for run in runs[
                mode
            ]

            if run[
                group
            ]
            is not None
        ]
    )


aggregate = {}


for mode in MODES:

    aggregate[
        mode
    ] = {

        "p50":
            median_from_runs(
                mode,
                "itl",
                "p50",
            ),

        "p95":
            median_from_runs(
                mode,
                "itl",
                "p95",
            ),

        "p99":
            median_from_runs(
                mode,
                "itl",
                "p99",
            ),

        "update_p50":
            median_from_runs(
                mode,
                "update_token_itl",
                "p50",
            ),

        "normal_p50":
            median_from_runs(
                mode,
                "normal_token_itl",
                "p50",
            ),

        "tok_s":
            statistics.median(
                [
                    r[
                        "tok_s"
                    ]

                    for r in runs[
                        mode
                    ]
                ]
            ),
    }


    ttts = [

        r[
            "ttt"
        ][
            "p50"
        ]

        for r in runs[
            mode
        ]

        if r[
            "ttt"
        ]
        is not None
    ]


    aggregate[
        mode
    ][
        "ttt_ms"
    ] = (
        statistics.median(
            ttts
        )
        if ttts
        else None
    )


    lags = [

        r[
            "visibility_lag"
        ][
            "p50"
        ]

        for r in runs[
            mode
        ]

        if r[
            "visibility_lag"
        ]
        is not None
    ]


    aggregate[
        mode
    ][
        "lag"
    ] = (
        statistics.median(
            lags
        )
        if lags
        else None
    )


# =============================================================================
# FINAL TABLE
# =============================================================================

print()
print("=" * 155)
print("FINAL REAL FAST-WEIGHT RESULTS")
print("=" * 155)


print(
    f"{'Mode':<25}"
    f"{'p50':>11}"
    f"{'p95':>11}"
    f"{'p99':>11}"
    f"{'update':>12}"
    f"{'normal':>12}"
    f"{'TTT':>11}"
    f"{'lag':>8}"
    f"{'tok/s':>11}"
)


print(
    "-" * 155
)


for mode in MODES:

    r = (
        aggregate[
            mode
        ]
    )


    ttt = (
        "-"
        if r[
            "ttt_ms"
        ]
        is None
        else
        f"{r['ttt_ms']:.3f}"
    )


    lag = (
        "-"
        if r[
            "lag"
        ]
        is None
        else
        f"{r['lag']:.1f}"
    )


    print(
        f"{mode:<25}"
        f"{r['p50']:>9.3f}ms"
        f"{r['p95']:>9.3f}ms"
        f"{r['p99']:>9.3f}ms"
        f"{r['update_p50']:>10.3f}ms"
        f"{r['normal_p50']:>10.3f}ms"
        f"{ttt:>11}"
        f"{lag:>8}"
        f"{r['tok_s']:>11.2f}"
    )


# =============================================================================
# VERSION / POINTER PROOF
# =============================================================================

async_example = (
    runs[
        "async_triton"
    ][
        0
    ]
)


print()
print("=" * 120)
print("ASYNC VERSION TRACE — FIRST RUN")
print("=" * 120)


print(
    "token : version : pointer"
)


for i in range(
    min(
        20,
        DECODE_TOKENS,
    )
):

    print(
        f"{i:>5} : "
        f"{async_example['used_versions'][i]:>7} : "
        f"{async_example['used_pointers'][i]}"
    )


print()
print(
    "COMMITS"
)


for commit in (
    async_example[
        "commits"
    ]
):

    print(
        commit
    )


# =============================================================================
# GENERATED-SEQUENCE DIFFERENCE
# =============================================================================

baseline_tokens = (
    runs[
        "baseline_fastweight"
    ][
        0
    ][
        "generated_tokens"
    ]
)


async_tokens = (
    async_example[
        "generated_tokens"
    ]
)


different_positions = [

    i

    for i, (
        a,
        b,
    )

    in enumerate(
        zip(
            baseline_tokens,
            async_tokens,
        )
    )

    if a != b
]


print()
print("=" * 120)
print("ADAPTATION EFFECT ON GREEDY GENERATION")
print("=" * 120)


print(
    "different generated positions:",
    len(
        different_positions
    ),
    "/",
    DECODE_TOKENS,
)


print(
    "first differing positions:",
    different_positions[
        :20
    ],
)


# =============================================================================
# SAVE
# =============================================================================

RESULT = {

    "model":
        MODEL_ID,

    "gpu":
        torch.cuda.get_device_name(
            0
        ),

    "shape": {

        "M":
            M,

        "N":
            N,

        "K":
            TTT_K,

        "layer":
            TTT_LAYER,
    },

    "paper_config_precision_test": {

        "lr":
            PERF_LR,

        "clip":
            PERF_CLIP,

        "changed_count":
            paper_changed_count,

        "changed_fraction":
            paper_changed_fraction,

        "weight_delta_l2":
            paper_weight_delta,

        "weight_delta_max":
            paper_max_delta,
    },

    "semantic_config": {

        "lr":
            SEMANTIC_LR,

        "clip":
            SEMANTIC_CLIP,

        "changed_count":
            semantic_changed_count,

        "weight_delta_l2":
            semantic_weight_delta,

        "weight_delta_max":
            semantic_max_weight_delta,

        "logit_relative_l2":
            logit_rel,

        "logit_max_abs":
            logit_max,

        "top1_before":
            top0,

        "top1_after":
            top1,

        "pointer_switch_valid":
            bool(
                v1_expected_ptr
                ==
                v1_actual_ptr
                and
                v0_ptr
                !=
                v1_actual_ptr
            ),
    },

    "static_fastweight_vs_original": {

        "relative_l2":
            baseline_path_rel,

        "max_abs":
            baseline_path_max,
    },

    "aggregate":
        aggregate,

    "runs":
        runs,

    "different_generated_positions":
        different_positions,
}


RESULT_PATH = Path(
    "async_ttt_qwen7b_real_fastweight.json"
)


with open(
    RESULT_PATH,
    "w",
) as f:

    json.dump(
        RESULT,
        f,
        indent=2,
    )


print()
print("=" * 120)
print("DONE")
print("=" * 120)


print(
    "Saved:",
    RESULT_PATH.resolve(),
)


# =============================================================================
# RESTORE ORIGINAL MODEL FORWARD
# =============================================================================

target_layer.mlp.down_proj.forward = (
    original_down_forward
)


torch.cuda.synchronize()


print(
    "Original down_proj.forward restored."
)


free_bytes, total_bytes = (
    torch.cuda.mem_get_info()
)


print(
    "Free VRAM after test:",
    f"{free_bytes / 1024**3:.2f} GiB",
)

FLUSHING PREVIOUS GPU STATE
Free VRAM : 38.56 GiB
Total VRAM: 39.49 GiB

QWEN2.5-7B — REAL FAST-WEIGHT CONSUMPTION
Model                : Qwen/Qwen2.5-7B-Instruct
Python               : 3.12.11
PyTorch              : 2.8.0+cu128
CUDA                 : 12.8
Triton               : 3.4.0
GPU                  : NVIDIA A100-SXM4-40GB
SMs                  : 108
VRAM                 : 39.49 GiB

LOADING QWEN2.5-7B


Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

hidden_size          : 3584
intermediate_size    : 18944
layers               : 28
TTT layer            : 13
model dtype          : torch.bfloat16
FP32 fast weight     : 271.58 MB
Free after load      : 24.33 GiB
Input tokens         : 512

CAPTURING REAL QWEN ACTIVATIONS
VT                   : (3584, 512)
Z                    : (512, 18944)
W_down               : (3584, 18944)

PAPER-CONFIG WEIGHT-VISIBILITY TEST
LR                    : 0.001
clip                  : 1e-05
changed elements      : 46,465 / 67,895,296
changed fraction      : 0.06843626%
actual ||W1-W0||      : 2.87057533e-09
actual max |W1-W0|    : 9.31322575e-10

IMPORTANT: the current LR/clip is near/below the FP32 weight precision floor.
This is a hyperparameter/numerical issue, NOT a kernel correctness issue.

FAST-WEIGHT PROJECTION INSTALLED
W_A pointer           : 132570090045440
W_B pointer           : 132570362675200
model target layer    : 13

Patched static-path difference vs original BF16 Qwen
relative L2     

In [11]:
# =============================================================================
# ASYNC-TTT — QWEN2.5-7B WITH A REAL LM GRADIENT
#
# This experiment fixes the major mathematical gap in the previous benchmark.
#
# REAL gradient:
#
#   Z = input to target down_proj
#   G = d(LM loss) / d(down_proj output)
#
# therefore:
#
#   dW = G^T @ Z
#
# and gradient descent is:
#
#   W_new = W_old - eta * dW
#
# The Triton kernel:
#   - computes exact global Frobenius norm
#   - NEVER materializes full dW in HBM
#   - rematerializes dW tiles on-chip in pass 2
#
# This script:
#
#   1. Flushes previous GPU state.
#   2. Loads Qwen2.5-7B.
#   3. Freezes all model parameters.
#   4. Cuts autograd at layer-13 down_proj output.
#   5. Computes REAL next-token LM loss.
#   6. Captures real grad_output G.
#   7. Forms the mathematically correct W_down gradient.
#   8. Validates Triton against PyTorch.
#   9. Searches a reasonable update magnitude.
#  10. Shows loss before/after adaptation.
#  11. Shows held-out loss before/after adaptation.
#  12. Wires A/B fast weights into REAL Qwen inference.
#  13. Proves the pointer/version swap.
#  14. Runs one real gradient-derived update:
#          baseline
#          serialized
#          async
#  15. Measures ITL, update stall, hidden fraction and visibility lag.
#
# NOTE:
#   "on-chip rematerialization" is safe terminology.
#   Exact SRAM/shared-memory/register/HBM traffic still needs Nsight Compute.
# =============================================================================


# =============================================================================
# IMPORTS
# =============================================================================

import gc
import json
import math
import statistics
import sys
import time

from pathlib import Path

import torch
import torch.nn.functional as F

import triton
import triton.language as tl

from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
)


# =============================================================================
# FLUSH EXISTING GPU STATE
# =============================================================================

print("=" * 120)
print("FLUSHING PREVIOUS GPU STATE")
print("=" * 120)


cleanup_names = [
    "model",
    "tokenizer",
    "target_layer",
    "original_forward",
    "original_down_forward",
    "W_MASTER",
    "W_A",
    "W_B",
    "V_ALL",
    "Z_ALL",
    "VT",
    "fast_state",
    "output",
    "out",
    "past_key_values",
    "next_token",
    "captured",
    "capture",
    "runs",
    "warmup",
]


for name in cleanup_names:

    if name in globals():

        try:
            del globals()[name]
        except Exception:
            pass


gc.collect()


if torch.cuda.is_available():

    torch.cuda.synchronize()

    torch.cuda.empty_cache()

    try:
        torch.cuda.ipc_collect()
    except Exception:
        pass

    gc.collect()

    torch.cuda.empty_cache()


free_bytes, total_bytes = (
    torch.cuda.mem_get_info()
)


print(
    "Free VRAM :",
    f"{free_bytes / 1024**3:.2f} GiB",
)

print(
    "Total VRAM:",
    f"{total_bytes / 1024**3:.2f} GiB",
)


# =============================================================================
# CONFIG
# =============================================================================

MODEL_ID = (
    "Qwen/Qwen2.5-7B-Instruct"
)

DEVICE = "cuda"

TTT_LAYER = 13


# 512 observed tokens are used to compute the actual LM gradient.
ADAPT_LEN = 512

# Separate held-out block.
EVAL_LEN = 512

TOTAL_LEN = (
    ADAPT_LEN
    +
    EVAL_LEN
)


DECODE_TOKENS = 64


torch.manual_seed(
    0
)

torch.set_grad_enabled(
    True
)

torch.backends.cuda.matmul.allow_tf32 = False
torch.backends.cudnn.allow_tf32 = False


# =============================================================================
# ENVIRONMENT
# =============================================================================

assert torch.cuda.is_available()


print()
print("=" * 120)
print("QWEN2.5-7B — REAL GRADIENT ASYNC-TTT")
print("=" * 120)


print(
    "Model       :",
    MODEL_ID,
)

print(
    "Python      :",
    sys.version.split()[0],
)

print(
    "PyTorch     :",
    torch.__version__,
)

print(
    "CUDA        :",
    torch.version.cuda,
)

print(
    "Triton      :",
    triton.__version__,
)

print(
    "GPU         :",
    torch.cuda.get_device_name(0),
)


# =============================================================================
# LOAD MODEL
# =============================================================================

print()
print("=" * 120)
print("LOADING QWEN2.5-7B")
print("=" * 120)


tokenizer = (
    AutoTokenizer.from_pretrained(
        MODEL_ID,
        trust_remote_code=True,
    )
)


model = (
    AutoModelForCausalLM.from_pretrained(
        MODEL_ID,

        dtype=
            torch.bfloat16,

        device_map={
            "": 0
        },

        low_cpu_mem_usage=True,

        trust_remote_code=True,

        attn_implementation=
            "sdpa",
    )
)


model.eval()


# Freeze EVERY model parameter.
#
# Autograd will begin only at the explicitly detached down_proj output leaf.

for parameter in model.parameters():

    parameter.requires_grad_(
        False
    )


M = int(
    model.config.hidden_size
)

N = int(
    model.config.intermediate_size
)


target_layer = (
    model.model.layers[
        TTT_LAYER
    ]
)


print(
    "hidden_size         :",
    M,
)

print(
    "intermediate_size   :",
    N,
)

print(
    "layers              :",
    model.config.num_hidden_layers,
)

print(
    "TTT layer           :",
    TTT_LAYER,
)

print(
    "FP32 fast weight    :",
    f"{M * N * 4 / 1e6:.2f} MB",
)


free_bytes, _ = (
    torch.cuda.mem_get_info()
)


print(
    "Free after load     :",
    f"{free_bytes / 1024**3:.2f} GiB",
)


# =============================================================================
# BUILD 1024 REAL TOKENS
# =============================================================================

base_text = """
A production language model may need to adapt continuously to incoming
information. Test-time learning changes an internal learned state using recently
observed tokens while autoregressive inference continues. An efficient serving
runtime must overlap adaptation with inference while controlling GPU memory
traffic, synchronization, staleness and user-visible tail latency.
"""


text = "\n".join(
    [base_text] * 2500
)


tokenized = tokenizer(
    text,

    return_tensors=
        "pt",

    truncation=
        True,

    max_length=
        TOTAL_LEN,
)


all_ids = (
    tokenized[
        "input_ids"
    ]
    .to(
        DEVICE
    )
)


if (
    all_ids.shape[1]
    <
    TOTAL_LEN
):

    raise RuntimeError(
        f"Need {TOTAL_LEN} tokens; "
        f"got {all_ids.shape[1]}"
    )


adapt_ids = (
    all_ids[
        :,
        :ADAPT_LEN
    ]
    .contiguous()
)


eval_ids = (
    all_ids[
        :,
        ADAPT_LEN:
        ADAPT_LEN + EVAL_LEN
    ]
    .contiguous()
)


print(
    "Adapt tokens        :",
    adapt_ids.shape[1],
)

print(
    "Held-out tokens     :",
    eval_ids.shape[1],
)


del tokenized
del text

gc.collect()


# =============================================================================
# REAL GRADIENT CAPTURE
#
# Critical trick:
#
#   - Lower half of model requires no gradient graph.
#   - target down_proj executes normally.
#   - its output is detached and turned into a fresh leaf requiring grad.
#   - upper model layers build an autograd graph from that leaf.
#
# This gives:
#
#       G = dLoss / d(down_proj output)
#
# without storing backward activations for the entire 7B model.
# =============================================================================

print()
print("=" * 120)
print("CAPTURING REAL LM GRADIENT SIGNAL")
print("=" * 120)


original_down_forward = (
    target_layer
    .mlp
    .down_proj
    .forward
)


capture = {}


def gradient_capture_forward(
    x,
):

    # Real input to down_proj.
    capture[
        "z"
    ] = (
        x
        .detach()
        .float()
        .contiguous()
    )


    # Execute the ORIGINAL Qwen BF16 projection.
    y = original_down_forward(
        x
    )


    # Cut graph here.
    #
    # Everything before this layer remains gradient-free.
    # Everything above it can differentiate with respect to y_leaf.
    y_leaf = (
        y
        .detach()
        .requires_grad_(
            True
        )
    )


    y_leaf.retain_grad()


    capture[
        "y_leaf"
    ] = (
        y_leaf
    )


    return y_leaf


target_layer.mlp.down_proj.forward = (
    gradient_capture_forward
)


# Clear any old gradients.
model.zero_grad(
    set_to_none=True
)


torch.cuda.synchronize()


gradient_wall_start = (
    time.perf_counter()
)


# Real next-token LM objective.
#
# QwenForCausalLM performs the standard one-token shift internally.

adapt_output = model(
    input_ids=
        adapt_ids,

    labels=
        adapt_ids,

    use_cache=
        False,
)


loss_before_original_path = (
    adapt_output.loss
)


print(
    "LM loss before backward:",
    float(
        loss_before_original_path
        .detach()
        .item()
    ),
)


loss_before_original_path.backward()


torch.cuda.synchronize()


gradient_wall_ms = (
    time.perf_counter()
    -
    gradient_wall_start
) * 1000.0


if (
    capture[
        "y_leaf"
    ].grad
    is None
):

    raise RuntimeError(
        "No gradient reached the target down_proj output."
    )


# =============================================================================
# THE REAL TTT MATRICES
#
# down_proj:
#
#       y = z @ W^T
#
# gradient:
#
#       dW = G^T @ Z
#
# where:
#
#       G = dLoss / dy        [K, M]
#       Z = down_proj input   [K, N]
#
# =============================================================================

Z_REAL = (
    capture[
        "z"
    ][
        0
    ]
    .float()
    .contiguous()
)


G_REAL = (
    capture[
        "y_leaf"
    ]
    .grad[
        0
    ]
    .float()
    .contiguous()
)


VT_REAL = (
    G_REAL
    .T
    .contiguous()
)


W_MASTER = (
    target_layer
    .mlp
    .down_proj
    .weight
    .detach()
    .float()
    .contiguous()
)


assert Z_REAL.shape == (
    ADAPT_LEN,
    N,
)


assert G_REAL.shape == (
    ADAPT_LEN,
    M,
)


assert VT_REAL.shape == (
    M,
    ADAPT_LEN,
)


assert W_MASTER.shape == (
    M,
    N,
)


print()
print(
    "Z = down_proj input  :",
    tuple(
        Z_REAL.shape
    ),
)

print(
    "G = dLoss/dOutput    :",
    tuple(
        G_REAL.shape
    ),
)

print(
    "G^T                  :",
    tuple(
        VT_REAL.shape
    ),
)

print(
    "W_down               :",
    tuple(
        W_MASTER.shape
    ),
)

print(
    "Gradient capture time:",
    f"{gradient_wall_ms:.3f} ms",
)


# Restore original Qwen projection immediately.
target_layer.mlp.down_proj.forward = (
    original_down_forward
)


# Release autograd graph.
del adapt_output
del loss_before_original_path
del capture

gc.collect()

torch.cuda.empty_cache()


# =============================================================================
# TRITON PASS 1
#
# Computes:
#
#       tile = G^T @ Z
#
#       global_norm² += ||tile||²
#
# Full dW is NEVER written to HBM.
# =============================================================================

@triton.jit
def real_ttt_norm_kernel(
    gt_ptr,
    z_ptr,
    norm_ptr,

    stride_gm,
    stride_gk,

    stride_zk,
    stride_zn,

    M: tl.constexpr,
    N: tl.constexpr,
    K: tl.constexpr,

    BLOCK_M: tl.constexpr,
    BLOCK_N: tl.constexpr,
    BLOCK_K: tl.constexpr,
):

    pid_m = tl.program_id(
        0
    )

    pid_n = tl.program_id(
        1
    )


    offs_m = (
        pid_m * BLOCK_M
        +
        tl.arange(
            0,
            BLOCK_M,
        )
    )


    offs_n = (
        pid_n * BLOCK_N
        +
        tl.arange(
            0,
            BLOCK_N,
        )
    )


    # Temporary dW tile.
    #
    # Not a global-memory dW tensor.
    acc = tl.zeros(
        (
            BLOCK_M,
            BLOCK_N,
        ),
        dtype=
            tl.float32,
    )


    for k in range(
        0,
        K,
        BLOCK_K,
    ):

        offs_k = (
            k
            +
            tl.arange(
                0,
                BLOCK_K,
            )
        )


        g = tl.load(
            gt_ptr
            +
            offs_m[:, None]
            *
            stride_gm
            +
            offs_k[None, :]
            *
            stride_gk,

            mask=(
                (
                    offs_m[:, None]
                    <
                    M
                )
                &
                (
                    offs_k[None, :]
                    <
                    K
                )
            ),

            other=
                0.0,
        )


        z = tl.load(
            z_ptr
            +
            offs_k[:, None]
            *
            stride_zk
            +
            offs_n[None, :]
            *
            stride_zn,

            mask=(
                (
                    offs_k[:, None]
                    <
                    K
                )
                &
                (
                    offs_n[None, :]
                    <
                    N
                )
            ),

            other=
                0.0,
        )


        acc = tl.dot(
            g,
            z,
            acc,

            input_precision=
                "ieee",
        )


    local_norm_sq = (
        tl.sum(
            acc
            *
            acc
        )
    )


    tl.atomic_add(
        norm_ptr,
        local_norm_sq,
    )


# =============================================================================
# TRITON PASS 2
#
# True gradient descent:
#
#       W_new =
#           W_old
#           -
#           learning_rate
#           *
#           clipped(dW)
# =============================================================================

@triton.jit
def real_ttt_update_kernel(
    gt_ptr,
    z_ptr,

    src_ptr,
    dst_ptr,

    norm_ptr,

    stride_gm,
    stride_gk,

    stride_zk,
    stride_zn,

    stride_sm,
    stride_sn,

    stride_dm,
    stride_dn,

    learning_rate,
    clip_threshold,

    M: tl.constexpr,
    N: tl.constexpr,
    K: tl.constexpr,

    BLOCK_M: tl.constexpr,
    BLOCK_N: tl.constexpr,
    BLOCK_K: tl.constexpr,
):

    pid_m = tl.program_id(
        0
    )

    pid_n = tl.program_id(
        1
    )


    offs_m = (
        pid_m * BLOCK_M
        +
        tl.arange(
            0,
            BLOCK_M,
        )
    )


    offs_n = (
        pid_n * BLOCK_N
        +
        tl.arange(
            0,
            BLOCK_N,
        )
    )


    norm_sq = tl.load(
        norm_ptr
    )


    raw_scale = (
        clip_threshold
        /
        (
            tl.sqrt(
                norm_sq
            )
            +
            1e-12
        )
    )


    scale = tl.minimum(
        raw_scale,
        1.0,
    )


    acc = tl.zeros(
        (
            BLOCK_M,
            BLOCK_N,
        ),
        dtype=
            tl.float32,
    )


    for k in range(
        0,
        K,
        BLOCK_K,
    ):

        offs_k = (
            k
            +
            tl.arange(
                0,
                BLOCK_K,
            )
        )


        g = tl.load(
            gt_ptr
            +
            offs_m[:, None]
            *
            stride_gm
            +
            offs_k[None, :]
            *
            stride_gk,

            mask=(
                (
                    offs_m[:, None]
                    <
                    M
                )
                &
                (
                    offs_k[None, :]
                    <
                    K
                )
            ),

            other=
                0.0,
        )


        z = tl.load(
            z_ptr
            +
            offs_k[:, None]
            *
            stride_zk
            +
            offs_n[None, :]
            *
            stride_zn,

            mask=(
                (
                    offs_k[:, None]
                    <
                    K
                )
                &
                (
                    offs_n[None, :]
                    <
                    N
                )
            ),

            other=
                0.0,
        )


        acc = tl.dot(
            g,
            z,
            acc,

            input_precision=
                "ieee",
        )


    mask = (
        (
            offs_m[:, None]
            <
            M
        )
        &
        (
            offs_n[None, :]
            <
            N
        )
    )


    weight = tl.load(
        src_ptr
        +
        offs_m[:, None]
        *
        stride_sm
        +
        offs_n[None, :]
        *
        stride_sn,

        mask=
            mask,

        other=
            0.0,
    )


    # IMPORTANT:
    #
    # G is a true gradient signal.
    #
    # Therefore gradient descent uses MINUS.
    updated = (
        weight
        -
        learning_rate
        *
        scale
        *
        acc
    )


    tl.store(
        dst_ptr
        +
        offs_m[:, None]
        *
        stride_dm
        +
        offs_n[None, :]
        *
        stride_dn,

        updated,

        mask=
            mask,
    )


# =============================================================================
# A100 CONFIG
# =============================================================================

NORM_BM = 64
NORM_BN = 128
NORM_BK = 64
NORM_WARPS = 4


UPDATE_BM = 64
UPDATE_BN = 64
UPDATE_BK = 64
UPDATE_WARPS = 8


norm_ws = torch.zeros(
    1,

    device=
        DEVICE,

    dtype=
        torch.float32,
)


norm_grid = (
    triton.cdiv(
        M,
        NORM_BM,
    ),

    triton.cdiv(
        N,
        NORM_BN,
    ),
)


update_grid = (
    triton.cdiv(
        M,
        UPDATE_BM,
    ),

    triton.cdiv(
        N,
        UPDATE_BN,
    ),
)


# =============================================================================
# WRAPPERS
# =============================================================================

def compute_triton_gradient_norm():

    norm_ws.zero_()


    real_ttt_norm_kernel[
        norm_grid
    ](
        VT_REAL,
        Z_REAL,

        norm_ws,

        VT_REAL.stride(0),
        VT_REAL.stride(1),

        Z_REAL.stride(0),
        Z_REAL.stride(1),

        M=M,
        N=N,
        K=ADAPT_LEN,

        BLOCK_M=
            NORM_BM,

        BLOCK_N=
            NORM_BN,

        BLOCK_K=
            NORM_BK,

        num_warps=
            NORM_WARPS,
    )


    torch.cuda.synchronize()


    return math.sqrt(
        float(
            norm_ws.item()
        )
    )


def triton_real_ttt_update(
    src,
    dst,

    learning_rate,
    clip_threshold,
):

    norm_ws.zero_()


    real_ttt_norm_kernel[
        norm_grid
    ](
        VT_REAL,
        Z_REAL,

        norm_ws,

        VT_REAL.stride(0),
        VT_REAL.stride(1),

        Z_REAL.stride(0),
        Z_REAL.stride(1),

        M=M,
        N=N,
        K=ADAPT_LEN,

        BLOCK_M=
            NORM_BM,

        BLOCK_N=
            NORM_BN,

        BLOCK_K=
            NORM_BK,

        num_warps=
            NORM_WARPS,
    )


    real_ttt_update_kernel[
        update_grid
    ](
        VT_REAL,
        Z_REAL,

        src,
        dst,

        norm_ws,

        VT_REAL.stride(0),
        VT_REAL.stride(1),

        Z_REAL.stride(0),
        Z_REAL.stride(1),

        src.stride(0),
        src.stride(1),

        dst.stride(0),
        dst.stride(1),

        learning_rate,
        clip_threshold,

        M=M,
        N=N,
        K=ADAPT_LEN,

        BLOCK_M=
            UPDATE_BM,

        BLOCK_N=
            UPDATE_BN,

        BLOCK_K=
            UPDATE_BK,

        num_warps=
            UPDATE_WARPS,
    )


# =============================================================================
# A/B BUFFERS
# =============================================================================

W_A = (
    W_MASTER
    .clone()
    .contiguous()
)


W_B = (
    torch.empty_like(
        W_A
    )
)


# JIT.
triton_real_ttt_update(
    W_A,
    W_B,

    learning_rate=
        1.0,

    clip_threshold=
        1e30,
)


torch.cuda.synchronize()


# =============================================================================
# GRADIENT NORM VALIDATION
# =============================================================================

print()
print("=" * 120)
print("TRUE GRADIENT VALIDATION")
print("=" * 120)


triton_grad_norm = (
    compute_triton_gradient_norm()
)


# PyTorch intentionally materializes full dW here ONLY as a reference.
#
# This is exactly the allocation our Triton path eliminates.

torch.cuda.reset_peak_memory_stats()

baseline_alloc = (
    torch.cuda.memory_allocated()
)


dW_reference = (
    VT_REAL
    @
    Z_REAL
)


torch.cuda.synchronize()


peak_alloc = (
    torch.cuda.max_memory_allocated()
)


pytorch_gradient_temp_mb = (
    peak_alloc
    -
    baseline_alloc
) / 1e6


pytorch_grad_norm = float(
    torch.linalg.vector_norm(
        dW_reference
    ).item()
)


norm_relative_error = (
    abs(
        triton_grad_norm
        -
        pytorch_grad_norm
    )
    /
    max(
        pytorch_grad_norm,
        1e-30,
    )
)


print(
    "PyTorch gradient norm :",
    f"{pytorch_grad_norm:.8e}",
)

print(
    "Triton gradient norm  :",
    f"{triton_grad_norm:.8e}",
)

print(
    "Norm relative error   :",
    f"{norm_relative_error:.8e}",
)

print(
    "PyTorch dW temp       :",
    f"{pytorch_gradient_temp_mb:.2f} MB",
)

print(
    "Triton dW temp        :",
    "0.00 MB",
)


# =============================================================================
# UPDATE CORRECTNESS
#
# Compare the actual update, not base W itself.
# =============================================================================

VALIDATION_DELTA_NORM = (
    1e-3
)


validation_lr = (
    VALIDATION_DELTA_NORM
    /
    max(
        pytorch_grad_norm,
        1e-30,
    )
)


triton_real_ttt_update(
    W_A,
    W_B,

    learning_rate=
        validation_lr,

    clip_threshold=
        1e30,
)


torch.cuda.synchronize()


reference_delta = (
    -validation_lr
    *
    dW_reference
)


triton_delta = (
    W_B
    -
    W_A
)


delta_error = (
    triton_delta
    -
    reference_delta
)


update_relative_l2 = float(
    (
        torch.linalg.vector_norm(
            delta_error
        )
        /
        torch.linalg.vector_norm(
            reference_delta
        ).clamp_min(
            1e-30
        )
    ).item()
)


update_max_abs = float(
    delta_error
    .abs()
    .max()
    .item()
)


print()
print(
    "Validation delta norm :",
    VALIDATION_DELTA_NORM,
)

print(
    "Update relative L2    :",
    f"{update_relative_l2:.8e}",
)

print(
    "Update max abs        :",
    f"{update_max_abs:.8e}",
)


# Reference no longer needed.
del dW_reference
del reference_delta
del triton_delta
del delta_error

gc.collect()

torch.cuda.empty_cache()


# =============================================================================
# FAST-WEIGHT MODEL PATH
# =============================================================================

class FastWeightState:

    def __init__(
        self,
        active,
        staging,
    ):

        self.active = active
        self.staging = staging

        self.version = 0

        self.last_version = None
        self.last_pointer = None


fast_state = FastWeightState(
    W_A,
    W_B,
)


def reset_fast_state():

    W_A.copy_(
        W_MASTER
    )


    torch.cuda.synchronize()


    fast_state.active = W_A

    fast_state.staging = W_B

    fast_state.version = 0

    fast_state.last_version = None

    fast_state.last_pointer = None


def commit_fast_weight():

    fast_state.active, fast_state.staging = (
        fast_state.staging,
        fast_state.active,
    )


    fast_state.version += 1


def fast_weight_forward(
    x,
):

    fast_state.last_version = (
        fast_state.version
    )


    fast_state.last_pointer = (
        fast_state.active.data_ptr()
    )


    # FP32 master fast weight.
    #
    # Baseline and adapted loss below BOTH use this exact path,
    # so the comparison is not mixed with stock-BF16 vs FP32 differences.

    y = F.linear(
        x.float(),
        fast_state.active,
        bias=None,
    )


    return y.to(
        dtype=
            x.dtype
    )


target_layer.mlp.down_proj.forward = (
    fast_weight_forward
)


# =============================================================================
# LOSS HELPER
# =============================================================================

@torch.no_grad()
def model_loss(
    ids,
):

    result = model(
        input_ids=
            ids,

        labels=
            ids,

        use_cache=
            False,
    )


    return float(
        result.loss.item()
    )


# =============================================================================
# BASELINE LOSS UNDER THE EXACT SAME FAST-WEIGHT PATH
# =============================================================================

reset_fast_state()


adapt_loss_before = (
    model_loss(
        adapt_ids
    )
)


heldout_loss_before = (
    model_loss(
        eval_ids
    )
)


print()
print("=" * 120)
print("FAST-WEIGHT PATH BASELINE")
print("=" * 120)


print(
    "Adapt loss before    :",
    f"{adapt_loss_before:.8f}",
)

print(
    "Held-out loss before :",
    f"{heldout_loss_before:.8f}",
)


# =============================================================================
# STEP-SIZE SEARCH
#
# We search directly by desired ||Delta W||_F.
#
# Because:
#
#       lr = desired_delta_norm / ||gradient||
#
# this makes the experiment interpretable across models.
#
# Every candidate is a TRUE gradient descent step.
# =============================================================================

TARGET_DELTA_NORMS = [
    1e-4,
    3e-4,
    1e-3,
    3e-3,
    1e-2,
    3e-2,
]


line_search = []


print()
print("=" * 120)
print("REAL-GRADIENT STEP-SIZE SEARCH")
print("=" * 120)


print(
    f"{'||DeltaW||':>14}"
    f"{'LR':>16}"
    f"{'adapt loss':>16}"
    f"{'delta':>16}"
    f"{'heldout':>16}"
    f"{'heldout Δ':>16}"
)


print(
    "-" * 100
)


for target_delta_norm in TARGET_DELTA_NORMS:

    learning_rate = (
        target_delta_norm
        /
        max(
            triton_grad_norm,
            1e-30,
        )
    )


    # Always update from the pristine W0.
    triton_real_ttt_update(
        W_A,
        W_B,

        learning_rate=
            learning_rate,

        clip_threshold=
            1e30,
    )


    torch.cuda.synchronize()


    fast_state.active = W_B


    candidate_adapt_loss = (
        model_loss(
            adapt_ids
        )
    )


    candidate_heldout_loss = (
        model_loss(
            eval_ids
        )
    )


    result = {

        "target_delta_norm":
            target_delta_norm,

        "learning_rate":
            learning_rate,

        "adapt_loss":
            candidate_adapt_loss,

        "adapt_improvement":
            adapt_loss_before
            -
            candidate_adapt_loss,

        "heldout_loss":
            candidate_heldout_loss,

        "heldout_improvement":
            heldout_loss_before
            -
            candidate_heldout_loss,
    }


    line_search.append(
        result
    )


    print(
        f"{target_delta_norm:>14.3e}"
        f"{learning_rate:>16.3e}"
        f"{candidate_adapt_loss:>16.8f}"
        f"{result['adapt_improvement']:>16.8f}"
        f"{candidate_heldout_loss:>16.8f}"
        f"{result['heldout_improvement']:>16.8f}"
    )


    fast_state.active = W_A


# =============================================================================
# SELECT STEP
#
# Primary criterion:
#     adaptation objective decreases.
#
# Tie-breaker:
#     held-out loss.
# =============================================================================

improving_candidates = [

    x

    for x in line_search

    if x[
        "adapt_loss"
    ]
    <
    adapt_loss_before
]


if improving_candidates:

    best_step = min(
        improving_candidates,

        key=lambda x: (
            x[
                "adapt_loss"
            ],

            x[
                "heldout_loss"
            ],
        ),
    )

else:

    # If something unexpected happens, use smallest stable step
    # rather than inventing a large update.

    best_step = (
        line_search[
            0
        ]
    )


CHOSEN_DELTA_NORM = (
    best_step[
        "target_delta_norm"
    ]
)


CHOSEN_LR = (
    best_step[
        "learning_rate"
    ]
)


print()
print(
    "Chosen ||DeltaW||   :",
    f"{CHOSEN_DELTA_NORM:.3e}",
)

print(
    "Chosen learning rate:",
    f"{CHOSEN_LR:.8e}",
)


# =============================================================================
# FINAL QUALITY / SEMANTIC PROOF
# =============================================================================

reset_fast_state()


# Fixed-input logits before update.
probe_ids = (
    adapt_ids[
        :,
        :128
    ]
)


with torch.no_grad():

    logits_before = (
        model(
            input_ids=
                probe_ids,

            use_cache=
                False,
        )
        .logits[
            :,
            -1,
            :
        ]
        .float()
        .clone()
    )


W0_pointer = (
    fast_state.last_pointer
)


# Real gradient-derived Triton update.
triton_real_ttt_update(
    fast_state.active,
    fast_state.staging,

    learning_rate=
        CHOSEN_LR,

    clip_threshold=
        1e30,
)


torch.cuda.synchronize()


actual_delta_norm = float(
    torch.linalg.vector_norm(
        fast_state.staging
        -
        fast_state.active
    ).item()
)


# O(1) A/B commit.
commit_fast_weight()


W1_expected_pointer = (
    fast_state.active.data_ptr()
)


with torch.no_grad():

    logits_after = (
        model(
            input_ids=
                probe_ids,

            use_cache=
                False,
        )
        .logits[
            :,
            -1,
            :
        ]
        .float()
        .clone()
    )


W1_actual_pointer = (
    fast_state.last_pointer
)


logit_difference = (
    logits_after
    -
    logits_before
)


logit_relative_l2 = float(
    (
        torch.linalg.vector_norm(
            logit_difference
        )
        /
        torch.linalg.vector_norm(
            logits_before
        ).clamp_min(
            1e-30
        )
    ).item()
)


logit_max_abs = float(
    logit_difference
    .abs()
    .max()
    .item()
)


adapt_loss_after = (
    model_loss(
        adapt_ids
    )
)


heldout_loss_after = (
    model_loss(
        eval_ids
    )
)


print()
print("=" * 120)
print("REAL TTT SEMANTIC RESULT")
print("=" * 120)


print(
    "Actual ||DeltaW||     :",
    f"{actual_delta_norm:.8e}",
)

print()
print(
    "W0 pointer            :",
    W0_pointer,
)

print(
    "W1 expected pointer   :",
    W1_expected_pointer,
)

print(
    "W1 actual pointer     :",
    W1_actual_pointer,
)

print(
    "Pointer switch valid  :",
    (
        W0_pointer
        !=
        W1_actual_pointer
        and
        W1_expected_pointer
        ==
        W1_actual_pointer
    ),
)

print()
print(
    "Adapt loss before     :",
    f"{adapt_loss_before:.8f}",
)

print(
    "Adapt loss after      :",
    f"{adapt_loss_after:.8f}",
)

print(
    "Adapt improvement     :",
    f"{adapt_loss_before - adapt_loss_after:.8f}",
)

print()
print(
    "Held-out before       :",
    f"{heldout_loss_before:.8f}",
)

print(
    "Held-out after        :",
    f"{heldout_loss_after:.8f}",
)

print(
    "Held-out improvement  :",
    f"{heldout_loss_before - heldout_loss_after:.8f}",
)

print()
print(
    "Logit relative L2     :",
    f"{logit_relative_l2:.8e}",
)

print(
    "Logit max abs         :",
    f"{logit_max_abs:.8e}",
)


# =============================================================================
# CUDA STREAMS
# =============================================================================

try:

    inference_stream = (
        torch.cuda.Stream(
            priority=-1
        )
    )

except Exception:

    inference_stream = (
        torch.cuda.Stream()
    )


learning_stream = (
    torch.cuda.Stream(
        priority=0
    )
)


torch.cuda.synchronize()


# =============================================================================
# STATS
# =============================================================================

def percentile(
    values,
    p,
):

    values = sorted(
        values
    )


    if not values:
        return 0.0


    position = (
        len(values)
        -
        1
    ) * p


    lo = int(
        math.floor(
            position
        )
    )


    hi = int(
        math.ceil(
            position
        )
    )


    if lo == hi:

        return values[
            lo
        ]


    fraction = (
        position
        -
        lo
    )


    return (
        values[
            lo
        ]
        *
        (
            1.0
            -
            fraction
        )
        +
        values[
            hi
        ]
        *
        fraction
    )


def summarize(
    values,
):

    if not values:
        return None


    return {

        "p50":
            percentile(
                values,
                0.50,
            ),

        "p95":
            percentile(
                values,
                0.95,
            ),

        "p99":
            percentile(
                values,
                0.99,
            ),

        "mean":
            statistics.mean(
                values
            ),

        "min":
            min(
                values
            ),

        "max":
            max(
                values
            ),
    }


# =============================================================================
# REAL ONE-UPDATE SERVING BENCHMARK
#
# baseline:
#
#       W0 token 0 ... W0 token 63
#
# serialized:
#
#       update W0 -> W1
#       wait
#       commit
#       token 0 reads W1
#
# async:
#
#       launch W0 -> W1
#       token 0 reads W0
#       commit at next safe token boundary
#       token 1+ reads W1
#
# This uses the REAL gradient captured from the actual Qwen LM objective.
# =============================================================================

MODES = [
    "baseline",
    "serialized",
    "async",
]


def run_serving(
    mode,
):

    if mode not in MODES:

        raise ValueError(
            mode
        )


    reset_fast_state()


    prompt = (
        adapt_ids
        .contiguous()
    )


    token_latencies = []

    version_trace = []

    pointer_trace = []


    update_latency_ms = None

    visibility_lag = None

    update_in_flight = False

    update_start = None

    update_end = None


    torch.cuda.synchronize()


    # =========================================================================
    # PREFILL
    # =========================================================================

    with torch.cuda.stream(
        inference_stream
    ):

        with torch.no_grad():

            output = model(
                input_ids=
                    prompt,

                use_cache=
                    True,
            )


        past_key_values = (
            output.past_key_values
        )


        next_token = (
            output
            .logits[
                :,
                -1,
                :
            ]
            .argmax(
                dim=-1,

                keepdim=
                    True,
            )
        )


    inference_stream.synchronize()


    foreground_start = (
        time.perf_counter()
    )


    # =========================================================================
    # TOKENS
    # =========================================================================

    for token_idx in range(
        DECODE_TOKENS
    ):

        # ---------------------------------------------------------------------
        # Async update becomes visible only at a token boundary.
        # ---------------------------------------------------------------------

        if (
            mode
            ==
            "async"
            and
            update_in_flight
            and
            update_end.query()
        ):

            update_end.synchronize()


            update_latency_ms = (
                update_start.elapsed_time(
                    update_end
                )
            )


            commit_fast_weight()


            visibility_lag = (
                token_idx
            )


            update_in_flight = False


        token_wall_start = (
            time.perf_counter()
        )


        # =====================================================================
        # SERIALIZED REAL TTT
        # =====================================================================

        if (
            mode
            ==
            "serialized"
            and
            token_idx
            ==
            0
        ):

            serial_start = (
                torch.cuda.Event(
                    enable_timing=True
                )
            )


            serial_end = (
                torch.cuda.Event(
                    enable_timing=True
                )
            )


            with torch.cuda.stream(
                inference_stream
            ):

                serial_start.record(
                    inference_stream
                )


                triton_real_ttt_update(
                    fast_state.active,
                    fast_state.staging,

                    learning_rate=
                        CHOSEN_LR,

                    clip_threshold=
                        1e30,
                )


                serial_end.record(
                    inference_stream
                )


            serial_end.synchronize()


            update_latency_ms = (
                serial_start.elapsed_time(
                    serial_end
                )
            )


            commit_fast_weight()


            visibility_lag = 0


        # =====================================================================
        # ASYNC REAL TTT
        # =====================================================================

        if (
            mode
            ==
            "async"
            and
            token_idx
            ==
            0
        ):

            update_start = (
                torch.cuda.Event(
                    enable_timing=True
                )
            )


            update_end = (
                torch.cuda.Event(
                    enable_timing=True
                )
            )


            with torch.cuda.stream(
                learning_stream
            ):

                update_start.record(
                    learning_stream
                )


                triton_real_ttt_update(
                    fast_state.active,
                    fast_state.staging,

                    learning_rate=
                        CHOSEN_LR,

                    clip_threshold=
                        1e30,
                )


                update_end.record(
                    learning_stream
                )


            update_in_flight = True


        # =====================================================================
        # REAL QWEN INFERENCE
        # =====================================================================

        fast_state.last_version = None
        fast_state.last_pointer = None


        with torch.cuda.stream(
            inference_stream
        ):

            with torch.no_grad():

                output = model(
                    input_ids=
                        next_token,

                    past_key_values=
                        past_key_values,

                    use_cache=
                        True,
                )


            past_key_values = (
                output.past_key_values
            )


            next_token = (
                output
                .logits[
                    :,
                    -1,
                    :
                ]
                .argmax(
                    dim=-1,

                    keepdim=
                        True,
                )
            )


        # Only wait for inference.
        inference_stream.synchronize()


        _ = int(
            next_token.item()
        )


        token_latency_ms = (
            time.perf_counter()
            -
            token_wall_start
        ) * 1000.0


        token_latencies.append(
            token_latency_ms
        )


        version_trace.append(
            int(
                fast_state.last_version
            )
        )


        pointer_trace.append(
            int(
                fast_state.last_pointer
            )
        )


    foreground_ms = (
        time.perf_counter()
        -
        foreground_start
    ) * 1000.0


    # =========================================================================
    # FINISH OUTSTANDING ASYNC UPDATE
    # =========================================================================

    if update_in_flight:

        update_end.synchronize()


        update_latency_ms = (
            update_start.elapsed_time(
                update_end
            )
        )


        commit_fast_weight()


        visibility_lag = (
            DECODE_TOKENS
        )


    torch.cuda.synchronize()


    result = {

        "mode":
            mode,

        "itl":
            summarize(
                token_latencies
            ),

        "foreground_ms":
            foreground_ms,

        "tokens_per_second":
            DECODE_TOKENS
            /
            (
                foreground_ms
                /
                1000.0
            ),

        "update_latency_ms":
            update_latency_ms,

        "visibility_lag_tokens":
            visibility_lag,

        "version_trace":
            version_trace,

        "pointer_trace":
            pointer_trace,
    }


    del output
    del past_key_values
    del next_token


    gc.collect()

    torch.cuda.empty_cache()


    return result


# =============================================================================
# WARMUP
# =============================================================================

print()
print("=" * 120)
print("SERVING WARMUP")
print("=" * 120)


warmup = run_serving(
    "baseline"
)


print(
    "Baseline warmup p50:",
    f"{warmup['itl']['p50']:.3f} ms",
)


# =============================================================================
# BENCHMARK
# =============================================================================

RUNS = 3


results = {
    mode: []
    for mode in MODES
}


print()
print("=" * 145)
print("REAL-GRADIENT FAST-WEIGHT SERVING")
print("=" * 145)


for mode in MODES:

    print()
    print("-" * 100)
    print(
        mode.upper()
    )
    print("-" * 100)


    for run_idx in range(
        RUNS
    ):

        result = run_serving(
            mode
        )


        results[
            mode
        ].append(
            result
        )


        print(
            f"run={run_idx + 1} | "
            f"p50={result['itl']['p50']:.3f} ms | "
            f"p95={result['itl']['p95']:.3f} ms | "
            f"p99={result['itl']['p99']:.3f} ms | "
            f"tok/s={result['tokens_per_second']:.2f} | "
            f"TTT={result['update_latency_ms'] if result['update_latency_ms'] is not None else '-'} | "
            f"lag={result['visibility_lag_tokens']}"
        )


# =============================================================================
# AGGREGATE
# =============================================================================

aggregate = {}


for mode in MODES:

    aggregate[
        mode
    ] = {

        "p50_ms":
            statistics.median(
                [
                    x[
                        "itl"
                    ][
                        "p50"
                    ]
                    for x in results[
                        mode
                    ]
                ]
            ),

        "p95_ms":
            statistics.median(
                [
                    x[
                        "itl"
                    ][
                        "p95"
                    ]
                    for x in results[
                        mode
                    ]
                ]
            ),

        "p99_ms":
            statistics.median(
                [
                    x[
                        "itl"
                    ][
                        "p99"
                    ]
                    for x in results[
                        mode
                    ]
                ]
            ),

        "tok_s":
            statistics.median(
                [
                    x[
                        "tokens_per_second"
                    ]
                    for x in results[
                        mode
                    ]
                ]
            ),
    }


    update_values = [
        x[
            "update_latency_ms"
        ]

        for x in results[
            mode
        ]

        if x[
            "update_latency_ms"
        ]
        is not None
    ]


    aggregate[
        mode
    ][
        "update_ms"
    ] = (
        statistics.median(
            update_values
        )
        if update_values
        else None
    )


    lag_values = [
        x[
            "visibility_lag_tokens"
        ]

        for x in results[
            mode
        ]

        if x[
            "visibility_lag_tokens"
        ]
        is not None
    ]


    aggregate[
        mode
    ][
        "lag"
    ] = (
        statistics.median(
            lag_values
        )
        if lag_values
        else None
    )


# =============================================================================
# SERIALIZED VS ASYNC EXPOSURE
# =============================================================================

baseline_p50 = (
    aggregate[
        "baseline"
    ][
        "p50_ms"
    ]
)


serialized_p50 = (
    aggregate[
        "serialized"
    ][
        "p50_ms"
    ]
)


async_p50 = (
    aggregate[
        "async"
    ][
        "p50_ms"
    ]
)


serialized_exposed = (
    results[
        "serialized"
    ][
        0
    ][
        "itl"
    ][
        "max"
    ]
    -
    aggregate[
        "baseline"
    ][
        "p50_ms"
    ]
)


async_exposed = (
    results[
        "async"
    ][
        0
    ][
        "itl"
    ][
        "max"
    ]
    -
    aggregate[
        "baseline"
    ][
        "p50_ms"
    ]
)


async_update_ms = (
    aggregate[
        "async"
    ][
        "update_ms"
    ]
)


hidden_fraction = None


if (
    async_update_ms
    is not None
    and
    async_update_ms
    >
    0
):

    hidden_fraction = (
        1.0
        -
        max(
            0.0,
            async_exposed,
        )
        /
        async_update_ms
    )


# =============================================================================
# FINAL TABLE
# =============================================================================

print()
print("=" * 145)
print("FINAL REAL-GRADIENT ASYNC-TTT RESULTS")
print("=" * 145)


print(
    f"{'Mode':<18}"
    f"{'p50':>12}"
    f"{'p95':>12}"
    f"{'p99':>12}"
    f"{'tok/s':>12}"
    f"{'update ms':>14}"
    f"{'lag':>10}"
)


print("-" * 100)


for mode in MODES:

    row = aggregate[
        mode
    ]


    update_text = (
        "-"
        if row[
            "update_ms"
        ]
        is None
        else
        f"{row['update_ms']:.3f}"
    )


    lag_text = (
        "-"
        if row[
            "lag"
        ]
        is None
        else
        f"{row['lag']:.1f}"
    )


    print(
        f"{mode:<18}"
        f"{row['p50_ms']:>10.3f}ms"
        f"{row['p95_ms']:>10.3f}ms"
        f"{row['p99_ms']:>10.3f}ms"
        f"{row['tok_s']:>12.2f}"
        f"{update_text:>14}"
        f"{lag_text:>10}"
    )


print()
print("=" * 120)
print("ASYNC REAL-GRADIENT SUMMARY")
print("=" * 120)


print(
    "Real LM gradient norm        :",
    f"{triton_grad_norm:.8e}",
)

print(
    "Chosen ||DeltaW||            :",
    f"{CHOSEN_DELTA_NORM:.8e}",
)

print(
    "Adapt loss improvement       :",
    f"{adapt_loss_before - adapt_loss_after:.8f}",
)

print(
    "Held-out loss improvement    :",
    f"{heldout_loss_before - heldout_loss_after:.8f}",
)

print(
    "Serialized TTT latency       :",
    f"{aggregate['serialized']['update_ms']:.3f} ms",
)

print(
    "Async TTT latency            :",
    f"{aggregate['async']['update_ms']:.3f} ms",
)

print(
    "Async visibility lag         :",
    f"{aggregate['async']['lag']:.1f} token(s)",
)


if hidden_fraction is not None:

    print(
        "Approx async hidden fraction:",
        f"{hidden_fraction * 100:.2f}%",
    )


# =============================================================================
# VERSION TRACE
# =============================================================================

print()
print("=" * 120)
print("ASYNC VERSION TRACE")
print("=" * 120)


example = (
    results[
        "async"
    ][
        0
    ]
)


for i in range(
    min(
        12,
        DECODE_TOKENS,
    )
):

    print(
        f"token {i:>2} | "
        f"version={example['version_trace'][i]} | "
        f"ptr={example['pointer_trace'][i]}"
    )


# =============================================================================
# SAVE
# =============================================================================

RESULT_PATH = Path(
    "async_ttt_qwen7b_real_gradient.json"
)


result_json = {

    "model":
        MODEL_ID,

    "shape": {

        "M":
            M,

        "N":
            N,

        "K":
            ADAPT_LEN,

        "layer":
            TTT_LAYER,
    },

    "gradient": {

        "pytorch_norm":
            pytorch_grad_norm,

        "triton_norm":
            triton_grad_norm,

        "norm_relative_error":
            norm_relative_error,

        "pytorch_materialized_gradient_mb":
            pytorch_gradient_temp_mb,

        "triton_materialized_gradient_mb":
            0.0,

        "update_relative_l2":
            update_relative_l2,

        "update_max_abs":
            update_max_abs,
    },

    "step_search":
        line_search,

    "chosen_step": {

        "target_delta_norm":
            CHOSEN_DELTA_NORM,

        "learning_rate":
            CHOSEN_LR,
    },

    "quality": {

        "adapt_loss_before":
            adapt_loss_before,

        "adapt_loss_after":
            adapt_loss_after,

        "adapt_improvement":
            adapt_loss_before
            -
            adapt_loss_after,

        "heldout_loss_before":
            heldout_loss_before,

        "heldout_loss_after":
            heldout_loss_after,

        "heldout_improvement":
            heldout_loss_before
            -
            heldout_loss_after,

        "logit_relative_l2":
            logit_relative_l2,

        "logit_max_abs":
            logit_max_abs,
    },

    "serving":
        aggregate,

    "raw_serving":
        results,
}


with open(
    RESULT_PATH,
    "w",
) as file:

    json.dump(
        result_json,
        file,
        indent=2,
    )


print()
print(
    "Saved:",
    RESULT_PATH.resolve(),
)


# =============================================================================
# RESTORE ORIGINAL QWEN MODULE
# =============================================================================

target_layer.mlp.down_proj.forward = (
    original_down_forward
)


torch.cuda.synchronize()


print(
    "Original down_proj.forward restored."
)


free_bytes, _ = (
    torch.cuda.mem_get_info()
)


print(
    "Free VRAM:",
    f"{free_bytes / 1024**3:.2f} GiB",
)

FLUSHING PREVIOUS GPU STATE
Free VRAM : 38.53 GiB
Total VRAM: 39.49 GiB

QWEN2.5-7B — REAL GRADIENT ASYNC-TTT
Model       : Qwen/Qwen2.5-7B-Instruct
Python      : 3.12.11
PyTorch     : 2.8.0+cu128
CUDA        : 12.8
Triton      : 3.4.0
GPU         : NVIDIA A100-SXM4-40GB

LOADING QWEN2.5-7B


Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

hidden_size         : 3584
intermediate_size   : 18944
layers              : 28
TTT layer           : 13
FP32 fast weight    : 271.58 MB
Free after load     : 24.32 GiB
Adapt tokens        : 512
Held-out tokens     : 512

CAPTURING REAL LM GRADIENT SIGNAL
LM loss before backward: 0.5967137217521667

Z = down_proj input  : (512, 18944)
G = dLoss/dOutput    : (512, 3584)
G^T                  : (3584, 512)
W_down               : (3584, 18944)
Gradient capture time: 137.340 ms

TRUE GRADIENT VALIDATION
PyTorch gradient norm : 3.26891303e-01
Triton gradient norm  : 3.26891100e-01
Norm relative error   : 6.19864623e-07
PyTorch dW temp       : 272.63 MB
Triton dW temp        : 0.00 MB

Validation delta norm : 0.001
Update relative L2    : 3.38084111e-03
Update max abs        : 2.94205620e-08

FAST-WEIGHT PATH BASELINE
Adapt loss before    : 0.59655070
Held-out loss before : 0.58373791

REAL-GRADIENT STEP-SIZE SEARCH
    ||DeltaW||              LR      adapt loss           delta         heldou

In [12]:
# =============================================================================
# ASYNC-TTT — QWEN2.5-7B
# FP32 MASTER FAST WEIGHTS + BF16 SERVING FAST WEIGHTS
#
# Architecture:
#
#        REAL LM GRADIENT
#              |
#              v
#     FP32 MASTER ACTIVE
#              |
#       Triton TTT update
#              |
#              v
#     FP32 MASTER STAGING
#              |
#      FP32 -> BF16 publish
#              |
#              v
#     BF16 SERVING STAGING
#              |
#        O(1) pointer swap
#              |
#              v
#     BF16 SERVING ACTIVE
#              |
#              v
#        REAL QWEN DECODE
#
#
# This removes the previous serving confound:
#
#     F.linear(x.float(), FP32_weight)
#
# Qwen now executes:
#
#     BF16 activations x BF16 fast weight
#
# exactly like the original nn.Linear projection.
#
#
# TESTS:
#   1. Flush previous GPU state.
#   2. Load Qwen2.5-7B.
#   3. Capture REAL LM gradient:
#          dW = G^T @ Z
#   4. Validate Triton vs PyTorch.
#   5. FP32 master A/B update.
#   6. Preallocated BF16 serving A/B buffers.
#   7. Measure FP32 -> BF16 publication latency.
#   8. Verify stock-Qwen vs BF16 fast-weight path parity.
#   9. Verify BF16 publication actually changes weights.
#  10. Verify loss before/after real TTT.
#  11. Compare:
#          stock Qwen
#          BF16 fast-weight baseline
#          serialized real TTT
#          async real TTT
#  12. Measure:
#          p50 / p95 / p99
#          throughput
#          TTT compute time
#          BF16 publish time
#          total update time
#          exposed async overhead
#          hidden fraction
#          visibility lag
#          pointer/version trace
#
# NOTE:
#   Full Delta-W is NOT stored in HBM by Triton.
#   Nsight Compute is still required before making exact shared-memory/SRAM
#   traffic or residency claims.
# =============================================================================


# =============================================================================
# IMPORTS
# =============================================================================

import gc
import json
import math
import statistics
import sys
import time

from pathlib import Path

import torch
import torch.nn.functional as F

import triton
import triton.language as tl

from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
)


# =============================================================================
# FLUSH PREVIOUS GPU STATE
# =============================================================================

print("=" * 120)
print("FLUSHING PREVIOUS GPU STATE")
print("=" * 120)


_cleanup_names = [

    "model",
    "tokenizer",
    "target_layer",

    "W_MASTER",
    "W_A",
    "W_B",

    "MASTER_A",
    "MASTER_B",

    "SERVE_A",
    "SERVE_B",

    "Z_REAL",
    "G_REAL",
    "GT_REAL",

    "VT_REAL",

    "capture",
    "captured",

    "output",
    "out",

    "past_key_values",
    "next_token",

    "fast_state",
    "state",

    "results",
    "runs",

]


for _name in _cleanup_names:

    if _name in globals():

        try:

            del globals()[
                _name
            ]

        except Exception:

            pass


gc.collect()


if torch.cuda.is_available():

    torch.cuda.synchronize()

    torch.cuda.empty_cache()

    try:

        torch.cuda.ipc_collect()

    except Exception:

        pass

    gc.collect()

    torch.cuda.empty_cache()


free_bytes, total_bytes = (
    torch.cuda.mem_get_info()
)


print(
    "Free VRAM :",
    f"{free_bytes / 1024**3:.2f} GiB",
)

print(
    "Total VRAM:",
    f"{total_bytes / 1024**3:.2f} GiB",
)


if (
    free_bytes
    /
    1024**3
    <
    30
):

    print()
    print(
        "WARNING: less than 30 GiB free."
    )

    print(
        "Restart the kernel if an older model is still referenced."
    )


# =============================================================================
# CONFIG
# =============================================================================

MODEL_ID = (
    "Qwen/Qwen2.5-7B-Instruct"
)


DEVICE = "cuda"


TTT_LAYER = 13


ADAPT_LEN = 512

EVAL_LEN = 512


TOTAL_LEN = (
    ADAPT_LEN
    +
    EVAL_LEN
)


DECODE_TOKENS = 64


RUNS = 5


# Based on the previous real-gradient experiment.
TARGET_DELTA_NORM = (
    3e-2
)


torch.manual_seed(
    0
)


torch.backends.cuda.matmul.allow_tf32 = False

torch.backends.cudnn.allow_tf32 = False


# =============================================================================
# ENVIRONMENT
# =============================================================================

assert torch.cuda.is_available()


props = (
    torch.cuda.get_device_properties(
        0
    )
)


print()
print("=" * 120)
print("ASYNC-TTT — QWEN2.5-7B BF16 SERVING")
print("=" * 120)


print(
    "Model                  :",
    MODEL_ID,
)

print(
    "Python                 :",
    sys.version.split()[0],
)

print(
    "PyTorch                :",
    torch.__version__,
)

print(
    "CUDA                   :",
    torch.version.cuda,
)

print(
    "Triton                 :",
    triton.__version__,
)

print(
    "GPU                    :",
    torch.cuda.get_device_name(0),
)

print(
    "SMs                    :",
    props.multi_processor_count,
)

print(
    "VRAM                   :",
    f"{props.total_memory / 1024**3:.2f} GiB",
)


# =============================================================================
# LOAD MODEL
# =============================================================================

print()
print("=" * 120)
print("LOADING QWEN2.5-7B")
print("=" * 120)


tokenizer = (
    AutoTokenizer.from_pretrained(
        MODEL_ID,
        trust_remote_code=True,
    )
)


model = (
    AutoModelForCausalLM.from_pretrained(

        MODEL_ID,

        dtype=
            torch.bfloat16,

        device_map={
            "": 0
        },

        low_cpu_mem_usage=True,

        trust_remote_code=True,

        attn_implementation=
            "sdpa",
    )
)


model.eval()


# We only need a gradient with respect to the target down_proj output.
#
# Freeze all model parameters.

for parameter in model.parameters():

    parameter.requires_grad_(
        False
    )


M = int(
    model.config.hidden_size
)


N = int(
    model.config.intermediate_size
)


NUM_LAYERS = int(
    model.config.num_hidden_layers
)


target_layer = (
    model.model.layers[
        TTT_LAYER
    ]
)


print(
    "hidden_size            :",
    M,
)

print(
    "intermediate_size      :",
    N,
)

print(
    "layers                 :",
    NUM_LAYERS,
)

print(
    "TTT layer              :",
    TTT_LAYER,
)

print(
    "model dtype            :",
    next(
        model.parameters()
    ).dtype,
)


FP32_WEIGHT_MB = (
    M
    *
    N
    *
    4
    /
    1e6
)


BF16_WEIGHT_MB = (
    M
    *
    N
    *
    2
    /
    1e6
)


print(
    "FP32 fast weight       :",
    f"{FP32_WEIGHT_MB:.2f} MB",
)

print(
    "BF16 serving weight    :",
    f"{BF16_WEIGHT_MB:.2f} MB",
)


free_bytes, _ = (
    torch.cuda.mem_get_info()
)


print(
    "Free after load        :",
    f"{free_bytes / 1024**3:.2f} GiB",
)


# =============================================================================
# REAL TOKEN DATA
# =============================================================================

base_text = """
Test-time adaptation allows a language model to modify an internal learned
state using recently observed information. A production inference runtime must
perform this learning concurrently with autoregressive generation while
minimizing synchronization, temporary memory allocation, GPU memory traffic,
tail latency and update staleness.
"""


text = "\n".join(
    [base_text] * 2500
)


tokenized = (
    tokenizer(

        text,

        return_tensors=
            "pt",

        truncation=
            True,

        max_length=
            TOTAL_LEN,
    )
)


all_ids = (
    tokenized[
        "input_ids"
    ]
    .to(
        DEVICE
    )
)


if (
    all_ids.shape[1]
    <
    TOTAL_LEN
):

    raise RuntimeError(
        f"Need {TOTAL_LEN} tokens but got "
        f"{all_ids.shape[1]}"
    )


adapt_ids = (
    all_ids[
        :,
        :ADAPT_LEN
    ]
    .contiguous()
)


eval_ids = (
    all_ids[
        :,
        ADAPT_LEN:
        ADAPT_LEN + EVAL_LEN
    ]
    .contiguous()
)


print(
    "Adapt tokens           :",
    adapt_ids.shape[1],
)

print(
    "Held-out tokens        :",
    eval_ids.shape[1],
)


del tokenized
del text

gc.collect()


# =============================================================================
# CAPTURE REAL GRADIENT
#
# down_proj:
#
#       Y = Z W^T
#
# therefore:
#
#       dW = G^T Z
#
# where:
#
#       Z = input to down_proj
#       G = dLoss / dY
#
# =============================================================================

print()
print("=" * 120)
print("CAPTURING REAL LM GRADIENT")
print("=" * 120)


original_down_forward = (
    target_layer
    .mlp
    .down_proj
    .forward
)


capture = {}


def capture_gradient_forward(
    x,
):

    # Real down_proj input.
    capture[
        "z"
    ] = (
        x
        .detach()
        .float()
        .contiguous()
    )


    # Original BF16 Qwen projection.
    y = original_down_forward(
        x
    )


    # Cut graph here.
    #
    # The lower model does not participate in backward.
    y_leaf = (
        y
        .detach()
        .requires_grad_(
            True
        )
    )


    y_leaf.retain_grad()


    capture[
        "y"
    ] = y_leaf


    return y_leaf


target_layer.mlp.down_proj.forward = (
    capture_gradient_forward
)


model.zero_grad(
    set_to_none=True
)


gradient_start = (
    time.perf_counter()
)


adapt_output = model(

    input_ids=
        adapt_ids,

    labels=
        adapt_ids,

    use_cache=
        False,
)


original_adapt_loss = float(
    adapt_output
    .loss
    .detach()
    .item()
)


adapt_output.loss.backward()


torch.cuda.synchronize()


gradient_capture_ms = (
    time.perf_counter()
    -
    gradient_start
) * 1000.0


if (
    capture[
        "y"
    ].grad
    is None
):

    raise RuntimeError(
        "Target layer gradient was not captured."
    )


Z_REAL = (
    capture[
        "z"
    ][
        0
    ]
    .float()
    .contiguous()
)


G_REAL = (
    capture[
        "y"
    ]
    .grad[
        0
    ]
    .float()
    .contiguous()
)


GT_REAL = (
    G_REAL
    .T
    .contiguous()
)


assert Z_REAL.shape == (
    ADAPT_LEN,
    N,
)


assert G_REAL.shape == (
    ADAPT_LEN,
    M,
)


assert GT_REAL.shape == (
    M,
    ADAPT_LEN,
)


print(
    "Original LM loss      :",
    f"{original_adapt_loss:.8f}",
)

print(
    "Z                     :",
    tuple(
        Z_REAL.shape
    ),
)

print(
    "G                     :",
    tuple(
        G_REAL.shape
    ),
)

print(
    "G^T                   :",
    tuple(
        GT_REAL.shape
    ),
)

print(
    "Gradient capture      :",
    f"{gradient_capture_ms:.3f} ms",
)


# Restore original projection.
target_layer.mlp.down_proj.forward = (
    original_down_forward
)


del adapt_output
del capture


gc.collect()

torch.cuda.empty_cache()


# =============================================================================
# FAST-WEIGHT BUFFERS
#
# SERVE_A reuses the ORIGINAL Qwen BF16 parameter storage.
#
# This means the baseline does NOT require another active BF16 copy.
# =============================================================================

ORIGINAL_BF16_WEIGHT = (
    target_layer
    .mlp
    .down_proj
    .weight
    .detach()
)


assert ORIGINAL_BF16_WEIGHT.dtype == (
    torch.bfloat16
)


# FP32 master state.
MASTER_A = (
    ORIGINAL_BF16_WEIGHT
    .float()
    .contiguous()
)


MASTER_B = (
    torch.empty_like(
        MASTER_A
    )
)


# BF16 serving state.
#
# A = original model parameter.
# B = one preallocated publication buffer.

SERVE_A = (
    ORIGINAL_BF16_WEIGHT
)


SERVE_B = (
    torch.empty_like(
        SERVE_A
    )
)


print()
print("=" * 120)
print("FAST-WEIGHT MEMORY")
print("=" * 120)


print(
    "FP32 master A         :",
    f"{MASTER_A.numel() * MASTER_A.element_size() / 1e6:.2f} MB",
)

print(
    "FP32 master B         :",
    f"{MASTER_B.numel() * MASTER_B.element_size() / 1e6:.2f} MB",
)

print(
    "BF16 serving A        :",
    f"{SERVE_A.numel() * SERVE_A.element_size() / 1e6:.2f} MB",
    "(reuses model weight)",
)

print(
    "BF16 serving B        :",
    f"{SERVE_B.numel() * SERVE_B.element_size() / 1e6:.2f} MB",
)


# =============================================================================
# TRITON PASS 1
# =============================================================================

@triton.jit
def gradient_norm_kernel(
    gt_ptr,
    z_ptr,

    norm_ptr,

    stride_gm,
    stride_gk,

    stride_zk,
    stride_zn,

    M: tl.constexpr,
    N: tl.constexpr,
    K: tl.constexpr,

    BLOCK_M: tl.constexpr,
    BLOCK_N: tl.constexpr,
    BLOCK_K: tl.constexpr,
):

    pid_m = (
        tl.program_id(0)
    )


    pid_n = (
        tl.program_id(1)
    )


    offs_m = (
        pid_m
        *
        BLOCK_M
        +
        tl.arange(
            0,
            BLOCK_M,
        )
    )


    offs_n = (
        pid_n
        *
        BLOCK_N
        +
        tl.arange(
            0,
            BLOCK_N,
        )
    )


    acc = tl.zeros(

        (
            BLOCK_M,
            BLOCK_N,
        ),

        dtype=
            tl.float32,
    )


    for k in range(
        0,
        K,
        BLOCK_K,
    ):

        offs_k = (
            k
            +
            tl.arange(
                0,
                BLOCK_K,
            )
        )


        g = tl.load(

            gt_ptr
            +
            offs_m[:, None]
            *
            stride_gm
            +
            offs_k[None, :]
            *
            stride_gk,

            mask=(
                (
                    offs_m[:, None]
                    <
                    M
                )
                &
                (
                    offs_k[None, :]
                    <
                    K
                )
            ),

            other=
                0.0,
        )


        z = tl.load(

            z_ptr
            +
            offs_k[:, None]
            *
            stride_zk
            +
            offs_n[None, :]
            *
            stride_zn,

            mask=(
                (
                    offs_k[:, None]
                    <
                    K
                )
                &
                (
                    offs_n[None, :]
                    <
                    N
                )
            ),

            other=
                0.0,
        )


        acc = tl.dot(

            g,
            z,
            acc,

            input_precision=
                "ieee",
        )


    local_sq = (
        tl.sum(
            acc
            *
            acc
        )
    )


    tl.atomic_add(
        norm_ptr,
        local_sq,
    )


# =============================================================================
# TRITON PASS 2
# =============================================================================

@triton.jit
def gradient_update_kernel(
    gt_ptr,
    z_ptr,

    src_ptr,
    dst_ptr,

    norm_ptr,

    stride_gm,
    stride_gk,

    stride_zk,
    stride_zn,

    stride_sm,
    stride_sn,

    stride_dm,
    stride_dn,

    learning_rate,
    clip_threshold,

    M: tl.constexpr,
    N: tl.constexpr,
    K: tl.constexpr,

    BLOCK_M: tl.constexpr,
    BLOCK_N: tl.constexpr,
    BLOCK_K: tl.constexpr,
):

    pid_m = (
        tl.program_id(0)
    )


    pid_n = (
        tl.program_id(1)
    )


    offs_m = (
        pid_m
        *
        BLOCK_M
        +
        tl.arange(
            0,
            BLOCK_M,
        )
    )


    offs_n = (
        pid_n
        *
        BLOCK_N
        +
        tl.arange(
            0,
            BLOCK_N,
        )
    )


    norm_sq = (
        tl.load(
            norm_ptr
        )
    )


    scale = (
        clip_threshold
        /
        (
            tl.sqrt(
                norm_sq
            )
            +
            1e-12
        )
    )


    scale = (
        tl.minimum(
            scale,
            1.0,
        )
    )


    acc = tl.zeros(

        (
            BLOCK_M,
            BLOCK_N,
        ),

        dtype=
            tl.float32,
    )


    for k in range(
        0,
        K,
        BLOCK_K,
    ):

        offs_k = (
            k
            +
            tl.arange(
                0,
                BLOCK_K,
            )
        )


        g = tl.load(

            gt_ptr
            +
            offs_m[:, None]
            *
            stride_gm
            +
            offs_k[None, :]
            *
            stride_gk,

            mask=(
                (
                    offs_m[:, None]
                    <
                    M
                )
                &
                (
                    offs_k[None, :]
                    <
                    K
                )
            ),

            other=
                0.0,
        )


        z = tl.load(

            z_ptr
            +
            offs_k[:, None]
            *
            stride_zk
            +
            offs_n[None, :]
            *
            stride_zn,

            mask=(
                (
                    offs_k[:, None]
                    <
                    K
                )
                &
                (
                    offs_n[None, :]
                    <
                    N
                )
            ),

            other=
                0.0,
        )


        acc = tl.dot(

            g,
            z,
            acc,

            input_precision=
                "ieee",
        )


    mask = (
        (
            offs_m[:, None]
            <
            M
        )
        &
        (
            offs_n[None, :]
            <
            N
        )
    )


    weight = tl.load(

        src_ptr
        +
        offs_m[:, None]
        *
        stride_sm
        +
        offs_n[None, :]
        *
        stride_sn,

        mask=
            mask,

        other=
            0.0,
    )


    # True gradient descent.
    updated = (
        weight
        -
        learning_rate
        *
        scale
        *
        acc
    )


    tl.store(

        dst_ptr
        +
        offs_m[:, None]
        *
        stride_dm
        +
        offs_n[None, :]
        *
        stride_dn,

        updated,

        mask=
            mask,
    )


# =============================================================================
# KERNEL CONFIG
# =============================================================================

NORM_BM = 64

NORM_BN = 128

NORM_BK = 64

NORM_WARPS = 4


UPDATE_BM = 64

UPDATE_BN = 64

UPDATE_BK = 64

UPDATE_WARPS = 8


norm_ws = torch.zeros(

    1,

    device=
        DEVICE,

    dtype=
        torch.float32,
)


norm_grid = (

    triton.cdiv(
        M,
        NORM_BM,
    ),

    triton.cdiv(
        N,
        NORM_BN,
    ),
)


update_grid = (

    triton.cdiv(
        M,
        UPDATE_BM,
    ),

    triton.cdiv(
        N,
        UPDATE_BN,
    ),
)


# =============================================================================
# TRITON UPDATE WRAPPER
# =============================================================================

def triton_master_update(
    src,
    dst,

    learning_rate,
    clip_threshold,
):

    norm_ws.zero_()


    gradient_norm_kernel[
        norm_grid
    ](

        GT_REAL,
        Z_REAL,

        norm_ws,

        GT_REAL.stride(0),
        GT_REAL.stride(1),

        Z_REAL.stride(0),
        Z_REAL.stride(1),

        M=M,
        N=N,
        K=ADAPT_LEN,

        BLOCK_M=
            NORM_BM,

        BLOCK_N=
            NORM_BN,

        BLOCK_K=
            NORM_BK,

        num_warps=
            NORM_WARPS,
    )


    gradient_update_kernel[
        update_grid
    ](

        GT_REAL,
        Z_REAL,

        src,
        dst,

        norm_ws,

        GT_REAL.stride(0),
        GT_REAL.stride(1),

        Z_REAL.stride(0),
        Z_REAL.stride(1),

        src.stride(0),
        src.stride(1),

        dst.stride(0),
        dst.stride(1),

        learning_rate,
        clip_threshold,

        M=M,
        N=N,
        K=ADAPT_LEN,

        BLOCK_M=
            UPDATE_BM,

        BLOCK_N=
            UPDATE_BN,

        BLOCK_K=
            UPDATE_BK,

        num_warps=
            UPDATE_WARPS,
    )


# =============================================================================
# JIT COMPILE
# =============================================================================

triton_master_update(

    MASTER_A,
    MASTER_B,

    learning_rate=
        1.0,

    clip_threshold=
        1e30,
)


torch.cuda.synchronize()


# =============================================================================
# REAL GRADIENT VALIDATION
# =============================================================================

print()
print("=" * 120)
print("REAL GRADIENT VALIDATION")
print("=" * 120)


# Materialized PyTorch dW only for reference.

torch.cuda.empty_cache()

torch.cuda.reset_peak_memory_stats()


baseline_memory = (
    torch.cuda.memory_allocated()
)


dW_reference = (
    GT_REAL
    @
    Z_REAL
)


torch.cuda.synchronize()


peak_memory = (
    torch.cuda.max_memory_allocated()
)


pytorch_temp_mb = (
    peak_memory
    -
    baseline_memory
) / 1e6


gradient_norm = float(

    torch.linalg.vector_norm(
        dW_reference
    )
    .item()
)


norm_ws.zero_()


gradient_norm_kernel[
    norm_grid
](

    GT_REAL,
    Z_REAL,

    norm_ws,

    GT_REAL.stride(0),
    GT_REAL.stride(1),

    Z_REAL.stride(0),
    Z_REAL.stride(1),

    M=M,
    N=N,
    K=ADAPT_LEN,

    BLOCK_M=
        NORM_BM,

    BLOCK_N=
        NORM_BN,

    BLOCK_K=
        NORM_BK,

    num_warps=
        NORM_WARPS,
)


torch.cuda.synchronize()


triton_gradient_norm = (
    math.sqrt(
        float(
            norm_ws.item()
        )
    )
)


gradient_norm_error = (
    abs(
        triton_gradient_norm
        -
        gradient_norm
    )
    /
    gradient_norm
)


print(
    "PyTorch norm          :",
    f"{gradient_norm:.8e}",
)

print(
    "Triton norm           :",
    f"{triton_gradient_norm:.8e}",
)

print(
    "Relative norm error   :",
    f"{gradient_norm_error:.8e}",
)

print(
    "PyTorch dW allocation :",
    f"{pytorch_temp_mb:.2f} MB",
)

print(
    "Triton dW allocation  :",
    "0.00 MB",
)


# =============================================================================
# CHOOSE UPDATE MAGNITUDE
# =============================================================================

LEARNING_RATE = (
    TARGET_DELTA_NORM
    /
    gradient_norm
)


CLIP_THRESHOLD = (
    1e30
)


print()
print(
    "Target ||DeltaW||     :",
    TARGET_DELTA_NORM,
)

print(
    "Learning rate         :",
    f"{LEARNING_RATE:.8e}",
)


# =============================================================================
# UPDATE CORRECTNESS
# =============================================================================

MASTER_A.copy_(
    ORIGINAL_BF16_WEIGHT
)


triton_master_update(

    MASTER_A,
    MASTER_B,

    learning_rate=
        LEARNING_RATE,

    clip_threshold=
        CLIP_THRESHOLD,
)


torch.cuda.synchronize()


reference_master = (
    MASTER_A
    -
    LEARNING_RATE
    *
    dW_reference
)


update_relative_error = float(

    (
        torch.linalg.vector_norm(
            MASTER_B
            -
            reference_master
        )
        /
        torch.linalg.vector_norm(
            reference_master
            -
            MASTER_A
        ).clamp_min(
            1e-30
        )
    )
    .item()
)


print()
print(
    "Update relative error :",
    f"{update_relative_error:.8e}",
)


del reference_master
del dW_reference


gc.collect()

torch.cuda.empty_cache()


# =============================================================================
# FAST-WEIGHT STATE
# =============================================================================

class FastWeightState:

    def __init__(
        self,
    ):

        self.master_active = (
            MASTER_A
        )

        self.master_staging = (
            MASTER_B
        )


        self.serve_active = (
            SERVE_A
        )

        self.serve_staging = (
            SERVE_B
        )


        self.version = 0


        self.last_version = None

        self.last_pointer = None


state = FastWeightState()


def reset_state():

    # Restore FP32 master state.
    MASTER_A.copy_(
        ORIGINAL_BF16_WEIGHT
    )


    # Restore serving staging just for deterministic state.
    SERVE_B.copy_(
        ORIGINAL_BF16_WEIGHT
    )


    torch.cuda.synchronize()


    state.master_active = (
        MASTER_A
    )

    state.master_staging = (
        MASTER_B
    )


    state.serve_active = (
        SERVE_A
    )

    state.serve_staging = (
        SERVE_B
    )


    state.version = 0


    state.last_version = None

    state.last_pointer = None


def commit_state():

    state.master_active, state.master_staging = (

        state.master_staging,
        state.master_active,
    )


    state.serve_active, state.serve_staging = (

        state.serve_staging,
        state.serve_active,
    )


    state.version += 1


# =============================================================================
# BF16 FAST-WEIGHT FORWARD
#
# IMPORTANT:
#
# No x.float().
# No FP32 projection.
#
# This is BF16 x BF16 just like original Qwen.
# =============================================================================

def bf16_fast_weight_forward(
    x,
):

    state.last_version = (
        state.version
    )


    state.last_pointer = (
        state.serve_active.data_ptr()
    )


    return F.linear(

        x,

        state.serve_active,

        bias=
            None,
    )


# =============================================================================
# STATIC PARITY TEST
# =============================================================================

print()
print("=" * 120)
print("BF16 FAST-WEIGHT STATIC PARITY")
print("=" * 120)


probe_ids = (
    adapt_ids[
        :,
        :128
    ]
    .contiguous()
)


# -------------------------------------------------------------------------
# Original Qwen
# -------------------------------------------------------------------------

target_layer.mlp.down_proj.forward = (
    original_down_forward
)


with torch.no_grad():

    stock_logits = (

        model(

            input_ids=
                probe_ids,

            use_cache=
                False,
        )

        .logits[
            :,
            -1,
            :
        ]

        .float()
        .clone()
    )


torch.cuda.synchronize()


# -------------------------------------------------------------------------
# BF16 fast-weight path
# -------------------------------------------------------------------------

reset_state()


target_layer.mlp.down_proj.forward = (
    bf16_fast_weight_forward
)


with torch.no_grad():

    fast_logits = (

        model(

            input_ids=
                probe_ids,

            use_cache=
                False,
        )

        .logits[
            :,
            -1,
            :
        ]

        .float()
        .clone()
    )


torch.cuda.synchronize()


parity_delta = (
    fast_logits
    -
    stock_logits
)


parity_relative_l2 = float(

    (
        torch.linalg.vector_norm(
            parity_delta
        )
        /
        torch.linalg.vector_norm(
            stock_logits
        ).clamp_min(
            1e-30
        )
    )
    .item()
)


parity_max_abs = float(

    parity_delta
    .abs()
    .max()
    .item()
)


print(
    "Relative L2           :",
    f"{parity_relative_l2:.8e}",
)

print(
    "Max abs               :",
    f"{parity_max_abs:.8e}",
)


# =============================================================================
# PUBLISH ONE REAL UPDATE
# =============================================================================

print()
print("=" * 120)
print("FP32 MASTER -> BF16 SERVING PUBLICATION")
print("=" * 120)


reset_state()


update_start = torch.cuda.Event(
    enable_timing=True
)


kernel_done = torch.cuda.Event(
    enable_timing=True
)


publish_done = torch.cuda.Event(
    enable_timing=True
)


update_start.record()


triton_master_update(

    state.master_active,
    state.master_staging,

    learning_rate=
        LEARNING_RATE,

    clip_threshold=
        CLIP_THRESHOLD,
)


kernel_done.record()


# Preallocated FP32 -> BF16 conversion.
#
# No new serving tensor is allocated.

state.serve_staging.copy_(
    state.master_staging
)


publish_done.record()


publish_done.synchronize()


kernel_ms = (
    update_start.elapsed_time(
        kernel_done
    )
)


publish_ms = (
    kernel_done.elapsed_time(
        publish_done
    )
)


total_update_ms = (
    update_start.elapsed_time(
        publish_done
    )
)


print(
    "Triton FP32 update    :",
    f"{kernel_ms:.3f} ms",
)

print(
    "FP32 -> BF16 publish  :",
    f"{publish_ms:.3f} ms",
)

print(
    "Total ready latency   :",
    f"{total_update_ms:.3f} ms",
)


# =============================================================================
# QUANTIZATION / PUBLICATION ANALYSIS
# =============================================================================

master_delta = (
    state.master_staging
    -
    state.master_active
)


bf16_delta = (

    state.serve_staging
    .float()

    -
    state.serve_active
    .float()
)


master_delta_norm = float(

    torch.linalg.vector_norm(
        master_delta
    )
    .item()
)


bf16_delta_norm = float(

    torch.linalg.vector_norm(
        bf16_delta
    )
    .item()
)


bf16_changed_count = int(

    (
        state.serve_staging
        !=
        state.serve_active
    )
    .sum()
    .item()
)


bf16_changed_fraction = (

    bf16_changed_count
    /
    state.serve_active.numel()
)


publication_error = float(

    (
        torch.linalg.vector_norm(
            bf16_delta
            -
            master_delta
        )
        /
        torch.linalg.vector_norm(
            master_delta
        ).clamp_min(
            1e-30
        )
    )
    .item()
)


print()
print(
    "FP32 ||DeltaW||       :",
    f"{master_delta_norm:.8e}",
)

print(
    "BF16 ||DeltaW||       :",
    f"{bf16_delta_norm:.8e}",
)

print(
    "BF16 changed weights  :",
    f"{bf16_changed_count:,}",
)

print(
    "BF16 changed fraction :",
    f"{bf16_changed_fraction * 100:.4f}%",
)

print(
    "Publication rel error :",
    f"{publication_error:.8e}",
)


# =============================================================================
# COMMIT + POINTER PROOF
# =============================================================================

before_pointer = (
    state.serve_active.data_ptr()
)


commit_state()


after_pointer = (
    state.serve_active.data_ptr()
)


with torch.no_grad():

    adapted_probe_logits = (

        model(

            input_ids=
                probe_ids,

            use_cache=
                False,
        )

        .logits[
            :,
            -1,
            :
        ]

        .float()
        .clone()
    )


torch.cuda.synchronize()


actual_forward_pointer = (
    state.last_pointer
)


adapted_logit_delta = (

    adapted_probe_logits
    -
    fast_logits
)


adapted_logit_relative_l2 = float(

    (
        torch.linalg.vector_norm(
            adapted_logit_delta
        )
        /
        torch.linalg.vector_norm(
            fast_logits
        ).clamp_min(
            1e-30
        )
    )
    .item()
)


print()
print(
    "Old serving pointer   :",
    before_pointer,
)

print(
    "New serving pointer   :",
    after_pointer,
)

print(
    "Forward used pointer  :",
    actual_forward_pointer,
)

print(
    "Pointer switch valid  :",
    (
        after_pointer
        ==
        actual_forward_pointer
        and
        before_pointer
        !=
        after_pointer
    ),
)

print(
    "Adapted logit rel L2  :",
    f"{adapted_logit_relative_l2:.8e}",
)


# =============================================================================
# LOSS QUALITY UNDER BF16 SERVING
# =============================================================================

@torch.no_grad()
def evaluate_loss(
    ids,
):

    result = model(

        input_ids=
            ids,

        labels=
            ids,

        use_cache=
            False,
    )


    return float(
        result.loss.item()
    )


# -------------------------------------------------------------------------
# W0
# -------------------------------------------------------------------------

reset_state()


adapt_loss_before = (
    evaluate_loss(
        adapt_ids
    )
)


eval_loss_before = (
    evaluate_loss(
        eval_ids
    )
)


# -------------------------------------------------------------------------
# W1
# -------------------------------------------------------------------------

triton_master_update(

    state.master_active,
    state.master_staging,

    learning_rate=
        LEARNING_RATE,

    clip_threshold=
        CLIP_THRESHOLD,
)


state.serve_staging.copy_(
    state.master_staging
)


torch.cuda.synchronize()


commit_state()


adapt_loss_after = (
    evaluate_loss(
        adapt_ids
    )
)


eval_loss_after = (
    evaluate_loss(
        eval_ids
    )
)


print()
print("=" * 120)
print("BF16 REAL-TTT QUALITY")
print("=" * 120)


print(
    "Adapt loss before     :",
    f"{adapt_loss_before:.8f}",
)

print(
    "Adapt loss after      :",
    f"{adapt_loss_after:.8f}",
)

print(
    "Adapt improvement     :",
    f"{adapt_loss_before - adapt_loss_after:.8f}",
)


print()
print(
    "Held-out before       :",
    f"{eval_loss_before:.8f}",
)

print(
    "Held-out after        :",
    f"{eval_loss_after:.8f}",
)

print(
    "Held-out improvement  :",
    f"{eval_loss_before - eval_loss_after:.8f}",
)


# =============================================================================
# CUDA STREAMS
# =============================================================================

try:

    inference_stream = (
        torch.cuda.Stream(
            priority=-1
        )
    )

except Exception:

    inference_stream = (
        torch.cuda.Stream()
    )


learning_stream = (
    torch.cuda.Stream(
        priority=0
    )
)


torch.cuda.synchronize()


# =============================================================================
# STATS
# =============================================================================

def percentile(
    values,
    p,
):

    if not values:

        return 0.0


    values = sorted(
        values
    )


    position = (
        len(values)
        -
        1
    ) * p


    lo = int(
        math.floor(
            position
        )
    )


    hi = int(
        math.ceil(
            position
        )
    )


    if lo == hi:

        return values[
            lo
        ]


    f = (
        position
        -
        lo
    )


    return (

        values[
            lo
        ]
        *
        (
            1.0
            -
            f
        )

        +

        values[
            hi
        ]
        *
        f
    )


def summarize(
    values,
):

    if not values:

        return None


    return {

        "mean":
            statistics.mean(
                values
            ),

        "p50":
            percentile(
                values,
                0.50,
            ),

        "p95":
            percentile(
                values,
                0.95,
            ),

        "p99":
            percentile(
                values,
                0.99,
            ),

        "min":
            min(
                values
            ),

        "max":
            max(
                values
            ),
    }


# =============================================================================
# SERVING BENCHMARK
# =============================================================================

MODES = [

    "stock",

    "fastweight_baseline",

    "serialized",

    "async",

]


def run_decode(
    mode,
):

    if mode not in MODES:

        raise ValueError(
            mode
        )


    # -------------------------------------------------------------------------
    # Select projection path.
    # -------------------------------------------------------------------------

    if (
        mode
        ==
        "stock"
    ):

        target_layer.mlp.down_proj.forward = (
            original_down_forward
        )

    else:

        target_layer.mlp.down_proj.forward = (
            bf16_fast_weight_forward
        )


        reset_state()


    token_latencies = []

    update_token_latencies = []

    normal_token_latencies = []


    versions = []

    pointers = []


    update_kernel_ms = None

    publish_ms_local = None

    total_update_ms_local = None


    visibility_lag = None


    update_in_flight = False


    update_start_event = None

    kernel_done_event = None

    publish_done_event = None


    torch.cuda.synchronize()


    # =========================================================================
    # PREFILL
    # =========================================================================

    with torch.cuda.stream(
        inference_stream
    ):

        with torch.no_grad():

            output = model(

                input_ids=
                    adapt_ids,

                use_cache=
                    True,
            )


        past_key_values = (
            output.past_key_values
        )


        next_token = (

            output.logits[
                :,
                -1,
                :
            ]

            .argmax(

                dim=
                    -1,

                keepdim=
                    True,
            )
        )


    inference_stream.synchronize()


    foreground_start = (
        time.perf_counter()
    )


    # =========================================================================
    # DECODE
    # =========================================================================

    for token_idx in range(
        DECODE_TOKENS
    ):

        # ---------------------------------------------------------------------
        # Commit async update at a token boundary.
        # ---------------------------------------------------------------------

        if (
            mode
            ==
            "async"

            and
            update_in_flight

            and
            publish_done_event.query()
        ):

            publish_done_event.synchronize()


            update_kernel_ms = (
                update_start_event.elapsed_time(
                    kernel_done_event
                )
            )


            publish_ms_local = (
                kernel_done_event.elapsed_time(
                    publish_done_event
                )
            )


            total_update_ms_local = (
                update_start_event.elapsed_time(
                    publish_done_event
                )
            )


            commit_state()


            visibility_lag = (
                token_idx
            )


            update_in_flight = (
                False
            )


        token_wall_start = (
            time.perf_counter()
        )


        # =====================================================================
        # SERIALIZED
        #
        # FP32 update -> BF16 publication -> commit -> inference.
        # =====================================================================

        if (
            mode
            ==
            "serialized"

            and
            token_idx
            ==
            0
        ):

            update_start_event = (
                torch.cuda.Event(
                    enable_timing=True
                )
            )


            kernel_done_event = (
                torch.cuda.Event(
                    enable_timing=True
                )
            )


            publish_done_event = (
                torch.cuda.Event(
                    enable_timing=True
                )
            )


            with torch.cuda.stream(
                inference_stream
            ):

                update_start_event.record(
                    inference_stream
                )


                triton_master_update(

                    state.master_active,
                    state.master_staging,

                    learning_rate=
                        LEARNING_RATE,

                    clip_threshold=
                        CLIP_THRESHOLD,
                )


                kernel_done_event.record(
                    inference_stream
                )


                state.serve_staging.copy_(
                    state.master_staging
                )


                publish_done_event.record(
                    inference_stream
                )


            publish_done_event.synchronize()


            update_kernel_ms = (
                update_start_event.elapsed_time(
                    kernel_done_event
                )
            )


            publish_ms_local = (
                kernel_done_event.elapsed_time(
                    publish_done_event
                )
            )


            total_update_ms_local = (
                update_start_event.elapsed_time(
                    publish_done_event
                )
            )


            commit_state()


            visibility_lag = 0


        # =====================================================================
        # ASYNC
        #
        # token 0 sees W0.
        #
        # learning stream computes:
        #
        #     master W0 -> master W1
        #     master W1 -> BF16 serving W1
        #
        # token 1 should normally see W1.
        # =====================================================================

        if (
            mode
            ==
            "async"

            and
            token_idx
            ==
            0
        ):

            update_start_event = (
                torch.cuda.Event(
                    enable_timing=True
                )
            )


            kernel_done_event = (
                torch.cuda.Event(
                    enable_timing=True
                )
            )


            publish_done_event = (
                torch.cuda.Event(
                    enable_timing=True
                )
            )


            with torch.cuda.stream(
                learning_stream
            ):

                update_start_event.record(
                    learning_stream
                )


                triton_master_update(

                    state.master_active,
                    state.master_staging,

                    learning_rate=
                        LEARNING_RATE,

                    clip_threshold=
                        CLIP_THRESHOLD,
                )


                kernel_done_event.record(
                    learning_stream
                )


                # Preallocated conversion.
                state.serve_staging.copy_(
                    state.master_staging
                )


                publish_done_event.record(
                    learning_stream
                )


            update_in_flight = True


        # =====================================================================
        # REAL TOKEN DECODE
        # =====================================================================

        if (
            mode
            !=
            "stock"
        ):

            state.last_version = None

            state.last_pointer = None


        with torch.cuda.stream(
            inference_stream
        ):

            with torch.no_grad():

                output = model(

                    input_ids=
                        next_token,

                    past_key_values=
                        past_key_values,

                    use_cache=
                        True,
                )


            past_key_values = (
                output.past_key_values
            )


            next_token = (

                output.logits[
                    :,
                    -1,
                    :
                ]

                .argmax(

                    dim=
                        -1,

                    keepdim=
                        True,
                )
            )


        # Wait only for foreground inference.
        inference_stream.synchronize()


        _ = int(
            next_token.item()
        )


        token_ms = (

            time.perf_counter()

            -
            token_wall_start

        ) * 1000.0


        token_latencies.append(
            token_ms
        )


        if token_idx == 0:

            update_token_latencies.append(
                token_ms
            )

        else:

            normal_token_latencies.append(
                token_ms
            )


        if (
            mode
            !=
            "stock"
        ):

            versions.append(
                int(
                    state.last_version
                )
            )


            pointers.append(
                int(
                    state.last_pointer
                )
            )


    foreground_ms = (

        time.perf_counter()

        -
        foreground_start

    ) * 1000.0


    # =========================================================================
    # FINAL ASYNC UPDATE
    # =========================================================================

    if (
        mode
        ==
        "async"

        and
        update_in_flight
    ):

        publish_done_event.synchronize()


        update_kernel_ms = (
            update_start_event.elapsed_time(
                kernel_done_event
            )
        )


        publish_ms_local = (
            kernel_done_event.elapsed_time(
                publish_done_event
            )
        )


        total_update_ms_local = (
            update_start_event.elapsed_time(
                publish_done_event
            )
        )


        commit_state()


        visibility_lag = (
            DECODE_TOKENS
        )


        update_in_flight = (
            False
        )


    torch.cuda.synchronize()


    result = {

        "mode":
            mode,

        "itl":
            summarize(
                token_latencies
            ),

        "update_token":
            summarize(
                update_token_latencies
            ),

        "normal_tokens":
            summarize(
                normal_token_latencies
            ),

        "foreground_ms":
            foreground_ms,

        "tok_s":
            DECODE_TOKENS
            /
            (
                foreground_ms
                /
                1000.0
            ),

        "kernel_ms":
            update_kernel_ms,

        "publish_ms":
            publish_ms_local,

        "total_update_ms":
            total_update_ms_local,

        "visibility_lag":
            visibility_lag,

        "versions":
            versions,

        "pointers":
            pointers,
    }


    del output
    del past_key_values
    del next_token


    gc.collect()

    torch.cuda.empty_cache()


    return result


# =============================================================================
# WARMUPS
# =============================================================================

print()
print("=" * 120)
print("SERVING WARMUP")
print("=" * 120)


stock_warmup = (
    run_decode(
        "stock"
    )
)


fast_warmup = (
    run_decode(
        "fastweight_baseline"
    )
)


print(
    "Stock p50             :",
    f"{stock_warmup['itl']['p50']:.3f} ms",
)

print(
    "Fast-weight BF16 p50  :",
    f"{fast_warmup['itl']['p50']:.3f} ms",
)


# =============================================================================
# BENCHMARK
# =============================================================================

runs = {

    mode: []

    for mode in MODES
}


print()
print("=" * 155)
print("QWEN2.5-7B — BF16 SERVING + FP32 MASTER ASYNC-TTT")
print("=" * 155)


for mode in MODES:

    print()
    print("-" * 115)
    print(
        mode.upper()
    )
    print("-" * 115)


    for run_idx in range(
        RUNS
    ):

        result = (
            run_decode(
                mode
            )
        )


        runs[
            mode
        ].append(
            result
        )


        kernel_text = (
            "-"
            if result[
                "kernel_ms"
            ]
            is None
            else
            f"{result['kernel_ms']:.3f}"
        )


        publish_text = (
            "-"
            if result[
                "publish_ms"
            ]
            is None
            else
            f"{result['publish_ms']:.3f}"
        )


        total_text = (
            "-"
            if result[
                "total_update_ms"
            ]
            is None
            else
            f"{result['total_update_ms']:.3f}"
        )


        print(
            f"run={run_idx + 1} | "
            f"p50={result['itl']['p50']:.3f} | "
            f"p95={result['itl']['p95']:.3f} | "
            f"p99={result['itl']['p99']:.3f} | "
            f"token0={result['update_token']['p50']:.3f} | "
            f"normal={result['normal_tokens']['p50']:.3f} | "
            f"kernel={kernel_text} | "
            f"publish={publish_text} | "
            f"total={total_text} | "
            f"lag={result['visibility_lag']}"
        )


# =============================================================================
# AGGREGATE
# =============================================================================

def median_metric(
    mode,
    group,
    metric,
):

    return statistics.median(
        [
            x[
                group
            ][
                metric
            ]

            for x in runs[
                mode
            ]
        ]
    )


aggregate = {}


for mode in MODES:

    aggregate[
        mode
    ] = {

        "p50":
            median_metric(
                mode,
                "itl",
                "p50",
            ),

        "p95":
            median_metric(
                mode,
                "itl",
                "p95",
            ),

        "p99":
            median_metric(
                mode,
                "itl",
                "p99",
            ),

        "token0":
            median_metric(
                mode,
                "update_token",
                "p50",
            ),

        "normal":
            median_metric(
                mode,
                "normal_tokens",
                "p50",
            ),

        "tok_s":
            statistics.median(
                [
                    x[
                        "tok_s"
                    ]

                    for x in runs[
                        mode
                    ]
                ]
            ),
    }


    for key in [

        "kernel_ms",
        "publish_ms",
        "total_update_ms",

    ]:

        values = [

            x[
                key
            ]

            for x in runs[
                mode
            ]

            if x[
                key
            ]
            is not None
        ]


        aggregate[
            mode
        ][
            key
        ] = (

            statistics.median(
                values
            )

            if values

            else None
        )


    lag_values = [

        x[
            "visibility_lag"
        ]

        for x in runs[
            mode
        ]

        if x[
            "visibility_lag"
        ]
        is not None
    ]


    aggregate[
        mode
    ][
        "lag"
    ] = (

        statistics.median(
            lag_values
        )

        if lag_values

        else None
    )


# =============================================================================
# FINAL TABLE
# =============================================================================

print()
print("=" * 175)
print("FINAL BF16-SERVING RESULTS")
print("=" * 175)


print(
    f"{'Mode':<24}"
    f"{'p50':>11}"
    f"{'p95':>11}"
    f"{'p99':>11}"
    f"{'token0':>12}"
    f"{'normal':>12}"
    f"{'kernel':>11}"
    f"{'publish':>11}"
    f"{'total':>11}"
    f"{'lag':>8}"
    f"{'tok/s':>11}"
)


print(
    "-" * 175
)


for mode in MODES:

    row = (
        aggregate[
            mode
        ]
    )


    def fmt(
        value
    ):

        if value is None:

            return "-"

        return (
            f"{value:.3f}"
        )


    lag_text = (

        "-"

        if row[
            "lag"
        ]
        is None

        else
        f"{row['lag']:.1f}"
    )


    print(
        f"{mode:<24}"
        f"{row['p50']:>9.3f}ms"
        f"{row['p95']:>9.3f}ms"
        f"{row['p99']:>9.3f}ms"
        f"{row['token0']:>10.3f}ms"
        f"{row['normal']:>10.3f}ms"
        f"{fmt(row['kernel_ms']):>11}"
        f"{fmt(row['publish_ms']):>11}"
        f"{fmt(row['total_update_ms']):>11}"
        f"{lag_text:>8}"
        f"{row['tok_s']:>11.2f}"
    )


# =============================================================================
# PERFORMANCE CONFOUND CHECK
# =============================================================================

stock = (
    aggregate[
        "stock"
    ]
)


fast = (
    aggregate[
        "fastweight_baseline"
    ]
)


fastweight_overhead_pct = (

    (
        fast[
            "p50"
        ]
        -
        stock[
            "p50"
        ]
    )

    /
    stock[
        "p50"
    ]

    *
    100.0
)


print()
print("=" * 120)
print("BF16 SERVING CONFOUND CHECK")
print("=" * 120)


print(
    "Stock Qwen p50        :",
    f"{stock['p50']:.3f} ms",
)

print(
    "BF16 fast-weight p50  :",
    f"{fast['p50']:.3f} ms",
)

print(
    "Fast-weight overhead  :",
    f"{fastweight_overhead_pct:.3f}%",
)

print(
    "Static logit rel L2   :",
    f"{parity_relative_l2:.8e}",
)

print(
    "Static logit max abs  :",
    f"{parity_max_abs:.8e}",
)


# =============================================================================
# SERIALIZED STALL CHECK
# =============================================================================

serialized = (
    aggregate[
        "serialized"
    ]
)


serialized_exposed_stall = (

    serialized[
        "token0"
    ]

    -
    serialized[
        "normal"
    ]
)


print()
print("=" * 120)
print("SERIALIZED UPDATE CHECK")
print("=" * 120)


print(
    "Normal-token p50      :",
    f"{serialized['normal']:.3f} ms",
)

print(
    "Update-token latency  :",
    f"{serialized['token0']:.3f} ms",
)

print(
    "Exposed stall         :",
    f"{serialized_exposed_stall:.3f} ms",
)

print(
    "Kernel                :",
    f"{serialized['kernel_ms']:.3f} ms",
)

print(
    "BF16 publication      :",
    f"{serialized['publish_ms']:.3f} ms",
)

print(
    "Total update          :",
    f"{serialized['total_update_ms']:.3f} ms",
)


# =============================================================================
# ASYNC HIDING CHECK
# =============================================================================

async_result = (
    aggregate[
        "async"
    ]
)


async_exposed = (

    async_result[
        "token0"
    ]

    -
    async_result[
        "normal"
    ]
)


async_hidden_fraction = (

    1.0

    -
    max(
        0.0,
        async_exposed,
    )

    /
    async_result[
        "total_update_ms"
    ]
)


print()
print("=" * 120)
print("ASYNC HIDING CHECK")
print("=" * 120)


print(
    "Async normal p50      :",
    f"{async_result['normal']:.3f} ms",
)

print(
    "Async update-token    :",
    f"{async_result['token0']:.3f} ms",
)

print(
    "Exposed overhead      :",
    f"{async_exposed:.3f} ms",
)

print(
    "FP32 Triton kernel    :",
    f"{async_result['kernel_ms']:.3f} ms",
)

print(
    "BF16 publication      :",
    f"{async_result['publish_ms']:.3f} ms",
)

print(
    "Total background      :",
    f"{async_result['total_update_ms']:.3f} ms",
)

print(
    "Visibility lag        :",
    f"{async_result['lag']:.1f} token(s)",
)

print(
    "Approx hidden fraction:",
    f"{async_hidden_fraction * 100:.2f}%",
)


# =============================================================================
# VERSION TRACE
# =============================================================================

example_async = (
    runs[
        "async"
    ][
        0
    ]
)


print()
print("=" * 120)
print("ASYNC BF16 VERSION TRACE")
print("=" * 120)


for token_idx in range(
    min(
        12,
        DECODE_TOKENS,
    )
):

    print(

        f"token {token_idx:>2} | "
        f"version={example_async['versions'][token_idx]} | "
        f"ptr={example_async['pointers'][token_idx]}"
    )


# =============================================================================
# SAVE
# =============================================================================

RESULT_PATH = Path(
    "async_ttt_qwen7b_bf16_serving.json"
)


result_json = {

    "model":
        MODEL_ID,

    "shape": {

        "M":
            M,

        "N":
            N,

        "K":
            ADAPT_LEN,

        "layer":
            TTT_LAYER,
    },

    "gradient": {

        "norm":
            gradient_norm,

        "triton_norm":
            triton_gradient_norm,

        "relative_norm_error":
            gradient_norm_error,

        "pytorch_materialized_dw_mb":
            pytorch_temp_mb,

        "triton_materialized_dw_mb":
            0.0,

        "target_delta_norm":
            TARGET_DELTA_NORM,

        "learning_rate":
            LEARNING_RATE,

        "update_relative_error":
            update_relative_error,
    },

    "memory": {

        "fp32_master_buffer_mb":
            FP32_WEIGHT_MB,

        "bf16_serving_buffer_mb":
            BF16_WEIGHT_MB,

        "extra_fp32_master_buffers":
            2,

        "additional_bf16_staging_buffers":
            1,
    },

    "publication": {

        "fp32_delta_norm":
            master_delta_norm,

        "bf16_delta_norm":
            bf16_delta_norm,

        "bf16_changed_count":
            bf16_changed_count,

        "bf16_changed_fraction":
            bf16_changed_fraction,

        "publication_relative_error":
            publication_error,

        "single_kernel_ms":
            kernel_ms,

        "single_publish_ms":
            publish_ms,

        "single_total_ms":
            total_update_ms,
    },

    "parity": {

        "relative_l2":
            parity_relative_l2,

        "max_abs":
            parity_max_abs,
    },

    "quality": {

        "adapt_loss_before":
            adapt_loss_before,

        "adapt_loss_after":
            adapt_loss_after,

        "adapt_improvement":
            adapt_loss_before
            -
            adapt_loss_after,

        "heldout_loss_before":
            eval_loss_before,

        "heldout_loss_after":
            eval_loss_after,

        "heldout_improvement":
            eval_loss_before
            -
            eval_loss_after,
    },

    "aggregate":
        aggregate,

    "raw":
        runs,
}


with open(

    RESULT_PATH,

    "w",

) as file:

    json.dump(

        result_json,

        file,

        indent=
            2,
    )


print()
print("=" * 120)
print("DONE")
print("=" * 120)


print(
    "Saved:",
    RESULT_PATH.resolve(),
)


# =============================================================================
# RESTORE ORIGINAL QWEN FORWARD
# =============================================================================

target_layer.mlp.down_proj.forward = (
    original_down_forward
)


torch.cuda.synchronize()


free_bytes, total_bytes = (
    torch.cuda.mem_get_info()
)


print(
    "Original down_proj restored."
)

print(
    "Free VRAM:",
    f"{free_bytes / 1024**3:.2f} GiB",
)

FLUSHING PREVIOUS GPU STATE
Free VRAM : 24.40 GiB
Total VRAM: 39.49 GiB

Restart the kernel if an older model is still referenced.

ASYNC-TTT — QWEN2.5-7B BF16 SERVING
Model                  : Qwen/Qwen2.5-7B-Instruct
Python                 : 3.12.11
PyTorch                : 2.8.0+cu128
CUDA                   : 12.8
Triton                 : 3.4.0
GPU                    : NVIDIA A100-SXM4-40GB
SMs                    : 108
VRAM                   : 39.49 GiB

LOADING QWEN2.5-7B


Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

hidden_size            : 3584
intermediate_size      : 18944
layers                 : 28
TTT layer              : 13
model dtype            : torch.bfloat16
FP32 fast weight       : 271.58 MB
BF16 serving weight    : 135.79 MB
Free after load        : 23.16 GiB
Adapt tokens           : 512
Held-out tokens        : 512

CAPTURING REAL LM GRADIENT
Original LM loss      : 0.50794744
Z                     : (512, 18944)
G                     : (512, 3584)
G^T                   : (3584, 512)
Gradient capture      : 93.249 ms

FAST-WEIGHT MEMORY
FP32 master A         : 271.58 MB
FP32 master B         : 271.58 MB
BF16 serving A        : 135.79 MB (reuses model weight)
BF16 serving B        : 135.79 MB

REAL GRADIENT VALIDATION
PyTorch norm          : 2.83145487e-01
Triton norm           : 2.83145386e-01
Relative norm error   : 3.59053712e-07
PyTorch dW allocation : 271.58 MB
Triton dW allocation  : 0.00 MB

Target ||DeltaW||     : 0.03
Learning rate         : 1.05952598e-01

Update relative e

In [1]:
# =============================================================================
# ASYNC-TTT — QWEN2.5-7B
#
# MULTI-STEP ONLINE ADAPTATION
# +
# FP32 MASTER / BF16 SERVING QUANTIZATION ACCUMULATION
# +
# STALENESS SWEEP L = 0, 1, 2, 4, 8
#
#
# WHAT THIS TEST ANSWERS
# ----------------------
#
# 1. Do small FP32 updates accumulate across many online steps?
#
# 2. Even though one FP32 -> BF16 publication has large quantization error,
#    do previously-invisible FP32 changes eventually cross BF16 quantization
#    boundaries and become visible to inference?
#
# 3. How does:
#
#       cumulative FP32 ||W_t - W_0||
#       cumulative BF16 ||Q(W_t) - Q(W_0)||
#       BF16 changed fraction
#       publication error
#
#    evolve over 16 updates?
#
# 4. Does the LM adaptation loss continue improving?
#
# 5. What happens to held-out loss?
#
# 6. If the update is delayed by:
#
#       L = 0, 1, 2, 4, 8 tokens
#
#    how does next-token NLL / perplexity change?
#
#
# IMPORTANT
# ---------
#
# Gradient generation and fast-weight application are measured separately.
#
# "gradient capture ms" below is the cost of obtaining the real LM gradient
# signal with autograd.
#
# "kernel / publish / total update ms" measures only:
#
#       G,Z already available
#              |
#              v
#       Triton FP32 update
#              |
#              v
#       BF16 publication
#
# Do NOT combine these into one claim unless explicitly stated.
#
#
# MEMORY HIERARCHY
# ----------------
#
# HBM:
#     G
#     Z
#     FP32 master A/B
#     BF16 serving A/B
#
# Triton:
#     Delta-W tiles are rematerialized during pass 2.
#     No full Delta-W tensor is materialized in HBM.
#
# Exact register/shared-memory/L2/HBM traffic still requires Nsight Compute.
# =============================================================================


# =============================================================================
# IMPORTS
# =============================================================================

import gc
import json
import math
import statistics
import sys
import time

from pathlib import Path

import torch
import torch.nn.functional as F

import triton
import triton.language as tl

from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
)


# =============================================================================
# AGGRESSIVE GPU CLEANUP
# =============================================================================

print("=" * 120)
print("FLUSHING PREVIOUS GPU STATE")
print("=" * 120)


cleanup_names = [
    "model",
    "tokenizer",
    "target_layer",
    "state",
    "MASTER_A",
    "MASTER_B",
    "MASTER_BASE",
    "SERVE_A",
    "SERVE_B",
    "BASE_BF16",
    "GT_REAL",
    "G_REAL",
    "Z_REAL",
    "output",
    "past_key_values",
    "next_token",
    "capture",
    "runs",
]


for name in cleanup_names:

    if name in globals():

        try:
            del globals()[name]
        except Exception:
            pass


gc.collect()


if torch.cuda.is_available():

    torch.cuda.synchronize()

    torch.cuda.empty_cache()

    try:
        torch.cuda.ipc_collect()
    except Exception:
        pass

    gc.collect()

    torch.cuda.empty_cache()


free_bytes, total_bytes = (
    torch.cuda.mem_get_info()
)


print(
    "Free VRAM :",
    f"{free_bytes / 1024**3:.2f} GiB",
)

print(
    "Total VRAM:",
    f"{total_bytes / 1024**3:.2f} GiB",
)


# A freshly restarted A100-40GB session should be around 38+ GiB free.
#
# Don't silently continue with a previous model still alive.

if (
    free_bytes
    /
    1024**3
    <
    30.0
):

    raise RuntimeError(
        "\nLess than 30 GiB is free.\n"
        "Restart the kernel first, then rerun this cell."
    )


# =============================================================================
# CONFIG
# =============================================================================

MODEL_ID = (
    "Qwen/Qwen2.5-7B-Instruct"
)


DEVICE = "cuda"


TTT_LAYER = 13


NUM_UPDATES = 16


# Real gradient per online update.
ADAPT_CHUNK = 256


# Independent evaluation sequence.
HELDOUT_LEN = 512


TOTAL_REQUIRED_TOKENS = (
    NUM_UPDATES
    *
    ADAPT_CHUNK
    +
    HELDOUT_LEN
)


# Keep the same useful update magnitude from our previous experiment.
TARGET_DELTA_NORM = 3e-2


# Staleness quality experiment.
STALENESS_LAGS = [
    None,   # W0 baseline: never publish W1
    0,
    1,
    2,
    4,
    8,
]


STALENESS_PREFIX = 64

STALENESS_STEPS = 64


torch.manual_seed(
    0
)


torch.backends.cuda.matmul.allow_tf32 = False

torch.backends.cudnn.allow_tf32 = False


# =============================================================================
# ENVIRONMENT
# =============================================================================

assert torch.cuda.is_available()


props = (
    torch.cuda.get_device_properties(
        0
    )
)


print()
print("=" * 120)
print("QWEN2.5-7B — MULTI-STEP ASYNC-TTT")
print("=" * 120)


print(
    "Model                  :",
    MODEL_ID,
)

print(
    "Python                 :",
    sys.version.split()[0],
)

print(
    "PyTorch                :",
    torch.__version__,
)

print(
    "CUDA                   :",
    torch.version.cuda,
)

print(
    "Triton                 :",
    triton.__version__,
)

print(
    "GPU                    :",
    torch.cuda.get_device_name(0),
)

print(
    "SMs                    :",
    props.multi_processor_count,
)

print(
    "VRAM                   :",
    f"{props.total_memory / 1024**3:.2f} GiB",
)


# =============================================================================
# LOAD QWEN
# =============================================================================

print()
print("=" * 120)
print("LOADING QWEN2.5-7B")
print("=" * 120)


tokenizer = (
    AutoTokenizer.from_pretrained(
        MODEL_ID,
        trust_remote_code=True,
    )
)


model = (
    AutoModelForCausalLM.from_pretrained(

        MODEL_ID,

        dtype=
            torch.bfloat16,

        device_map={
            "": 0
        },

        low_cpu_mem_usage=True,

        trust_remote_code=True,

        attn_implementation=
            "sdpa",
    )
)


model.eval()


# Freeze the real model.
#
# We create the gradient leaf explicitly at the target down_proj output.

for parameter in model.parameters():

    parameter.requires_grad_(
        False
    )


M = int(
    model.config.hidden_size
)


N = int(
    model.config.intermediate_size
)


target_layer = (
    model.model.layers[
        TTT_LAYER
    ]
)


original_down_forward = (
    target_layer
    .mlp
    .down_proj
    .forward
)


print(
    "hidden_size            :",
    M,
)

print(
    "intermediate_size      :",
    N,
)

print(
    "layers                 :",
    model.config.num_hidden_layers,
)

print(
    "TTT layer              :",
    TTT_LAYER,
)

print(
    "model dtype            :",
    next(
        model.parameters()
    ).dtype,
)


FP32_MB = (
    M
    *
    N
    *
    4
    /
    1e6
)


BF16_MB = (
    M
    *
    N
    *
    2
    /
    1e6
)


print(
    "FP32 fast weight       :",
    f"{FP32_MB:.2f} MB",
)

print(
    "BF16 serving weight    :",
    f"{BF16_MB:.2f} MB",
)


free_bytes, _ = (
    torch.cuda.mem_get_info()
)


print(
    "Free after load        :",
    f"{free_bytes / 1024**3:.2f} GiB",
)


# =============================================================================
# BUILD MULTI-DOMAIN TOKEN STREAM
#
# This is NOT our eventual quality benchmark.
#
# It is only a deterministic online-adaptation stream for:
#
#       accumulation
#       quantization
#       staleness
#
# The complex-prompt memory test comes next.
# =============================================================================

paragraphs = [

    """
    Distributed systems must preserve correctness while requests execute
    concurrently across unreliable machines. Replication, consensus,
    idempotency and backpressure determine whether a service remains stable
    when traffic increases or individual components fail.
    """,

    """
    Modern processors exploit instruction-level parallelism, cache locality
    and speculative execution. Memory hierarchy frequently determines
    application performance because arithmetic can complete much faster than
    data can be transferred from distant memory.
    """,

    """
    In statistical learning, generalization depends on more than minimizing
    training error. Regularization, validation methodology and distribution
    shift determine whether a learned representation remains useful on
    observations that were not available during optimization.
    """,

    """
    Database systems organize persistent information using indexes, pages,
    transactions and concurrency control. Query optimizers estimate the cost
    of alternative execution plans before selecting joins, scans and access
    paths.
    """,

    """
    Compilers transform high-level programs through parsing, intermediate
    representations, optimization and machine-code generation. Data-flow
    analysis exposes relationships between definitions, uses and control-flow
    paths.
    """,

    """
    Network protocols coordinate independent machines through layers of
    addressing, routing, congestion control and reliability. Latency,
    bandwidth and packet loss interact differently depending on workload and
    communication pattern.
    """,

    """
    Numerical optimization studies how an objective changes as parameters are
    modified. Gradient-based methods use local derivative information while
    curvature and conditioning influence how rapidly optimization progresses.
    """,

    """
    Information retrieval systems map queries and documents into structures
    that make relevance search efficient. Sparse lexical methods and dense
    vector representations make different assumptions about semantic
    similarity.
    """,
]


long_text = "\n".join(
    paragraphs
    *
    500
)


tokens = (
    tokenizer(

        long_text,

        return_tensors=
            "pt",

        truncation=
            True,

        max_length=
            TOTAL_REQUIRED_TOKENS,

    )[
        "input_ids"
    ]
    .to(
        DEVICE
    )
)


if (
    tokens.shape[1]
    <
    TOTAL_REQUIRED_TOKENS
):

    raise RuntimeError(
        f"Need {TOTAL_REQUIRED_TOKENS} tokens, "
        f"got {tokens.shape[1]}"
    )


adapt_stream = (

    tokens[
        :,
        :
        NUM_UPDATES
        *
        ADAPT_CHUNK
    ]
    .contiguous()
)


heldout_ids = (

    tokens[
        :,
        NUM_UPDATES
        *
        ADAPT_CHUNK:
        NUM_UPDATES
        *
        ADAPT_CHUNK
        +
        HELDOUT_LEN
    ]
    .contiguous()
)


del tokens
del long_text


gc.collect()


print()
print(
    "Adaptation stream     :",
    adapt_stream.shape[1],
    "tokens",
)

print(
    "Held-out stream       :",
    heldout_ids.shape[1],
    "tokens",
)


# =============================================================================
# FAST-WEIGHT BUFFERS
# =============================================================================

ORIGINAL_PARAMETER = (
    target_layer
    .mlp
    .down_proj
    .weight
    .detach()
)


# Immutable base snapshots.
#
# Needed because SERVE_A itself will become a staging buffer on alternating
# versions.

BASE_BF16 = (
    ORIGINAL_PARAMETER
    .clone()
    .contiguous()
)


MASTER_BASE = (
    BASE_BF16
    .float()
    .contiguous()
)


# Master A/B.
MASTER_A = (
    MASTER_BASE
    .clone()
    .contiguous()
)


MASTER_B = (
    MASTER_BASE
    .clone()
    .contiguous()
)


# BF16 serving A/B.
#
# A reuses the actual model parameter storage.
SERVE_A = (
    ORIGINAL_PARAMETER
)


SERVE_B = (
    BASE_BF16
    .clone()
    .contiguous()
)


print()
print("=" * 120)
print("FAST-WEIGHT STATE")
print("=" * 120)


print(
    "Master base           :",
    f"{FP32_MB:.2f} MB",
)

print(
    "Master A              :",
    f"{FP32_MB:.2f} MB",
)

print(
    "Master B              :",
    f"{FP32_MB:.2f} MB",
)

print(
    "BF16 base             :",
    f"{BF16_MB:.2f} MB",
)

print(
    "Serving A             :",
    f"{BF16_MB:.2f} MB",
)

print(
    "Serving B             :",
    f"{BF16_MB:.2f} MB",
)


# =============================================================================
# STATE
# =============================================================================

class FastWeightState:

    def __init__(
        self,
    ):

        self.master_active = MASTER_A

        self.master_staging = MASTER_B


        self.serve_active = SERVE_A

        self.serve_staging = SERVE_B


        self.version = 0


        self.last_version = None

        self.last_pointer = None


state = FastWeightState()


def reset_all_state():

    MASTER_A.copy_(
        MASTER_BASE
    )

    MASTER_B.copy_(
        MASTER_BASE
    )


    SERVE_A.copy_(
        BASE_BF16
    )

    SERVE_B.copy_(
        BASE_BF16
    )


    torch.cuda.synchronize()


    state.master_active = MASTER_A

    state.master_staging = MASTER_B


    state.serve_active = SERVE_A

    state.serve_staging = SERVE_B


    state.version = 0


    state.last_version = None

    state.last_pointer = None


def commit_state():

    state.master_active, state.master_staging = (
        state.master_staging,
        state.master_active,
    )


    state.serve_active, state.serve_staging = (
        state.serve_staging,
        state.serve_active,
    )


    state.version += 1


# =============================================================================
# BF16 MODEL FORWARD
# =============================================================================

def fast_weight_forward(
    x,
):

    state.last_version = (
        state.version
    )


    state.last_pointer = (
        state.serve_active.data_ptr()
    )


    return F.linear(
        x,
        state.serve_active,
        bias=None,
    )


target_layer.mlp.down_proj.forward = (
    fast_weight_forward
)


# =============================================================================
# TRITON PASS 1
# =============================================================================

@triton.jit
def norm_kernel(
    gt_ptr,
    z_ptr,

    norm_ptr,

    stride_gm,
    stride_gk,

    stride_zk,
    stride_zn,

    M: tl.constexpr,
    N: tl.constexpr,
    K: tl.constexpr,

    BM: tl.constexpr,
    BN: tl.constexpr,
    BK: tl.constexpr,
):

    pid_m = tl.program_id(
        0
    )

    pid_n = tl.program_id(
        1
    )


    offs_m = (
        pid_m * BM
        +
        tl.arange(
            0,
            BM,
        )
    )


    offs_n = (
        pid_n * BN
        +
        tl.arange(
            0,
            BN,
        )
    )


    acc = tl.zeros(

        (
            BM,
            BN,
        ),

        dtype=
            tl.float32,
    )


    for k in range(
        0,
        K,
        BK,
    ):

        offs_k = (
            k
            +
            tl.arange(
                0,
                BK,
            )
        )


        g = tl.load(

            gt_ptr
            +
            offs_m[:, None]
            *
            stride_gm
            +
            offs_k[None, :]
            *
            stride_gk,

            mask=(
                (
                    offs_m[:, None]
                    <
                    M
                )
                &
                (
                    offs_k[None, :]
                    <
                    K
                )
            ),

            other=0.0,
        )


        z = tl.load(

            z_ptr
            +
            offs_k[:, None]
            *
            stride_zk
            +
            offs_n[None, :]
            *
            stride_zn,

            mask=(
                (
                    offs_k[:, None]
                    <
                    K
                )
                &
                (
                    offs_n[None, :]
                    <
                    N
                )
            ),

            other=0.0,
        )


        acc = tl.dot(

            g,
            z,
            acc,

            input_precision=
                "ieee",
        )


    local_sq = tl.sum(
        acc
        *
        acc
    )


    tl.atomic_add(
        norm_ptr,
        local_sq,
    )


# =============================================================================
# TRITON PASS 2
# =============================================================================

@triton.jit
def update_kernel(
    gt_ptr,
    z_ptr,

    src_ptr,
    dst_ptr,

    norm_ptr,

    stride_gm,
    stride_gk,

    stride_zk,
    stride_zn,

    stride_sm,
    stride_sn,

    stride_dm,
    stride_dn,

    learning_rate,
    clip_threshold,

    M: tl.constexpr,
    N: tl.constexpr,
    K: tl.constexpr,

    BM: tl.constexpr,
    BN: tl.constexpr,
    BK: tl.constexpr,
):

    pid_m = tl.program_id(
        0
    )

    pid_n = tl.program_id(
        1
    )


    offs_m = (
        pid_m * BM
        +
        tl.arange(
            0,
            BM,
        )
    )


    offs_n = (
        pid_n * BN
        +
        tl.arange(
            0,
            BN,
        )
    )


    norm_sq = tl.load(
        norm_ptr
    )


    scale = (
        clip_threshold
        /
        (
            tl.sqrt(
                norm_sq
            )
            +
            1e-12
        )
    )


    scale = tl.minimum(
        scale,
        1.0,
    )


    acc = tl.zeros(

        (
            BM,
            BN,
        ),

        dtype=
            tl.float32,
    )


    for k in range(
        0,
        K,
        BK,
    ):

        offs_k = (
            k
            +
            tl.arange(
                0,
                BK,
            )
        )


        g = tl.load(

            gt_ptr
            +
            offs_m[:, None]
            *
            stride_gm
            +
            offs_k[None, :]
            *
            stride_gk,

            mask=(
                (
                    offs_m[:, None]
                    <
                    M
                )
                &
                (
                    offs_k[None, :]
                    <
                    K
                )
            ),

            other=0.0,
        )


        z = tl.load(

            z_ptr
            +
            offs_k[:, None]
            *
            stride_zk
            +
            offs_n[None, :]
            *
            stride_zn,

            mask=(
                (
                    offs_k[:, None]
                    <
                    K
                )
                &
                (
                    offs_n[None, :]
                    <
                    N
                )
            ),

            other=0.0,
        )


        acc = tl.dot(

            g,
            z,
            acc,

            input_precision=
                "ieee",
        )


    mask = (
        (
            offs_m[:, None]
            <
            M
        )
        &
        (
            offs_n[None, :]
            <
            N
        )
    )


    w = tl.load(

        src_ptr
        +
        offs_m[:, None]
        *
        stride_sm
        +
        offs_n[None, :]
        *
        stride_sn,

        mask=
            mask,

        other=
            0.0,
    )


    # Real gradient descent.
    updated = (
        w
        -
        learning_rate
        *
        scale
        *
        acc
    )


    tl.store(

        dst_ptr
        +
        offs_m[:, None]
        *
        stride_dm
        +
        offs_n[None, :]
        *
        stride_dn,

        updated,

        mask=
            mask,
    )


# =============================================================================
# A100 CONFIG
# =============================================================================

NORM_BM = 64
NORM_BN = 128
NORM_BK = 64
NORM_WARPS = 4


UPDATE_BM = 64
UPDATE_BN = 64
UPDATE_BK = 64
UPDATE_WARPS = 8


norm_ws = torch.zeros(

    1,

    device=
        DEVICE,

    dtype=
        torch.float32,
)


norm_grid = (

    triton.cdiv(
        M,
        NORM_BM,
    ),

    triton.cdiv(
        N,
        NORM_BN,
    ),
)


update_grid = (

    triton.cdiv(
        M,
        UPDATE_BM,
    ),

    triton.cdiv(
        N,
        UPDATE_BN,
    ),
)


# =============================================================================
# KERNEL HELPERS
# =============================================================================

def gradient_norm(
    gt,
    z,
):

    K = int(
        z.shape[0]
    )


    norm_ws.zero_()


    norm_kernel[
        norm_grid
    ](

        gt,
        z,

        norm_ws,

        gt.stride(0),
        gt.stride(1),

        z.stride(0),
        z.stride(1),

        M=M,
        N=N,
        K=K,

        BM=
            NORM_BM,

        BN=
            NORM_BN,

        BK=
            NORM_BK,

        num_warps=
            NORM_WARPS,
    )


    torch.cuda.synchronize()


    return math.sqrt(
        float(
            norm_ws.item()
        )
    )


def triton_update(
    gt,
    z,

    src,
    dst,

    learning_rate,
):

    K = int(
        z.shape[0]
    )


    norm_ws.zero_()


    norm_kernel[
        norm_grid
    ](

        gt,
        z,

        norm_ws,

        gt.stride(0),
        gt.stride(1),

        z.stride(0),
        z.stride(1),

        M=M,
        N=N,
        K=K,

        BM=
            NORM_BM,

        BN=
            NORM_BN,

        BK=
            NORM_BK,

        num_warps=
            NORM_WARPS,
    )


    update_kernel[
        update_grid
    ](

        gt,
        z,

        src,
        dst,

        norm_ws,

        gt.stride(0),
        gt.stride(1),

        z.stride(0),
        z.stride(1),

        src.stride(0),
        src.stride(1),

        dst.stride(0),
        dst.stride(1),

        learning_rate,
        1e30,

        M=M,
        N=N,
        K=K,

        BM=
            UPDATE_BM,

        BN=
            UPDATE_BN,

        BK=
            UPDATE_BK,

        num_warps=
            UPDATE_WARPS,
    )


# =============================================================================
# REAL GRADIENT CAPTURE
# =============================================================================

def capture_real_gradient(
    ids,
):

    capture = {}


    def gradient_forward(
        x,
    ):

        capture[
            "z"
        ] = (
            x
            .detach()
            .float()
            .contiguous()
        )


        # Current BF16 fast weight.
        y = F.linear(

            x,

            state.serve_active,

            bias=None,
        )


        # Cut lower graph.
        y_leaf = (
            y
            .detach()
            .requires_grad_(
                True
            )
        )


        y_leaf.retain_grad()


        capture[
            "y"
        ] = y_leaf


        return y_leaf


    target_layer.mlp.down_proj.forward = (
        gradient_forward
    )


    model.zero_grad(
        set_to_none=True
    )


    torch.cuda.synchronize()


    start = (
        time.perf_counter()
    )


    output = model(

        input_ids=
            ids,

        labels=
            ids,

        use_cache=
            False,
    )


    loss_before = float(
        output.loss
        .detach()
        .item()
    )


    output.loss.backward()


    torch.cuda.synchronize()


    capture_ms = (

        time.perf_counter()
        -
        start

    ) * 1000.0


    z = (

        capture[
            "z"
        ][
            0
        ]

        .float()
        .contiguous()
    )


    g = (

        capture[
            "y"
        ]
        .grad[
            0
        ]

        .float()
        .contiguous()
    )


    gt = (
        g
        .T
        .contiguous()
    )


    target_layer.mlp.down_proj.forward = (
        fast_weight_forward
    )


    del output
    del capture


    gc.collect()

    torch.cuda.empty_cache()


    return (
        gt,
        z,
        loss_before,
        capture_ms,
    )


# =============================================================================
# LOSS HELPER
# =============================================================================

@torch.no_grad()
def evaluate_loss(
    ids,
):

    output = model(

        input_ids=
            ids,

        labels=
            ids,

        use_cache=
            False,
    )


    value = float(
        output.loss.item()
    )


    del output


    return value


# =============================================================================
# CHUNKED WEIGHT METRICS
#
# Avoid allocating another full 271 MB difference tensor just to calculate
# statistics.
# =============================================================================

@torch.no_grad()
def count_changed(
    a,
    b,
    rows=128,
):

    count = 0


    for start in range(
        0,
        a.shape[0],
        rows,
    ):

        end = min(
            start + rows,
            a.shape[0],
        )


        count += int(

            (
                a[
                    start:end
                ]
                !=
                b[
                    start:end
                ]
            )
            .sum()
            .item()
        )


    return count


@torch.no_grad()
def cumulative_publication_stats():

    fp32_sq = 0.0

    bf16_sq = 0.0

    error_sq = 0.0

    changed = 0


    rows = 64


    for start in range(
        0,
        M,
        rows,
    ):

        end = min(
            start + rows,
            M,
        )


        master_delta = (

            state.master_active[
                start:end
            ]

            -
            MASTER_BASE[
                start:end
            ]
        )


        bf16_delta = (

            state.serve_active[
                start:end
            ]
            .float()

            -
            BASE_BF16[
                start:end
            ]
            .float()
        )


        fp32_sq += float(

            (
                master_delta
                *
                master_delta
            )
            .sum()
            .item()
        )


        bf16_sq += float(

            (
                bf16_delta
                *
                bf16_delta
            )
            .sum()
            .item()
        )


        error = (
            bf16_delta
            -
            master_delta
        )


        error_sq += float(

            (
                error
                *
                error
            )
            .sum()
            .item()
        )


        changed += int(

            (
                state.serve_active[
                    start:end
                ]
                !=
                BASE_BF16[
                    start:end
                ]
            )
            .sum()
            .item()
        )


    fp32_norm = math.sqrt(
        fp32_sq
    )


    bf16_norm = math.sqrt(
        bf16_sq
    )


    error_norm = math.sqrt(
        error_sq
    )


    relative_publication_error = (

        error_norm
        /
        max(
            fp32_norm,
            1e-30,
        )
    )


    changed_fraction = (

        changed
        /
        state.serve_active.numel()
    )


    return {

        "fp32_cumulative_norm":
            fp32_norm,

        "bf16_cumulative_norm":
            bf16_norm,

        "publication_relative_error":
            relative_publication_error,

        "bf16_changed_count":
            changed,

        "bf16_changed_fraction":
            changed_fraction,
    }


# =============================================================================
# JIT WARMUP
# =============================================================================

print()
print("=" * 120)
print("JIT WARMUP")
print("=" * 120)


reset_all_state()


warmup_ids = (
    adapt_stream[
        :,
        :ADAPT_CHUNK
    ]
)


warm_gt, warm_z, _, _ = (
    capture_real_gradient(
        warmup_ids
    )
)


warm_norm = (
    gradient_norm(
        warm_gt,
        warm_z,
    )
)


warm_lr = (
    TARGET_DELTA_NORM
    /
    max(
        warm_norm,
        1e-30,
    )
)


triton_update(

    warm_gt,
    warm_z,

    state.master_active,
    state.master_staging,

    warm_lr,
)


torch.cuda.synchronize()


print(
    "Warmup gradient norm  :",
    f"{warm_norm:.8e}",
)

print(
    "JIT warmup passed."
)


del warm_gt
del warm_z


reset_all_state()


# =============================================================================
# INITIAL HELD-OUT LOSS
# =============================================================================

initial_heldout_loss = (
    evaluate_loss(
        heldout_ids
    )
)


print()
print(
    "Initial held-out loss :",
    f"{initial_heldout_loss:.8f}",
)


# =============================================================================
# MULTI-STEP ONLINE ADAPTATION
# =============================================================================

print()
print("=" * 185)
print("16-STEP ONLINE ADAPTATION")
print("=" * 185)


print(
    f"{'step':>5}"
    f"{'grad norm':>14}"
    f"{'LR':>13}"
    f"{'grad ms':>11}"
    f"{'kernel':>10}"
    f"{'publish':>10}"
    f"{'total':>10}"
    f"{'ΔFP32 cum':>13}"
    f"{'ΔBF16 cum':>13}"
    f"{'inc changed':>13}"
    f"{'cum changed':>13}"
    f"{'pub err':>11}"
    f"{'adapt pre':>12}"
    f"{'adapt post':>12}"
    f"{'heldout':>12}"
)


print(
    "-" * 185
)


step_results = []


for step in range(
    NUM_UPDATES
):

    # -------------------------------------------------------------------------
    # Current 256-token adaptation block.
    # -------------------------------------------------------------------------

    start_token = (
        step
        *
        ADAPT_CHUNK
    )


    end_token = (
        start_token
        +
        ADAPT_CHUNK
    )


    chunk = (

        adapt_stream[
            :,
            start_token:
            end_token
        ]

        .contiguous()
    )


    # -------------------------------------------------------------------------
    # Obtain REAL gradient under current BF16 serving version.
    # -------------------------------------------------------------------------

    gt, z, adapt_loss_before, gradient_capture_ms = (

        capture_real_gradient(
            chunk
        )
    )


    grad_norm = (
        gradient_norm(
            gt,
            z,
        )
    )


    learning_rate = (

        TARGET_DELTA_NORM
        /
        max(
            grad_norm,
            1e-30,
        )
    )


    # -------------------------------------------------------------------------
    # Remember old serving buffer so we can measure this publication.
    # -------------------------------------------------------------------------

    old_serving = (
        state.serve_active
    )


    new_serving = (
        state.serve_staging
    )


    # -------------------------------------------------------------------------
    # Update + BF16 publication timing.
    # -------------------------------------------------------------------------

    update_start = (
        torch.cuda.Event(
            enable_timing=True
        )
    )


    kernel_done = (
        torch.cuda.Event(
            enable_timing=True
        )
    )


    publication_done = (
        torch.cuda.Event(
            enable_timing=True
        )
    )


    update_start.record()


    triton_update(

        gt,
        z,

        state.master_active,
        state.master_staging,

        learning_rate,
    )


    kernel_done.record()


    # FP32 master staging -> BF16 serving staging.
    state.serve_staging.copy_(
        state.master_staging
    )


    publication_done.record()


    publication_done.synchronize()


    kernel_ms = (
        update_start.elapsed_time(
            kernel_done
        )
    )


    publish_ms = (
        kernel_done.elapsed_time(
            publication_done
        )
    )


    total_update_ms = (
        update_start.elapsed_time(
            publication_done
        )
    )


    # -------------------------------------------------------------------------
    # Incremental BF16 visibility.
    # -------------------------------------------------------------------------

    incremental_changed = (
        count_changed(
            new_serving,
            old_serving,
        )
    )


    incremental_changed_fraction = (

        incremental_changed
        /
        new_serving.numel()
    )


    # -------------------------------------------------------------------------
    # Atomic logical version transition.
    # -------------------------------------------------------------------------

    commit_state()


    # -------------------------------------------------------------------------
    # Cumulative FP32/BF16 quantization statistics.
    # -------------------------------------------------------------------------

    publication_stats = (
        cumulative_publication_stats()
    )


    # -------------------------------------------------------------------------
    # Quality after publication.
    # -------------------------------------------------------------------------

    adapt_loss_after = (
        evaluate_loss(
            chunk
        )
    )


    heldout_loss = (
        evaluate_loss(
            heldout_ids
        )
    )


    result = {

        "step":
            step + 1,

        "version":
            state.version,

        "gradient_norm":
            grad_norm,

        "learning_rate":
            learning_rate,

        "gradient_capture_ms":
            gradient_capture_ms,

        "kernel_ms":
            kernel_ms,

        "publish_ms":
            publish_ms,

        "total_update_ms":
            total_update_ms,

        "incremental_changed_count":
            incremental_changed,

        "incremental_changed_fraction":
            incremental_changed_fraction,

        "fp32_cumulative_norm":
            publication_stats[
                "fp32_cumulative_norm"
            ],

        "bf16_cumulative_norm":
            publication_stats[
                "bf16_cumulative_norm"
            ],

        "publication_relative_error":
            publication_stats[
                "publication_relative_error"
            ],

        "bf16_changed_count":
            publication_stats[
                "bf16_changed_count"
            ],

        "bf16_changed_fraction":
            publication_stats[
                "bf16_changed_fraction"
            ],

        "adapt_loss_before":
            adapt_loss_before,

        "adapt_loss_after":
            adapt_loss_after,

        "adapt_improvement":
            adapt_loss_before
            -
            adapt_loss_after,

        "heldout_loss":
            heldout_loss,

        "heldout_change_vs_initial":
            initial_heldout_loss
            -
            heldout_loss,
    }


    step_results.append(
        result
    )


    print(
        f"{step + 1:>5}"
        f"{grad_norm:>14.4e}"
        f"{learning_rate:>13.3e}"
        f"{gradient_capture_ms:>11.2f}"
        f"{kernel_ms:>10.3f}"
        f"{publish_ms:>10.3f}"
        f"{total_update_ms:>10.3f}"
        f"{publication_stats['fp32_cumulative_norm']:>13.4e}"
        f"{publication_stats['bf16_cumulative_norm']:>13.4e}"
        f"{incremental_changed_fraction * 100:>12.3f}%"
        f"{publication_stats['bf16_changed_fraction'] * 100:>12.3f}%"
        f"{publication_stats['publication_relative_error']:>11.4f}"
        f"{adapt_loss_before:>12.6f}"
        f"{adapt_loss_after:>12.6f}"
        f"{heldout_loss:>12.6f}"
    )


    del gt
    del z
    del chunk


    gc.collect()

    torch.cuda.empty_cache()


# =============================================================================
# MULTI-STEP SUMMARY
# =============================================================================

first_step = (
    step_results[
        0
    ]
)


last_step = (
    step_results[
        -1
    ]
)


print()
print("=" * 120)
print("MULTI-STEP SUMMARY")
print("=" * 120)


print(
    "Updates completed              :",
    NUM_UPDATES,
)

print(
    "Initial held-out loss          :",
    f"{initial_heldout_loss:.8f}",
)

print(
    "Final held-out loss            :",
    f"{last_step['heldout_loss']:.8f}",
)

print(
    "Held-out improvement           :",
    f"{initial_heldout_loss - last_step['heldout_loss']:.8f}",
)


print()
print(
    "Step-1 FP32 cumulative ||ΔW||  :",
    f"{first_step['fp32_cumulative_norm']:.8e}",
)

print(
    "Step-16 FP32 cumulative ||ΔW|| :",
    f"{last_step['fp32_cumulative_norm']:.8e}",
)


print()
print(
    "Step-1 BF16 cumulative ||ΔW||  :",
    f"{first_step['bf16_cumulative_norm']:.8e}",
)

print(
    "Step-16 BF16 cumulative ||ΔW|| :",
    f"{last_step['bf16_cumulative_norm']:.8e}",
)


print()
print(
    "Step-1 BF16 changed fraction   :",
    f"{first_step['bf16_changed_fraction'] * 100:.4f}%",
)

print(
    "Step-16 BF16 changed fraction  :",
    f"{last_step['bf16_changed_fraction'] * 100:.4f}%",
)


print()
print(
    "Step-1 publication error       :",
    f"{first_step['publication_relative_error']:.6f}",
)

print(
    "Step-16 publication error      :",
    f"{last_step['publication_relative_error']:.6f}",
)


print()
print(
    "Median kernel time             :",
    f"{statistics.median([x['kernel_ms'] for x in step_results]):.3f} ms",
)

print(
    "Median BF16 publication        :",
    f"{statistics.median([x['publish_ms'] for x in step_results]):.3f} ms",
)

print(
    "Median total ready latency     :",
    f"{statistics.median([x['total_update_ms'] for x in step_results]):.3f} ms",
)

print(
    "Median real-gradient capture   :",
    f"{statistics.median([x['gradient_capture_ms'] for x in step_results]):.3f} ms",
)


# =============================================================================
# PREPARE A SINGLE REAL W0 -> W1 UPDATE FOR STALENESS EXPERIMENT
#
# We intentionally isolate staleness from the 16-step cumulative state.
# =============================================================================

print()
print("=" * 120)
print("PREPARING SINGLE UPDATE FOR STALENESS SWEEP")
print("=" * 120)


reset_all_state()


first_chunk = (

    adapt_stream[
        :,
        :ADAPT_CHUNK
    ]

    .contiguous()
)


stale_gt, stale_z, stale_loss_before, _ = (

    capture_real_gradient(
        first_chunk
    )
)


stale_grad_norm = (
    gradient_norm(
        stale_gt,
        stale_z,
    )
)


stale_lr = (

    TARGET_DELTA_NORM
    /
    max(
        stale_grad_norm,
        1e-30,
    )
)


# Compute W1 into staging but DO NOT COMMIT.
triton_update(

    stale_gt,
    stale_z,

    MASTER_A,
    MASTER_B,

    stale_lr,
)


SERVE_B.copy_(
    MASTER_B
)


torch.cuda.synchronize()


# At this point:
#
# MASTER_A / SERVE_A = W0
# MASTER_B / SERVE_B = W1


w1_changed_count = (
    count_changed(
        SERVE_B,
        SERVE_A,
    )
)


print(
    "Gradient norm         :",
    f"{stale_grad_norm:.8e}",
)

print(
    "Learning rate         :",
    f"{stale_lr:.8e}",
)

print(
    "BF16 W1 changed       :",
    f"{w1_changed_count:,}",
)

print(
    "BF16 W1 changed frac  :",
    f"{100 * w1_changed_count / SERVE_A.numel():.4f}%",
)


del stale_gt
del stale_z


# =============================================================================
# STALENESS EVALUATION
#
# Teacher-forced autoregressive evaluation.
#
# Prefix KV:
#       always produced with W0.
#
# Then:
#
#   L=None -> never switch, always W0
#   L=0    -> W1 used from first measured decode
#   L=1    -> 1 measured token uses W0, then W1
#   L=2    -> 2 tokens use W0, then W1
#   ...
#
# This models delayed update visibility while retaining the already-produced
# KV cache.
# =============================================================================

def percentile(
    values,
    p,
):

    values = sorted(
        values
    )


    if not values:
        return 0.0


    position = (
        len(values)
        -
        1
    ) * p


    lo = int(
        math.floor(
            position
        )
    )


    hi = int(
        math.ceil(
            position
        )
    )


    if lo == hi:
        return values[
            lo
        ]


    fraction = (
        position
        -
        lo
    )


    return (

        values[
            lo
        ]
        *
        (
            1.0
            -
            fraction
        )

        +

        values[
            hi
        ]
        *
        fraction
    )


def run_staleness_eval(
    lag,
):

    # -------------------------------------------------------------------------
    # Restore logical W0 / W1 pointer configuration.
    #
    # Do NOT overwrite buffers:
    #
    # SERVE_A = W0
    # SERVE_B = W1
    # -------------------------------------------------------------------------

    state.serve_active = (
        SERVE_A
    )

    state.serve_staging = (
        SERVE_B
    )


    state.version = 0

    state.last_version = None

    state.last_pointer = None


    seq = (
        heldout_ids[
            :,
            :
            STALENESS_PREFIX
            +
            STALENESS_STEPS
        ]
        .contiguous()
    )


    # -------------------------------------------------------------------------
    # Build almost all prefix KV with W0.
    #
    # We intentionally leave the final prefix token to the measured loop.
    # This allows L=0 to make W1 visible before the first measured prediction.
    # -------------------------------------------------------------------------

    prefix = (
        seq[
            :,
            :
            STALENESS_PREFIX
            -
            1
        ]
    )


    with torch.no_grad():

        output = model(

            input_ids=
                prefix,

            use_cache=
                True,
        )


    past = (
        output.past_key_values
    )


    del output


    losses = []

    latencies = []

    versions = []


    for step in range(
        STALENESS_STEPS
    ):

        # ---------------------------------------------------------------------
        # Version publication boundary.
        # ---------------------------------------------------------------------

        if (
            lag is not None
            and
            step == lag
        ):

            state.serve_active, state.serve_staging = (
                state.serve_staging,
                state.serve_active,
            )


            state.version = 1


        input_position = (
            STALENESS_PREFIX
            -
            1
            +
            step
        )


        target_position = (
            input_position
            +
            1
        )


        input_token = (
            seq[
                :,
                input_position:
                input_position
                +
                1
            ]
        )


        target_token = (
            seq[
                :,
                target_position
            ]
        )


        torch.cuda.synchronize()


        token_start = (
            time.perf_counter()
        )


        with torch.no_grad():

            output = model(

                input_ids=
                    input_token,

                past_key_values=
                    past,

                use_cache=
                    True,
            )


        past = (
            output.past_key_values
        )


        torch.cuda.synchronize()


        token_ms = (

            time.perf_counter()
            -
            token_start

        ) * 1000.0


        logits = (
            output.logits[
                :,
                -1,
                :
            ]
            .float()
        )


        nll = float(

            F.cross_entropy(
                logits,
                target_token,
            )
            .item()
        )


        losses.append(
            nll
        )


        latencies.append(
            token_ms
        )


        versions.append(
            state.version
        )


        del output


    mean_nll = (
        statistics.mean(
            losses
        )
    )


    perplexity = (
        math.exp(
            min(
                mean_nll,
                20.0,
            )
        )
    )


    result = {

        "lag":
            lag,

        "mean_nll":
            mean_nll,

        "perplexity":
            perplexity,

        "p50_ms":
            percentile(
                latencies,
                0.50,
            ),

        "p95_ms":
            percentile(
                latencies,
                0.95,
            ),

        "versions":
            versions,

        "losses":
            losses,
    }


    del past


    gc.collect()

    torch.cuda.empty_cache()


    return result


# =============================================================================
# RUN STALENESS SWEEP
# =============================================================================

print()
print("=" * 120)
print("STALENESS SWEEP")
print("=" * 120)


staleness_results = []


print(
    f"{'lag':>10}"
    f"{'mean NLL':>15}"
    f"{'perplexity':>15}"
    f"{'p50 ms':>12}"
    f"{'p95 ms':>12}"
)


print(
    "-" * 70
)


for lag in STALENESS_LAGS:

    result = (
        run_staleness_eval(
            lag
        )
    )


    staleness_results.append(
        result
    )


    lag_label = (

        "W0-only"

        if lag is None

        else
        str(
            lag
        )
    )


    print(
        f"{lag_label:>10}"
        f"{result['mean_nll']:>15.8f}"
        f"{result['perplexity']:>15.8f}"
        f"{result['p50_ms']:>12.3f}"
        f"{result['p95_ms']:>12.3f}"
    )


# =============================================================================
# STALENESS DELTAS
# =============================================================================

baseline_stale = next(

    x

    for x in staleness_results

    if x[
        "lag"
    ]
    is None
)


lag0 = next(

    x

    for x in staleness_results

    if x[
        "lag"
    ]
    ==
    0
)


print()
print("=" * 120)
print("STALENESS QUALITY SUMMARY")
print("=" * 120)


print(
    "W0-only NLL           :",
    f"{baseline_stale['mean_nll']:.8f}",
)

print(
    "Immediate W1 NLL      :",
    f"{lag0['mean_nll']:.8f}",
)

print(
    "Immediate improvement :",
    f"{baseline_stale['mean_nll'] - lag0['mean_nll']:.8f}",
)


print()


for result in staleness_results:

    if result[
        "lag"
    ] is None:

        continue


    quality_gain = (

        baseline_stale[
            "mean_nll"
        ]

        -
        result[
            "mean_nll"
        ]
    )


    retained_fraction = None


    immediate_gain = (

        baseline_stale[
            "mean_nll"
        ]

        -
        lag0[
            "mean_nll"
        ]
    )


    if abs(
        immediate_gain
    ) > 1e-12:

        retained_fraction = (

            quality_gain
            /
            immediate_gain
        )


    print(
        f"L={result['lag']:<2} | "
        f"NLL={result['mean_nll']:.8f} | "
        f"gain vs W0={quality_gain:+.8f} | "
        f"retained="
        +
        (
            "-"
            if retained_fraction is None
            else
            f"{retained_fraction * 100:.2f}%"
        )
    )


# =============================================================================
# SAVE
# =============================================================================

RESULT_PATH = Path(
    "async_ttt_qwen7b_multistep_staleness.json"
)


output_json = {

    "model":
        MODEL_ID,

    "shape": {

        "M":
            M,

        "N":
            N,

        "layer":
            TTT_LAYER,

        "adapt_chunk":
            ADAPT_CHUNK,
    },

    "config": {

        "num_updates":
            NUM_UPDATES,

        "target_delta_norm":
            TARGET_DELTA_NORM,

        "staleness_lags":
            STALENESS_LAGS,

        "staleness_prefix":
            STALENESS_PREFIX,

        "staleness_steps":
            STALENESS_STEPS,
    },

    "initial_heldout_loss":
        initial_heldout_loss,

    "multi_step":
        step_results,

    "staleness":
        staleness_results,
}


with open(

    RESULT_PATH,

    "w",

) as file:

    json.dump(

        output_json,

        file,

        indent=2,
    )


print()
print("=" * 120)
print("DONE")
print("=" * 120)


print(
    "Saved:",
    RESULT_PATH.resolve(),
)


# =============================================================================
# RESTORE ORIGINAL MODULE
# =============================================================================

target_layer.mlp.down_proj.forward = (
    original_down_forward
)


SERVE_A.copy_(
    BASE_BF16
)


torch.cuda.synchronize()


free_bytes, total_bytes = (
    torch.cuda.mem_get_info()
)


print(
    "Original down_proj restored."
)

print(
    "Free VRAM:",
    f"{free_bytes / 1024**3:.2f} GiB",
)

FLUSHING PREVIOUS GPU STATE


Free VRAM : 39.08 GiB
Total VRAM: 39.49 GiB

QWEN2.5-7B — MULTI-STEP ASYNC-TTT
Model                  : Qwen/Qwen2.5-7B-Instruct
Python                 : 3.12.11
PyTorch                : 2.8.0+cu128
CUDA                   : 12.8
Triton                 : 3.4.0
GPU                    : NVIDIA A100-SXM4-40GB
SMs                    : 108
VRAM                   : 39.49 GiB

LOADING QWEN2.5-7B


Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

hidden_size            : 3584
intermediate_size      : 18944
layers                 : 28
TTT layer              : 13
model dtype            : torch.bfloat16
FP32 fast weight       : 271.58 MB
BF16 serving weight    : 135.79 MB
Free after load        : 24.89 GiB

Adaptation stream     : 4096 tokens
Held-out stream       : 512 tokens

FAST-WEIGHT STATE
Master base           : 271.58 MB
Master A              : 271.58 MB
Master B              : 271.58 MB
BF16 base             : 135.79 MB
Serving A             : 135.79 MB
Serving B             : 135.79 MB

JIT WARMUP
Warmup gradient norm  : 1.04329182e+00
JIT warmup passed.

Initial held-out loss : 1.60790229

16-STEP ONLINE ADAPTATION
 step     grad norm           LR    grad ms    kernel   publish     total    ΔFP32 cum    ΔBF16 cum  inc changed  cum changed    pub err   adapt pre  adapt post     heldout
-----------------------------------------------------------------------------------------------------------------------------------------

In [1]:
# =============================================================================
# ASYNC-TTT — QWEN2.5-7B SYNTHETIC WORLD MEMORY BENCHMARK
#
# QUESTION:
#
# Can information supplied through a complex prompt be transferred into
# Qwen's fast weight and remain usable after the teaching prompt disappears?
#
#
# CONDITIONS
# ----------
#
# 1. W0 + NO CONTEXT
#       Base model has never seen the synthetic world.
#
# 2. W0 + FULL CONTEXT
#       Normal Transformer in-context-learning upper/reference condition.
#
# 3. W1 + NO CONTEXT
#       Teaching context removed.
#       Only adapted fast weight remains.
#
# 4. W1 + TINY CUE
#       Teaching context removed.
#       Model receives only:
#           "This question concerns the Helion Registry."
#
#
# QUESTION TYPES
# --------------
#
# direct_guild:
#       Entity -> guild
#
# direct_status:
#       Entity -> status
#
# compositional:
#       Entity
#          -> guild
#          -> guild base key
#          -> status
#          -> status suffix
#          -> final access key
#
# exception:
#       Entity has an override key which supersedes normal composition.
#
# paraphrase:
#       Same world relation, but wording differs from teaching prompt.
#
#
# METRICS
# -------
#
# - candidate accuracy
# - correct candidate probability among valid candidates
# - answer-token NLL
# - free-generation exact match
# - category accuracy
#
#
# TTT PATH
# --------
#
# real LM gradient:
#
#       Z = input to down_proj
#       G = dLoss / d(down_proj output)
#
#       dW = G^T Z
#
# FP32 master update:
#
#       W1 = W0 - eta * dW
#
# followed by:
#
#       FP32 master staging
#               |
#               v
#       BF16 serving staging
#               |
#               v
#          pointer swap
#
#
# IMPORTANT:
#
# This is a controlled synthetic-memory experiment.
# It is NOT yet a general benchmark of continual learning.
# =============================================================================


# =============================================================================
# IMPORTS
# =============================================================================

import gc
import json
import math
import random
import re
import statistics
import sys
import time

from collections import defaultdict
from pathlib import Path

import torch
import torch.nn.functional as F

import triton
import triton.language as tl

from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
)


# =============================================================================
# FLUSH GPU
# =============================================================================

print("=" * 120)
print("FLUSHING PREVIOUS GPU STATE")
print("=" * 120)


for name in list(globals().keys()):

    if name in {
        "model",
        "tokenizer",
        "target_layer",
        "state",
        "MASTER_A",
        "MASTER_B",
        "MASTER_BASE",
        "SERVE_A",
        "SERVE_B",
        "BASE_BF16",
        "capture",
        "output",
        "past_key_values",
        "next_token",
    }:

        try:
            del globals()[name]
        except Exception:
            pass


gc.collect()


if torch.cuda.is_available():

    torch.cuda.synchronize()

    torch.cuda.empty_cache()

    try:
        torch.cuda.ipc_collect()
    except Exception:
        pass

    gc.collect()

    torch.cuda.empty_cache()


free_bytes, total_bytes = (
    torch.cuda.mem_get_info()
)


print(
    "Free VRAM :",
    f"{free_bytes / 1024**3:.2f} GiB",
)

print(
    "Total VRAM:",
    f"{total_bytes / 1024**3:.2f} GiB",
)


if (
    free_bytes
    /
    1024**3
    <
    30
):

    raise RuntimeError(
        "Less than 30 GiB is free. "
        "Restart the kernel and rerun."
    )


# =============================================================================
# CONFIG
# =============================================================================

MODEL_ID = (
    "Qwen/Qwen2.5-7B-Instruct"
)


DEVICE = "cuda"


TTT_LAYER = 13


WORLD_SEED = 728419


NUM_ENTITIES = 12


NUM_ADAPT_STEPS = 8


TARGET_DELTA_NORM = 3e-2


MAX_ADAPT_TOKENS = 512


MAX_NEW_TOKENS = 12


DO_GENERATION = True


torch.manual_seed(
    0
)


random.seed(
    WORLD_SEED
)


torch.backends.cuda.matmul.allow_tf32 = False

torch.backends.cudnn.allow_tf32 = False


# =============================================================================
# SYNTHETIC WORLD GENERATION
# =============================================================================

print()
print("=" * 120)
print("CREATING SYNTHETIC WORLD")
print("=" * 120)


rng = random.Random(
    WORLD_SEED
)


# -----------------------------------------------------------------------------
# Generate nonce-like names.
#
# They are deliberately constructed rather than drawn from known proper nouns.
# -----------------------------------------------------------------------------

entity_prefixes = [
    "Velo",
    "Qem",
    "Zor",
    "Pellu",
    "Nivo",
    "Tarse",
    "Mire",
    "Sol",
    "Dra",
    "Kel",
    "Ruvo",
    "Xani",
    "Brel",
    "Ome",
    "Yari",
    "Cavo",
]


entity_suffixes = [
    "rin",
    "tra",
    "vane",
    "lux",
    "ra",
    "sen",
    "vek",
    "nix",
    "dor",
    "mir",
    "th",
    "zor",
]


guild_names = [
    "Nask",
    "Oriv",
    "Talm",
    "Phex",
]


statuses = [
    "amber",
    "cobalt",
    "ivory",
    "jade",
]


status_suffix = {
    "amber": "-R",
    "cobalt": "-S",
    "ivory": "-T",
    "jade": "-U",
}


def random_code(
    prefix_len=2,
):

    letters = "".join(
        rng.choice(
            "ABCDEFGHJKLMNPQRSTUVWXYZ"
        )
        for _ in range(
            prefix_len
        )
    )


    digits = "".join(
        rng.choice(
            "0123456789"
        )
        for _ in range(
            4
        )
    )


    return (
        f"{letters}-{digits}"
    )


# -----------------------------------------------------------------------------
# Entity names
# -----------------------------------------------------------------------------

entities = []


used_names = set()


while (
    len(entities)
    <
    NUM_ENTITIES
):

    name = (
        rng.choice(
            entity_prefixes
        )
        +
        rng.choice(
            entity_suffixes
        )
        +
        "-"
        +
        str(
            rng.randint(
                17,
                97,
            )
        )
    )


    if name not in used_names:

        entities.append(
            name
        )

        used_names.add(
            name
        )


# -----------------------------------------------------------------------------
# Guild base keys
# -----------------------------------------------------------------------------

guild_keys = {
    guild:
        random_code()
    for guild in guild_names
}


# -----------------------------------------------------------------------------
# Assign entities evenly enough across guild/status.
# -----------------------------------------------------------------------------

world_entities = []


for index, entity in enumerate(
    entities
):

    guild = (
        guild_names[
            index
            %
            len(
                guild_names
            )
        ]
    )


    status = (
        statuses[
            (
                index
                *
                3
            )
            %
            len(
                statuses
            )
        ]
    )


    world_entities.append(
        {
            "entity":
                entity,

            "guild":
                guild,

            "status":
                status,

            "override":
                None,
        }
    )


rng.shuffle(
    world_entities
)


# -----------------------------------------------------------------------------
# Two explicit exception entities.
# -----------------------------------------------------------------------------

exception_indices = [
    2,
    9,
]


for idx in exception_indices:

    world_entities[
        idx
    ][
        "override"
    ] = (
        random_code()
        +
        "-X"
    )


def valid_key(
    record,
):

    if (
        record[
            "override"
        ]
        is not None
    ):

        return (
            record[
                "override"
            ]
        )


    return (
        guild_keys[
            record[
                "guild"
            ]
        ]
        +
        status_suffix[
            record[
                "status"
            ]
        ]
    )


for record in world_entities:

    record[
        "valid_key"
    ] = (
        valid_key(
            record
        )
    )


WORLD = {

    "name":
        "Helion Registry",

    "guild_keys":
        guild_keys,

    "status_suffix":
        status_suffix,

    "entities":
        world_entities,
}


# =============================================================================
# TEACHING TEXT GENERATOR
#
# We generate several paraphrased descriptions of exactly the SAME world.
#
# This lets repeated adaptation steps see equivalent knowledge without simply
# repeating one byte-identical prompt.
# =============================================================================

def teaching_text(
    variant=0,
):

    lines = []


    if (
        variant
        %
        3
        ==
        0
    ):

        lines.append(
            "HELION REGISTRY — INTERNAL REFERENCE"
        )

        lines.append(
            "The following registry data is authoritative."
        )


    elif (
        variant
        %
        3
        ==
        1
    ):

        lines.append(
            "Authoritative Helion Registry briefing."
        )

        lines.append(
            "Memorize the relationships and operational rules below."
        )


    else:

        lines.append(
            "Helion Registry operational records."
        )

        lines.append(
            "All names, assignments, statuses and keys below are exact."
        )


    lines.append(
        ""
    )


    # -------------------------------------------------------------------------
    # Guild facts
    # -------------------------------------------------------------------------

    lines.append(
        "Guild access-key records:"
    )


    guild_items = list(
        guild_keys.items()
    )


    rng_variant = random.Random(
        WORLD_SEED
        +
        variant
        *
        101
    )


    rng_variant.shuffle(
        guild_items
    )


    for guild, key in guild_items:

        templates = [

            f"The {guild} guild uses base access key {key}.",

            f"Base credential for guild {guild}: {key}.",

            f"Registry mapping: {guild} -> base key {key}.",
        ]


        lines.append(
            rng_variant.choice(
                templates
            )
        )


    lines.append(
        ""
    )


    # -------------------------------------------------------------------------
    # Status rules
    # -------------------------------------------------------------------------

    lines.append(
        "Status transformation rules:"
    )


    for status in statuses:

        suffix = (
            status_suffix[
                status
            ]
        )


        templates = [

            (
                f"A {status}-status member appends "
                f"{suffix} to its guild base key."
            ),

            (
                f"For status {status}, the normal access-key suffix "
                f"is {suffix}."
            ),

            (
                f"Status rule: {status} => append {suffix}."
            ),
        ]


        lines.append(
            rng_variant.choice(
                templates
            )
        )


    lines.append(
        ""
    )


    # -------------------------------------------------------------------------
    # Entity facts
    # -------------------------------------------------------------------------

    lines.append(
        "Entity records:"
    )


    records = list(
        world_entities
    )


    rng_variant.shuffle(
        records
    )


    for record in records:

        entity = (
            record[
                "entity"
            ]
        )

        guild = (
            record[
                "guild"
            ]
        )

        status = (
            record[
                "status"
            ]
        )


        templates = [

            (
                f"{entity} belongs to guild {guild} "
                f"and currently has {status} status."
            ),

            (
                f"Registry entry {entity}: guild={guild}; "
                f"status={status}."
            ),

            (
                f"The assigned guild of {entity} is {guild}. "
                f"Its status classification is {status}."
            ),
        ]


        lines.append(
            rng_variant.choice(
                templates
            )
        )


        if (
            record[
                "override"
            ]
            is not None
        ):

            override = (
                record[
                    "override"
                ]
            )


            exception_templates = [

                (
                    f"{entity} has mirror clearance. "
                    f"Its override key is {override}. "
                    f"Mirror clearance overrides all guild/status key rules."
                ),

                (
                    f"Exception: {entity} uses override credential "
                    f"{override}; do not construct its key from guild "
                    f"and status."
                ),

                (
                    f"Special registry override for {entity}: {override}. "
                    f"This value supersedes its normal derived key."
                ),
            ]


            lines.append(
                rng_variant.choice(
                    exception_templates
                )
            )


    lines.append(
        ""
    )


    lines.append(
        "Key derivation rule:"
    )

    lines.append(
        "For a normal entity, take its guild base key and append the "
        "suffix corresponding to the entity's status."
    )

    lines.append(
        "If an explicit override exists, the override is the complete "
        "valid key and takes precedence."
    )


    return "\n".join(
        lines
    )


TEACHING_TEXT = (
    teaching_text(
        0
    )
)


print(
    TEACHING_TEXT
)


# =============================================================================
# BUILD QUESTIONS
# =============================================================================

QUESTIONS = []


def add_question(
    qid,
    category,
    question,
    answer,
    candidates,
):

    QUESTIONS.append(
        {
            "id":
                qid,

            "category":
                category,

            "question":
                question,

            "answer":
                answer,

            "candidates":
                list(
                    candidates
                ),
        }
    )


# -----------------------------------------------------------------------------
# Direct guild recall.
# -----------------------------------------------------------------------------

for i, record in enumerate(
    world_entities[
        :4
    ]
):

    add_question(

        f"guild_{i}",

        "direct_guild",

        (
            f"Which guild is {record['entity']} assigned to?"
        ),

        record[
            "guild"
        ],

        guild_names,
    )


# -----------------------------------------------------------------------------
# Direct status recall.
# -----------------------------------------------------------------------------

for i, record in enumerate(
    world_entities[
        4:8
    ]
):

    add_question(

        f"status_{i}",

        "direct_status",

        (
            f"What status classification does "
            f"{record['entity']} have?"
        ),

        record[
            "status"
        ],

        statuses,
    )


# -----------------------------------------------------------------------------
# Normal compositional keys.
# -----------------------------------------------------------------------------

normal_records = [

    record

    for record in world_entities

    if record[
        "override"
    ]
    is None
]


all_valid_keys = [

    record[
        "valid_key"
    ]

    for record in world_entities
]


for i, record in enumerate(
    normal_records[
        :4
    ]
):

    prompts = [

        (
            f"What is the complete valid access key for "
            f"{record['entity']}?"
        ),

        (
            f"Using its guild assignment and status rule, "
            f"which credential should {record['entity']} present?"
        ),

        (
            f"Derive the final Helion access credential belonging "
            f"to {record['entity']}."
        ),

        (
            f"If {record['entity']} authenticates now, what exact "
            f"key should it use?"
        ),
    ]


    add_question(

        f"compose_{i}",

        "compositional",

        prompts[
            i
            %
            len(
                prompts
            )
        ],

        record[
            "valid_key"
        ],

        all_valid_keys,
    )


# -----------------------------------------------------------------------------
# Explicit exception questions.
# -----------------------------------------------------------------------------

exception_records = [

    record

    for record in world_entities

    if record[
        "override"
    ]
    is not None
]


for i, record in enumerate(
    exception_records
):

    add_question(

        f"exception_{i}",

        "exception",

        (
            f"{record['entity']} has to authenticate. "
            f"What exact access key is valid for it?"
        ),

        record[
            "valid_key"
        ],

        all_valid_keys,
    )


# -----------------------------------------------------------------------------
# Paraphrased compositional questions not phrased like teaching statements.
# -----------------------------------------------------------------------------

for i, record in enumerate(
    normal_records[
        4:6
    ]
):

    add_question(

        f"paraphrase_{i}",

        "paraphrase",

        (
            f"Suppose the checkpoint receives identity "
            f"{record['entity']}. Which final credential string "
            f"should the checkpoint accept?"
        ),

        record[
            "valid_key"
        ],

        all_valid_keys,
    )


print()
print("=" * 120)
print("QUESTION SET")
print("=" * 120)


for q in QUESTIONS:

    print(
        f"{q['id']:<14} | "
        f"{q['category']:<16} | "
        f"{q['question']} -> {q['answer']}"
    )


print()
print(
    "Total questions:",
    len(
        QUESTIONS
    ),
)


# =============================================================================
# SAVE WORLD BEFORE MODEL WORK
# =============================================================================

WORLD_PATH = Path(
    "synthetic_helion_world.json"
)


with open(
    WORLD_PATH,
    "w",
) as f:

    json.dump(
        {
            "world":
                WORLD,

            "teaching_text":
                TEACHING_TEXT,

            "questions":
                QUESTIONS,
        },
        f,
        indent=2,
    )


print(
    "World saved:",
    WORLD_PATH.resolve(),
)


# =============================================================================
# LOAD MODEL
# =============================================================================

print()
print("=" * 120)
print("LOADING QWEN2.5-7B")
print("=" * 120)


tokenizer = (
    AutoTokenizer.from_pretrained(
        MODEL_ID,
        trust_remote_code=True,
    )
)


model = (
    AutoModelForCausalLM.from_pretrained(

        MODEL_ID,

        dtype=
            torch.bfloat16,

        device_map={
            "": 0
        },

        low_cpu_mem_usage=True,

        trust_remote_code=True,

        attn_implementation=
            "sdpa",
    )
)


model.eval()


for parameter in model.parameters():

    parameter.requires_grad_(
        False
    )


M = int(
    model.config.hidden_size
)


N = int(
    model.config.intermediate_size
)


target_layer = (
    model.model.layers[
        TTT_LAYER
    ]
)


original_down_forward = (
    target_layer
    .mlp
    .down_proj
    .forward
)


print(
    "M                     :",
    M,
)

print(
    "N                     :",
    N,
)

print(
    "TTT layer             :",
    TTT_LAYER,
)


# =============================================================================
# FAST-WEIGHT BUFFERS
# =============================================================================

ORIGINAL_WEIGHT = (
    target_layer
    .mlp
    .down_proj
    .weight
    .detach()
)


BASE_BF16 = (
    ORIGINAL_WEIGHT
    .clone()
    .contiguous()
)


MASTER_BASE = (
    BASE_BF16
    .float()
    .contiguous()
)


MASTER_A = (
    MASTER_BASE
    .clone()
    .contiguous()
)


MASTER_B = (
    MASTER_BASE
    .clone()
    .contiguous()
)


SERVE_A = (
    ORIGINAL_WEIGHT
)


SERVE_B = (
    BASE_BF16
    .clone()
    .contiguous()
)


# =============================================================================
# STATE
# =============================================================================

class FastWeightState:

    def __init__(
        self,
    ):

        self.master_active = MASTER_A

        self.master_staging = MASTER_B


        self.serve_active = SERVE_A

        self.serve_staging = SERVE_B


        self.version = 0

        self.last_pointer = None

        self.last_version = None


state = FastWeightState()


def reset_state():

    MASTER_A.copy_(
        MASTER_BASE
    )

    MASTER_B.copy_(
        MASTER_BASE
    )


    SERVE_A.copy_(
        BASE_BF16
    )

    SERVE_B.copy_(
        BASE_BF16
    )


    torch.cuda.synchronize()


    state.master_active = MASTER_A

    state.master_staging = MASTER_B


    state.serve_active = SERVE_A

    state.serve_staging = SERVE_B


    state.version = 0

    state.last_pointer = None

    state.last_version = None


def commit_state():

    state.master_active, state.master_staging = (
        state.master_staging,
        state.master_active,
    )


    state.serve_active, state.serve_staging = (
        state.serve_staging,
        state.serve_active,
    )


    state.version += 1


def fast_weight_forward(
    x,
):

    state.last_pointer = (
        state.serve_active.data_ptr()
    )


    state.last_version = (
        state.version
    )


    return F.linear(
        x,
        state.serve_active,
        bias=None,
    )


target_layer.mlp.down_proj.forward = (
    fast_weight_forward
)


# =============================================================================
# TRITON KERNELS
# =============================================================================

@triton.jit
def norm_kernel(
    gt_ptr,
    z_ptr,

    norm_ptr,

    stride_gm,
    stride_gk,

    stride_zk,
    stride_zn,

    M: tl.constexpr,
    N: tl.constexpr,
    K: tl.constexpr,

    BM: tl.constexpr,
    BN: tl.constexpr,
    BK: tl.constexpr,
):

    pid_m = tl.program_id(
        0
    )

    pid_n = tl.program_id(
        1
    )


    offs_m = (
        pid_m * BM
        +
        tl.arange(
            0,
            BM,
        )
    )


    offs_n = (
        pid_n * BN
        +
        tl.arange(
            0,
            BN,
        )
    )


    acc = tl.zeros(
        (
            BM,
            BN,
        ),
        dtype=
            tl.float32,
    )


    for k in range(
        0,
        K,
        BK,
    ):

        offs_k = (
            k
            +
            tl.arange(
                0,
                BK,
            )
        )


        g = tl.load(
            gt_ptr
            +
            offs_m[:, None]
            *
            stride_gm
            +
            offs_k[None, :]
            *
            stride_gk,

            mask=(
                (
                    offs_m[:, None]
                    <
                    M
                )
                &
                (
                    offs_k[None, :]
                    <
                    K
                )
            ),

            other=
                0.0,
        )


        z = tl.load(
            z_ptr
            +
            offs_k[:, None]
            *
            stride_zk
            +
            offs_n[None, :]
            *
            stride_zn,

            mask=(
                (
                    offs_k[:, None]
                    <
                    K
                )
                &
                (
                    offs_n[None, :]
                    <
                    N
                )
            ),

            other=
                0.0,
        )


        acc = tl.dot(
            g,
            z,
            acc,

            input_precision=
                "ieee",
        )


    local_sq = tl.sum(
        acc
        *
        acc
    )


    tl.atomic_add(
        norm_ptr,
        local_sq,
    )


@triton.jit
def update_kernel(
    gt_ptr,
    z_ptr,

    src_ptr,
    dst_ptr,

    norm_ptr,

    stride_gm,
    stride_gk,

    stride_zk,
    stride_zn,

    stride_sm,
    stride_sn,

    stride_dm,
    stride_dn,

    learning_rate,

    M: tl.constexpr,
    N: tl.constexpr,
    K: tl.constexpr,

    BM: tl.constexpr,
    BN: tl.constexpr,
    BK: tl.constexpr,
):

    pid_m = tl.program_id(
        0
    )

    pid_n = tl.program_id(
        1
    )


    offs_m = (
        pid_m * BM
        +
        tl.arange(
            0,
            BM,
        )
    )


    offs_n = (
        pid_n * BN
        +
        tl.arange(
            0,
            BN,
        )
    )


    acc = tl.zeros(
        (
            BM,
            BN,
        ),
        dtype=
            tl.float32,
    )


    for k in range(
        0,
        K,
        BK,
    ):

        offs_k = (
            k
            +
            tl.arange(
                0,
                BK,
            )
        )


        g = tl.load(
            gt_ptr
            +
            offs_m[:, None]
            *
            stride_gm
            +
            offs_k[None, :]
            *
            stride_gk,

            mask=(
                (
                    offs_m[:, None]
                    <
                    M
                )
                &
                (
                    offs_k[None, :]
                    <
                    K
                )
            ),

            other=
                0.0,
        )


        z = tl.load(
            z_ptr
            +
            offs_k[:, None]
            *
            stride_zk
            +
            offs_n[None, :]
            *
            stride_zn,

            mask=(
                (
                    offs_k[:, None]
                    <
                    K
                )
                &
                (
                    offs_n[None, :]
                    <
                    N
                )
            ),

            other=
                0.0,
        )


        acc = tl.dot(
            g,
            z,
            acc,

            input_precision=
                "ieee",
        )


    mask = (
        (
            offs_m[:, None]
            <
            M
        )
        &
        (
            offs_n[None, :]
            <
            N
        )
    )


    w = tl.load(
        src_ptr
        +
        offs_m[:, None]
        *
        stride_sm
        +
        offs_n[None, :]
        *
        stride_sn,

        mask=
            mask,

        other=
            0.0,
    )


    updated = (
        w
        -
        learning_rate
        *
        acc
    )


    tl.store(
        dst_ptr
        +
        offs_m[:, None]
        *
        stride_dm
        +
        offs_n[None, :]
        *
        stride_dn,

        updated,

        mask=
            mask,
    )


# =============================================================================
# KERNEL CONFIG
# =============================================================================

NBM = 64
NBN = 128
NBK = 64


UBM = 64
UBN = 64
UBK = 64


norm_ws = torch.zeros(
    1,
    device=
        DEVICE,
    dtype=
        torch.float32,
)


norm_grid = (
    triton.cdiv(
        M,
        NBM,
    ),

    triton.cdiv(
        N,
        NBN,
    ),
)


update_grid = (
    triton.cdiv(
        M,
        UBM,
    ),

    triton.cdiv(
        N,
        UBN,
    ),
)


def calculate_gradient_norm(
    gt,
    z,
):

    K = int(
        z.shape[0]
    )


    norm_ws.zero_()


    norm_kernel[
        norm_grid
    ](
        gt,
        z,

        norm_ws,

        gt.stride(0),
        gt.stride(1),

        z.stride(0),
        z.stride(1),

        M=M,
        N=N,
        K=K,

        BM=NBM,
        BN=NBN,
        BK=NBK,

        num_warps=4,
    )


    torch.cuda.synchronize()


    return math.sqrt(
        float(
            norm_ws.item()
        )
    )


def ttt_update(
    gt,
    z,

    src,
    dst,

    learning_rate,
):

    K = int(
        z.shape[0]
    )


    update_kernel[
        update_grid
    ](
        gt,
        z,

        src,
        dst,

        norm_ws,

        gt.stride(0),
        gt.stride(1),

        z.stride(0),
        z.stride(1),

        src.stride(0),
        src.stride(1),

        dst.stride(0),
        dst.stride(1),

        learning_rate,

        M=M,
        N=N,
        K=K,

        BM=UBM,
        BN=UBN,
        BK=UBK,

        num_warps=8,
    )


# =============================================================================
# REAL GRADIENT CAPTURE
# =============================================================================

def capture_world_gradient(
    text,
):

    encoded = tokenizer(
        text,

        return_tensors=
            "pt",

        truncation=
            True,

        max_length=
            MAX_ADAPT_TOKENS,
    )


    ids = (
        encoded[
            "input_ids"
        ]
        .to(
            DEVICE
        )
    )


    capture = {}


    def gradient_forward(
        x,
    ):

        capture[
            "z"
        ] = (
            x
            .detach()
            .float()
            .contiguous()
        )


        y = F.linear(
            x,
            state.serve_active,
            bias=None,
        )


        y_leaf = (
            y
            .detach()
            .requires_grad_(
                True
            )
        )


        y_leaf.retain_grad()


        capture[
            "y"
        ] = y_leaf


        return y_leaf


    target_layer.mlp.down_proj.forward = (
        gradient_forward
    )


    model.zero_grad(
        set_to_none=True
    )


    torch.cuda.synchronize()


    start = (
        time.perf_counter()
    )


    output = model(
        input_ids=
            ids,

        labels=
            ids,

        use_cache=
            False,
    )


    loss = float(
        output.loss
        .detach()
        .item()
    )


    output.loss.backward()


    torch.cuda.synchronize()


    capture_ms = (
        time.perf_counter()
        -
        start
    ) * 1000.0


    z = (
        capture[
            "z"
        ][
            0
        ]
        .float()
        .contiguous()
    )


    g = (
        capture[
            "y"
        ]
        .grad[
            0
        ]
        .float()
        .contiguous()
    )


    gt = (
        g
        .T
        .contiguous()
    )


    target_layer.mlp.down_proj.forward = (
        fast_weight_forward
    )


    del output
    del capture
    del ids
    del encoded


    gc.collect()

    torch.cuda.empty_cache()


    return (
        gt,
        z,
        loss,
        capture_ms,
    )


# =============================================================================
# QUESTION PROMPT BUILDERS
# =============================================================================

def no_context_prompt(
    question,
):

    return (
        "Answer the question using only the exact requested value. "
        "Do not explain.\n\n"
        f"Question: {question}\n"
        "Answer:"
    )


def tiny_cue_prompt(
    question,
):

    return (
        "This question concerns the Helion Registry that you previously "
        "learned. Answer with only the exact requested value.\n\n"
        f"Question: {question}\n"
        "Answer:"
    )


def full_context_prompt(
    question,
):

    return (
        TEACHING_TEXT
        +
        "\n\n"
        "Using the registry above, answer with only the exact requested "
        "value.\n\n"
        f"Question: {question}\n"
        "Answer:"
    )


# =============================================================================
# CANDIDATE SCORING
# =============================================================================

@torch.no_grad()
def score_answer(
    prompt,
    answer,
):

    prompt_ids = (
        tokenizer(
            prompt,
            add_special_tokens=True,
            return_tensors="pt",
        )[
            "input_ids"
        ]
        .to(
            DEVICE
        )
    )


    answer_ids = (
        tokenizer(
            " " + answer,
            add_special_tokens=False,
            return_tensors="pt",
        )[
            "input_ids"
        ]
        .to(
            DEVICE
        )
    )


    full_ids = torch.cat(
        [
            prompt_ids,
            answer_ids,
        ],
        dim=1,
    )


    output = model(
        input_ids=
            full_ids,

        use_cache=
            False,
    )


    logits = (
        output.logits
        .float()
    )


    prompt_len = int(
        prompt_ids.shape[1]
    )


    answer_len = int(
        answer_ids.shape[1]
    )


    prediction_logits = (
        logits[
            :,
            prompt_len - 1:
            prompt_len - 1 + answer_len,
            :
        ]
    )


    log_probs = F.log_softmax(
        prediction_logits,
        dim=-1,
    )


    token_log_probs = (
        log_probs
        .gather(
            -1,
            answer_ids.unsqueeze(
                -1
            ),
        )
        .squeeze(
            -1
        )
    )


    total_logprob = float(
        token_log_probs
        .sum()
        .item()
    )


    mean_nll = float(
        (
            -token_log_probs
            .mean()
        )
        .item()
    )


    del output
    del logits
    del log_probs
    del token_log_probs
    del full_ids
    del prompt_ids
    del answer_ids


    return (
        total_logprob,
        mean_nll,
    )


# =============================================================================
# FREE GENERATION
# =============================================================================

def normalize_answer(
    text,
):

    text = (
        text
        .strip()
        .lower()
    )


    text = re.sub(
        r"^[\"'`]+|[\"'`]+$",
        "",
        text,
    )


    text = re.sub(
        r"\s+",
        "",
        text,
    )


    text = text.rstrip(
        ".,;:!?"
    )


    return text


@torch.no_grad()
def generate_answer(
    prompt,
):

    encoded = tokenizer(
        prompt,
        return_tensors=
            "pt",
    )


    input_ids = (
        encoded[
            "input_ids"
        ]
        .to(
            DEVICE
        )
    )


    output_ids = model.generate(

        input_ids=
            input_ids,

        max_new_tokens=
            MAX_NEW_TOKENS,

        do_sample=
            False,

        use_cache=
            True,

        pad_token_id=
            tokenizer.eos_token_id,
    )


    generated = (
        output_ids[
            0,
            input_ids.shape[1]:
        ]
    )


    text = tokenizer.decode(
        generated,
        skip_special_tokens=True,
    )


    del encoded
    del input_ids
    del output_ids


    return text.strip()


# =============================================================================
# CONDITION EVALUATOR
# =============================================================================

def evaluate_condition(
    name,
    prompt_builder,
):

    rows = []


    print()
    print("-" * 120)
    print(name)
    print("-" * 120)


    for q in QUESTIONS:

        prompt = (
            prompt_builder(
                q[
                    "question"
                ]
            )
        )


        candidate_scores = []


        correct_nll = None


        for candidate in q[
            "candidates"
        ]:

            score, nll = (
                score_answer(
                    prompt,
                    candidate,
                )
            )


            candidate_scores.append(
                (
                    candidate,
                    score,
                )
            )


            if (
                candidate
                ==
                q[
                    "answer"
                ]
            ):

                correct_nll = (
                    nll
                )


        score_tensor = torch.tensor(
            [
                x[
                    1
                ]
                for x in candidate_scores
            ],
            dtype=
                torch.float64,
        )


        candidate_probs = torch.softmax(
            score_tensor,
            dim=0,
        )


        predicted_index = int(
            torch.argmax(
                score_tensor
            )
            .item()
        )


        predicted_candidate = (
            candidate_scores[
                predicted_index
            ][
                0
            ]
        )


        correct_index = next(

            i

            for i, (
                candidate,
                _
            )

            in enumerate(
                candidate_scores
            )

            if (
                candidate
                ==
                q[
                    "answer"
                ]
            )
        )


        correct_candidate_prob = float(
            candidate_probs[
                correct_index
            ]
            .item()
        )


        candidate_correct = (
            predicted_candidate
            ==
            q[
                "answer"
            ]
        )


        generated_text = ""


        generation_correct = False


        if DO_GENERATION:

            generated_text = (
                generate_answer(
                    prompt
                )
            )


            expected_norm = (
                normalize_answer(
                    q[
                        "answer"
                    ]
                )
            )


            generated_norm = (
                normalize_answer(
                    generated_text
                )
            )


            generation_correct = (

                generated_norm
                ==
                expected_norm

                or

                generated_norm.startswith(
                    expected_norm
                )
            )


        row = {

            "id":
                q[
                    "id"
                ],

            "category":
                q[
                    "category"
                ],

            "question":
                q[
                    "question"
                ],

            "answer":
                q[
                    "answer"
                ],

            "predicted_candidate":
                predicted_candidate,

            "candidate_correct":
                candidate_correct,

            "correct_candidate_probability":
                correct_candidate_prob,

            "answer_nll":
                correct_nll,

            "generated":
                generated_text,

            "generation_correct":
                generation_correct,
        }


        rows.append(
            row
        )


        print(
            f"{q['id']:<14} | "
            f"expected={q['answer']:<12} | "
            f"candidate={predicted_candidate:<12} | "
            f"P*={correct_candidate_prob:.4f} | "
            f"NLL={correct_nll:.4f} | "
            f"gen={generated_text[:30]!r}"
        )


    # -------------------------------------------------------------------------
    # Aggregate
    # -------------------------------------------------------------------------

    candidate_accuracy = (
        sum(
            row[
                "candidate_correct"
            ]
            for row in rows
        )
        /
        len(
            rows
        )
    )


    generation_accuracy = (
        sum(
            row[
                "generation_correct"
            ]
            for row in rows
        )
        /
        len(
            rows
        )
    )


    mean_candidate_prob = statistics.mean(
        row[
            "correct_candidate_probability"
        ]
        for row in rows
    )


    mean_nll = statistics.mean(
        row[
            "answer_nll"
        ]
        for row in rows
    )


    by_category = {}


    categories = sorted(
        set(
            row[
                "category"
            ]
            for row in rows
        )
    )


    for category in categories:

        subset = [
            row

            for row in rows

            if (
                row[
                    "category"
                ]
                ==
                category
            )
        ]


        by_category[
            category
        ] = {

            "candidate_accuracy":
                sum(
                    row[
                        "candidate_correct"
                    ]
                    for row in subset
                )
                /
                len(
                    subset
                ),

            "generation_accuracy":
                sum(
                    row[
                        "generation_correct"
                    ]
                    for row in subset
                )
                /
                len(
                    subset
                ),

            "mean_correct_probability":
                statistics.mean(
                    row[
                        "correct_candidate_probability"
                    ]
                    for row in subset
                ),

            "mean_nll":
                statistics.mean(
                    row[
                        "answer_nll"
                    ]
                    for row in subset
                ),
        }


    return {

        "name":
            name,

        "candidate_accuracy":
            candidate_accuracy,

        "generation_accuracy":
            generation_accuracy,

        "mean_correct_probability":
            mean_candidate_prob,

        "mean_nll":
            mean_nll,

        "by_category":
            by_category,

        "rows":
            rows,
    }


# =============================================================================
# RESET W0
# =============================================================================

reset_state()


# =============================================================================
# CONDITION 1
#
# W0 + NO CONTEXT
# =============================================================================

print()
print("=" * 120)
print("EVALUATING W0")
print("=" * 120)


W0_NO_CONTEXT = (
    evaluate_condition(
        "W0 + NO CONTEXT",
        no_context_prompt,
    )
)


# =============================================================================
# CONDITION 2
#
# W0 + FULL TEACHING CONTEXT
# =============================================================================

W0_FULL_CONTEXT = (
    evaluate_condition(
        "W0 + FULL TEACHING CONTEXT",
        full_context_prompt,
    )
)


# =============================================================================
# OPTIONAL W0 TINY-CUE CONTROL
# =============================================================================

W0_TINY_CUE = (
    evaluate_condition(
        "W0 + TINY CUE",
        tiny_cue_prompt,
    )
)


# =============================================================================
# ADAPT MODEL
# =============================================================================

print()
print("=" * 140)
print("REAL-GRADIENT WORLD ADAPTATION")
print("=" * 140)


adaptation_log = []


for step in range(
    NUM_ADAPT_STEPS
):

    text_variant = (
        teaching_text(
            step
        )
    )


    gt, z, loss_before, capture_ms = (
        capture_world_gradient(
            text_variant
        )
    )


    grad_norm = (
        calculate_gradient_norm(
            gt,
            z,
        )
    )


    learning_rate = (
        TARGET_DELTA_NORM
        /
        max(
            grad_norm,
            1e-30,
        )
    )


    start_event = (
        torch.cuda.Event(
            enable_timing=True
        )
    )


    kernel_event = (
        torch.cuda.Event(
            enable_timing=True
        )
    )


    publish_event = (
        torch.cuda.Event(
            enable_timing=True
        )
    )


    start_event.record()


    # Pass 1 norm has already been populated by calculate_gradient_norm().
    #
    # Pass 2 performs the actual gradient step.
    ttt_update(

        gt,
        z,

        state.master_active,
        state.master_staging,

        learning_rate,
    )


    kernel_event.record()


    state.serve_staging.copy_(
        state.master_staging
    )


    publish_event.record()


    publish_event.synchronize()


    kernel_ms = (
        start_event.elapsed_time(
            kernel_event
        )
    )


    publish_ms = (
        kernel_event.elapsed_time(
            publish_event
        )
    )


    total_ms = (
        start_event.elapsed_time(
            publish_event
        )
    )


    old_ptr = (
        state.serve_active.data_ptr()
    )


    commit_state()


    new_ptr = (
        state.serve_active.data_ptr()
    )


    with torch.no_grad():

        check_ids = (
            tokenizer(
                text_variant,

                return_tensors=
                    "pt",

                truncation=
                    True,

                max_length=
                    MAX_ADAPT_TOKENS,
            )[
                "input_ids"
            ]
            .to(
                DEVICE
            )
        )


        output = model(
            input_ids=
                check_ids,

            labels=
                check_ids,

            use_cache=
                False,
        )


        loss_after = float(
            output.loss.item()
        )


        del output
        del check_ids


    adaptation_log.append(
        {
            "step":
                step + 1,

            "version":
                state.version,

            "loss_before":
                loss_before,

            "loss_after":
                loss_after,

            "loss_improvement":
                loss_before
                -
                loss_after,

            "gradient_norm":
                grad_norm,

            "learning_rate":
                learning_rate,

            "gradient_capture_ms":
                capture_ms,

            "kernel_ms":
                kernel_ms,

            "publish_ms":
                publish_ms,

            "total_ready_ms":
                total_ms,

            "old_pointer":
                old_ptr,

            "new_pointer":
                new_ptr,
        }
    )


    print(
        f"step={step + 1:>2} | "
        f"version={state.version:>2} | "
        f"loss {loss_before:.6f} -> {loss_after:.6f} | "
        f"grad={grad_norm:.4e} | "
        f"lr={learning_rate:.4e} | "
        f"capture={capture_ms:.2f} ms | "
        f"kernel={kernel_ms:.3f} ms | "
        f"publish={publish_ms:.3f} ms | "
        f"total={total_ms:.3f} ms"
    )


    del gt
    del z


    gc.collect()

    torch.cuda.empty_cache()


# =============================================================================
# CONDITION 3
#
# W1 + NO CONTEXT
# =============================================================================

print()
print("=" * 120)
print("EVALUATING ADAPTED FAST WEIGHT")
print("=" * 120)


W1_NO_CONTEXT = (
    evaluate_condition(
        "W1 + NO CONTEXT",
        no_context_prompt,
    )
)


# =============================================================================
# CONDITION 4
#
# W1 + TINY CUE
# =============================================================================

W1_TINY_CUE = (
    evaluate_condition(
        "W1 + TINY CUE",
        tiny_cue_prompt,
    )
)


# =============================================================================
# OPTIONAL: W1 + FULL CONTEXT
# =============================================================================

W1_FULL_CONTEXT = (
    evaluate_condition(
        "W1 + FULL CONTEXT",
        full_context_prompt,
    )
)


# =============================================================================
# FINAL SUMMARY
# =============================================================================

conditions = [

    W0_NO_CONTEXT,

    W0_TINY_CUE,

    W0_FULL_CONTEXT,

    W1_NO_CONTEXT,

    W1_TINY_CUE,

    W1_FULL_CONTEXT,
]


print()
print("=" * 145)
print("SYNTHETIC WORLD MEMORY RESULTS")
print("=" * 145)


print(
    f"{'Condition':<32}"
    f"{'Cand Acc':>12}"
    f"{'Gen Acc':>12}"
    f"{'P(correct)':>14}"
    f"{'Answer NLL':>14}"
)


print(
    "-" * 90
)


for result in conditions:

    print(
        f"{result['name']:<32}"
        f"{result['candidate_accuracy'] * 100:>11.2f}%"
        f"{result['generation_accuracy'] * 100:>11.2f}%"
        f"{result['mean_correct_probability']:>14.4f}"
        f"{result['mean_nll']:>14.4f}"
    )


# =============================================================================
# MEMORY TRANSFER SCORE
#
# How much of the context advantage was recovered by W1 with no context?
#
# Candidate probability:
#
#       W0/no-context   = baseline
#       W0/full-context = ordinary context upper/reference
#       W1/no-context   = fast-weight memory
#
# transfer =
#
#       (W1_no - W0_no)
#       ----------------
#       (W0_ctx - W0_no)
#
# =============================================================================

p_base = (
    W0_NO_CONTEXT[
        "mean_correct_probability"
    ]
)


p_context = (
    W0_FULL_CONTEXT[
        "mean_correct_probability"
    ]
)


p_adapted = (
    W1_NO_CONTEXT[
        "mean_correct_probability"
    ]
)


denominator = (
    p_context
    -
    p_base
)


if abs(
    denominator
) > 1e-12:

    memory_transfer_fraction = (

        (
            p_adapted
            -
            p_base
        )
        /
        denominator
    )

else:

    memory_transfer_fraction = None


print()
print("=" * 120)
print("FAST-WEIGHT MEMORY TRANSFER")
print("=" * 120)


print(
    "W0 no-context P(correct) :",
    f"{p_base:.6f}",
)

print(
    "W0 full-context P(correct):",
    f"{p_context:.6f}",
)

print(
    "W1 no-context P(correct) :",
    f"{p_adapted:.6f}",
)


if memory_transfer_fraction is not None:

    print(
        "Recovered context advantage :",
        f"{memory_transfer_fraction * 100:.2f}%",
    )


# =============================================================================
# CATEGORY TABLE
# =============================================================================

print()
print("=" * 120)
print("CATEGORY ACCURACY")
print("=" * 120)


categories = sorted(
    set(
        q[
            "category"
        ]
        for q in QUESTIONS
    )
)


print(
    f"{'Category':<20}"
    f"{'W0 none':>12}"
    f"{'W0 context':>14}"
    f"{'W1 none':>12}"
    f"{'W1 cue':>12}"
)


print(
    "-" * 75
)


for category in categories:

    print(
        f"{category:<20}"
        f"{W0_NO_CONTEXT['by_category'][category]['candidate_accuracy'] * 100:>11.1f}%"
        f"{W0_FULL_CONTEXT['by_category'][category]['candidate_accuracy'] * 100:>13.1f}%"
        f"{W1_NO_CONTEXT['by_category'][category]['candidate_accuracy'] * 100:>11.1f}%"
        f"{W1_TINY_CUE['by_category'][category]['candidate_accuracy'] * 100:>11.1f}%"
    )


# =============================================================================
# SAVE EVERYTHING
# =============================================================================

RESULT_PATH = Path(
    "async_ttt_qwen7b_synthetic_world_memory.json"
)


with open(
    RESULT_PATH,
    "w",
) as f:

    json.dump(
        {
            "world":
                WORLD,

            "teaching_text":
                TEACHING_TEXT,

            "questions":
                QUESTIONS,

            "config": {
                "model":
                    MODEL_ID,

                "layer":
                    TTT_LAYER,

                "adapt_steps":
                    NUM_ADAPT_STEPS,

                "target_delta_norm":
                    TARGET_DELTA_NORM,

                "max_adapt_tokens":
                    MAX_ADAPT_TOKENS,
            },

            "adaptation":
                adaptation_log,

            "conditions": {
                "W0_no_context":
                    W0_NO_CONTEXT,

                "W0_tiny_cue":
                    W0_TINY_CUE,

                "W0_full_context":
                    W0_FULL_CONTEXT,

                "W1_no_context":
                    W1_NO_CONTEXT,

                "W1_tiny_cue":
                    W1_TINY_CUE,

                "W1_full_context":
                    W1_FULL_CONTEXT,
            },

            "memory_transfer_fraction":
                memory_transfer_fraction,
        },
        f,
        indent=2,
    )


print()
print("=" * 120)
print("DONE")
print("=" * 120)


print(
    "Saved world :",
    WORLD_PATH.resolve(),
)

print(
    "Saved result:",
    RESULT_PATH.resolve(),
)


# =============================================================================
# RESTORE ORIGINAL QWEN
# =============================================================================

target_layer.mlp.down_proj.forward = (
    original_down_forward
)


SERVE_A.copy_(
    BASE_BF16
)


torch.cuda.synchronize()


free_bytes, _ = (
    torch.cuda.mem_get_info()
)


print(
    "Original down_proj restored."
)

print(
    "Free VRAM:",
    f"{free_bytes / 1024**3:.2f} GiB",
)

FLUSHING PREVIOUS GPU STATE
Free VRAM : 39.08 GiB
Total VRAM: 39.49 GiB

CREATING SYNTHETIC WORLD
HELION REGISTRY — INTERNAL REFERENCE
The following registry data is authoritative.

Guild access-key records:
Base credential for guild Nask: KP-7374.
Registry mapping: Talm -> base key AX-7672.
Base credential for guild Phex: RM-6806.
The Oriv guild uses base access key MD-9742.

Status transformation rules:
Status rule: amber => append -R.
A cobalt-status member appends -S to its guild base key.
Status rule: ivory => append -T.
For status jade, the normal access-key suffix is -U.

Entity records:
Tarseth-60 belongs to guild Phex and currently has cobalt status.
Tarseth-60 has mirror clearance. Its override key is SQ-2424-X. Mirror clearance overrides all guild/status key rules.
Registry entry Velonix-86: guild=Talm; status=ivory.
Xanivane-94 belongs to guild Talm and currently has ivory status.
Registry entry Qemdor-62: guild=Talm; status=ivory.
Qemdor-62 has mirror clearance. Its overri

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

M                     : 3584
N                     : 18944
TTT layer             : 13

EVALUATING W0

------------------------------------------------------------------------------------------------------------------------
W0 + NO CONTEXT
------------------------------------------------------------------------------------------------------------------------
guild_0        | expected=Nask         | candidate=Talm         | P*=0.0000 | NLL=13.5215 | gen="Blacksmiths' Guild\nYou are an "
guild_1        | expected=Phex         | candidate=Talm         | P*=0.0093 | NLL=10.5211 | gen='Blacksmithing Guild\nYou are an'
guild_2        | expected=Talm         | candidate=Talm         | P*=0.9747 | NLL=7.6751 | gen='Blacksmithing\n```'
guild_3        | expected=Oriv         | candidate=Talm         | P*=0.0100 | NLL=9.9168 | gen="Blacksmiths' Guild\nYou are an "
status_0       | expected=amber        | candidate=amber        | P*=0.9348 | NLL=16.4928 | gen='Investigational drug'
status_1       |

In [1]:
# =============================================================================
# ASYNC-TTT — QWEN2.5-7B
#
# TRUE ONLINE ASSOCIATIVE MEMORY BENCHMARK
#
# Tests:
#
#   1. Synthetic unknown world
#   2. Fixed-K Triton kernel (no varying-K JIT pollution)
#   3. Answer-only QA write objective
#   4. Real LM gradient at Qwen layer 13
#   5. FP32 master A/B state
#   6. BF16 serving A/B state
#   7. Online packet-by-packet learning
#   8. Retrieval after EVERY new packet
#   9. Held-out paraphrases never used for writing
#  10. Direct entity memory
#  11. Guild-key / status-rule memory
#  12. Compositional derived-key reasoning
#  13. Exception/override memory
#  14. Full-context reference
#  15. No-context fast-weight retrieval
#  16. Online correction / overwrite of an old fact
#  17. Retention after correction
#
#
# IMPORTANT:
#
# This is the SEMANTIC online-learning benchmark.
#
# We already established separately that the Triton update/publish operation
# can execute asynchronously beside autoregressive decoding.
#
# Here we ask:
#
#       can the fast-weight state actually LEARN useful associations online?
#
#
# ONLINE WRITE:
#
#    new packet arrives
#          |
#          v
#    answer-only QA objective
#          |
#          v
#    real G = dL/dY
#          |
#          v
#      dW = G^T Z
#          |
#          v
#    FP32 master staging
#          |
#          v
#    BF16 serving staging
#          |
#          v
#      pointer commit
#          |
#          v
#    immediately query W_new
#
#
# The original teaching context is NOT supplied during retrieval.
# =============================================================================


# =============================================================================
# IMPORTS
# =============================================================================

import gc
import json
import math
import random
import re
import statistics
import sys
import time

from collections import defaultdict
from pathlib import Path

import torch
import torch.nn.functional as F

import triton
import triton.language as tl

from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
)


# =============================================================================
# FLUSH GPU
# =============================================================================

print("=" * 120)
print("FLUSHING PREVIOUS GPU STATE")
print("=" * 120)


_cleanup_names = [
    "model",
    "tokenizer",
    "target_layer",
    "state",
    "MASTER_BASE",
    "MASTER_A",
    "MASTER_B",
    "BASE_BF16",
    "SERVE_A",
    "SERVE_B",
    "output",
    "capture",
    "past_key_values",
    "next_token",
]


for _name in _cleanup_names:

    if _name in globals():

        try:
            del globals()[_name]
        except Exception:
            pass


gc.collect()


if torch.cuda.is_available():

    torch.cuda.synchronize()
    torch.cuda.empty_cache()

    try:
        torch.cuda.ipc_collect()
    except Exception:
        pass

    gc.collect()
    torch.cuda.empty_cache()


free_bytes, total_bytes = (
    torch.cuda.mem_get_info()
)


print(
    "Free VRAM :",
    f"{free_bytes / 1024**3:.2f} GiB",
)

print(
    "Total VRAM:",
    f"{total_bytes / 1024**3:.2f} GiB",
)


if (
    free_bytes
    /
    1024**3
    <
    30
):

    raise RuntimeError(
        "\nLess than 30 GiB free.\n"
        "Restart the kernel before running this experiment."
    )


# =============================================================================
# CONFIG
# =============================================================================

MODEL_ID = (
    "Qwen/Qwen2.5-7B-Instruct"
)


DEVICE = "cuda"


TTT_LAYER = 13


WORLD_SEED = 928371


NUM_ENTITIES = 8


# -----------------------------------------------------------------------------
# FIXED K.
#
# Every write packet is padded to exactly this length.
#
# Therefore Triton sees ONE compile-time K for the entire experiment.
# -----------------------------------------------------------------------------

WRITE_K = 384


# -----------------------------------------------------------------------------
# Multiple gradient steps are allowed on a packet, but NO previous packet
# needs to be replayed.
#
# This is genuine incremental/online updating.
# -----------------------------------------------------------------------------

UPDATES_PER_PACKET = 3


# Each normalized gradient update gets this FP32 Frobenius magnitude.
#
# Lower than our previous 0.03 × many updates because this experiment performs
# repeated packet-level writes.
# -----------------------------------------------------------------------------

TARGET_DELTA_NORM = 1.5e-2


MAX_NEW_TOKENS = 12


DO_GENERATION_AT_FINAL = True


torch.manual_seed(
    0
)


random.seed(
    WORLD_SEED
)


torch.backends.cuda.matmul.allow_tf32 = False
torch.backends.cudnn.allow_tf32 = False


# =============================================================================
# CREATE WORLD
# =============================================================================

print()
print("=" * 120)
print("CREATING ONLINE SYNTHETIC WORLD")
print("=" * 120)


rng = random.Random(
    WORLD_SEED
)


GUILDS = [
    "Nask",
    "Oriv",
    "Talm",
    "Phex",
]


STATUSES = [
    "amber",
    "cobalt",
    "ivory",
    "jade",
]


STATUS_SUFFIX = {
    "amber":
        "-R",

    "cobalt":
        "-S",

    "ivory":
        "-T",

    "jade":
        "-U",
}


def random_code():

    letters = "".join(
        rng.choice(
            "ABCDEFGHJKLMNPQRSTUVWXYZ"
        )
        for _ in range(
            2
        )
    )


    digits = "".join(
        rng.choice(
            "0123456789"
        )
        for _ in range(
            4
        )
    )


    return (
        f"{letters}-{digits}"
    )


GUILD_KEYS = {
    guild:
        random_code()
    for guild in GUILDS
}


prefixes = [
    "Velo",
    "Qem",
    "Zar",
    "Pellu",
    "Nivo",
    "Tarse",
    "Mire",
    "Sol",
    "Dra",
    "Kel",
    "Ruvo",
    "Xani",
]


suffixes = [
    "rin",
    "tra",
    "vane",
    "lux",
    "sen",
    "vek",
    "nix",
    "dor",
    "mir",
    "zor",
]


ENTITIES = []


while (
    len(
        ENTITIES
    )
    <
    NUM_ENTITIES
):

    name = (
        rng.choice(
            prefixes
        )
        +
        rng.choice(
            suffixes
        )
        +
        "-"
        +
        str(
            rng.randint(
                20,
                98,
            )
        )
    )


    if (
        name
        not in ENTITIES
    ):

        ENTITIES.append(
            name
        )


WORLD_RECORDS = []


for i, entity in enumerate(
    ENTITIES
):

    guild = (
        GUILDS[
            i
            %
            len(
                GUILDS
            )
        ]
    )


    status = (
        STATUSES[
            (
                i
                *
                3
                +
                1
            )
            %
            len(
                STATUSES
            )
        ]
    )


    WORLD_RECORDS.append(
        {
            "entity":
                entity,

            "guild":
                guild,

            "status":
                status,

            "override":
                None,
        }
    )


rng.shuffle(
    WORLD_RECORDS
)


# Two exception entities.
exception_indices = [
    2,
    6,
]


for idx in exception_indices:

    WORLD_RECORDS[
        idx
    ][
        "override"
    ] = (
        random_code()
        +
        "-X"
    )


def final_key(
    record,
):

    if (
        record[
            "override"
        ]
        is not None
    ):

        return (
            record[
                "override"
            ]
        )


    return (
        GUILD_KEYS[
            record[
                "guild"
            ]
        ]
        +
        STATUS_SUFFIX[
            record[
                "status"
            ]
        ]
    )


for record in WORLD_RECORDS:

    record[
        "valid_key"
    ] = (
        final_key(
            record
        )
    )


# =============================================================================
# DECLARATIVE FULL-CONTEXT REFERENCE
# =============================================================================

def build_full_context():

    lines = [

        "HELION REGISTRY — AUTHORITATIVE RECORD",

        "",

        "Guild base keys:",
    ]


    for guild in GUILDS:

        lines.append(
            f"{guild} uses base key "
            f"{GUILD_KEYS[guild]}."
        )


    lines.append(
        ""
    )


    lines.append(
        "Status suffix rules:"
    )


    for status in STATUSES:

        lines.append(
            f"{status} status appends "
            f"{STATUS_SUFFIX[status]}."
        )


    lines.append(
        ""
    )


    lines.append(
        "Entity records:"
    )


    for record in WORLD_RECORDS:

        lines.append(
            f"{record['entity']} belongs to "
            f"{record['guild']} and has "
            f"{record['status']} status."
        )


        if (
            record[
                "override"
            ]
            is not None
        ):

            lines.append(
                f"{record['entity']} has override "
                f"{record['override']}. "
                f"The override supersedes the derived key."
            )


    lines.append(
        ""
    )


    lines.append(
        "For a normal entity, concatenate its guild base key "
        "with its status suffix."
    )


    return "\n".join(
        lines
    )


FULL_CONTEXT = (
    build_full_context()
)


print(
    FULL_CONTEXT
)


# =============================================================================
# ONLINE WRITE PACKETS
#
# IMPORTANT:
#
# WRITE wording != TEST wording.
#
# We train associations with one phrasing and evaluate retrieval using
# held-out paraphrases.
# =============================================================================

def rule_write_examples():

    examples = []


    for guild in GUILDS:

        key = (
            GUILD_KEYS[
                guild
            ]
        )


        examples.extend(
            [
                (
                    f"State the registered base credential "
                    f"for guild {guild}.",
                    key,
                ),

                (
                    f"In the Helion registry, what code is "
                    f"mapped to guild {guild}?",
                    key,
                ),
            ]
        )


    for status in STATUSES:

        suffix = (
            STATUS_SUFFIX[
                status
            ]
        )


        examples.extend(
            [
                (
                    f"What suffix does {status} status contribute "
                    f"to an access credential?",
                    suffix,
                ),

                (
                    f"State the Helion suffix associated with "
                    f"status {status}.",
                    suffix,
                ),
            ]
        )


    return examples


def entity_write_examples(
    record,
):

    entity = (
        record[
            "entity"
        ]
    )


    guild = (
        record[
            "guild"
        ]
    )


    status = (
        record[
            "status"
        ]
    )


    examples = [

        (
            f"Give the assigned guild for registry identity "
            f"{entity}.",
            guild,
        ),

        (
            f"In the Helion database, which guild contains "
            f"{entity}?",
            guild,
        ),

        (
            f"Give the current status for registry identity "
            f"{entity}.",
            status,
        ),

        (
            f"Which status is recorded for {entity}?",
            status,
        ),
    ]


    if (
        record[
            "override"
        ]
        is not None
    ):

        override = (
            record[
                "override"
            ]
        )


        examples.extend(
            [
                (
                    f"What override credential is registered "
                    f"for {entity}?",
                    override,
                ),

                (
                    f"State the exceptional authentication key "
                    f"assigned to {entity}.",
                    override,
                ),
            ]
        )


    return examples


ONLINE_PACKETS = [

    {
        "name":
            "GLOBAL_RULES",

        "type":
            "rules",

        "examples":
            rule_write_examples(),

        "record":
            None,
    }
]


for record in WORLD_RECORDS:

    ONLINE_PACKETS.append(
        {
            "name":
                record[
                    "entity"
                ],

            "type":
                "entity",

            "examples":
                entity_write_examples(
                    record
                ),

            "record":
                record,
        }
    )


# =============================================================================
# TEST QUESTIONS — NEVER USED FOR WRITING
# =============================================================================

def dedupe(
    values,
):

    return list(
        dict.fromkeys(
            values
        )
    )


BASE_KEY_CANDIDATES = dedupe(
    list(
        GUILD_KEYS.values()
    )
)


SUFFIX_CANDIDATES = dedupe(
    list(
        STATUS_SUFFIX.values()
    )
)


VALID_KEY_CANDIDATES = dedupe(
    [
        record[
            "valid_key"
        ]
        for record in WORLD_RECORDS
    ]
)


def rule_test_questions():

    questions = []


    for guild in GUILDS:

        questions.append(
            {
                "id":
                    f"rule_guild_{guild}",

                "category":
                    "guild_key_rule",

                "question":
                    (
                        f"A Helion operator sees guild {guild}. "
                        f"Which base-key string belongs to that guild?"
                    ),

                "answer":
                    GUILD_KEYS[
                        guild
                    ],

                "candidates":
                    BASE_KEY_CANDIDATES,
            }
        )


    for status in STATUSES:

        questions.append(
            {
                "id":
                    f"rule_status_{status}",

                "category":
                    "status_suffix_rule",

                "question":
                    (
                        f"When an identity carries {status} status, "
                        f"which suffix must be appended?"
                    ),

                "answer":
                    STATUS_SUFFIX[
                        status
                    ],

                "candidates":
                    SUFFIX_CANDIDATES,
            }
        )


    return questions


def entity_test_questions(
    record,
):

    entity = (
        record[
            "entity"
        ]
    )


    questions = [

        {
            "id":
                f"{entity}_guild",

            "category":
                "direct_guild",

            "question":
                (
                    f"An operator receives identity {entity}. "
                    f"To which guild does the registry assign it?"
                ),

            "answer":
                record[
                    "guild"
                ],

            "candidates":
                GUILDS,
        },

        {
            "id":
                f"{entity}_status",

            "category":
                "direct_status",

            "question":
                (
                    f"Look up identity {entity}. "
                    f"What status classification should be returned?"
                ),

            "answer":
                record[
                    "status"
                ],

            "candidates":
                STATUSES,
        },

        {
            "id":
                f"{entity}_compose",

            "category":
                (
                    "exception"
                    if (
                        record[
                            "override"
                        ]
                        is not None
                    )
                    else
                    "compositional"
                ),

            "question":
                (
                    f"Identity {entity} reaches the Helion checkpoint. "
                    f"What exact final access credential should "
                    f"the checkpoint accept?"
                ),

            "answer":
                record[
                    "valid_key"
                ],

            "candidates":
                VALID_KEY_CANDIDATES,
        },
    ]


    return questions


RULE_TESTS = (
    rule_test_questions()
)


ENTITY_TESTS = {
    record[
        "entity"
    ]:
        entity_test_questions(
            record
        )

    for record in WORLD_RECORDS
}


ALL_TESTS = list(
    RULE_TESTS
)


for record in WORLD_RECORDS:

    ALL_TESTS.extend(
        ENTITY_TESTS[
            record[
                "entity"
            ]
        ]
    )


print()
print("=" * 120)
print("ONLINE PACKETS")
print("=" * 120)


for index, packet in enumerate(
    ONLINE_PACKETS
):

    print(
        f"{index:>2} | "
        f"{packet['type']:<8} | "
        f"{packet['name']:<18} | "
        f"{len(packet['examples'])} write QAs"
    )


print()
print(
    "Final held-out questions:",
    len(
        ALL_TESTS
    ),
)


# =============================================================================
# LOAD QWEN
# =============================================================================

print()
print("=" * 120)
print("LOADING QWEN2.5-7B")
print("=" * 120)


tokenizer = (
    AutoTokenizer.from_pretrained(
        MODEL_ID,
        trust_remote_code=True,
    )
)


if (
    tokenizer.pad_token_id
    is None
):

    tokenizer.pad_token = (
        tokenizer.eos_token
    )


model = (
    AutoModelForCausalLM.from_pretrained(

        MODEL_ID,

        dtype=
            torch.bfloat16,

        device_map={
            "": 0
        },

        low_cpu_mem_usage=True,

        trust_remote_code=True,

        attn_implementation=
            "sdpa",
    )
)


model.eval()


for parameter in model.parameters():

    parameter.requires_grad_(
        False
    )


M = int(
    model.config.hidden_size
)


N = int(
    model.config.intermediate_size
)


target_layer = (
    model.model.layers[
        TTT_LAYER
    ]
)


original_down_forward = (
    target_layer
    .mlp
    .down_proj
    .forward
)


print(
    "M                     :",
    M,
)

print(
    "N                     :",
    N,
)

print(
    "TTT layer             :",
    TTT_LAYER,
)

print(
    "Fixed write K         :",
    WRITE_K,
)


# =============================================================================
# FAST WEIGHTS
# =============================================================================

ORIGINAL_WEIGHT = (
    target_layer
    .mlp
    .down_proj
    .weight
    .detach()
)


BASE_BF16 = (
    ORIGINAL_WEIGHT
    .clone()
    .contiguous()
)


MASTER_BASE = (
    BASE_BF16
    .float()
    .contiguous()
)


MASTER_A = (
    MASTER_BASE
    .clone()
    .contiguous()
)


MASTER_B = (
    MASTER_BASE
    .clone()
    .contiguous()
)


# Serving A reuses Qwen's actual BF16 weight storage.
SERVE_A = (
    ORIGINAL_WEIGHT
)


SERVE_B = (
    BASE_BF16
    .clone()
    .contiguous()
)


# =============================================================================
# STATE
# =============================================================================

class FastWeightState:

    def __init__(
        self,
    ):

        self.master_active = MASTER_A
        self.master_staging = MASTER_B

        self.serve_active = SERVE_A
        self.serve_staging = SERVE_B

        self.version = 0

        self.last_version = None
        self.last_pointer = None


state = (
    FastWeightState()
)


def reset_state():

    MASTER_A.copy_(
        MASTER_BASE
    )

    MASTER_B.copy_(
        MASTER_BASE
    )


    SERVE_A.copy_(
        BASE_BF16
    )

    SERVE_B.copy_(
        BASE_BF16
    )


    torch.cuda.synchronize()


    state.master_active = MASTER_A
    state.master_staging = MASTER_B

    state.serve_active = SERVE_A
    state.serve_staging = SERVE_B

    state.version = 0

    state.last_version = None
    state.last_pointer = None


def commit_state():

    state.master_active, state.master_staging = (
        state.master_staging,
        state.master_active,
    )


    state.serve_active, state.serve_staging = (
        state.serve_staging,
        state.serve_active,
    )


    state.version += 1


def fast_weight_forward(
    x,
):

    state.last_version = (
        state.version
    )


    state.last_pointer = (
        state.serve_active.data_ptr()
    )


    return F.linear(
        x,
        state.serve_active,
        bias=None,
    )


target_layer.mlp.down_proj.forward = (
    fast_weight_forward
)


# =============================================================================
# FIXED-K WRITE SEQUENCE
#
# Only ANSWER tokens receive labels.
#
# Everything else:
#
#       labels = -100
#
# So this is not generic LM adaptation.
#
# It explicitly teaches:
#
#       query -> answer
# =============================================================================

def build_write_tensor(
    examples,
):

    header = (
        "Helion Registry online memory write.\n"
        "Store each exact association.\n"
    )


    header_ids = (
        tokenizer(
            header,
            add_special_tokens=True,
        )[
            "input_ids"
        ]
    )


    token_ids = list(
        header_ids
    )


    labels = [
        -100
    ] * len(
        token_ids
    )


    for question, answer in examples:

        prefix = (
            "\nQuestion: "
            +
            question
            +
            "\nAnswer:"
        )


        prefix_ids = (
            tokenizer(
                prefix,
                add_special_tokens=False,
            )[
                "input_ids"
            ]
        )


        answer_ids = (
            tokenizer(
                " "
                +
                answer,

                add_special_tokens=False,
            )[
                "input_ids"
            ]
        )


        separator_ids = (
            tokenizer(
                "\n",
                add_special_tokens=False,
            )[
                "input_ids"
            ]
        )


        required = (
            len(
                prefix_ids
            )
            +
            len(
                answer_ids
            )
            +
            len(
                separator_ids
            )
        )


        if (
            len(
                token_ids
            )
            +
            required
            >
            WRITE_K
        ):

            raise RuntimeError(
                f"Write packet exceeds fixed K={WRITE_K}. "
                f"Increase WRITE_K."
            )


        token_ids.extend(
            prefix_ids
        )


        labels.extend(
            [
                -100
            ]
            *
            len(
                prefix_ids
            )
        )


        token_ids.extend(
            answer_ids
        )


        # Only the answer is supervised.
        labels.extend(
            answer_ids
        )


        token_ids.extend(
            separator_ids
        )


        labels.extend(
            [
                -100
            ]
            *
            len(
                separator_ids
            )
        )


    actual_length = len(
        token_ids
    )


    pad_count = (
        WRITE_K
        -
        actual_length
    )


    token_ids.extend(
        [
            tokenizer.pad_token_id
        ]
        *
        pad_count
    )


    labels.extend(
        [
            -100
        ]
        *
        pad_count
    )


    attention_mask = (
        [
            1
        ]
        *
        actual_length
        +
        [
            0
        ]
        *
        pad_count
    )


    input_tensor = torch.tensor(
        [
            token_ids
        ],
        dtype=
            torch.long,
        device=
            DEVICE,
    )


    label_tensor = torch.tensor(
        [
            labels
        ],
        dtype=
            torch.long,
        device=
            DEVICE,
    )


    attention_tensor = torch.tensor(
        [
            attention_mask
        ],
        dtype=
            torch.long,
        device=
            DEVICE,
    )


    return (
        input_tensor,
        attention_tensor,
        label_tensor,
        actual_length,
    )


# =============================================================================
# TRITON PASS 1
# =============================================================================

@triton.jit
def norm_kernel(
    gt_ptr,
    z_ptr,

    norm_ptr,

    stride_gm,
    stride_gk,

    stride_zk,
    stride_zn,

    M: tl.constexpr,
    N: tl.constexpr,
    K: tl.constexpr,

    BM: tl.constexpr,
    BN: tl.constexpr,
    BK: tl.constexpr,
):

    pid_m = tl.program_id(
        0
    )

    pid_n = tl.program_id(
        1
    )


    offs_m = (
        pid_m * BM
        +
        tl.arange(
            0,
            BM,
        )
    )


    offs_n = (
        pid_n * BN
        +
        tl.arange(
            0,
            BN,
        )
    )


    acc = tl.zeros(
        (
            BM,
            BN,
        ),
        dtype=
            tl.float32,
    )


    for k in range(
        0,
        K,
        BK,
    ):

        offs_k = (
            k
            +
            tl.arange(
                0,
                BK,
            )
        )


        g = tl.load(
            gt_ptr
            +
            offs_m[:, None]
            *
            stride_gm
            +
            offs_k[None, :]
            *
            stride_gk,

            mask=(
                (
                    offs_m[:, None]
                    <
                    M
                )
                &
                (
                    offs_k[None, :]
                    <
                    K
                )
            ),

            other=
                0.0,
        )


        z = tl.load(
            z_ptr
            +
            offs_k[:, None]
            *
            stride_zk
            +
            offs_n[None, :]
            *
            stride_zn,

            mask=(
                (
                    offs_k[:, None]
                    <
                    K
                )
                &
                (
                    offs_n[None, :]
                    <
                    N
                )
            ),

            other=
                0.0,
        )


        acc = tl.dot(
            g,
            z,
            acc,

            input_precision=
                "ieee",
        )


    tl.atomic_add(
        norm_ptr,
        tl.sum(
            acc
            *
            acc
        ),
    )


# =============================================================================
# TRITON PASS 2
# =============================================================================

@triton.jit
def update_kernel(
    gt_ptr,
    z_ptr,

    src_ptr,
    dst_ptr,

    stride_gm,
    stride_gk,

    stride_zk,
    stride_zn,

    stride_sm,
    stride_sn,

    stride_dm,
    stride_dn,

    step_scale,

    M: tl.constexpr,
    N: tl.constexpr,
    K: tl.constexpr,

    BM: tl.constexpr,
    BN: tl.constexpr,
    BK: tl.constexpr,
):

    pid_m = tl.program_id(
        0
    )

    pid_n = tl.program_id(
        1
    )


    offs_m = (
        pid_m * BM
        +
        tl.arange(
            0,
            BM,
        )
    )


    offs_n = (
        pid_n * BN
        +
        tl.arange(
            0,
            BN,
        )
    )


    acc = tl.zeros(
        (
            BM,
            BN,
        ),
        dtype=
            tl.float32,
    )


    for k in range(
        0,
        K,
        BK,
    ):

        offs_k = (
            k
            +
            tl.arange(
                0,
                BK,
            )
        )


        g = tl.load(
            gt_ptr
            +
            offs_m[:, None]
            *
            stride_gm
            +
            offs_k[None, :]
            *
            stride_gk,

            mask=(
                (
                    offs_m[:, None]
                    <
                    M
                )
                &
                (
                    offs_k[None, :]
                    <
                    K
                )
            ),

            other=
                0.0,
        )


        z = tl.load(
            z_ptr
            +
            offs_k[:, None]
            *
            stride_zk
            +
            offs_n[None, :]
            *
            stride_zn,

            mask=(
                (
                    offs_k[:, None]
                    <
                    K
                )
                &
                (
                    offs_n[None, :]
                    <
                    N
                )
            ),

            other=
                0.0,
        )


        acc = tl.dot(
            g,
            z,
            acc,

            input_precision=
                "ieee",
        )


    mask = (
        (
            offs_m[:, None]
            <
            M
        )
        &
        (
            offs_n[None, :]
            <
            N
        )
    )


    weight = tl.load(
        src_ptr
        +
        offs_m[:, None]
        *
        stride_sm
        +
        offs_n[None, :]
        *
        stride_sn,

        mask=
            mask,

        other=
            0.0,
    )


    updated = (
        weight
        -
        step_scale
        *
        acc
    )


    tl.store(
        dst_ptr
        +
        offs_m[:, None]
        *
        stride_dm
        +
        offs_n[None, :]
        *
        stride_dn,

        updated,

        mask=
            mask,
    )


# =============================================================================
# A100 CONFIG
# =============================================================================

NBM = 64
NBN = 128
NBK = 64


UBM = 64
UBN = 64
UBK = 64


norm_grid = (
    triton.cdiv(
        M,
        NBM,
    ),

    triton.cdiv(
        N,
        NBN,
    ),
)


update_grid = (
    triton.cdiv(
        M,
        UBM,
    ),

    triton.cdiv(
        N,
        UBN,
    ),
)


norm_ws = torch.zeros(
    1,
    device=
        DEVICE,
    dtype=
        torch.float32,
)


# =============================================================================
# REAL QA GRADIENT
# =============================================================================

def capture_qa_gradient(
    examples,
):

    (
        input_ids,
        attention_mask,
        labels,
        actual_length,
    ) = build_write_tensor(
        examples
    )


    capture = {}


    def gradient_forward(
        x,
    ):

        capture[
            "z"
        ] = (
            x
            .detach()
            .float()
            .contiguous()
        )


        y = F.linear(
            x,
            state.serve_active,
            bias=None,
        )


        y_leaf = (
            y
            .detach()
            .requires_grad_(
                True
            )
        )


        y_leaf.retain_grad()


        capture[
            "y"
        ] = (
            y_leaf
        )


        return y_leaf


    target_layer.mlp.down_proj.forward = (
        gradient_forward
    )


    model.zero_grad(
        set_to_none=True
    )


    torch.cuda.synchronize()


    start = (
        time.perf_counter()
    )


    output = model(
        input_ids=
            input_ids,

        attention_mask=
            attention_mask,

        labels=
            labels,

        use_cache=
            False,
    )


    loss_before = float(
        output.loss
        .detach()
        .item()
    )


    output.loss.backward()


    torch.cuda.synchronize()


    capture_ms = (
        time.perf_counter()
        -
        start
    ) * 1000.0


    z = (
        capture[
            "z"
        ][
            0
        ]
        .float()
        .contiguous()
    )


    g = (
        capture[
            "y"
        ]
        .grad[
            0
        ]
        .float()
        .contiguous()
    )


    gt = (
        g
        .T
        .contiguous()
    )


    assert z.shape == (
        WRITE_K,
        N,
    )


    assert gt.shape == (
        M,
        WRITE_K,
    )


    target_layer.mlp.down_proj.forward = (
        fast_weight_forward
    )


    del output
    del capture
    del input_ids
    del attention_mask
    del labels


    gc.collect()
    torch.cuda.empty_cache()


    return (
        gt,
        z,
        loss_before,
        capture_ms,
        actual_length,
    )


# =============================================================================
# PASS 1 — EXACT GRADIENT NORM
# =============================================================================

def calculate_gradient_norm(
    gt,
    z,
):

    norm_ws.zero_()


    norm_kernel[
        norm_grid
    ](
        gt,
        z,

        norm_ws,

        gt.stride(0),
        gt.stride(1),

        z.stride(0),
        z.stride(1),

        M=M,
        N=N,
        K=WRITE_K,

        BM=NBM,
        BN=NBN,
        BK=NBK,

        num_warps=4,
    )


    torch.cuda.synchronize()


    return math.sqrt(
        float(
            norm_ws.item()
        )
    )


# =============================================================================
# PASS 2 — REMATERIALIZED UPDATE
# =============================================================================

def apply_gradient_update(
    gt,
    z,
    step_scale,
):

    update_kernel[
        update_grid
    ](
        gt,
        z,

        state.master_active,
        state.master_staging,

        gt.stride(0),
        gt.stride(1),

        z.stride(0),
        z.stride(1),

        state.master_active.stride(0),
        state.master_active.stride(1),

        state.master_staging.stride(0),
        state.master_staging.stride(1),

        step_scale,

        M=M,
        N=N,
        K=WRITE_K,

        BM=UBM,
        BN=UBN,
        BK=UBK,

        num_warps=8,
    )


# =============================================================================
# WRITE LOSS
# =============================================================================

@torch.no_grad()
def evaluate_write_loss(
    examples,
):

    (
        input_ids,
        attention_mask,
        labels,
        _,
    ) = build_write_tensor(
        examples
    )


    output = model(
        input_ids=
            input_ids,

        attention_mask=
            attention_mask,

        labels=
            labels,

        use_cache=
            False,
    )


    loss = float(
        output.loss.item()
    )


    del output
    del input_ids
    del attention_mask
    del labels


    return loss


# =============================================================================
# ONE ONLINE WRITE
# =============================================================================

def online_write(
    packet,
    repeats=
        UPDATES_PER_PACKET,
):

    history = []


    for repeat in range(
        repeats
    ):

        (
            gt,
            z,
            loss_before,
            capture_ms,
            actual_length,
        ) = capture_qa_gradient(
            packet[
                "examples"
            ]
        )


        grad_norm = (
            calculate_gradient_norm(
                gt,
                z,
            )
        )


        step_scale = (
            TARGET_DELTA_NORM
            /
            max(
                grad_norm,
                1e-30,
            )
        )


        start_event = torch.cuda.Event(
            enable_timing=True
        )


        kernel_event = torch.cuda.Event(
            enable_timing=True
        )


        publish_event = torch.cuda.Event(
            enable_timing=True
        )


        start_event.record()


        apply_gradient_update(
            gt,
            z,
            step_scale,
        )


        kernel_event.record()


        # FP32 -> BF16 publication into preallocated staging.
        state.serve_staging.copy_(
            state.master_staging
        )


        publish_event.record()


        publish_event.synchronize()


        kernel_ms = (
            start_event.elapsed_time(
                kernel_event
            )
        )


        publish_ms = (
            kernel_event.elapsed_time(
                publish_event
            )
        )


        total_ready_ms = (
            start_event.elapsed_time(
                publish_event
            )
        )


        old_pointer = (
            state.serve_active.data_ptr()
        )


        commit_state()


        new_pointer = (
            state.serve_active.data_ptr()
        )


        loss_after = (
            evaluate_write_loss(
                packet[
                    "examples"
                ]
            )
        )


        history.append(
            {
                "repeat":
                    repeat + 1,

                "version":
                    state.version,

                "actual_tokens":
                    actual_length,

                "loss_before":
                    loss_before,

                "loss_after":
                    loss_after,

                "loss_improvement":
                    loss_before
                    -
                    loss_after,

                "gradient_norm":
                    grad_norm,

                "step_scale":
                    step_scale,

                "gradient_capture_ms":
                    capture_ms,

                "kernel_ms":
                    kernel_ms,

                "publish_ms":
                    publish_ms,

                "total_ready_ms":
                    total_ready_ms,

                "old_pointer":
                    old_pointer,

                "new_pointer":
                    new_pointer,
            }
        )


        del gt
        del z


        gc.collect()
        torch.cuda.empty_cache()


    return history


# =============================================================================
# CANDIDATE SCORING — BATCHED
# =============================================================================

def no_context_prompt(
    question,
):

    return (
        "Answer the Helion Registry question with only the exact "
        "requested value. Do not explain.\n\n"
        f"Question: {question}\n"
        "Answer:"
    )


def full_context_prompt(
    question,
):

    return (
        FULL_CONTEXT
        +
        "\n\n"
        "Using only the registry above, answer with the exact "
        "requested value.\n\n"
        f"Question: {question}\n"
        "Answer:"
    )


@torch.no_grad()
def candidate_scores(
    prompt,
    candidates,
):

    candidates = dedupe(
        candidates
    )


    encoded_rows = []


    label_rows = []


    for candidate in candidates:

        prefix_ids = (
            tokenizer(
                prompt,
                add_special_tokens=True,
            )[
                "input_ids"
            ]
        )


        answer_ids = (
            tokenizer(
                " "
                +
                candidate,

                add_special_tokens=False,
            )[
                "input_ids"
            ]
        )


        row = (
            prefix_ids
            +
            answer_ids
        )


        labels = (
            [
                -100
            ]
            *
            len(
                prefix_ids
            )
            +
            answer_ids
        )


        encoded_rows.append(
            row
        )


        label_rows.append(
            labels
        )


    max_len = max(
        len(
            row
        )
        for row in encoded_rows
    )


    padded_ids = []

    padded_labels = []

    masks = []


    for row, labels in zip(
        encoded_rows,
        label_rows,
    ):

        pad = (
            max_len
            -
            len(
                row
            )
        )


        padded_ids.append(
            row
            +
            [
                tokenizer.pad_token_id
            ]
            *
            pad
        )


        padded_labels.append(
            labels
            +
            [
                -100
            ]
            *
            pad
        )


        masks.append(
            [
                1
            ]
            *
            len(
                row
            )
            +
            [
                0
            ]
            *
            pad
        )


    input_ids = torch.tensor(
        padded_ids,
        dtype=
            torch.long,
        device=
            DEVICE,
    )


    labels = torch.tensor(
        padded_labels,
        dtype=
            torch.long,
        device=
            DEVICE,
    )


    attention_mask = torch.tensor(
        masks,
        dtype=
            torch.long,
        device=
            DEVICE,
    )


    output = model(
        input_ids=
            input_ids,

        attention_mask=
            attention_mask,

        use_cache=
            False,
    )


    logits = (
        output.logits
        .float()
    )


    shift_logits = (
        logits[
            :,
            :-1,
            :
        ]
    )


    shift_labels = (
        labels[
            :,
            1:
        ]
    )


    log_probs = F.log_softmax(
        shift_logits,
        dim=-1,
    )


    valid = (
        shift_labels
        !=
        -100
    )


    safe_labels = (
        shift_labels
        .clamp_min(
            0
        )
    )


    token_scores = (
        log_probs
        .gather(
            -1,
            safe_labels.unsqueeze(
                -1
            ),
        )
        .squeeze(
            -1
        )
    )


    token_scores = (
        token_scores
        *
        valid
    )


    sequence_scores = (
        token_scores
        .sum(
            dim=1
        )
    )


    answer_token_counts = (
        valid
        .sum(
            dim=1
        )
        .clamp_min(
            1
        )
    )


    mean_nll = (
        -sequence_scores
        /
        answer_token_counts
    )


    candidate_probability = (
        torch.softmax(
            sequence_scores,
            dim=0,
        )
    )


    result = {}


    for i, candidate in enumerate(
        candidates
    ):

        result[
            candidate
        ] = {
            "log_probability":
                float(
                    sequence_scores[
                        i
                    ]
                    .item()
                ),

            "candidate_probability":
                float(
                    candidate_probability[
                        i
                    ]
                    .item()
                ),

            "mean_nll":
                float(
                    mean_nll[
                        i
                    ]
                    .item()
                ),
        }


    del output
    del logits
    del log_probs
    del input_ids
    del labels
    del attention_mask


    return result


# =============================================================================
# EVALUATE QUESTIONS
# =============================================================================

def evaluate_questions(
    questions,
    context_mode=
        "none",
):

    rows = []


    for question in questions:

        if (
            context_mode
            ==
            "full"
        ):

            prompt = (
                full_context_prompt(
                    question[
                        "question"
                    ]
                )
            )

        else:

            prompt = (
                no_context_prompt(
                    question[
                        "question"
                    ]
                )
            )


        scores = (
            candidate_scores(
                prompt,
                question[
                    "candidates"
                ],
            )
        )


        predicted = max(
            scores,
            key=lambda candidate:
                scores[
                    candidate
                ][
                    "log_probability"
                ],
        )


        answer = (
            question[
                "answer"
            ]
        )


        rows.append(
            {
                "id":
                    question[
                        "id"
                    ],

                "category":
                    question[
                        "category"
                    ],

                "question":
                    question[
                        "question"
                    ],

                "answer":
                    answer,

                "predicted":
                    predicted,

                "correct":
                    predicted
                    ==
                    answer,

                "correct_probability":
                    scores[
                        answer
                    ][
                        "candidate_probability"
                    ],

                "answer_nll":
                    scores[
                        answer
                    ][
                        "mean_nll"
                    ],

                "scores":
                    scores,
            }
        )


    accuracy = (
        sum(
            row[
                "correct"
            ]
            for row in rows
        )
        /
        len(
            rows
        )
        if rows
        else 0.0
    )


    mean_probability = (
        statistics.mean(
            row[
                "correct_probability"
            ]
            for row in rows
        )
        if rows
        else 0.0
    )


    mean_nll = (
        statistics.mean(
            row[
                "answer_nll"
            ]
            for row in rows
        )
        if rows
        else 0.0
    )


    categories = {}


    for category in sorted(
        set(
            row[
                "category"
            ]
            for row in rows
        )
    ):

        subset = [
            row

            for row in rows

            if (
                row[
                    "category"
                ]
                ==
                category
            )
        ]


        categories[
            category
        ] = {
            "accuracy":
                sum(
                    row[
                        "correct"
                    ]
                    for row in subset
                )
                /
                len(
                    subset
                ),

            "mean_probability":
                statistics.mean(
                    row[
                        "correct_probability"
                    ]
                    for row in subset
                ),

            "mean_nll":
                statistics.mean(
                    row[
                        "answer_nll"
                    ]
                    for row in subset
                ),
        }


    return {
        "accuracy":
            accuracy,

        "mean_correct_probability":
            mean_probability,

        "mean_answer_nll":
            mean_nll,

        "categories":
            categories,

        "rows":
            rows,
    }


# =============================================================================
# ONLINE DIRECT RETRIEVAL EVALUATION
#
# Rules + direct guild/status facts only.
#
# Composition is evaluated at the final checkpoint.
# =============================================================================

def evaluate_online_state(
    seen_entities,
):

    tests = list(
        RULE_TESTS
    )


    for entity in seen_entities:

        entity_questions = (
            ENTITY_TESTS[
                entity
            ]
        )


        tests.extend(
            [
                q

                for q in entity_questions

                if q[
                    "category"
                ]
                in {
                    "direct_guild",
                    "direct_status",
                    "exception",
                }
            ]
        )


    return evaluate_questions(
        tests,
        context_mode=
            "none",
    )


# =============================================================================
# GENERATION — FINAL DIAGNOSTIC ONLY
# =============================================================================

def normalize(
    text,
):

    return re.sub(
        r"\s+",
        "",
        text
        .strip()
        .lower()
        .rstrip(
            ".,;:!?"
        ),
    )


@torch.no_grad()
def generate_answer(
    question,
):

    prompt = (
        no_context_prompt(
            question[
                "question"
            ]
        )
    )


    encoded = tokenizer(
        prompt,
        return_tensors=
            "pt",
    )


    input_ids = (
        encoded[
            "input_ids"
        ]
        .to(
            DEVICE
        )
    )


    output = model.generate(
        input_ids=
            input_ids,

        max_new_tokens=
            MAX_NEW_TOKENS,

        do_sample=
            False,

        use_cache=
            True,

        pad_token_id=
            tokenizer.eos_token_id,
    )


    generated = tokenizer.decode(
        output[
            0,
            input_ids.shape[1]:
        ],
        skip_special_tokens=True,
    )


    return generated.strip()


# =============================================================================
# JIT WARMUP
# =============================================================================

print()
print("=" * 120)
print("FIXED-K JIT WARMUP")
print("=" * 120)


reset_state()


warmup_packet = (
    ONLINE_PACKETS[
        0
    ]
)


gt, z, _, _, actual_len = (
    capture_qa_gradient(
        warmup_packet[
            "examples"
        ]
    )
)


warm_norm = (
    calculate_gradient_norm(
        gt,
        z,
    )
)


warm_scale = (
    TARGET_DELTA_NORM
    /
    max(
        warm_norm,
        1e-30,
    )
)


apply_gradient_update(
    gt,
    z,
    warm_scale,
)


state.serve_staging.copy_(
    state.master_staging
)


torch.cuda.synchronize()


print(
    "Fixed K               :",
    WRITE_K,
)

print(
    "Actual warmup tokens  :",
    actual_len,
)

print(
    "Gradient norm         :",
    f"{warm_norm:.8e}",
)

print(
    "JIT compiled."
)


del gt
del z


# Crucially remove the warmup update.
reset_state()


# =============================================================================
# W0 BASELINES
# =============================================================================

print()
print("=" * 120)
print("W0 BASELINES")
print("=" * 120)


W0_NO_CONTEXT = (
    evaluate_questions(
        ALL_TESTS,
        context_mode=
            "none",
    )
)


W0_FULL_CONTEXT = (
    evaluate_questions(
        ALL_TESTS,
        context_mode=
            "full",
    )
)


print(
    "W0 no-context accuracy :",
    f"{W0_NO_CONTEXT['accuracy'] * 100:.2f}%",
)

print(
    "W0 no-context P*       :",
    f"{W0_NO_CONTEXT['mean_correct_probability']:.4f}",
)

print(
    "W0 full-context acc    :",
    f"{W0_FULL_CONTEXT['accuracy'] * 100:.2f}%",
)

print(
    "W0 full-context P*     :",
    f"{W0_FULL_CONTEXT['mean_correct_probability']:.4f}",
)


# =============================================================================
# TRUE ONLINE LEARNING STREAM
# =============================================================================

print()
print("=" * 165)
print("ONLINE LEARNING — ONE PACKET AT A TIME")
print("=" * 165)


print(
    f"{'packet':<20}"
    f"{'version':>9}"
    f"{'write pre':>12}"
    f"{'write post':>12}"
    f"{'grad ms':>11}"
    f"{'kernel':>10}"
    f"{'publish':>10}"
    f"{'ready':>10}"
    f"{'online acc':>13}"
    f"{'P(correct)':>13}"
)


print(
    "-" * 165
)


online_history = []


seen_entities = []


for packet_index, packet in enumerate(
    ONLINE_PACKETS
):

    write_history = (
        online_write(
            packet
        )
    )


    if (
        packet[
            "type"
        ]
        ==
        "entity"
    ):

        seen_entities.append(
            packet[
                "name"
            ]
        )


    retrieval = (
        evaluate_online_state(
            seen_entities
        )
    )


    last_write = (
        write_history[
            -1
        ]
    )


    record = {
        "packet_index":
            packet_index,

        "packet":
            packet[
                "name"
            ],

        "packet_type":
            packet[
                "type"
            ],

        "version":
            state.version,

        "seen_entities":
            list(
                seen_entities
            ),

        "writes":
            write_history,

        "online_retrieval":
            retrieval,
    }


    online_history.append(
        record
    )


    print(
        f"{packet['name']:<20}"
        f"{state.version:>9}"
        f"{last_write['loss_before']:>12.4f}"
        f"{last_write['loss_after']:>12.4f}"
        f"{last_write['gradient_capture_ms']:>11.2f}"
        f"{last_write['kernel_ms']:>10.3f}"
        f"{last_write['publish_ms']:>10.3f}"
        f"{last_write['total_ready_ms']:>10.3f}"
        f"{retrieval['accuracy'] * 100:>12.2f}%"
        f"{retrieval['mean_correct_probability']:>13.4f}"
    )


# =============================================================================
# FINAL NO-CONTEXT MEMORY TEST
# =============================================================================

print()
print("=" * 120)
print("FINAL ONLINE-LEARNED MEMORY")
print("=" * 120)


W_FINAL_NO_CONTEXT = (
    evaluate_questions(
        ALL_TESTS,
        context_mode=
            "none",
    )
)


print(
    "Candidate accuracy    :",
    f"{W_FINAL_NO_CONTEXT['accuracy'] * 100:.2f}%",
)

print(
    "Mean P(correct)       :",
    f"{W_FINAL_NO_CONTEXT['mean_correct_probability']:.6f}",
)

print(
    "Mean answer NLL       :",
    f"{W_FINAL_NO_CONTEXT['mean_answer_nll']:.6f}",
)


# =============================================================================
# CATEGORY BREAKDOWN
# =============================================================================

print()
print("=" * 120)
print("FINAL CATEGORY BREAKDOWN")
print("=" * 120)


print(
    f"{'Category':<22}"
    f"{'W0 none':>12}"
    f"{'W0 context':>14}"
    f"{'Online W':>12}"
    f"{'P* online':>12}"
)


print(
    "-" * 75
)


all_categories = sorted(
    set(
        q[
            "category"
        ]
        for q in ALL_TESTS
    )
)


for category in all_categories:

    b = (
        W0_NO_CONTEXT[
            "categories"
        ][
            category
        ]
    )


    c = (
        W0_FULL_CONTEXT[
            "categories"
        ][
            category
        ]
    )


    o = (
        W_FINAL_NO_CONTEXT[
            "categories"
        ][
            category
        ]
    )


    print(
        f"{category:<22}"
        f"{b['accuracy'] * 100:>11.1f}%"
        f"{c['accuracy'] * 100:>13.1f}%"
        f"{o['accuracy'] * 100:>11.1f}%"
        f"{o['mean_probability']:>12.4f}"
    )


# =============================================================================
# MEMORY TRANSFER FRACTION
# =============================================================================

p0 = (
    W0_NO_CONTEXT[
        "mean_correct_probability"
    ]
)


p_ctx = (
    W0_FULL_CONTEXT[
        "mean_correct_probability"
    ]
)


p_online = (
    W_FINAL_NO_CONTEXT[
        "mean_correct_probability"
    ]
)


if abs(
    p_ctx
    -
    p0
) > 1e-12:

    transfer_fraction = (
        (
            p_online
            -
            p0
        )
        /
        (
            p_ctx
            -
            p0
        )
    )

else:

    transfer_fraction = None


print()
print("=" * 120)
print("ONLINE MEMORY TRANSFER")
print("=" * 120)


print(
    "W0 no-context P*      :",
    f"{p0:.6f}",
)

print(
    "W0 full-context P*    :",
    f"{p_ctx:.6f}",
)

print(
    "Online W no-context P*:",
    f"{p_online:.6f}",
)


if (
    transfer_fraction
    is not None
):

    print(
        "Recovered context gain:",
        f"{transfer_fraction * 100:.2f}%",
    )


# =============================================================================
# ONLINE CORRECTION / KNOWLEDGE UPDATE
#
# This is important:
#
# Online learning is not just "add information".
#
# We need to be able to change old information WITHOUT:
#
#       restarting model
#       resetting fast state
#       replaying all old packets
#
# =============================================================================

print()
print("=" * 120)
print("ONLINE CORRECTION TEST")
print("=" * 120)


correction_record = next(

    record

    for record in WORLD_RECORDS

    if (
        record[
            "override"
        ]
        is None
    )
)


entity = (
    correction_record[
        "entity"
    ]
)


old_status = (
    correction_record[
        "status"
    ]
)


new_status = next(

    status

    for status in STATUSES

    if status
    !=
    old_status
)


old_key = (
    GUILD_KEYS[
        correction_record[
            "guild"
        ]
    ]
    +
    STATUS_SUFFIX[
        old_status
    ]
)


new_key = (
    GUILD_KEYS[
        correction_record[
            "guild"
        ]
    ]
    +
    STATUS_SUFFIX[
        new_status
    ]
)


correction_status_question = {
    "id":
        "correction_status",

    "category":
        "online_correction",

    "question":
        (
            f"The registry is queried for {entity}. "
            f"What is its CURRENT status?"
        ),

    "answer":
        new_status,

    "candidates":
        STATUSES,
}


correction_key_question = {
    "id":
        "correction_key",

    "category":
        "online_correction_composition",

    "question":
        (
            f"Using the current registry state, what final "
            f"credential should {entity} now present?"
        ),

    "answer":
        new_key,

    "candidates":
        dedupe(
            VALID_KEY_CANDIDATES
            +
            [
                old_key,
                new_key,
            ]
        ),
}


# -----------------------------------------------------------------------------
# Measure probabilities BEFORE correction.
# -----------------------------------------------------------------------------

status_prompt = (
    no_context_prompt(
        correction_status_question[
            "question"
        ]
    )
)


status_before_scores = (
    candidate_scores(
        status_prompt,
        STATUSES,
    )
)


key_prompt = (
    no_context_prompt(
        correction_key_question[
            "question"
        ]
    )
)


key_before_scores = (
    candidate_scores(
        key_prompt,
        correction_key_question[
            "candidates"
        ],
    )
)


print(
    "Entity                 :",
    entity,
)

print(
    "Old status             :",
    old_status,
)

print(
    "New status             :",
    new_status,
)

print(
    "Old derived key        :",
    old_key,
)

print(
    "New derived key        :",
    new_key,
)


print()
print(
    "BEFORE correction:"
)

print(
    "P(old status)          :",
    f"{status_before_scores[old_status]['candidate_probability']:.6f}",
)

print(
    "P(new status)          :",
    f"{status_before_scores[new_status]['candidate_probability']:.6f}",
)

print(
    "P(old key)             :",
    f"{key_before_scores[old_key]['candidate_probability']:.6f}",
)

print(
    "P(new key)             :",
    f"{key_before_scores[new_key]['candidate_probability']:.6f}",
)


# -----------------------------------------------------------------------------
# Stream ONLY THE CORRECTION.
#
# No world replay.
# -----------------------------------------------------------------------------

correction_packet = {

    "name":
        f"CORRECT_{entity}",

    "type":
        "correction",

    "examples": [

        (
            f"The previous status for {entity} is obsolete. "
            f"What is its current registry status?",
            new_status,
        ),

        (
            f"After the registry update, which status now belongs "
            f"to {entity}?",
            new_status,
        ),

        (
            f"What status value supersedes the old assignment "
            f"for {entity}?",
            new_status,
        ),

        (
            f"State the current Helion status for {entity}.",
            new_status,
        ),
    ],
}


correction_write_history = (
    online_write(
        correction_packet,
        repeats=4,
    )
)


# -----------------------------------------------------------------------------
# Evaluate AFTER correction.
# -----------------------------------------------------------------------------

status_after_scores = (
    candidate_scores(
        status_prompt,
        STATUSES,
    )
)


key_after_scores = (
    candidate_scores(
        key_prompt,
        correction_key_question[
            "candidates"
        ],
    )
)


print()
print(
    "AFTER correction:"
)


print(
    "P(old status)          :",
    f"{status_after_scores[old_status]['candidate_probability']:.6f}",
)

print(
    "P(new status)          :",
    f"{status_after_scores[new_status]['candidate_probability']:.6f}",
)

print(
    "P(old key)             :",
    f"{key_after_scores[old_key]['candidate_probability']:.6f}",
)

print(
    "P(new key)             :",
    f"{key_after_scores[new_key]['candidate_probability']:.6f}",
)


# =============================================================================
# RETENTION AFTER CORRECTION
#
# Exclude corrected entity and make sure learning a new value did not erase
# the rest of the fast-weight memory.
# =============================================================================

retention_questions = []


for record in WORLD_RECORDS:

    if (
        record[
            "entity"
        ]
        ==
        entity
    ):

        continue


    retention_questions.extend(
        ENTITY_TESTS[
            record[
                "entity"
            ]
        ]
    )


retention_before = (
    evaluate_questions(
        retention_questions,
        context_mode=
            "none",
    )
)


# NOTE:
# retention_before is evaluated after correction in this cell.
# We also recover pre-correction accuracy from W_FINAL_NO_CONTEXT rows.

retained_ids = {
    q[
        "id"
    ]
    for q in retention_questions
}


pre_correction_rows = [

    row

    for row in W_FINAL_NO_CONTEXT[
        "rows"
    ]

    if row[
        "id"
    ]
    in retained_ids
]


pre_correction_retention_accuracy = (
    sum(
        row[
            "correct"
        ]
        for row in pre_correction_rows
    )
    /
    len(
        pre_correction_rows
    )
)


post_correction_retention_accuracy = (
    retention_before[
        "accuracy"
    ]
)


print()
print("=" * 120)
print("RETENTION AFTER ONLINE CORRECTION")
print("=" * 120)


print(
    "Other-fact accuracy before:",
    f"{pre_correction_retention_accuracy * 100:.2f}%",
)

print(
    "Other-fact accuracy after :",
    f"{post_correction_retention_accuracy * 100:.2f}%",
)

print(
    "Retention delta           :",
    f"{(post_correction_retention_accuracy - pre_correction_retention_accuracy) * 100:+.2f} pp",
)


# =============================================================================
# FINAL GENERATION DIAGNOSTIC
# =============================================================================

generation_rows = []


if DO_GENERATION_AT_FINAL:

    print()
    print("=" * 120)
    print("FINAL NO-CONTEXT GREEDY GENERATION")
    print("=" * 120)


    for question in ALL_TESTS:

        generated = (
            generate_answer(
                question
            )
        )


        expected = (
            question[
                "answer"
            ]
        )


        correct = (

            normalize(
                generated
            )
            .startswith(
                normalize(
                    expected
                )
            )
        )


        generation_rows.append(
            {
                "id":
                    question[
                        "id"
                    ],

                "expected":
                    expected,

                "generated":
                    generated,

                "correct":
                    correct,
            }
        )


        print(
            f"{question['id']:<30} | "
            f"expected={expected:<12} | "
            f"generated={generated[:45]!r}"
        )


    generation_accuracy = (
        sum(
            row[
                "correct"
            ]
            for row in generation_rows
        )
        /
        len(
            generation_rows
        )
    )


    print()
    print(
        "Generation exact-match:",
        f"{generation_accuracy * 100:.2f}%",
    )


# =============================================================================
# SAVE
# =============================================================================

RESULT_PATH = Path(
    "async_ttt_qwen7b_true_online_memory.json"
)


WORLD_PATH = Path(
    "async_ttt_online_world.json"
)


with open(
    WORLD_PATH,
    "w",
) as file:

    json.dump(
        {
            "guild_keys":
                GUILD_KEYS,

            "status_suffix":
                STATUS_SUFFIX,

            "records":
                WORLD_RECORDS,

            "full_context":
                FULL_CONTEXT,

            "online_packets":
                ONLINE_PACKETS,

            "tests":
                ALL_TESTS,
        },
        file,
        indent=2,
    )


with open(
    RESULT_PATH,
    "w",
) as file:

    json.dump(
        {
            "config": {
                "model":
                    MODEL_ID,

                "layer":
                    TTT_LAYER,

                "fixed_write_k":
                    WRITE_K,

                "updates_per_packet":
                    UPDATES_PER_PACKET,

                "target_delta_norm":
                    TARGET_DELTA_NORM,
            },

            "W0_no_context":
                W0_NO_CONTEXT,

            "W0_full_context":
                W0_FULL_CONTEXT,

            "online_history":
                online_history,

            "final_no_context":
                W_FINAL_NO_CONTEXT,

            "memory_transfer_fraction":
                transfer_fraction,

            "correction": {
                "entity":
                    entity,

                "old_status":
                    old_status,

                "new_status":
                    new_status,

                "old_key":
                    old_key,

                "new_key":
                    new_key,

                "status_before":
                    status_before_scores,

                "status_after":
                    status_after_scores,

                "key_before":
                    key_before_scores,

                "key_after":
                    key_after_scores,

                "write_history":
                    correction_write_history,

                "other_fact_accuracy_before":
                    pre_correction_retention_accuracy,

                "other_fact_accuracy_after":
                    post_correction_retention_accuracy,
            },

            "generation":
                generation_rows,
        },
        file,
        indent=2,
    )


print()
print("=" * 120)
print("DONE")
print("=" * 120)


print(
    "Saved world :",
    WORLD_PATH.resolve(),
)

print(
    "Saved result:",
    RESULT_PATH.resolve(),
)


# =============================================================================
# RESTORE STOCK QWEN
# =============================================================================

target_layer.mlp.down_proj.forward = (
    original_down_forward
)


SERVE_A.copy_(
    BASE_BF16
)


torch.cuda.synchronize()


free_bytes, _ = (
    torch.cuda.mem_get_info()
)


print(
    "Original down_proj restored."
)

print(
    "Free VRAM:",
    f"{free_bytes / 1024**3:.2f} GiB",
)

FLUSHING PREVIOUS GPU STATE


Free VRAM : 39.08 GiB
Total VRAM: 39.49 GiB

CREATING ONLINE SYNTHETIC WORLD
HELION REGISTRY — AUTHORITATIVE RECORD

Guild base keys:
Nask uses base key MS-1427.
Oriv uses base key MZ-2006.
Talm uses base key FB-6789.
Phex uses base key YT-1066.

Status suffix rules:
amber status appends -R.
cobalt status appends -S.
ivory status appends -T.
jade status appends -U.

Entity records:
Zarsen-91 belongs to Phex and has ivory status.
Velolux-22 belongs to Nask and has cobalt status.
Solrin-79 belongs to Oriv and has amber status.
Solrin-79 has override JQ-0930-X. The override supersedes the derived key.
Mirezor-72 belongs to Nask and has cobalt status.
Xanidor-95 belongs to Talm and has jade status.
Nivonix-27 belongs to Phex and has ivory status.
Zartra-86 belongs to Oriv and has amber status.
Zartra-86 has override DS-5894-X. The override supersedes the derived key.
Kellux-52 belongs to Talm and has jade status.

For a normal entity, concatenate its guild base key with its status suffix.


Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

M                     : 3584
N                     : 18944
TTT layer             : 13
Fixed write K         : 384

FIXED-K JIT WARMUP
Fixed K               : 384
Actual warmup tokens  : 360
Gradient norm         : 5.95867453e+00
JIT compiled.

W0 BASELINES
W0 no-context accuracy : 21.88%
W0 no-context P*       : 0.2096
W0 full-context acc    : 84.38%
W0 full-context P*     : 0.8269

ONLINE LEARNING — ONE PACKET AT A TIME
packet                version   write pre  write post    grad ms    kernel   publish     ready   online acc   P(correct)
---------------------------------------------------------------------------------------------------------------------------------------------------------------------
GLOBAL_RULES                3      1.6049      1.5635      70.33     3.085     0.314     3.399       37.50%       0.2405
Zarsen-91                   6      7.4470      7.2598      70.42     3.121     0.316     3.437       30.00%       0.1957
Velolux-22                  9      5.7076     

In [1]:
# =============================================================================
# ASYNC-TTT — QWEN2.5-7B
#
# COMPLETE SINGLE-ASSOCIATION ONLINE-LEARNING CAPACITY EXPERIMENT
#
# Goals
# -----
# 1. Test whether unknown context can be written into fast weights.
# 2. Remove the original context and test retrieval using unseen paraphrases.
# 3. Sweep which transformer MLP layers are writable.
# 4. Sweep update magnitude and number of online updates.
# 5. Remove answer-token prior bias with matched-token synthetic secrets.
# 6. Use REAL answer-only LM gradients.
# 7. Never materialize the full dW tensor in the TTT path.
# 8. Use a SAFE per-tile gradient-norm workspace (NO global atomic).
# 9. Maintain FP32 master A/B + BF16 serving A/B.
# 10. Test online overwrite:
#
#         QEVAX-731 -> SECRET_A
#
#     then, without reset:
#
#         QEVAX-731 -> SECRET_B
#
# 11. Calculate fixed-state vs KV-cache break-even.
#
#
# IMPORTANT INTERPRETATION
# ------------------------
# This is deliberately a capacity diagnostic.
#
# If an ordinary Qwen W_down retrofit cannot robustly store ONE unknown
# association even after a multi-layer sweep, do not keep increasing the LR
# indefinitely. The correct architectural next step is a dedicated small
# TTT / associative-memory layer using the SAME asynchronous A/B runtime.
#
#
# RESTART THE NOTEBOOK KERNEL BEFORE RUNNING THIS CELL.
# =============================================================================


# =============================================================================
# IMPORTS
# =============================================================================

import gc
import json
import math
import os
import random
import re
import statistics
import sys
import time

from collections import defaultdict
from pathlib import Path

import torch
import torch.nn.functional as F

import triton
import triton.language as tl

from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
)


# =============================================================================
# GPU CLEANUP
# =============================================================================

print("=" * 120)
print("FLUSHING PREVIOUS GPU STATE")
print("=" * 120)


for _name in [
    "model",
    "tokenizer",
    "FAST",
    "ACTIVE_LAYERS",
    "CAPTURE_DATA",
    "output",
    "past_key_values",
    "next_token",
]:

    if _name in globals():

        try:
            del globals()[_name]
        except Exception:
            pass


gc.collect()


if torch.cuda.is_available():

    torch.cuda.synchronize()
    torch.cuda.empty_cache()

    try:
        torch.cuda.ipc_collect()
    except Exception:
        pass

    gc.collect()
    torch.cuda.empty_cache()


assert torch.cuda.is_available()


free_bytes, total_bytes = torch.cuda.mem_get_info()


print(
    "Free VRAM :",
    f"{free_bytes / 1024**3:.2f} GiB",
)

print(
    "Total VRAM:",
    f"{total_bytes / 1024**3:.2f} GiB",
)


if free_bytes / 1024**3 < 30:

    raise RuntimeError(
        "\nLess than 30 GiB is free.\n"
        "Restart the notebook kernel, then rerun this cell."
    )


# =============================================================================
# CONFIG
# =============================================================================

MODEL_ID = "Qwen/Qwen2.5-7B-Instruct"

DEVICE = "cuda"

SEED = 424242


# Completely synthetic identifier.
ENTITY = "QEVAX-731"


# Fixed write length.
#
# This ensures Triton sees the same compile-time K everywhere.
WRITE_K = 256


# -------------------------------------------------------------------------
# Writable-layer configurations.
# -------------------------------------------------------------------------

LAYER_CONFIGS = {

    "L13":
        [13],

    "L12_13":
        [12, 13],

    "L12_15":
        [12, 13, 14, 15],

    "L8_15":
        list(
            range(
                8,
                16,
            )
        ),
}


UNION_LAYERS = sorted(
    set(
        layer
        for layers in LAYER_CONFIGS.values()
        for layer in layers
    )
)


# -------------------------------------------------------------------------
# TOTAL Frobenius update budget across all writable layers.
#
# If n layers are writable:
#
#       per-layer target = TOTAL / sqrt(n)
#
# therefore:
#
#       sqrt(sum_i ||DeltaW_i||²) ~= TOTAL
#
# More layers do NOT automatically get a larger total update budget.
# -------------------------------------------------------------------------

TOTAL_DELTA_NORMS = [
    0.015,
    0.030,
    0.060,
    0.100,
]


# Evaluate after these many online write steps.
CHECKPOINT_STEPS = [
    1,
    2,
    4,
    8,
]


MAX_STEPS = max(
    CHECKPOINT_STEPS
)


NUM_CANDIDATES = 8


# Generate many random secrets, then choose a matched-prior subset.
CANDIDATE_POOL_TARGET = 96


MAX_NEW_TOKENS = 10


torch.manual_seed(
    SEED
)


random.seed(
    SEED
)


rng = random.Random(
    SEED
)


torch.backends.cuda.matmul.allow_tf32 = False
torch.backends.cudnn.allow_tf32 = False


# =============================================================================
# ENVIRONMENT
# =============================================================================

props = torch.cuda.get_device_properties(
    0
)


print()
print("=" * 120)
print("QWEN2.5-7B — UNKNOWN-CONTEXT ONLINE LEARNING")
print("=" * 120)


print(
    "Model               :",
    MODEL_ID,
)

print(
    "Python              :",
    sys.version.split()[0],
)

print(
    "PyTorch             :",
    torch.__version__,
)

print(
    "CUDA                :",
    torch.version.cuda,
)

print(
    "Triton              :",
    triton.__version__,
)

print(
    "GPU                 :",
    torch.cuda.get_device_name(0),
)

print(
    "SMs                 :",
    props.multi_processor_count,
)

print(
    "VRAM                :",
    f"{props.total_memory / 1024**3:.2f} GiB",
)


# =============================================================================
# LOAD QWEN
# =============================================================================

print()
print("=" * 120)
print("LOADING QWEN")
print("=" * 120)


tokenizer = AutoTokenizer.from_pretrained(
    MODEL_ID,
    trust_remote_code=True,
)


if tokenizer.pad_token_id is None:

    tokenizer.pad_token = (
        tokenizer.eos_token
    )


model = AutoModelForCausalLM.from_pretrained(

    MODEL_ID,

    dtype=torch.bfloat16,

    device_map={
        "": 0
    },

    low_cpu_mem_usage=True,

    trust_remote_code=True,

    attn_implementation="sdpa",
)


model.eval()


# Freeze ordinary Qwen parameters.
#
# We explicitly create our own autograd leaf inside the selected fast-weight
# region.

for parameter in model.parameters():

    parameter.requires_grad_(
        False
    )


M = int(
    model.config.hidden_size
)


N = int(
    model.config.intermediate_size
)


NUM_LAYERS = int(
    model.config.num_hidden_layers
)


NUM_ATTENTION_HEADS = int(
    model.config.num_attention_heads
)


NUM_KV_HEADS = int(
    getattr(
        model.config,
        "num_key_value_heads",
        NUM_ATTENTION_HEADS,
    )
)


HEAD_DIM = (
    M
    //
    NUM_ATTENTION_HEADS
)


print(
    "hidden_size         :",
    M,
)

print(
    "intermediate_size   :",
    N,
)

print(
    "layers              :",
    NUM_LAYERS,
)

print(
    "attention heads     :",
    NUM_ATTENTION_HEADS,
)

print(
    "KV heads            :",
    NUM_KV_HEADS,
)

print(
    "head dim            :",
    HEAD_DIM,
)

print(
    "candidate layers    :",
    UNION_LAYERS,
)

print(
    "fixed write K       :",
    WRITE_K,
)


free_bytes, _ = torch.cuda.mem_get_info()


print(
    "Free after model    :",
    f"{free_bytes / 1024**3:.2f} GiB",
)


# =============================================================================
# SYNTHETIC SECRET CANDIDATES
#
# We want all candidate secrets to:
#
#   - be synthetic
#   - have identical tokenizer length
#   - have roughly similar stock-model prior probability
#
# This removes the previous:
#
#       "amber" vs "ivory"
#
# style prior-bias problem.
# =============================================================================

print()
print("=" * 120)
print("BUILDING MATCHED-TOKEN SYNTHETIC SECRETS")
print("=" * 120)


def random_secret():

    letters = "".join(
        rng.choice(
            "ABCDEFGHJKLMNPQRSTUVWXYZ"
        )
        for _ in range(
            4
        )
    )


    digits = "".join(
        rng.choice(
            "0123456789"
        )
        for _ in range(
            3
        )
    )


    return (
        f"{letters}-{digits}"
    )


candidate_buckets = defaultdict(
    list
)


used_codes = set()


for _ in range(
    50000
):

    code = random_secret()


    if code in used_codes:
        continue


    used_codes.add(
        code
    )


    ids = tokenizer(
        " " + code,
        add_special_tokens=False,
    )[
        "input_ids"
    ]


    length = len(
        ids
    )


    if 2 <= length <= 6:

        candidate_buckets[
            length
        ].append(
            code
        )


eligible_lengths = [
    length

    for length, values in candidate_buckets.items()

    if len(
        values
    ) >= CANDIDATE_POOL_TARGET
]


if not eligible_lengths:

    raise RuntimeError(
        "Could not construct enough matched-token synthetic candidates."
    )


MATCHED_TOKEN_COUNT = min(
    eligible_lengths
)


raw_candidate_pool = (
    candidate_buckets[
        MATCHED_TOKEN_COUNT
    ][
        :CANDIDATE_POOL_TARGET
    ]
)


print(
    "Matched token count :",
    MATCHED_TOKEN_COUNT,
)

print(
    "Candidate pool      :",
    len(
        raw_candidate_pool
    ),
)


# =============================================================================
# STOCK CANDIDATE SCORING
#
# Used only to find synthetic answers with similar base-model prior.
# =============================================================================

@torch.no_grad()
def score_stock_prior_chunk(
    prompt,
    candidates,
):

    rows = []
    label_rows = []


    for candidate in candidates:

        prefix = tokenizer(
            prompt,
            add_special_tokens=True,
        )[
            "input_ids"
        ]


        answer = tokenizer(
            " " + candidate,
            add_special_tokens=False,
        )[
            "input_ids"
        ]


        if len(
            answer
        ) != MATCHED_TOKEN_COUNT:

            raise RuntimeError(
                "Candidate token-length mismatch."
            )


        rows.append(
            prefix
            +
            answer
        )


        label_rows.append(
            [-100]
            *
            len(
                prefix
            )
            +
            answer
        )


    max_len = max(
        len(
            row
        )
        for row in rows
    )


    padded_ids = []
    padded_labels = []
    masks = []


    for row, labels in zip(
        rows,
        label_rows,
    ):

        pad = (
            max_len
            -
            len(
                row
            )
        )


        padded_ids.append(
            row
            +
            [
                tokenizer.pad_token_id
            ]
            *
            pad
        )


        padded_labels.append(
            labels
            +
            [-100]
            *
            pad
        )


        masks.append(
            [1]
            *
            len(
                row
            )
            +
            [0]
            *
            pad
        )


    input_ids = torch.tensor(
        padded_ids,
        dtype=torch.long,
        device=DEVICE,
    )


    labels = torch.tensor(
        padded_labels,
        dtype=torch.long,
        device=DEVICE,
    )


    attention_mask = torch.tensor(
        masks,
        dtype=torch.long,
        device=DEVICE,
    )


    output = model(
        input_ids=input_ids,
        attention_mask=attention_mask,
        use_cache=False,
    )


    logits = (
        output.logits[
            :,
            :-1,
            :
        ]
        .float()
    )


    shifted_labels = (
        labels[
            :,
            1:
        ]
    )


    valid = (
        shifted_labels
        !=
        -100
    )


    safe_labels = (
        shifted_labels
        .clamp_min(
            0
        )
    )


    log_probs = F.log_softmax(
        logits,
        dim=-1,
    )


    selected = (
        log_probs
        .gather(
            -1,
            safe_labels.unsqueeze(
                -1
            ),
        )
        .squeeze(
            -1
        )
    )


    selected = (
        selected
        *
        valid
    )


    scores = (
        selected
        .sum(
            dim=1
        )
        .detach()
        .cpu()
        .tolist()
    )


    del output
    del logits
    del log_probs
    del input_ids
    del labels
    del attention_mask


    return scores


neutral_prompt = (
    "Return one exact private registry code.\n"
    "Answer:"
)


neutral_scores = []


# Chunk this so we don't produce a huge logits tensor for all 96 candidates.
PRIOR_BATCH = 16


for start in range(
    0,
    len(
        raw_candidate_pool
    ),
    PRIOR_BATCH,
):

    batch = (
        raw_candidate_pool[
            start:
            start + PRIOR_BATCH
        ]
    )


    neutral_scores.extend(
        score_stock_prior_chunk(
            neutral_prompt,
            batch,
        )
    )


scored_pool = sorted(

    zip(
        raw_candidate_pool,
        neutral_scores,
    ),

    key=lambda item:
        item[
            1
        ],
)


# Find contiguous set of 8 with smallest stock-prior span.

best_window = None

best_span = float(
    "inf"
)


for start in range(
    0,
    len(
        scored_pool
    )
    -
    NUM_CANDIDATES
    +
    1,
):

    window = (
        scored_pool[
            start:
            start + NUM_CANDIDATES
        ]
    )


    span = (
        window[
            -1
        ][
            1
        ]
        -
        window[
            0
        ][
            1
        ]
    )


    if span < best_span:

        best_span = span

        best_window = window


CANDIDATES = [
    value
    for value, _
    in best_window
]


rng.shuffle(
    CANDIDATES
)


SECRET_A = (
    CANDIDATES[
        0
    ]
)


SECRET_B = (
    CANDIDATES[
        1
    ]
)


print(
    "Candidates          :",
    CANDIDATES,
)

print(
    "SECRET_A            :",
    SECRET_A,
)

print(
    "SECRET_B            :",
    SECRET_B,
)

print(
    "Prior logprob span  :",
    f"{best_span:.6f}",
)


for candidate in CANDIDATES:

    ids = tokenizer(
        " " + candidate,
        add_special_tokens=False,
    )[
        "input_ids"
    ]


    print(
        f"{candidate:<12}",
        ids,
    )


# =============================================================================
# WRITE QUESTIONS
#
# Used ONLY for online memory writing.
# =============================================================================

WRITE_QUESTIONS_A = [

    (
        f"What private registry code is assigned to identity {ENTITY}?",
        SECRET_A,
    ),

    (
        f"State the exact stored code belonging to {ENTITY}.",
        SECRET_A,
    ),

    (
        f"In the Delta registry, which secret value maps to {ENTITY}?",
        SECRET_A,
    ),

    (
        f"Give the confidential lookup result for identifier {ENTITY}.",
        SECRET_A,
    ),

    (
        f"Which code should be returned when the registry key is {ENTITY}?",
        SECRET_A,
    ),

    (
        f"Recall the exact private value associated with {ENTITY}.",
        SECRET_A,
    ),
]


# =============================================================================
# HELD-OUT TEST QUESTIONS
#
# NEVER used in gradient construction.
# =============================================================================

TEST_QUESTIONS = [

    f"An operator queries {ENTITY}. What exact registry value should be returned?",

    f"What value is stored under lookup identity {ENTITY}?",

    f"Resolve identifier {ENTITY} to its secret registry code.",

    f"If a request references {ENTITY}, what confidential code belongs to it?",

    f"Which exact value corresponds to the record named {ENTITY}?",

    f"Retrieve the private code associated with {ENTITY}.",

    f"For registry key {ENTITY}, provide its stored secret value.",

    f"What is the hidden lookup result for {ENTITY}?",
]


FULL_CONTEXT_FACT = (
    f"Authoritative Delta Registry record: "
    f"identity {ENTITY} has private registry code {SECRET_A}."
)


print()
print("=" * 120)
print("UNKNOWN ASSOCIATION")
print("=" * 120)


print(
    "Entity              :",
    ENTITY,
)

print(
    "Secret A            :",
    SECRET_A,
)

print(
    "Held-out queries    :",
    len(
        TEST_QUESTIONS
    ),
)


# =============================================================================
# SAFE TRITON KERNELS
#
# PASS 1:
#
# Each gradient tile writes its squared norm to a UNIQUE output scalar.
#
# NO ATOMIC.
# NO full dW tensor.
#
#
# PASS 2:
#
# Recomputes the tile and directly applies:
#
#       W_new = W_old - scale * dW_tile
# =============================================================================


# Smaller norm tile than earlier experiments.
#
# This reduces accumulator/register pressure.

NBM = 32
NBN = 64
NBK = 64


UBM = 64
UBN = 64
UBK = 64


NUM_M_TILES = triton.cdiv(
    M,
    NBM,
)


NUM_N_TILES = triton.cdiv(
    N,
    NBN,
)


NUM_NORM_TILES = (
    NUM_M_TILES
    *
    NUM_N_TILES
)


norm_grid = (
    NUM_M_TILES,
    NUM_N_TILES,
)


update_grid = (

    triton.cdiv(
        M,
        UBM,
    ),

    triton.cdiv(
        N,
        UBN,
    ),
)


norm_tiles = torch.empty(

    NUM_NORM_TILES,

    dtype=torch.float32,

    device=DEVICE,
)


print()
print("=" * 120)
print("SAFE NORM WORKSPACE")
print("=" * 120)


print(
    "M tiles             :",
    NUM_M_TILES,
)

print(
    "N tiles             :",
    NUM_N_TILES,
)

print(
    "Norm entries        :",
    NUM_NORM_TILES,
)

print(
    "Norm workspace      :",
    f"{NUM_NORM_TILES * 4 / 1024:.2f} KiB",
)


@triton.jit
def safe_norm_kernel(
    gt_ptr,
    z_ptr,

    out_ptr,

    stride_gm,
    stride_gk,

    stride_zk,
    stride_zn,

    M: tl.constexpr,
    N: tl.constexpr,
    K: tl.constexpr,

    NUM_N_TILES: tl.constexpr,

    BM: tl.constexpr,
    BN: tl.constexpr,
    BK: tl.constexpr,
):

    pid_m = tl.program_id(
        0
    )

    pid_n = tl.program_id(
        1
    )


    offs_m = (
        pid_m * BM
        +
        tl.arange(
            0,
            BM,
        )
    )


    offs_n = (
        pid_n * BN
        +
        tl.arange(
            0,
            BN,
        )
    )


    acc = tl.zeros(
        (
            BM,
            BN,
        ),
        dtype=tl.float32,
    )


    for k_start in range(
        0,
        K,
        BK,
    ):

        offs_k = (
            k_start
            +
            tl.arange(
                0,
                BK,
            )
        )


        g = tl.load(
            gt_ptr
            +
            offs_m[:, None]
            *
            stride_gm
            +
            offs_k[None, :]
            *
            stride_gk,

            mask=(
                (
                    offs_m[:, None]
                    <
                    M
                )
                &
                (
                    offs_k[None, :]
                    <
                    K
                )
            ),

            other=0.0,
        )


        z = tl.load(
            z_ptr
            +
            offs_k[:, None]
            *
            stride_zk
            +
            offs_n[None, :]
            *
            stride_zn,

            mask=(
                (
                    offs_k[:, None]
                    <
                    K
                )
                &
                (
                    offs_n[None, :]
                    <
                    N
                )
            ),

            other=0.0,
        )


        acc = tl.dot(
            g,
            z,
            acc,
            input_precision="ieee",
        )


    valid = (
        (
            offs_m[:, None]
            <
            M
        )
        &
        (
            offs_n[None, :]
            <
            N
        )
    )


    acc = tl.where(
        valid,
        acc,
        0.0,
    )


    # Explicit two-axis reduction.
    #
    # This avoids ambiguity around reducing a 2-D block directly.

    row_squares = tl.sum(
        acc
        *
        acc,
        axis=1,
    )


    local_norm_sq = tl.sum(
        row_squares,
        axis=0,
    )


    linear_pid = (
        pid_m
        *
        NUM_N_TILES
        +
        pid_n
    )


    tl.store(
        out_ptr
        +
        linear_pid,

        local_norm_sq,
    )


@triton.jit
def safe_update_kernel(
    gt_ptr,
    z_ptr,

    src_ptr,
    dst_ptr,

    stride_gm,
    stride_gk,

    stride_zk,
    stride_zn,

    stride_sm,
    stride_sn,

    stride_dm,
    stride_dn,

    step_scale,

    M: tl.constexpr,
    N: tl.constexpr,
    K: tl.constexpr,

    BM: tl.constexpr,
    BN: tl.constexpr,
    BK: tl.constexpr,
):

    pid_m = tl.program_id(
        0
    )

    pid_n = tl.program_id(
        1
    )


    offs_m = (
        pid_m * BM
        +
        tl.arange(
            0,
            BM,
        )
    )


    offs_n = (
        pid_n * BN
        +
        tl.arange(
            0,
            BN,
        )
    )


    acc = tl.zeros(
        (
            BM,
            BN,
        ),
        dtype=tl.float32,
    )


    for k_start in range(
        0,
        K,
        BK,
    ):

        offs_k = (
            k_start
            +
            tl.arange(
                0,
                BK,
            )
        )


        g = tl.load(
            gt_ptr
            +
            offs_m[:, None]
            *
            stride_gm
            +
            offs_k[None, :]
            *
            stride_gk,

            mask=(
                (
                    offs_m[:, None]
                    <
                    M
                )
                &
                (
                    offs_k[None, :]
                    <
                    K
                )
            ),

            other=0.0,
        )


        z = tl.load(
            z_ptr
            +
            offs_k[:, None]
            *
            stride_zk
            +
            offs_n[None, :]
            *
            stride_zn,

            mask=(
                (
                    offs_k[:, None]
                    <
                    K
                )
                &
                (
                    offs_n[None, :]
                    <
                    N
                )
            ),

            other=0.0,
        )


        acc = tl.dot(
            g,
            z,
            acc,
            input_precision="ieee",
        )


    valid = (
        (
            offs_m[:, None]
            <
            M
        )
        &
        (
            offs_n[None, :]
            <
            N
        )
    )


    old_weight = tl.load(
        src_ptr
        +
        offs_m[:, None]
        *
        stride_sm
        +
        offs_n[None, :]
        *
        stride_sn,

        mask=valid,

        other=0.0,
    )


    new_weight = (
        old_weight
        -
        step_scale
        *
        acc
    )


    tl.store(
        dst_ptr
        +
        offs_m[:, None]
        *
        stride_dm
        +
        offs_n[None, :]
        *
        stride_dn,

        new_weight,

        mask=valid,
    )


# =============================================================================
# SAFE NORM HELPER
# =============================================================================

def get_gradient_norm(
    gt,
    z,
):

    assert gt.is_cuda
    assert z.is_cuda

    assert gt.dtype == torch.float32
    assert z.dtype == torch.float32

    assert gt.is_contiguous()
    assert z.is_contiguous()

    assert tuple(
        gt.shape
    ) == (
        M,
        WRITE_K,
    )


    assert tuple(
        z.shape
    ) == (
        WRITE_K,
        N,
    )


    # Surface any previous asynchronous CUDA fault BEFORE Triton.
    torch.cuda.synchronize()


    safe_norm_kernel[
        norm_grid
    ](
        gt,
        z,

        norm_tiles,

        gt.stride(0),
        gt.stride(1),

        z.stride(0),
        z.stride(1),

        M=M,
        N=N,
        K=WRITE_K,

        NUM_N_TILES=
            NUM_N_TILES,

        BM=
            NBM,

        BN=
            NBN,

        BK=
            NBK,

        num_warps=4,
    )


    # Ensure the error, if any, surfaces HERE.
    torch.cuda.synchronize()


    # Only ~100 KiB, not 271 MB.
    norm_sq = (
        norm_tiles
        .sum(
            dtype=torch.float64
        )
    )


    norm = float(
        torch.sqrt(
            norm_sq
        )
        .item()
    )


    if not math.isfinite(
        norm
    ):

        raise RuntimeError(
            f"Non-finite gradient norm: {norm}"
        )


    return norm


# =============================================================================
# TRITON RANDOM SMOKE TEST
#
# Do this BEFORE allocating all fast-weight layer states.
# =============================================================================

print()
print("=" * 120)
print("SAFE TRITON NORM SMOKE TEST")
print("=" * 120)


test_gt = (
    torch.randn(
        (
            M,
            WRITE_K,
        ),
        dtype=torch.float32,
        device=DEVICE,
    )
    *
    1e-3
)


test_z = (
    torch.randn(
        (
            WRITE_K,
            N,
        ),
        dtype=torch.float32,
        device=DEVICE,
    )
    *
    1e-3
)


torch.cuda.synchronize()


triton_norm = get_gradient_norm(
    test_gt,
    test_z,
)


# Reference only for validation.
#
# This deliberately materializes dW ONCE for the smoke test.
reference_dw = (
    test_gt
    @
    test_z
)


torch.cuda.synchronize()


reference_norm = float(
    torch.linalg.vector_norm(
        reference_dw
    )
    .item()
)


norm_relative_error = (
    abs(
        triton_norm
        -
        reference_norm
    )
    /
    max(
        reference_norm,
        1e-30,
    )
)


print(
    "PyTorch norm        :",
    f"{reference_norm:.8e}",
)

print(
    "Triton norm         :",
    f"{triton_norm:.8e}",
)

print(
    "Relative error      :",
    f"{norm_relative_error:.8e}",
)


if norm_relative_error > 1e-4:

    raise RuntimeError(
        "SAFE Triton norm kernel failed numerical validation."
    )


print(
    "SMOKE TEST PASSED."
)


del test_gt
del test_z
del reference_dw


gc.collect()
torch.cuda.empty_cache()


# =============================================================================
# FAST-WEIGHT LAYER STATE
# =============================================================================

class LayerFastState:

    def __init__(
        self,
        layer_index,
    ):

        self.layer_index = (
            layer_index
        )


        self.module = (
            model
            .model
            .layers[
                layer_index
            ]
            .mlp
            .down_proj
        )


        self.original_forward = (
            self.module.forward
        )


        self.original_weight = (
            self.module
            .weight
            .detach()
        )


        # -------------------------------------------------------------
        # FP32 master A/B.
        # -------------------------------------------------------------

        self.master_a = (
            self.original_weight
            .float()
            .contiguous()
        )


        self.master_b = (
            torch.empty_like(
                self.master_a
            )
        )


        # -------------------------------------------------------------
        # BF16 serving A/B.
        #
        # We deliberately keep the original Qwen parameter pristine.
        # -------------------------------------------------------------

        self.serve_a = (
            self.original_weight
            .clone()
            .contiguous()
        )


        self.serve_b = (
            torch.empty_like(
                self.serve_a
            )
        )


        self.master_active = (
            self.master_a
        )


        self.master_staging = (
            self.master_b
        )


        self.serve_active = (
            self.serve_a
        )


        self.serve_staging = (
            self.serve_b
        )


        self.version = 0


    def reset(
        self,
    ):

        self.master_a.copy_(
            self.original_weight
        )


        self.master_b.copy_(
            self.original_weight
        )


        self.serve_a.copy_(
            self.original_weight
        )


        self.serve_b.copy_(
            self.original_weight
        )


        self.master_active = (
            self.master_a
        )


        self.master_staging = (
            self.master_b
        )


        self.serve_active = (
            self.serve_a
        )


        self.serve_staging = (
            self.serve_b
        )


        self.version = 0


    def commit(
        self,
    ):

        self.master_active, self.master_staging = (
            self.master_staging,
            self.master_active,
        )


        self.serve_active, self.serve_staging = (
            self.serve_staging,
            self.serve_active,
        )


        self.version += 1


print()
print("=" * 120)
print("ALLOCATING FAST-WEIGHT STATES")
print("=" * 120)


FAST = {}


for layer_index in UNION_LAYERS:

    FAST[
        layer_index
    ] = LayerFastState(
        layer_index
    )


torch.cuda.synchronize()


free_bytes, _ = torch.cuda.mem_get_info()


print(
    "Fast layers         :",
    UNION_LAYERS,
)

print(
    "Free VRAM           :",
    f"{free_bytes / 1024**3:.2f} GiB",
)


# =============================================================================
# PATCH SELECTED MLP PROJECTIONS
# =============================================================================

ACTIVE_LAYERS = set()

CAPTURE_MODE = False

CAPTURE_DATA = {}


def activate_layers(
    layers,
):

    global ACTIVE_LAYERS

    ACTIVE_LAYERS = set(
        layers
    )


def reset_all_states():

    for layer_index in UNION_LAYERS:

        FAST[
            layer_index
        ].reset()


    torch.cuda.synchronize()


def commit_layers(
    layers,
):

    for layer_index in layers:

        FAST[
            layer_index
        ].commit()


def make_fast_forward(
    layer_index,
):

    fast = FAST[
        layer_index
    ]


    def forward(
        x,
    ):

        global CAPTURE_DATA


        # -------------------------------------------------------------
        # Layer is not part of the active fast-memory configuration.
        # -------------------------------------------------------------

        if layer_index not in ACTIVE_LAYERS:

            return fast.original_forward(
                x
            )


        # -------------------------------------------------------------
        # BF16 fast-weight projection.
        #
        # x shape:
        #
        #     [batch, sequence, intermediate]
        #
        # e.g.
        #
        #     [1, 256, 18944]
        #
        # y shape:
        #
        #     [batch, sequence, hidden]
        #
        # e.g.
        #
        #     [1, 256, 3584]
        # -------------------------------------------------------------

        y = F.linear(
            x,
            fast.serve_active,
            bias=None,
        )


        if not CAPTURE_MODE:

            return y


        # -------------------------------------------------------------
        # Capture Z for:
        #
        #     dW = G^T @ Z
        #
        # IMPORTANT:
        #
        # Kernel expects:
        #
        #     Z: [K, N]
        #
        # not:
        #
        #     [1, K, N]
        #
        # Batch size in this experiment is exactly 1.
        # -------------------------------------------------------------

        if x.ndim != 3:

            raise RuntimeError(
                f"Unexpected down_proj input rank at layer "
                f"{layer_index}: shape={tuple(x.shape)}"
            )


        if x.shape[0] != 1:

            raise RuntimeError(
                "This experiment currently assumes batch_size=1, "
                f"but got shape={tuple(x.shape)}"
            )


        z = (
            x[
                0
            ]
            .detach()
            .float()
            .contiguous()
        )


        if tuple(
            z.shape
        ) != (
            WRITE_K,
            N,
        ):

            raise RuntimeError(
                f"Captured Z shape mismatch at layer {layer_index}: "
                f"expected {(WRITE_K, N)}, got {tuple(z.shape)}"
            )


        CAPTURE_DATA[
            layer_index
        ] = {
            "z":
                z
        }


        first_active = min(
            ACTIVE_LAYERS
        )


        # -------------------------------------------------------------
        # Start autograd at the first writable layer.
        #
        # Everything below this point in the transformer remains frozen.
        #
        # The graph from this layer upward is retained so gradients can
        # propagate through later selected W_down layers as well.
        # -------------------------------------------------------------

        if layer_index == first_active:

            y = (
                y
                .detach()
                .requires_grad_(
                    True
                )
            )


        if not y.requires_grad:

            raise RuntimeError(
                f"Layer {layer_index} output does not require grad. "
                "The multi-layer gradient graph was unexpectedly broken."
            )


        y.retain_grad()


        CAPTURE_DATA[
            layer_index
        ][
            "y"
        ] = y


        return y


    return forward


# =============================================================================
# RE-PATCH FORWARDS
# =============================================================================

for layer_index in UNION_LAYERS:

    FAST[
        layer_index
    ].module.forward = (
        make_fast_forward(
            layer_index
        )
    )


print(
    "Corrected fast-weight forwards installed."
)


# =============================================================================
# STATIC BF16 PARITY
# =============================================================================

print()
print("=" * 120)
print("STATIC BF16 PARITY")
print("=" * 120)


parity_ids = tokenizer(
    "The quick brown fox jumps over the lazy dog.",
    return_tensors="pt",
)[
    "input_ids"
].to(
    DEVICE
)


activate_layers(
    []
)


with torch.no_grad():

    stock_logits = (
        model(
            input_ids=parity_ids,
            use_cache=False,
        )
        .logits[
            :,
            -1,
            :
        ]
        .float()
        .clone()
    )


reset_all_states()


activate_layers(
    LAYER_CONFIGS[
        "L8_15"
    ]
)


with torch.no_grad():

    fast_logits = (
        model(
            input_ids=parity_ids,
            use_cache=False,
        )
        .logits[
            :,
            -1,
            :
        ]
        .float()
        .clone()
    )


parity_difference = (
    fast_logits
    -
    stock_logits
)


parity_relative_l2 = float(
    (
        torch.linalg.vector_norm(
            parity_difference
        )
        /
        torch.linalg.vector_norm(
            stock_logits
        ).clamp_min(
            1e-30
        )
    )
    .item()
)


parity_max_abs = float(
    parity_difference
    .abs()
    .max()
    .item()
)


print(
    "Relative L2         :",
    f"{parity_relative_l2:.8e}",
)

print(
    "Max abs             :",
    f"{parity_max_abs:.8e}",
)


del stock_logits
del fast_logits
del parity_difference


# =============================================================================
# FIXED-K ANSWER-ONLY WRITE TENSOR
# =============================================================================

def build_write_tensor(
    examples,
):

    header = (
        "Delta Registry online memory write.\n"
        "Learn the exact association.\n"
    )


    input_tokens = tokenizer(
        header,
        add_special_tokens=True,
    )[
        "input_ids"
    ]


    labels = [
        -100
    ] * len(
        input_tokens
    )


    for question, answer in examples:

        prefix_text = (
            "\nQuestion: "
            +
            question
            +
            "\nAnswer:"
        )


        prefix_ids = tokenizer(
            prefix_text,
            add_special_tokens=False,
        )[
            "input_ids"
        ]


        answer_ids = tokenizer(
            " " + answer,
            add_special_tokens=False,
        )[
            "input_ids"
        ]


        separator_ids = tokenizer(
            "\n",
            add_special_tokens=False,
        )[
            "input_ids"
        ]


        if len(
            answer_ids
        ) != MATCHED_TOKEN_COUNT:

            raise RuntimeError(
                "Answer tokenizer length mismatch."
            )


        required = (
            len(
                prefix_ids
            )
            +
            len(
                answer_ids
            )
            +
            len(
                separator_ids
            )
        )


        if (
            len(
                input_tokens
            )
            +
            required
            >
            WRITE_K
        ):

            raise RuntimeError(
                f"Write sequence exceeds WRITE_K={WRITE_K}."
            )


        input_tokens.extend(
            prefix_ids
        )


        labels.extend(
            [-100]
            *
            len(
                prefix_ids
            )
        )


        input_tokens.extend(
            answer_ids
        )


        # -------------------------------------------------------------
        # ONLY answer tokens contribute to loss.
        # -------------------------------------------------------------

        labels.extend(
            answer_ids
        )


        input_tokens.extend(
            separator_ids
        )


        labels.extend(
            [-100]
            *
            len(
                separator_ids
            )
        )


    actual_length = len(
        input_tokens
    )


    padding = (
        WRITE_K
        -
        actual_length
    )


    input_tokens.extend(
        [
            tokenizer.pad_token_id
        ]
        *
        padding
    )


    labels.extend(
        [-100]
        *
        padding
    )


    attention = (
        [1]
        *
        actual_length
        +
        [0]
        *
        padding
    )


    return (

        torch.tensor(
            [
                input_tokens
            ],
            dtype=torch.long,
            device=DEVICE,
        ),

        torch.tensor(
            [
                attention
            ],
            dtype=torch.long,
            device=DEVICE,
        ),

        torch.tensor(
            [
                labels
            ],
            dtype=torch.long,
            device=DEVICE,
        ),

        actual_length,
    )


# =============================================================================
# REAL MULTI-LAYER GRADIENT CAPTURE
# =============================================================================

def capture_gradients(
    examples,
    layers,
):

    global CAPTURE_MODE
    global CAPTURE_DATA


    activate_layers(
        layers
    )


    (
        input_ids,
        attention_mask,
        labels,
        actual_length,
    ) = build_write_tensor(
        examples
    )


    CAPTURE_DATA = {}


    CAPTURE_MODE = True


    model.zero_grad(
        set_to_none=True
    )


    # Surface any prior CUDA problem before autograd.
    torch.cuda.synchronize()


    start = time.perf_counter()


    try:

        output = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            labels=labels,
            use_cache=False,
        )


        loss = float(
            output.loss
            .detach()
            .item()
        )


        output.loss.backward()


        torch.cuda.synchronize()


    finally:

        CAPTURE_MODE = False


    capture_ms = (
        time.perf_counter()
        -
        start
    ) * 1000.0


    gradients = {}


    for layer_index in layers:

        if layer_index not in CAPTURE_DATA:

            raise RuntimeError(
                f"Layer {layer_index} was not captured."
            )


        y = (
            CAPTURE_DATA[
                layer_index
            ][
                "y"
            ]
        )


        z = (
            CAPTURE_DATA[
                layer_index
            ][
                "z"
            ]
        )


        if y.grad is None:

            raise RuntimeError(
                f"No gradient for layer {layer_index}."
            )


        g = (
            y.grad[
                0
            ]
            .float()
            .contiguous()
        )


        gt = (
            g
            .T
            .contiguous()
        )


        if tuple(
            gt.shape
        ) != (
            M,
            WRITE_K,
        ):

            raise RuntimeError(
                f"GT shape for layer {layer_index}: "
                f"{tuple(gt.shape)}"
            )


        if tuple(
            z.shape
        ) != (
            WRITE_K,
            N,
        ):

            raise RuntimeError(
                f"Z shape for layer {layer_index}: "
                f"{tuple(z.shape)}"
            )


        gradients[
            layer_index
        ] = {
            "gt":
                gt,

            "z":
                z,
        }


    del output
    del input_ids
    del attention_mask
    del labels


    CAPTURE_DATA = {}


    gc.collect()
    torch.cuda.empty_cache()


    return (
        gradients,
        loss,
        capture_ms,
        actual_length,
    )


# =============================================================================
# MULTI-LAYER UPDATE
# =============================================================================

def apply_multilayer_update(
    gradients,
    layers,
    total_delta_norm,
):

    # Equal total update budget regardless of number of layers.

    per_layer_target = (
        total_delta_norm
        /
        math.sqrt(
            len(
                layers
            )
        )
    )


    gradient_norms = {}

    step_scales = {}


    # -------------------------------------------------------------------------
    # Exact global Frobenius norm per layer.
    # -------------------------------------------------------------------------

    for layer_index in layers:

        gt = (
            gradients[
                layer_index
            ][
                "gt"
            ]
        )


        z = (
            gradients[
                layer_index
            ][
                "z"
            ]
        )


        norm = get_gradient_norm(
            gt,
            z,
        )


        gradient_norms[
            layer_index
        ] = norm


        step_scales[
            layer_index
        ] = (
            per_layer_target
            /
            max(
                norm,
                1e-30,
            )
        )


    # -------------------------------------------------------------------------
    # Timing update + BF16 publication.
    # -------------------------------------------------------------------------

    start_event = torch.cuda.Event(
        enable_timing=True
    )


    kernels_done_event = torch.cuda.Event(
        enable_timing=True
    )


    publish_done_event = torch.cuda.Event(
        enable_timing=True
    )


    start_event.record()


    # FP32 update.
    for layer_index in layers:

        fast = (
            FAST[
                layer_index
            ]
        )


        gt = (
            gradients[
                layer_index
            ][
                "gt"
            ]
        )


        z = (
            gradients[
                layer_index
            ][
                "z"
            ]
        )


        safe_update_kernel[
            update_grid
        ](
            gt,
            z,

            fast.master_active,
            fast.master_staging,

            gt.stride(0),
            gt.stride(1),

            z.stride(0),
            z.stride(1),

            fast.master_active.stride(0),
            fast.master_active.stride(1),

            fast.master_staging.stride(0),
            fast.master_staging.stride(1),

            step_scales[
                layer_index
            ],

            M=M,
            N=N,
            K=WRITE_K,

            BM=UBM,
            BN=UBN,
            BK=UBK,

            num_warps=8,
        )


    kernels_done_event.record()


    # FP32 -> BF16 publication.
    for layer_index in layers:

        fast = (
            FAST[
                layer_index
            ]
        )


        fast.serve_staging.copy_(
            fast.master_staging
        )


    publish_done_event.record()


    publish_done_event.synchronize()


    kernel_ms = (
        start_event.elapsed_time(
            kernels_done_event
        )
    )


    publish_ms = (
        kernels_done_event.elapsed_time(
            publish_done_event
        )
    )


    total_ms = (
        start_event.elapsed_time(
            publish_done_event
        )
    )


    commit_layers(
        layers
    )


    return {

        "gradient_norms":
            gradient_norms,

        "step_scales":
            step_scales,

        "per_layer_target":
            per_layer_target,

        "kernel_ms":
            kernel_ms,

        "publish_ms":
            publish_ms,

        "total_ready_ms":
            total_ms,
    }


# =============================================================================
# WRITE LOSS
# =============================================================================

@torch.no_grad()
def evaluate_write_loss(
    examples,
):

    (
        input_ids,
        attention_mask,
        labels,
        _,
    ) = build_write_tensor(
        examples
    )


    output = model(
        input_ids=input_ids,
        attention_mask=attention_mask,
        labels=labels,
        use_cache=False,
    )


    value = float(
        output.loss.item()
    )


    del output
    del input_ids
    del attention_mask
    del labels


    return value


# =============================================================================
# RETRIEVAL PROMPTS
# =============================================================================

def no_context_prompt(
    question,
):

    return (
        "Answer the Delta Registry query with only the exact stored "
        "private code. Do not explain.\n\n"
        f"Question: {question}\n"
        "Answer:"
    )


def full_context_prompt(
    question,
):

    return (
        FULL_CONTEXT_FACT
        +
        "\n\n"
        "Using the authoritative record above, answer with only the "
        "exact stored private code.\n\n"
        f"Question: {question}\n"
        "Answer:"
    )


# =============================================================================
# BATCHED CANDIDATE SCORING
# =============================================================================

@torch.no_grad()
def score_candidates(
    prompt,
    candidates,
):

    rows = []
    labels_rows = []


    for candidate in candidates:

        prefix = tokenizer(
            prompt,
            add_special_tokens=True,
        )[
            "input_ids"
        ]


        answer = tokenizer(
            " " + candidate,
            add_special_tokens=False,
        )[
            "input_ids"
        ]


        if len(
            answer
        ) != MATCHED_TOKEN_COUNT:

            raise RuntimeError(
                "Candidate tokenizer length mismatch."
            )


        rows.append(
            prefix
            +
            answer
        )


        labels_rows.append(
            [-100]
            *
            len(
                prefix
            )
            +
            answer
        )


    max_len = max(
        len(
            row
        )
        for row in rows
    )


    padded_ids = []
    padded_labels = []
    masks = []


    for row, labels in zip(
        rows,
        labels_rows,
    ):

        pad = (
            max_len
            -
            len(
                row
            )
        )


        padded_ids.append(
            row
            +
            [
                tokenizer.pad_token_id
            ]
            *
            pad
        )


        padded_labels.append(
            labels
            +
            [-100]
            *
            pad
        )


        masks.append(
            [1]
            *
            len(
                row
            )
            +
            [0]
            *
            pad
        )


    input_ids = torch.tensor(
        padded_ids,
        dtype=torch.long,
        device=DEVICE,
    )


    labels = torch.tensor(
        padded_labels,
        dtype=torch.long,
        device=DEVICE,
    )


    attention_mask = torch.tensor(
        masks,
        dtype=torch.long,
        device=DEVICE,
    )


    output = model(
        input_ids=input_ids,
        attention_mask=attention_mask,
        use_cache=False,
    )


    logits = (
        output.logits[
            :,
            :-1,
            :
        ]
        .float()
    )


    shifted_labels = (
        labels[
            :,
            1:
        ]
    )


    valid = (
        shifted_labels
        !=
        -100
    )


    safe_labels = (
        shifted_labels
        .clamp_min(
            0
        )
    )


    log_probs = F.log_softmax(
        logits,
        dim=-1,
    )


    selected = (
        log_probs
        .gather(
            -1,
            safe_labels.unsqueeze(
                -1
            ),
        )
        .squeeze(
            -1
        )
    )


    selected = (
        selected
        *
        valid
    )


    sequence_logprobs = (
        selected
        .sum(
            dim=1
        )
    )


    answer_lengths = (
        valid
        .sum(
            dim=1
        )
        .clamp_min(
            1
        )
    )


    mean_nll = (
        -sequence_logprobs
        /
        answer_lengths
    )


    probabilities = torch.softmax(
        sequence_logprobs,
        dim=0,
    )


    result = {}


    for index, candidate in enumerate(
        candidates
    ):

        result[
            candidate
        ] = {
            "logprob":
                float(
                    sequence_logprobs[
                        index
                    ]
                    .item()
                ),

            "probability":
                float(
                    probabilities[
                        index
                    ]
                    .item()
                ),

            "nll":
                float(
                    mean_nll[
                        index
                    ]
                    .item()
                ),
        }


    del output
    del logits
    del log_probs
    del input_ids
    del labels
    del attention_mask


    return result


# =============================================================================
# HELD-OUT RETRIEVAL
# =============================================================================

def evaluate_secret(
    expected_secret,
    use_context=False,
):

    rows = []


    for question in TEST_QUESTIONS:

        prompt = (
            full_context_prompt(
                question
            )
            if use_context
            else
            no_context_prompt(
                question
            )
        )


        scores = score_candidates(
            prompt,
            CANDIDATES,
        )


        prediction = max(
            CANDIDATES,

            key=lambda candidate:
                scores[
                    candidate
                ][
                    "logprob"
                ],
        )


        correct_logprob = (
            scores[
                expected_secret
            ][
                "logprob"
            ]
        )


        best_distractor_logprob = max(

            scores[
                candidate
            ][
                "logprob"
            ]

            for candidate in CANDIDATES

            if candidate != expected_secret
        )


        rows.append(
            {
                "question":
                    question,

                "expected":
                    expected_secret,

                "prediction":
                    prediction,

                "correct":
                    prediction
                    ==
                    expected_secret,

                "p_correct":
                    scores[
                        expected_secret
                    ][
                        "probability"
                    ],

                "answer_nll":
                    scores[
                        expected_secret
                    ][
                        "nll"
                    ],

                "logprob_margin":
                    correct_logprob
                    -
                    best_distractor_logprob,
            }
        )


    return {

        "accuracy":
            sum(
                row[
                    "correct"
                ]
                for row in rows
            )
            /
            len(
                rows
            ),

        "mean_p_correct":
            statistics.mean(
                row[
                    "p_correct"
                ]
                for row in rows
            ),

        "mean_nll":
            statistics.mean(
                row[
                    "answer_nll"
                ]
                for row in rows
            ),

        "mean_margin":
            statistics.mean(
                row[
                    "logprob_margin"
                ]
                for row in rows
            ),

        "rows":
            rows,
    }


# =============================================================================
# GREEDY GENERATION
# =============================================================================

def normalize_answer(
    text,
):

    return re.sub(
        r"\s+",
        "",
        text
        .strip()
        .lower()
        .rstrip(
            ".,;:!?"
        ),
    )


@torch.no_grad()
def generate_answer(
    question,
):

    prompt = no_context_prompt(
        question
    )


    encoded = tokenizer(
        prompt,
        return_tensors="pt",
    )


    input_ids = (
        encoded[
            "input_ids"
        ]
        .to(
            DEVICE
        )
    )


    output_ids = model.generate(

        input_ids=input_ids,

        max_new_tokens=
            MAX_NEW_TOKENS,

        do_sample=False,

        use_cache=True,

        pad_token_id=
            tokenizer.eos_token_id,
    )


    generated = tokenizer.decode(
        output_ids[
            0,
            input_ids.shape[1]:
        ],
        skip_special_tokens=True,
    )


    del encoded
    del input_ids
    del output_ids


    return generated.strip()


# =============================================================================
# W0 BASELINES
# =============================================================================

print()
print("=" * 120)
print("W0 BASELINES")
print("=" * 120)


reset_all_states()


activate_layers(
    []
)


W0_NO_CONTEXT = evaluate_secret(
    SECRET_A,
    use_context=False,
)


W0_FULL_CONTEXT = evaluate_secret(
    SECRET_A,
    use_context=True,
)


print(
    "W0 no-context accuracy:",
    f"{W0_NO_CONTEXT['accuracy'] * 100:.2f}%",
)

print(
    "W0 no-context P*      :",
    f"{W0_NO_CONTEXT['mean_p_correct']:.6f}",
)

print(
    "W0 no-context margin  :",
    f"{W0_NO_CONTEXT['mean_margin']:.6f}",
)

print(
    "W0 no-context NLL     :",
    f"{W0_NO_CONTEXT['mean_nll']:.6f}",
)


print()


print(
    "W0 full-context acc   :",
    f"{W0_FULL_CONTEXT['accuracy'] * 100:.2f}%",
)

print(
    "W0 full-context P*    :",
    f"{W0_FULL_CONTEXT['mean_p_correct']:.6f}",
)

print(
    "W0 full-context margin:",
    f"{W0_FULL_CONTEXT['mean_margin']:.6f}",
)

print(
    "W0 full-context NLL   :",
    f"{W0_FULL_CONTEXT['mean_nll']:.6f}",
)


# =============================================================================
# REAL-QWEN FIXED-K WARMUP
# =============================================================================

print()
print("=" * 120)
print("REAL-QWEN FIXED-K TRITON WARMUP")
print("=" * 120)


reset_all_states()


activate_layers(
    [13]
)


(
    warm_gradients,
    warm_loss,
    warm_capture_ms,
    warm_actual_length,
) = capture_gradients(
    WRITE_QUESTIONS_A,
    [13],
)


# Explicitly prove capture itself didn't poison CUDA.
torch.cuda.synchronize()


warm_norm = get_gradient_norm(

    warm_gradients[
        13
    ][
        "gt"
    ],

    warm_gradients[
        13
    ][
        "z"
    ],
)


warm_stats = apply_multilayer_update(
    warm_gradients,
    [13],
    0.01,
)


print(
    "Actual write tokens :",
    warm_actual_length,
)

print(
    "Write loss          :",
    f"{warm_loss:.6f}",
)

print(
    "Gradient capture    :",
    f"{warm_capture_ms:.3f} ms",
)

print(
    "Gradient norm       :",
    f"{warm_norm:.8e}",
)

print(
    "Kernel              :",
    f"{warm_stats['kernel_ms']:.3f} ms",
)

print(
    "BF16 publication    :",
    f"{warm_stats['publish_ms']:.3f} ms",
)

print(
    "Total ready         :",
    f"{warm_stats['total_ready_ms']:.3f} ms",
)


del warm_gradients


# Remove warmup adaptation entirely.
reset_all_states()


torch.cuda.synchronize()


print(
    "REAL-QWEN WARMUP PASSED."
)


# =============================================================================
# CAPACITY SWEEP
# =============================================================================

print()
print("=" * 170)
print("SINGLE-ASSOCIATION ONLINE-LEARNING SWEEP")
print("=" * 170)


print(
    f"{'config':<12}"
    f"{'Δtotal':>9}"
    f"{'step':>6}"
    f"{'write pre':>12}"
    f"{'write post':>12}"
    f"{'grad ms':>10}"
    f"{'kernel':>9}"
    f"{'publish':>9}"
    f"{'ready':>9}"
    f"{'acc':>9}"
    f"{'P*':>10}"
    f"{'margin':>11}"
    f"{'NLL':>10}"
)


print(
    "-" * 170
)


SWEEP_RESULTS = []


for config_name, layers in LAYER_CONFIGS.items():

    for total_delta_norm in TOTAL_DELTA_NORMS:

        reset_all_states()


        activate_layers(
            layers
        )


        for step in range(
            1,
            MAX_STEPS
            +
            1,
        ):

            (
                gradients,
                write_pre,
                gradient_capture_ms,
                actual_length,
            ) = capture_gradients(
                WRITE_QUESTIONS_A,
                layers,
            )


            update_stats = apply_multilayer_update(
                gradients,
                layers,
                total_delta_norm,
            )


            write_post = evaluate_write_loss(
                WRITE_QUESTIONS_A
            )


            if step in CHECKPOINT_STEPS:

                retrieval = evaluate_secret(
                    SECRET_A,
                    use_context=False,
                )


                record = {

                    "config":
                        config_name,

                    "layers":
                        list(
                            layers
                        ),

                    "layer_count":
                        len(
                            layers
                        ),

                    "total_delta_norm":
                        total_delta_norm,

                    "step":
                        step,

                    "actual_write_tokens":
                        actual_length,

                    "write_loss_before":
                        write_pre,

                    "write_loss_after":
                        write_post,

                    "gradient_capture_ms":
                        gradient_capture_ms,

                    "kernel_ms":
                        update_stats[
                            "kernel_ms"
                        ],

                    "publish_ms":
                        update_stats[
                            "publish_ms"
                        ],

                    "total_ready_ms":
                        update_stats[
                            "total_ready_ms"
                        ],

                    "accuracy":
                        retrieval[
                            "accuracy"
                        ],

                    "mean_p_correct":
                        retrieval[
                            "mean_p_correct"
                        ],

                    "mean_margin":
                        retrieval[
                            "mean_margin"
                        ],

                    "mean_nll":
                        retrieval[
                            "mean_nll"
                        ],
                }


                SWEEP_RESULTS.append(
                    record
                )


                print(
                    f"{config_name:<12}"
                    f"{total_delta_norm:>9.3f}"
                    f"{step:>6}"
                    f"{write_pre:>12.4f}"
                    f"{write_post:>12.4f}"
                    f"{gradient_capture_ms:>10.2f}"
                    f"{update_stats['kernel_ms']:>9.3f}"
                    f"{update_stats['publish_ms']:>9.3f}"
                    f"{update_stats['total_ready_ms']:>9.3f}"
                    f"{retrieval['accuracy'] * 100:>8.1f}%"
                    f"{retrieval['mean_p_correct']:>10.4f}"
                    f"{retrieval['mean_margin']:>11.4f}"
                    f"{retrieval['mean_nll']:>10.4f}"
                )


            del gradients


            gc.collect()
            torch.cuda.empty_cache()


# =============================================================================
# SELECT BEST DIAGNOSTIC CONFIGURATION
#
# IMPORTANT:
#
# This uses the held-out paraphrases to diagnose whether the retrofit has
# capacity at all.
#
# For the paper, tune on separate synthetic worlds/seeds, then evaluate on
# untouched worlds.
# =============================================================================

BEST = max(

    SWEEP_RESULTS,

    key=lambda row: (

        row[
            "accuracy"
        ],

        row[
            "mean_p_correct"
        ],

        row[
            "mean_margin"
        ],

        -row[
            "mean_nll"
        ],
    ),
)


BEST_LAYERS = list(
    BEST[
        "layers"
    ]
)


BEST_DELTA = float(
    BEST[
        "total_delta_norm"
    ]
)


BEST_STEPS = int(
    BEST[
        "step"
    ]
)


print()
print("=" * 120)
print("BEST DIAGNOSTIC CONFIGURATION")
print("=" * 120)


print(
    "Configuration       :",
    BEST[
        "config"
    ],
)

print(
    "Layers              :",
    BEST_LAYERS,
)

print(
    "Total delta norm    :",
    BEST_DELTA,
)

print(
    "Steps               :",
    BEST_STEPS,
)

print(
    "Accuracy            :",
    f"{BEST['accuracy'] * 100:.2f}%",
)

print(
    "P(correct)          :",
    f"{BEST['mean_p_correct']:.6f}",
)

print(
    "Margin              :",
    f"{BEST['mean_margin']:.6f}",
)

print(
    "NLL                 :",
    f"{BEST['mean_nll']:.6f}",
)


# =============================================================================
# REBUILD BEST STATE FROM CLEAN W0
# =============================================================================

print()
print("=" * 120)
print("REBUILDING BEST ONLINE FAST-WEIGHT STATE")
print("=" * 120)


reset_all_states()


activate_layers(
    BEST_LAYERS
)


BEST_WRITE_HISTORY = []


for step in range(
    1,
    BEST_STEPS
    +
    1,
):

    (
        gradients,
        loss_before,
        capture_ms,
        actual_length,
    ) = capture_gradients(
        WRITE_QUESTIONS_A,
        BEST_LAYERS,
    )


    update_stats = apply_multilayer_update(
        gradients,
        BEST_LAYERS,
        BEST_DELTA,
    )


    loss_after = evaluate_write_loss(
        WRITE_QUESTIONS_A
    )


    BEST_WRITE_HISTORY.append(
        {
            "step":
                step,

            "loss_before":
                loss_before,

            "loss_after":
                loss_after,

            "gradient_capture_ms":
                capture_ms,

            "kernel_ms":
                update_stats[
                    "kernel_ms"
                ],

            "publish_ms":
                update_stats[
                    "publish_ms"
                ],

            "total_ready_ms":
                update_stats[
                    "total_ready_ms"
                ],
        }
    )


    print(
        f"step={step:>2} | "
        f"write={loss_before:.4f}->{loss_after:.4f} | "
        f"grad={capture_ms:.2f} ms | "
        f"kernel={update_stats['kernel_ms']:.3f} ms | "
        f"publish={update_stats['publish_ms']:.3f} ms | "
        f"ready={update_stats['total_ready_ms']:.3f} ms"
    )


    del gradients


# =============================================================================
# FINAL SECRET-A RETRIEVAL
# =============================================================================

FINAL_A = evaluate_secret(
    SECRET_A,
    use_context=False,
)


print()
print("=" * 120)
print("SECRET-A RETRIEVAL WITHOUT CONTEXT")
print("=" * 120)


print(
    "Accuracy            :",
    f"{FINAL_A['accuracy'] * 100:.2f}%",
)

print(
    "P(correct)          :",
    f"{FINAL_A['mean_p_correct']:.6f}",
)

print(
    "Mean margin         :",
    f"{FINAL_A['mean_margin']:.6f}",
)

print(
    "Mean NLL            :",
    f"{FINAL_A['mean_nll']:.6f}",
)


print()


for index, row in enumerate(
    FINAL_A[
        "rows"
    ]
):

    print(
        f"Q{index + 1:<2} | "
        f"prediction={row['prediction']:<12} | "
        f"expected={SECRET_A:<12} | "
        f"P*={row['p_correct']:.4f} | "
        f"margin={row['logprob_margin']:+.4f}"
    )


# =============================================================================
# CONTEXT-ADVANTAGE RECOVERY
# =============================================================================

base_probability = (
    W0_NO_CONTEXT[
        "mean_p_correct"
    ]
)


context_probability = (
    W0_FULL_CONTEXT[
        "mean_p_correct"
    ]
)


fast_probability = (
    FINAL_A[
        "mean_p_correct"
    ]
)


context_advantage = (
    context_probability
    -
    base_probability
)


if abs(
    context_advantage
) > 1e-12:

    recovered_context_advantage = (
        (
            fast_probability
            -
            base_probability
        )
        /
        context_advantage
    )

else:

    recovered_context_advantage = None


print()
print("=" * 120)
print("MEMORY TRANSFER")
print("=" * 120)


print(
    "W0 no-context P*    :",
    f"{base_probability:.6f}",
)

print(
    "W0 full-context P*  :",
    f"{context_probability:.6f}",
)

print(
    "Fast-state P*       :",
    f"{fast_probability:.6f}",
)


if recovered_context_advantage is not None:

    print(
        "Recovered context   :",
        f"{recovered_context_advantage * 100:.2f}%",
    )


# =============================================================================
# GREEDY GENERATION — SECRET A
# =============================================================================

print()
print("=" * 120)
print("GREEDY RETRIEVAL — SECRET A")
print("=" * 120)


GENERATION_A = []


for question in TEST_QUESTIONS:

    generated = generate_answer(
        question
    )


    correct = (
        normalize_answer(
            generated
        )
        .startswith(
            normalize_answer(
                SECRET_A
            )
        )
    )


    GENERATION_A.append(
        {
            "question":
                question,

            "generated":
                generated,

            "correct":
                correct,
        }
    )


    print(
        f"expected={SECRET_A:<12} | "
        f"generated={generated[:60]!r}"
    )


generation_a_accuracy = (
    sum(
        row[
            "correct"
        ]
        for row in GENERATION_A
    )
    /
    len(
        GENERATION_A
    )
)


print(
    "Exact-match generation:",
    f"{generation_a_accuracy * 100:.2f}%",
)


# =============================================================================
# ONLINE OVERWRITE
#
# No reset.
# No replay of SECRET_A.
#
# ENTITY -> SECRET_B
# =============================================================================

WRITE_QUESTIONS_B = [

    (
        f"The previous private code for {ENTITY} is obsolete. "
        f"What is its new code?",
        SECRET_B,
    ),

    (
        f"After the latest registry update, what exact value belongs "
        f"to {ENTITY}?",
        SECRET_B,
    ),

    (
        f"Which replacement secret now maps to identity {ENTITY}?",
        SECRET_B,
    ),

    (
        f"The Delta registry changed {ENTITY}. "
        f"What exact private code is current?",
        SECRET_B,
    ),

    (
        f"What new code supersedes the old record for {ENTITY}?",
        SECRET_B,
    ),

    (
        f"Return the latest private value for {ENTITY}.",
        SECRET_B,
    ),
]


def average_candidate_probability(
    candidate,
):

    values = []


    for question in TEST_QUESTIONS:

        scores = score_candidates(
            no_context_prompt(
                question
            ),
            CANDIDATES,
        )


        values.append(
            scores[
                candidate
            ][
                "probability"
            ]
        )


    return statistics.mean(
        values
    )


p_a_before = average_candidate_probability(
    SECRET_A
)


p_b_before = average_candidate_probability(
    SECRET_B
)


print()
print("=" * 120)
print("ONLINE OVERWRITE — SECRET A -> SECRET B")
print("=" * 120)


print(
    "Before overwrite:"
)

print(
    "P(secret A)         :",
    f"{p_a_before:.6f}",
)

print(
    "P(secret B)         :",
    f"{p_b_before:.6f}",
)


OVERWRITE_HISTORY = []


# Use same number of online steps as best Secret-A configuration.

for step in range(
    1,
    BEST_STEPS
    +
    1,
):

    (
        gradients,
        loss_before,
        capture_ms,
        _,
    ) = capture_gradients(
        WRITE_QUESTIONS_B,
        BEST_LAYERS,
    )


    update_stats = apply_multilayer_update(
        gradients,
        BEST_LAYERS,
        BEST_DELTA,
    )


    loss_after = evaluate_write_loss(
        WRITE_QUESTIONS_B
    )


    OVERWRITE_HISTORY.append(
        {
            "step":
                step,

            "loss_before":
                loss_before,

            "loss_after":
                loss_after,

            "gradient_capture_ms":
                capture_ms,

            "kernel_ms":
                update_stats[
                    "kernel_ms"
                ],

            "publish_ms":
                update_stats[
                    "publish_ms"
                ],

            "total_ready_ms":
                update_stats[
                    "total_ready_ms"
                ],
        }
    )


    print(
        f"step={step:>2} | "
        f"loss={loss_before:.4f}->{loss_after:.4f} | "
        f"grad={capture_ms:.2f} ms | "
        f"ready={update_stats['total_ready_ms']:.3f} ms"
    )


    del gradients


p_a_after = average_candidate_probability(
    SECRET_A
)


p_b_after = average_candidate_probability(
    SECRET_B
)


FINAL_B = evaluate_secret(
    SECRET_B,
    use_context=False,
)


print()
print(
    "After overwrite:"
)

print(
    "P(secret A)         :",
    f"{p_a_after:.6f}",
)

print(
    "P(secret B)         :",
    f"{p_b_after:.6f}",
)

print(
    "B retrieval accuracy:",
    f"{FINAL_B['accuracy'] * 100:.2f}%",
)

print(
    "B retrieval margin  :",
    f"{FINAL_B['mean_margin']:.6f}",
)


# =============================================================================
# GREEDY GENERATION — SECRET B
# =============================================================================

print()
print("=" * 120)
print("GREEDY RETRIEVAL — SECRET B")
print("=" * 120)


GENERATION_B = []


for question in TEST_QUESTIONS:

    generated = generate_answer(
        question
    )


    correct = (
        normalize_answer(
            generated
        )
        .startswith(
            normalize_answer(
                SECRET_B
            )
        )
    )


    GENERATION_B.append(
        {
            "question":
                question,

            "generated":
                generated,

            "correct":
                correct,
        }
    )


    print(
        f"expected={SECRET_B:<12} | "
        f"generated={generated[:60]!r}"
    )


generation_b_accuracy = (
    sum(
        row[
            "correct"
        ]
        for row in GENERATION_B
    )
    /
    len(
        GENERATION_B
    )
)


print(
    "Exact-match generation:",
    f"{generation_b_accuracy * 100:.2f}%",
)


# =============================================================================
# STATE-SIZE / KV BREAK-EVEN
#
# Current PROTOTYPE per fast layer:
#
#   FP32 master A = 4 bytes / parameter
#   FP32 master B = 4
#   BF16 serve A  = 2
#   BF16 serve B  = 2
#
#   total = 12 bytes / parameter
#
# This is intentionally conservative and is NOT the final compact memory
# architecture.
# =============================================================================

bytes_per_fast_layer = (
    M
    *
    N
    *
    12
)


best_fast_state_bytes = (
    bytes_per_fast_layer
    *
    len(
        BEST_LAYERS
    )
)


# Qwen KV cache per token:
#
# all layers *
# K + V *
# KV heads *
# head dimension *
# BF16 bytes

kv_bytes_per_token = (
    NUM_LAYERS
    *
    2
    *
    NUM_KV_HEADS
    *
    HEAD_DIM
    *
    2
)


kv_break_even_tokens = (
    best_fast_state_bytes
    /
    kv_bytes_per_token
)


print()
print("=" * 120)
print("BOUNDED-STATE VS KV MEMORY")
print("=" * 120)


print(
    "Best fast layers     :",
    len(
        BEST_LAYERS
    ),
)

print(
    "Prototype fast state :",
    f"{best_fast_state_bytes / 1024**2:.2f} MiB",
)

print(
    "KV bytes/token       :",
    f"{kv_bytes_per_token / 1024:.2f} KiB",
)

print(
    "KV break-even        :",
    f"{kv_break_even_tokens:,.0f} tokens",
)


for context_length in [
    1_000,
    4_000,
    8_000,
    16_000,
    32_000,
    64_000,
    128_000,
]:

    kv_memory = (
        kv_bytes_per_token
        *
        context_length
    )


    print(
        f"context={context_length:>7,} | "
        f"KV={kv_memory / 1024**2:>9.2f} MiB | "
        f"fast={best_fast_state_bytes / 1024**2:>9.2f} MiB"
    )


# =============================================================================
# SYSTEMS TIMING SUMMARY
# =============================================================================

best_rows = [

    row

    for row in SWEEP_RESULTS

    if (
        row[
            "config"
        ]
        ==
        BEST[
            "config"
        ]
        and
        row[
            "total_delta_norm"
        ]
        ==
        BEST[
            "total_delta_norm"
        ]
    )
]


median_update_ms = statistics.median(
    [
        row[
            "total_ready_ms"
        ]
        for row in best_rows
    ]
)


median_kernel_ms = statistics.median(
    [
        row[
            "kernel_ms"
        ]
        for row in best_rows
    ]
)


median_publish_ms = statistics.median(
    [
        row[
            "publish_ms"
        ]
        for row in best_rows
    ]
)


median_gradient_ms = statistics.median(
    [
        row[
            "gradient_capture_ms"
        ]
        for row in best_rows
    ]
)


print()
print("=" * 120)
print("SYSTEMS SUMMARY")
print("=" * 120)


print(
    "Median gradient capture:",
    f"{median_gradient_ms:.3f} ms",
)

print(
    "Median Triton update  :",
    f"{median_kernel_ms:.3f} ms",
)

print(
    "Median BF16 publish   :",
    f"{median_publish_ms:.3f} ms",
)

print(
    "Median ready latency  :",
    f"{median_update_ms:.3f} ms",
)

print(
    "Norm workspace        :",
    f"{NUM_NORM_TILES * 4 / 1024:.2f} KiB",
)

print(
    "Full dW materialized  :",
    "NO",
)


# =============================================================================
# ARCHITECTURAL DECISION
# =============================================================================

print()
print("=" * 120)
print("ARCHITECTURAL DIAGNOSTIC")
print("=" * 120)


print(
    "Secret-A accuracy     :",
    f"{FINAL_A['accuracy'] * 100:.2f}%",
)

print(
    "Secret-A P(correct)   :",
    f"{FINAL_A['mean_p_correct']:.6f}",
)

print(
    "Secret-A margin       :",
    f"{FINAL_A['mean_margin']:.6f}",
)

print(
    "Secret-B overwrite acc:",
    f"{FINAL_B['accuracy'] * 100:.2f}%",
)

print(
    "P(A) before overwrite:",
    f"{p_a_before:.6f}",
)

print(
    "P(A) after overwrite :",
    f"{p_a_after:.6f}",
)

print(
    "P(B) before overwrite:",
    f"{p_b_before:.6f}",
)

print(
    "P(B) after overwrite :",
    f"{p_b_after:.6f}",
)


if recovered_context_advantage is not None:

    print(
        "Context gain recovered:",
        f"{recovered_context_advantage * 100:.2f}%",
    )


print()


strong_insert = (
    FINAL_A[
        "accuracy"
    ]
    >=
    0.75
    and
    FINAL_A[
        "mean_margin"
    ]
    >
    0
)


strong_overwrite = (
    FINAL_B[
        "accuracy"
    ]
    >=
    0.75
    and
    p_b_after
    >
    p_a_after
)


if strong_insert and strong_overwrite:

    print(
        "RESULT: SUCCESSFUL ONLINE ASSOCIATIVE LEARNING."
    )

    print(
        "The selected Qwen fast-weight configuration stored an unknown "
        "association, retrieved it without the original context, and "
        "subsequently overwrote it online."
    )


elif (
    FINAL_A[
        "mean_p_correct"
    ]
    >
    W0_NO_CONTEXT[
        "mean_p_correct"
    ]
    *
    2
):

    print(
        "RESULT: STRONG MEMORY SIGNAL, BUT NOT YET ROBUST."
    )

    print(
        "The fast state substantially increases probability of the unknown "
        "association, but retrieval/overwrite is not yet reliable enough."
    )


else:

    print(
        "RESULT: ORDINARY QWEN W_down IS NOT A RELIABLE ASSOCIATIVE MEMORY."
    )

    print(
        "Keep the asynchronous A/B runtime and Triton rematerialization "
        "mechanism, but move the learned state into a dedicated compact "
        "TTT / associative-memory module."
    )


# =============================================================================
# SAVE
# =============================================================================

RESULT_PATH = Path(
    "async_ttt_qwen7b_single_association_complete.json"
)


results_json = {

    "environment": {
        "model":
            MODEL_ID,

        "pytorch":
            torch.__version__,

        "cuda":
            torch.version.cuda,

        "triton":
            triton.__version__,

        "gpu":
            torch.cuda.get_device_name(0),
    },

    "experiment": {
        "entity":
            ENTITY,

        "secret_a":
            SECRET_A,

        "secret_b":
            SECRET_B,

        "candidates":
            CANDIDATES,

        "candidate_token_count":
            MATCHED_TOKEN_COUNT,

        "write_k":
            WRITE_K,

        "layer_configs":
            LAYER_CONFIGS,

        "total_delta_norms":
            TOTAL_DELTA_NORMS,

        "checkpoint_steps":
            CHECKPOINT_STEPS,
    },

    "kernel_validation": {
        "norm_relative_error":
            norm_relative_error,

        "norm_workspace_bytes":
            NUM_NORM_TILES
            *
            4,

        "full_dw_materialized_in_ttt":
            False,
    },

    "parity": {
        "relative_l2":
            parity_relative_l2,

        "max_abs":
            parity_max_abs,
    },

    "baseline": {
        "no_context":
            W0_NO_CONTEXT,

        "full_context":
            W0_FULL_CONTEXT,
    },

    "sweep":
        SWEEP_RESULTS,

    "best":
        BEST,

    "best_rebuild":
        BEST_WRITE_HISTORY,

    "final_secret_a":
        FINAL_A,

    "generation_a":
        GENERATION_A,

    "memory_transfer": {
        "base_probability":
            base_probability,

        "context_probability":
            context_probability,

        "fast_probability":
            fast_probability,

        "recovered_context_advantage":
            recovered_context_advantage,
    },

    "overwrite": {
        "secret_a":
            SECRET_A,

        "secret_b":
            SECRET_B,

        "p_a_before":
            p_a_before,

        "p_b_before":
            p_b_before,

        "history":
            OVERWRITE_HISTORY,

        "p_a_after":
            p_a_after,

        "p_b_after":
            p_b_after,

        "final_secret_b":
            FINAL_B,

        "generation_b":
            GENERATION_B,
    },

    "memory": {
        "prototype_fast_state_bytes":
            best_fast_state_bytes,

        "kv_bytes_per_token":
            kv_bytes_per_token,

        "kv_break_even_tokens":
            kv_break_even_tokens,
    },

    "timing": {
        "median_gradient_capture_ms":
            median_gradient_ms,

        "median_kernel_ms":
            median_kernel_ms,

        "median_publish_ms":
            median_publish_ms,

        "median_ready_ms":
            median_update_ms,
    },
}


with open(
    RESULT_PATH,
    "w",
) as file:

    json.dump(
        results_json,
        file,
        indent=2,
    )


print()
print("=" * 120)
print("DONE")
print("=" * 120)


print(
    "Saved:",
    RESULT_PATH.resolve(),
)


# =============================================================================
# RESTORE STOCK QWEN
# =============================================================================

for layer_index in UNION_LAYERS:

    FAST[
        layer_index
    ].module.forward = (
        FAST[
            layer_index
        ].original_forward
    )


torch.cuda.synchronize()


free_bytes, total_bytes = torch.cuda.mem_get_info()


print(
    "Original Qwen forwards restored."
)

print(
    "Free VRAM:",
    f"{free_bytes / 1024**3:.2f} GiB",
)

FLUSHING PREVIOUS GPU STATE


Free VRAM : 39.08 GiB
Total VRAM: 39.49 GiB

QWEN2.5-7B — UNKNOWN-CONTEXT ONLINE LEARNING
Model               : Qwen/Qwen2.5-7B-Instruct
Python              : 3.12.11
PyTorch             : 2.8.0+cu128
CUDA                : 12.8
Triton              : 3.4.0
GPU                 : NVIDIA A100-SXM4-40GB
SMs                 : 108
VRAM                : 39.49 GiB

LOADING QWEN


Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

hidden_size         : 3584
intermediate_size   : 18944
layers              : 28
attention heads     : 28
KV heads            : 4
head dim            : 128
candidate layers    : [8, 9, 10, 11, 12, 13, 14, 15]
fixed write K       : 256
Free after model    : 24.89 GiB

BUILDING MATCHED-TOKEN SYNTHETIC SECRETS
Matched token count : 6
Candidate pool      : 96
Candidates          : ['TKMS-077', 'EGCR-046', 'NZRD-304', 'CXAT-679', 'GTGL-694', 'MBJD-397', 'SHLY-840', 'DJEB-600']
SECRET_A            : TKMS-077
SECRET_B            : EGCR-046
Prior logprob span  : 0.286716
TKMS-077     [44299, 4826, 12, 15, 22, 22]
EGCR-046     [56111, 8973, 12, 15, 19, 21]
NZRD-304     [41757, 36690, 12, 18, 15, 19]
CXAT-679     [48483, 828, 12, 21, 22, 24]
GTGL-694     [11911, 3825, 12, 21, 24, 19]
MBJD-397     [13339, 49915, 12, 18, 24, 22]
SHLY-840     [6434, 8932, 12, 23, 19, 15]
DJEB-600     [21387, 8264, 12, 21, 15, 15]

UNKNOWN ASSOCIATION
Entity              : QEVAX-731
Secret A            : TKMS-077
Hel

In [1]:
# =============================================================================
# ASYNC-TTT — QWEN2.5-7B
#
# SINGLE-TOKEN ONLINE MEMORY
# + OPEN-VOCAB GREEDY RETRIEVAL
# + CONTRASTIVE ONLINE OVERWRITE
#
#
# Based on previous successful result:
#
#   writable layers        = 12,13,14,15
#   total update norm      = 0.10
#   online steps           = 8
#
#
# WHY THIS EXPERIMENT
# -------------------
#
# Previous result:
#
#   W0 no-context P(correct)      = 0.0482
#   adapted P(correct)            = 0.7585
#   candidate accuracy            = 87.5%
#   recovered context advantage   = 74.62%
#
# BUT:
#
#   greedy exact generation       = 0%
#   overwrite B accuracy          = 12.5%
#
#
# Two remaining confounds:
#
# 1. The synthetic secret was SIX tokenizer tokens.
#
#    Ranking one 6-token sequence above seven candidate sequences does not
#    imply the model will greedily emit that exact sequence against the
#    entire vocabulary.
#
# 2. Ordinary CE on SECRET_B teaches the new association but does not
#    explicitly erase SECRET_A.
#
#
# THIS VERSION
# ------------
#
# - finds synthetic-looking SAFE SINGLE-TOKEN answers
# - matches their stock next-token priors
# - trains directly at the next-token answer position
# - uses full-vocabulary CE for insertion
# - uses held-out paraphrases for evaluation
# - tests actual greedy next-token generation
# - overwrite objective includes:
#
#       CE(new secret)
#       +
#       candidate CE(new secret)
#       +
#       explicit margin:
#
#           new_logit > old_logit + margin
#
# - keeps:
#
#       FP32 master A/B
#       BF16 serving A/B
#       rematerialized G^T Z Triton update
#       no full dW materialization
#
#
# IMPORTANT
# ---------
#
# This tests persistent learned context.
#
# Qwen itself STILL USES attention/KV cache for current-context inference.
#
# Do not call this a KV-free architecture yet.
# =============================================================================


# =============================================================================
# IMPORTS
# =============================================================================

import gc
import json
import math
import random
import re
import statistics
import sys
import time

from pathlib import Path

import torch
import torch.nn.functional as F

import triton
import triton.language as tl

from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
)


# =============================================================================
# GPU CLEANUP
# =============================================================================

print("=" * 120)
print("FLUSHING PREVIOUS GPU STATE")
print("=" * 120)


for name in [
    "model",
    "tokenizer",
    "FAST",
    "CAPTURE_DATA",
    "output",
]:

    if name in globals():

        try:
            del globals()[name]
        except Exception:
            pass


gc.collect()


if torch.cuda.is_available():

    torch.cuda.synchronize()
    torch.cuda.empty_cache()

    try:
        torch.cuda.ipc_collect()
    except Exception:
        pass

    gc.collect()
    torch.cuda.empty_cache()


assert torch.cuda.is_available()


free_bytes, total_bytes = (
    torch.cuda.mem_get_info()
)


print(
    "Free VRAM :",
    f"{free_bytes / 1024**3:.2f} GiB",
)

print(
    "Total VRAM:",
    f"{total_bytes / 1024**3:.2f} GiB",
)


if free_bytes / 1024**3 < 30:

    raise RuntimeError(
        "\nLess than 30 GiB free.\n"
        "Restart the kernel and rerun."
    )


# =============================================================================
# CONFIG
# =============================================================================

MODEL_ID = (
    "Qwen/Qwen2.5-7B-Instruct"
)


DEVICE = "cuda"


SEED = 730421


ENTITY = "QEVAX-731"


FAST_LAYERS = [
    12,
    13,
    14,
    15,
]


TOTAL_DELTA_NORM = 0.10


INSERT_STEPS = 8


OVERWRITE_STEPS = 8


# Number of write prompts processed simultaneously.
WRITE_BATCH = 8


# Every prompt is padded to this exact sequence length.
PROMPT_K = 96


# Triton reduction dimension:
#
# flatten:
#
#     batch × sequence
#
# into:
#
#     K
#
FLAT_K = (
    WRITE_BATCH
    *
    PROMPT_K
)


NUM_CANDIDATES = 8


# Weight of candidate-restricted CE in addition to full-vocab CE.
CANDIDATE_LOSS_WEIGHT = 0.50


# Explicit overwrite margin:
#
#     logit(new) >= logit(old) + margin
OVERWRITE_MARGIN = 4.0


OVERWRITE_MARGIN_WEIGHT = 1.0


torch.manual_seed(
    SEED
)


random.seed(
    SEED
)


rng = random.Random(
    SEED
)


torch.backends.cuda.matmul.allow_tf32 = False
torch.backends.cudnn.allow_tf32 = False


# =============================================================================
# ENVIRONMENT
# =============================================================================

props = (
    torch.cuda.get_device_properties(
        0
    )
)


print()
print("=" * 120)
print("QWEN2.5-7B — SINGLE-TOKEN ONLINE MEMORY")
print("=" * 120)


print(
    "Model               :",
    MODEL_ID,
)

print(
    "Python              :",
    sys.version.split()[0],
)

print(
    "PyTorch             :",
    torch.__version__,
)

print(
    "CUDA                :",
    torch.version.cuda,
)

print(
    "Triton              :",
    triton.__version__,
)

print(
    "GPU                 :",
    torch.cuda.get_device_name(0),
)

print(
    "Fast layers         :",
    FAST_LAYERS,
)

print(
    "Prompt K            :",
    PROMPT_K,
)

print(
    "Write batch         :",
    WRITE_BATCH,
)

print(
    "Flattened Triton K  :",
    FLAT_K,
)


# =============================================================================
# LOAD MODEL
# =============================================================================

print()
print("=" * 120)
print("LOADING QWEN")
print("=" * 120)


tokenizer = (
    AutoTokenizer.from_pretrained(
        MODEL_ID,
        trust_remote_code=True,
    )
)


if tokenizer.pad_token_id is None:

    tokenizer.pad_token = (
        tokenizer.eos_token
    )


model = (
    AutoModelForCausalLM.from_pretrained(

        MODEL_ID,

        dtype=
            torch.bfloat16,

        device_map={
            "": 0
        },

        low_cpu_mem_usage=True,

        trust_remote_code=True,

        attn_implementation=
            "sdpa",
    )
)


model.eval()


for parameter in model.parameters():

    parameter.requires_grad_(
        False
    )


M = int(
    model.config.hidden_size
)


N = int(
    model.config.intermediate_size
)


NUM_LAYERS = int(
    model.config.num_hidden_layers
)


NUM_ATTENTION_HEADS = int(
    model.config.num_attention_heads
)


NUM_KV_HEADS = int(
    getattr(
        model.config,
        "num_key_value_heads",
        NUM_ATTENTION_HEADS,
    )
)


HEAD_DIM = (
    M
    //
    NUM_ATTENTION_HEADS
)


print(
    "M                   :",
    M,
)

print(
    "N                   :",
    N,
)

print(
    "Layers              :",
    NUM_LAYERS,
)

print(
    "KV heads            :",
    NUM_KV_HEADS,
)

print(
    "Head dim            :",
    HEAD_DIM,
)


free_bytes, _ = (
    torch.cuda.mem_get_info()
)


print(
    "Free after load     :",
    f"{free_bytes / 1024**3:.2f} GiB",
)


# =============================================================================
# FIND SAFE SINGLE-TOKEN SECRETS
#
# We select tokens such that:
#
#       tokenizer(" " + DISPLAY) == [token_id]
#
# We also restrict decoded text to simple alphabetic strings so outputs are
# readable.
#
# The association itself remains completely unknown:
#
#       QEVAX-731 -> RANDOM TOKEN
# =============================================================================

print()
print("=" * 120)
print("SEARCHING FOR SINGLE-TOKEN SECRETS")
print("=" * 120)


valid_single_tokens = []


special_ids = set(
    tokenizer.all_special_ids
)


for token_id in range(
    tokenizer.vocab_size
):

    if token_id in special_ids:
        continue


    decoded = tokenizer.decode(
        [
            token_id
        ],
        clean_up_tokenization_spaces=False,
    )


    # Require normal single-leading-space word.
    if not re.fullmatch(
        r" [A-Za-z]{4,10}",
        decoded,
    ):

        continue


    display = decoded.strip()


    roundtrip = tokenizer(
        " " + display,
        add_special_tokens=False,
    )[
        "input_ids"
    ]


    if roundtrip != [
        token_id
    ]:

        continue


    valid_single_tokens.append(
        {
            "id":
                token_id,

            "text":
                display,
        }
    )


    if len(
        valid_single_tokens
    ) >= 1024:

        break


if len(
    valid_single_tokens
) < 64:

    raise RuntimeError(
        "Could not find enough safe single-token candidates."
    )


print(
    "Safe token pool      :",
    len(
        valid_single_tokens
    ),
)


# =============================================================================
# MATCH STOCK PRIORS
# =============================================================================

neutral_prompt = (
    "Return the exact private registry value.\n"
    "Answer:"
)


neutral_ids = tokenizer(
    neutral_prompt,
    return_tensors=
        "pt",
)[
    "input_ids"
].to(
    DEVICE
)


with torch.no_grad():

    neutral_logits = (
        model(
            input_ids=
                neutral_ids,

            use_cache=
                False,
        )
        .logits[
            0,
            -1,
            :
        ]
        .float()
    )


scored_tokens = []


for item in valid_single_tokens:

    scored_tokens.append(
        (
            item[
                "id"
            ],

            item[
                "text"
            ],

            float(
                neutral_logits[
                    item[
                        "id"
                    ]
                ]
                .item()
            ),
        )
    )


scored_tokens.sort(
    key=lambda x:
        x[
            2
        ]
)


best_window = None

best_span = float(
    "inf"
)


for start in range(
    len(
        scored_tokens
    )
    -
    NUM_CANDIDATES
    +
    1
):

    window = (
        scored_tokens[
            start:
            start + NUM_CANDIDATES
        ]
    )


    span = (
        window[
            -1
        ][
            2
        ]
        -
        window[
            0
        ][
            2
        ]
    )


    if span < best_span:

        best_span = span

        best_window = window


CANDIDATES = [
    {
        "id":
            token_id,

        "text":
            text,

        "neutral_logit":
            logit,
    }

    for token_id, text, logit
    in best_window
]


rng.shuffle(
    CANDIDATES
)


SECRET_A = (
    CANDIDATES[
        0
    ]
)


SECRET_B = (
    CANDIDATES[
        1
    ]
)


CANDIDATE_IDS = torch.tensor(
    [
        item[
            "id"
        ]
        for item in CANDIDATES
    ],

    dtype=
        torch.long,

    device=
        DEVICE,
)


SECRET_A_INDEX = next(

    index

    for index, item in enumerate(
        CANDIDATES
    )

    if item[
        "id"
    ]
    ==
    SECRET_A[
        "id"
    ]
)


SECRET_B_INDEX = next(

    index

    for index, item in enumerate(
        CANDIDATES
    )

    if item[
        "id"
    ]
    ==
    SECRET_B[
        "id"
    ]
)


print(
    "Prior logit span     :",
    f"{best_span:.6f}",
)


for item in CANDIDATES:

    print(
        f"id={item['id']:<7} | "
        f"text={item['text']!r:<14} | "
        f"neutral={item['neutral_logit']:+.4f}"
    )


print()


print(
    "SECRET_A             :",
    SECRET_A[
        "text"
    ],
    "| token",
    SECRET_A[
        "id"
    ],
)

print(
    "SECRET_B             :",
    SECRET_B[
        "text"
    ],
    "| token",
    SECRET_B[
        "id"
    ],
)


del neutral_logits
del neutral_ids


# =============================================================================
# WRITE AND TEST PROMPTS
# =============================================================================

INSERT_WRITE_QUESTIONS = [

    f"What private registry value is assigned to {ENTITY}?",

    f"State the stored private value belonging to {ENTITY}.",

    f"Which secret is mapped to identifier {ENTITY}?",

    f"Give the confidential lookup result for {ENTITY}.",

    f"What exact value should the registry return for {ENTITY}?",

    f"Recall the private value associated with {ENTITY}.",

    f"Resolve {ENTITY} to its stored confidential value.",

    f"Which private token belongs to registry key {ENTITY}?",
]


OVERWRITE_WRITE_QUESTIONS = [

    f"The previous value for {ENTITY} is obsolete. What is its new value?",

    f"After the registry update, what value now belongs to {ENTITY}?",

    f"Which replacement secret is currently mapped to {ENTITY}?",

    f"What new confidential value supersedes the old record for {ENTITY}?",

    f"Return the latest private value registered for {ENTITY}.",

    f"What is the current replacement value of {ENTITY}?",

    f"Resolve {ENTITY} using the newly updated registry value.",

    f"Which current private token should replace the old value for {ENTITY}?",
]


TEST_QUESTIONS = [

    f"An operator queries {ENTITY}. What exact private value should be returned?",

    f"What value is stored under lookup identity {ENTITY}?",

    f"Resolve identifier {ENTITY} to its secret registry value.",

    f"If a request references {ENTITY}, what confidential value belongs to it?",

    f"Which exact value corresponds to the record named {ENTITY}?",

    f"Retrieve the private value associated with {ENTITY}.",

    f"For registry key {ENTITY}, provide its stored secret.",

    f"What is the hidden lookup result for {ENTITY}?",
]


FULL_CONTEXT_A = (
    f"Authoritative Delta Registry record: "
    f"{ENTITY} has private value {SECRET_A['text']}."
)


print()
print("=" * 120)
print("UNKNOWN ASSOCIATION")
print("=" * 120)


print(
    ENTITY,
    "->",
    SECRET_A[
        "text"
    ],
)


# =============================================================================
# FAST-WEIGHT STATES
# =============================================================================

class LayerState:

    def __init__(
        self,
        layer_index,
    ):

        self.layer_index = (
            layer_index
        )


        self.module = (
            model
            .model
            .layers[
                layer_index
            ]
            .mlp
            .down_proj
        )


        self.original_forward = (
            self.module.forward
        )


        self.original_weight = (
            self.module
            .weight
            .detach()
        )


        # FP32 master A/B
        self.master_a = (
            self.original_weight
            .float()
            .contiguous()
        )


        self.master_b = (
            torch.empty_like(
                self.master_a
            )
        )


        # BF16 serving A/B
        self.serve_a = (
            self.original_weight
            .clone()
            .contiguous()
        )


        self.serve_b = (
            torch.empty_like(
                self.serve_a
            )
        )


        self.master_active = (
            self.master_a
        )

        self.master_staging = (
            self.master_b
        )


        self.serve_active = (
            self.serve_a
        )

        self.serve_staging = (
            self.serve_b
        )


        self.version = 0


    def reset(
        self,
    ):

        self.master_a.copy_(
            self.original_weight
        )

        self.master_b.copy_(
            self.original_weight
        )


        self.serve_a.copy_(
            self.original_weight
        )

        self.serve_b.copy_(
            self.original_weight
        )


        self.master_active = (
            self.master_a
        )

        self.master_staging = (
            self.master_b
        )


        self.serve_active = (
            self.serve_a
        )

        self.serve_staging = (
            self.serve_b
        )


        self.version = 0


    def commit(
        self,
    ):

        self.master_active, self.master_staging = (
            self.master_staging,
            self.master_active,
        )


        self.serve_active, self.serve_staging = (
            self.serve_staging,
            self.serve_active,
        )


        self.version += 1


print()
print("=" * 120)
print("ALLOCATING 4-LAYER FAST STATE")
print("=" * 120)


FAST = {
    layer:
        LayerState(
            layer
        )
    for layer in FAST_LAYERS
}


torch.cuda.synchronize()


free_bytes, _ = (
    torch.cuda.mem_get_info()
)


print(
    "Free after fast state:",
    f"{free_bytes / 1024**3:.2f} GiB",
)


# =============================================================================
# CAPTURE STATE
# =============================================================================

CAPTURE_MODE = False

CAPTURE_DATA = {}


def reset_states():

    for state in FAST.values():

        state.reset()


    torch.cuda.synchronize()


def commit_states():

    for layer in FAST_LAYERS:

        FAST[
            layer
        ].commit()


# =============================================================================
# PATCH FORWARDS
# =============================================================================

def make_forward(
    layer_index,
):

    fast = (
        FAST[
            layer_index
        ]
    )


    def forward(
        x,
    ):

        global CAPTURE_DATA


        y = F.linear(
            x,
            fast.serve_active,
            bias=None,
        )


        if not CAPTURE_MODE:

            return y


        # x:
        #
        # [B, S, N]
        #
        # flatten to:
        #
        # [B*S, N]

        z = (
            x
            .detach()
            .float()
            .reshape(
                -1,
                N,
            )
            .contiguous()
        )


        if tuple(
            z.shape
        ) != (
            FLAT_K,
            N,
        ):

            raise RuntimeError(
                f"Layer {layer_index}: "
                f"unexpected Z {tuple(z.shape)}, "
                f"expected {(FLAT_K, N)}"
            )


        CAPTURE_DATA[
            layer_index
        ] = {
            "z":
                z
        }


        # Create autograd leaf at first writable layer.
        if (
            layer_index
            ==
            FAST_LAYERS[
                0
            ]
        ):

            y = (
                y
                .detach()
                .requires_grad_(
                    True
                )
            )


        if not y.requires_grad:

            raise RuntimeError(
                f"Layer {layer_index} graph was broken."
            )


        y.retain_grad()


        CAPTURE_DATA[
            layer_index
        ][
            "y"
        ] = y


        return y


    return forward


for layer in FAST_LAYERS:

    FAST[
        layer
    ].module.forward = (
        make_forward(
            layer
        )
    )


# =============================================================================
# STATIC PARITY
# =============================================================================

print()
print("=" * 120)
print("STATIC BF16 PARITY")
print("=" * 120)


test_ids = tokenizer(
    "The quick brown fox jumps over the lazy dog.",
    return_tensors="pt",
)[
    "input_ids"
].to(
    DEVICE
)


# Temporarily original forwards.

for layer in FAST_LAYERS:

    FAST[
        layer
    ].module.forward = (
        FAST[
            layer
        ].original_forward
    )


with torch.no_grad():

    stock_logits = (
        model(
            input_ids=
                test_ids,

            use_cache=
                False,
        )
        .logits[
            :,
            -1,
            :
        ]
        .float()
        .clone()
    )


# Re-patch.

for layer in FAST_LAYERS:

    FAST[
        layer
    ].module.forward = (
        make_forward(
            layer
        )
    )


reset_states()


with torch.no_grad():

    fast_logits = (
        model(
            input_ids=
                test_ids,

            use_cache=
                False,
        )
        .logits[
            :,
            -1,
            :
        ]
        .float()
        .clone()
    )


difference = (
    fast_logits
    -
    stock_logits
)


parity_rel = float(
    (
        torch.linalg.vector_norm(
            difference
        )
        /
        torch.linalg.vector_norm(
            stock_logits
        )
    )
    .item()
)


parity_max = float(
    difference
    .abs()
    .max()
    .item()
)


print(
    "Relative L2         :",
    f"{parity_rel:.8e}",
)

print(
    "Max abs             :",
    f"{parity_max:.8e}",
)


del stock_logits
del fast_logits
del difference


# =============================================================================
# FIXED PROMPT BATCH
#
# No answer tokens are appended.
#
# Model predicts the desired secret as the NEXT token after:
#
#       Answer:
# =============================================================================

def build_prompt_batch(
    questions,
):

    if len(
        questions
    ) != WRITE_BATCH:

        raise RuntimeError(
            f"Expected {WRITE_BATCH} questions, "
            f"got {len(questions)}"
        )


    rows = []

    masks = []

    last_positions = []


    for question in questions:

        prompt = (
            "Delta Registry query.\n"
            "Return only the exact private value.\n\n"
            f"Question: {question}\n"
            "Answer:"
        )


        ids = tokenizer(
            prompt,
            add_special_tokens=True,
        )[
            "input_ids"
        ]


        if len(
            ids
        ) > PROMPT_K:

            raise RuntimeError(
                f"Prompt has {len(ids)} tokens "
                f"but PROMPT_K={PROMPT_K}."
            )


        last_positions.append(
            len(
                ids
            )
            -
            1
        )


        padding = (
            PROMPT_K
            -
            len(
                ids
            )
        )


        rows.append(
            ids
            +
            [
                tokenizer.pad_token_id
            ]
            *
            padding
        )


        masks.append(
            [1]
            *
            len(
                ids
            )
            +
            [0]
            *
            padding
        )


    input_ids = torch.tensor(
        rows,

        dtype=
            torch.long,

        device=
            DEVICE,
    )


    attention_mask = torch.tensor(
        masks,

        dtype=
            torch.long,

        device=
            DEVICE,
    )


    last_positions = torch.tensor(
        last_positions,

        dtype=
            torch.long,

        device=
            DEVICE,
    )


    return (
        input_ids,
        attention_mask,
        last_positions,
    )


# =============================================================================
# CUSTOM ONLINE WRITE LOSS
# =============================================================================

def capture_write_gradient(
    questions,
    target_secret,
    old_secret=None,
):

    global CAPTURE_MODE
    global CAPTURE_DATA


    (
        input_ids,
        attention_mask,
        last_positions,
    ) = build_prompt_batch(
        questions
    )


    CAPTURE_DATA = {}


    CAPTURE_MODE = True


    model.zero_grad(
        set_to_none=True
    )


    torch.cuda.synchronize()


    start = time.perf_counter()


    try:

        output = model(

            input_ids=
                input_ids,

            attention_mask=
                attention_mask,

            use_cache=
                False,
        )


        batch_indices = torch.arange(
            WRITE_BATCH,
            device=
                DEVICE,
        )


        answer_logits = (
            output.logits[
                batch_indices,
                last_positions,
                :
            ]
            .float()
        )


        target_ids = torch.full(

            (
                WRITE_BATCH,
            ),

            target_secret[
                "id"
            ],

            dtype=
                torch.long,

            device=
                DEVICE,
        )


        # -------------------------------------------------------------
        # 1. Full-vocabulary next-token CE.
        #
        # This is what should eventually make GREEDY generation work.
        # -------------------------------------------------------------

        full_ce = F.cross_entropy(
            answer_logits,
            target_ids,
        )


        # -------------------------------------------------------------
        # 2. Matched-candidate CE.
        #
        # Stabilizes association learning while still using the full-vocab CE.
        # -------------------------------------------------------------

        candidate_logits = (
            answer_logits[
                :,
                CANDIDATE_IDS
            ]
        )


        target_candidate_index = next(

            index

            for index, item in enumerate(
                CANDIDATES
            )

            if item[
                "id"
            ]
            ==
            target_secret[
                "id"
            ]
        )


        candidate_targets = torch.full(

            (
                WRITE_BATCH,
            ),

            target_candidate_index,

            dtype=
                torch.long,

            device=
                DEVICE,
        )


        candidate_ce = F.cross_entropy(
            candidate_logits,
            candidate_targets,
        )


        loss = (
            full_ce
            +
            CANDIDATE_LOSS_WEIGHT
            *
            candidate_ce
        )


        margin_loss = torch.tensor(
            0.0,
            device=
                DEVICE,
        )


        # -------------------------------------------------------------
        # 3. Explicit overwrite pressure.
        #
        # Push:
        #
        #       new_logit > old_logit + OVERWRITE_MARGIN
        #
        # so merely adding probability to B while retaining A is no longer
        # enough.
        # -------------------------------------------------------------

        if old_secret is not None:

            new_logits = (
                answer_logits[
                    :,
                    target_secret[
                        "id"
                    ]
                ]
            )


            old_logits = (
                answer_logits[
                    :,
                    old_secret[
                        "id"
                    ]
                ]
            )


            margin_loss = (
                F.softplus(
                    old_logits
                    -
                    new_logits
                    +
                    OVERWRITE_MARGIN
                )
                .mean()
            )


            loss = (
                loss
                +
                OVERWRITE_MARGIN_WEIGHT
                *
                margin_loss
            )


        loss_value = float(
            loss
            .detach()
            .item()
        )


        full_ce_value = float(
            full_ce
            .detach()
            .item()
        )


        candidate_ce_value = float(
            candidate_ce
            .detach()
            .item()
        )


        margin_value = float(
            margin_loss
            .detach()
            .item()
        )


        loss.backward()


        torch.cuda.synchronize()


    finally:

        CAPTURE_MODE = False


    capture_ms = (
        time.perf_counter()
        -
        start
    ) * 1000.0


    gradients = {}


    for layer in FAST_LAYERS:

        data = (
            CAPTURE_DATA[
                layer
            ]
        )


        y = (
            data[
                "y"
            ]
        )


        if y.grad is None:

            raise RuntimeError(
                f"No gradient for layer {layer}."
            )


        # [B,S,M] -> [B*S,M]
        g = (
            y.grad
            .float()
            .reshape(
                -1,
                M,
            )
            .contiguous()
        )


        # [M,B*S]
        gt = (
            g
            .T
            .contiguous()
        )


        z = (
            data[
                "z"
            ]
        )


        if tuple(
            gt.shape
        ) != (
            M,
            FLAT_K,
        ):

            raise RuntimeError(
                f"Layer {layer} GT mismatch: "
                f"{tuple(gt.shape)}"
            )


        gradients[
            layer
        ] = {
            "gt":
                gt,

            "z":
                z,
        }


    del output
    del input_ids
    del attention_mask
    del last_positions


    CAPTURE_DATA = {}


    gc.collect()
    torch.cuda.empty_cache()


    return (
        gradients,
        {
            "total_loss":
                loss_value,

            "full_ce":
                full_ce_value,

            "candidate_ce":
                candidate_ce_value,

            "margin_loss":
                margin_value,
        },
        capture_ms,
    )


# =============================================================================
# SAFE TRITON TWO-PASS UPDATE
# =============================================================================

NBM = 32
NBN = 64
NBK = 64


UBM = 64
UBN = 64
UBK = 64


NUM_M_TILES = (
    triton.cdiv(
        M,
        NBM,
    )
)


NUM_N_TILES = (
    triton.cdiv(
        N,
        NBN,
    )
)


NUM_NORM_TILES = (
    NUM_M_TILES
    *
    NUM_N_TILES
)


norm_tiles = torch.empty(

    NUM_NORM_TILES,

    dtype=
        torch.float32,

    device=
        DEVICE,
)


norm_grid = (
    NUM_M_TILES,
    NUM_N_TILES,
)


update_grid = (

    triton.cdiv(
        M,
        UBM,
    ),

    triton.cdiv(
        N,
        UBN,
    ),
)


print()
print("=" * 120)
print("TRITON WORKSPACE")
print("=" * 120)


print(
    "Flattened K          :",
    FLAT_K,
)

print(
    "Norm workspace       :",
    f"{NUM_NORM_TILES * 4 / 1024:.2f} KiB",
)


@triton.jit
def norm_kernel(
    gt_ptr,
    z_ptr,

    norm_ptr,

    stride_gm,
    stride_gk,

    stride_zk,
    stride_zn,

    M: tl.constexpr,
    N: tl.constexpr,
    K: tl.constexpr,

    NUM_N_TILES: tl.constexpr,

    BM: tl.constexpr,
    BN: tl.constexpr,
    BK: tl.constexpr,
):

    pid_m = tl.program_id(
        0
    )

    pid_n = tl.program_id(
        1
    )


    offs_m = (
        pid_m
        *
        BM
        +
        tl.arange(
            0,
            BM,
        )
    )


    offs_n = (
        pid_n
        *
        BN
        +
        tl.arange(
            0,
            BN,
        )
    )


    acc = tl.zeros(
        (
            BM,
            BN,
        ),

        dtype=
            tl.float32,
    )


    for k_start in range(
        0,
        K,
        BK,
    ):

        offs_k = (
            k_start
            +
            tl.arange(
                0,
                BK,
            )
        )


        g = tl.load(

            gt_ptr
            +
            offs_m[:, None]
            *
            stride_gm
            +
            offs_k[None, :]
            *
            stride_gk,

            mask=(
                (
                    offs_m[:, None]
                    <
                    M
                )
                &
                (
                    offs_k[None, :]
                    <
                    K
                )
            ),

            other=
                0.0,
        )


        z = tl.load(

            z_ptr
            +
            offs_k[:, None]
            *
            stride_zk
            +
            offs_n[None, :]
            *
            stride_zn,

            mask=(
                (
                    offs_k[:, None]
                    <
                    K
                )
                &
                (
                    offs_n[None, :]
                    <
                    N
                )
            ),

            other=
                0.0,
        )


        acc = tl.dot(
            g,
            z,
            acc,

            input_precision=
                "ieee",
        )


    valid = (
        (
            offs_m[:, None]
            <
            M
        )
        &
        (
            offs_n[None, :]
            <
            N
        )
    )


    acc = tl.where(
        valid,
        acc,
        0.0,
    )


    row_sum = tl.sum(
        acc
        *
        acc,
        axis=1,
    )


    tile_sum = tl.sum(
        row_sum,
        axis=0,
    )


    linear_pid = (
        pid_m
        *
        NUM_N_TILES
        +
        pid_n
    )


    tl.store(
        norm_ptr
        +
        linear_pid,

        tile_sum,
    )


@triton.jit
def update_kernel(
    gt_ptr,
    z_ptr,

    src_ptr,
    dst_ptr,

    stride_gm,
    stride_gk,

    stride_zk,
    stride_zn,

    stride_sm,
    stride_sn,

    stride_dm,
    stride_dn,

    step_scale,

    M: tl.constexpr,
    N: tl.constexpr,
    K: tl.constexpr,

    BM: tl.constexpr,
    BN: tl.constexpr,
    BK: tl.constexpr,
):

    pid_m = tl.program_id(
        0
    )

    pid_n = tl.program_id(
        1
    )


    offs_m = (
        pid_m
        *
        BM
        +
        tl.arange(
            0,
            BM,
        )
    )


    offs_n = (
        pid_n
        *
        BN
        +
        tl.arange(
            0,
            BN,
        )
    )


    acc = tl.zeros(
        (
            BM,
            BN,
        ),

        dtype=
            tl.float32,
    )


    for k_start in range(
        0,
        K,
        BK,
    ):

        offs_k = (
            k_start
            +
            tl.arange(
                0,
                BK,
            )
        )


        g = tl.load(

            gt_ptr
            +
            offs_m[:, None]
            *
            stride_gm
            +
            offs_k[None, :]
            *
            stride_gk,

            mask=(
                (
                    offs_m[:, None]
                    <
                    M
                )
                &
                (
                    offs_k[None, :]
                    <
                    K
                )
            ),

            other=
                0.0,
        )


        z = tl.load(

            z_ptr
            +
            offs_k[:, None]
            *
            stride_zk
            +
            offs_n[None, :]
            *
            stride_zn,

            mask=(
                (
                    offs_k[:, None]
                    <
                    K
                )
                &
                (
                    offs_n[None, :]
                    <
                    N
                )
            ),

            other=
                0.0,
        )


        acc = tl.dot(
            g,
            z,
            acc,

            input_precision=
                "ieee",
        )


    valid = (
        (
            offs_m[:, None]
            <
            M
        )
        &
        (
            offs_n[None, :]
            <
            N
        )
    )


    old_weight = tl.load(

        src_ptr
        +
        offs_m[:, None]
        *
        stride_sm
        +
        offs_n[None, :]
        *
        stride_sn,

        mask=
            valid,

        other=
            0.0,
    )


    new_weight = (
        old_weight
        -
        step_scale
        *
        acc
    )


    tl.store(

        dst_ptr
        +
        offs_m[:, None]
        *
        stride_dm
        +
        offs_n[None, :]
        *
        stride_dn,

        new_weight,

        mask=
            valid,
    )


# =============================================================================
# NORM HELPER
# =============================================================================

def gradient_norm(
    gt,
    z,
):

    assert tuple(
        gt.shape
    ) == (
        M,
        FLAT_K,
    )


    assert tuple(
        z.shape
    ) == (
        FLAT_K,
        N,
    )


    torch.cuda.synchronize()


    norm_kernel[
        norm_grid
    ](
        gt,
        z,

        norm_tiles,

        gt.stride(0),
        gt.stride(1),

        z.stride(0),
        z.stride(1),

        M=M,
        N=N,
        K=FLAT_K,

        NUM_N_TILES=
            NUM_N_TILES,

        BM=
            NBM,

        BN=
            NBN,

        BK=
            NBK,

        num_warps=
            4,
    )


    torch.cuda.synchronize()


    norm_sq = (
        norm_tiles
        .sum(
            dtype=
                torch.float64
        )
    )


    value = float(
        torch.sqrt(
            norm_sq
        )
        .item()
    )


    return value


# =============================================================================
# APPLY FOUR-LAYER UPDATE
# =============================================================================

def apply_update(
    gradients,
):

    per_layer_target = (
        TOTAL_DELTA_NORM
        /
        math.sqrt(
            len(
                FAST_LAYERS
            )
        )
    )


    scales = {}


    norms = {}


    for layer in FAST_LAYERS:

        norm = gradient_norm(

            gradients[
                layer
            ][
                "gt"
            ],

            gradients[
                layer
            ][
                "z"
            ],
        )


        norms[
            layer
        ] = norm


        scales[
            layer
        ] = (
            per_layer_target
            /
            max(
                norm,
                1e-30,
            )
        )


    start_event = torch.cuda.Event(
        enable_timing=True
    )


    kernel_event = torch.cuda.Event(
        enable_timing=True
    )


    publish_event = torch.cuda.Event(
        enable_timing=True
    )


    start_event.record()


    for layer in FAST_LAYERS:

        fast = (
            FAST[
                layer
            ]
        )


        gt = (
            gradients[
                layer
            ][
                "gt"
            ]
        )


        z = (
            gradients[
                layer
            ][
                "z"
            ]
        )


        update_kernel[
            update_grid
        ](
            gt,
            z,

            fast.master_active,
            fast.master_staging,

            gt.stride(0),
            gt.stride(1),

            z.stride(0),
            z.stride(1),

            fast.master_active.stride(0),
            fast.master_active.stride(1),

            fast.master_staging.stride(0),
            fast.master_staging.stride(1),

            scales[
                layer
            ],

            M=M,
            N=N,
            K=FLAT_K,

            BM=
                UBM,

            BN=
                UBN,

            BK=
                UBK,

            num_warps=
                8,
        )


    kernel_event.record()


    for layer in FAST_LAYERS:

        fast = (
            FAST[
                layer
            ]
        )


        fast.serve_staging.copy_(
            fast.master_staging
        )


    publish_event.record()


    publish_event.synchronize()


    kernel_ms = (
        start_event.elapsed_time(
            kernel_event
        )
    )


    publish_ms = (
        kernel_event.elapsed_time(
            publish_event
        )
    )


    total_ms = (
        start_event.elapsed_time(
            publish_event
        )
    )


    commit_states()


    return {
        "norms":
            norms,

        "kernel_ms":
            kernel_ms,

        "publish_ms":
            publish_ms,

        "total_ms":
            total_ms,
    }


# =============================================================================
# TRITON SMOKE TEST
# =============================================================================

print()
print("=" * 120)
print("TRITON SMOKE TEST")
print("=" * 120)


test_gt = (
    torch.randn(

        (
            M,
            FLAT_K,
        ),

        device=
            DEVICE,

        dtype=
            torch.float32,
    )
    *
    1e-3
)


test_z = (
    torch.randn(

        (
            FLAT_K,
            N,
        ),

        device=
            DEVICE,

        dtype=
            torch.float32,
    )
    *
    1e-3
)


triton_norm = gradient_norm(
    test_gt,
    test_z,
)


reference_dw = (
    test_gt
    @
    test_z
)


reference_norm = float(
    torch.linalg.vector_norm(
        reference_dw
    )
    .item()
)


relative_error = (
    abs(
        triton_norm
        -
        reference_norm
    )
    /
    reference_norm
)


print(
    "PyTorch norm         :",
    f"{reference_norm:.8e}",
)

print(
    "Triton norm          :",
    f"{triton_norm:.8e}",
)

print(
    "Relative error       :",
    f"{relative_error:.8e}",
)


if relative_error > 1e-4:

    raise RuntimeError(
        "Triton norm validation failed."
    )


del test_gt
del test_z
del reference_dw


gc.collect()
torch.cuda.empty_cache()


print(
    "SMOKE TEST PASSED."
)


# =============================================================================
# SINGLE-QUESTION EVALUATION
# =============================================================================

def eval_prompt(
    question,
    expected_secret,
    full_context=None,
):

    if full_context is None:

        prompt = (
            "Delta Registry query.\n"
            "Return only the exact private value.\n\n"
            f"Question: {question}\n"
            "Answer:"
        )

    else:

        prompt = (
            full_context
            +
            "\n\n"
            "Return only the exact private value.\n\n"
            f"Question: {question}\n"
            "Answer:"
        )


    encoded = tokenizer(
        prompt,
        return_tensors=
            "pt",
    )


    input_ids = (
        encoded[
            "input_ids"
        ]
        .to(
            DEVICE
        )
    )


    attention_mask = (
        encoded[
            "attention_mask"
        ]
        .to(
            DEVICE
        )
    )


    with torch.no_grad():

        logits = (
            model(

                input_ids=
                    input_ids,

                attention_mask=
                    attention_mask,

                use_cache=
                    False,
            )
            .logits[
                0,
                -1,
                :
            ]
            .float()
        )


    candidate_logits = (
        logits[
            CANDIDATE_IDS
        ]
    )


    candidate_probs = torch.softmax(
        candidate_logits,
        dim=0,
    )


    expected_index = next(

        index

        for index, item in enumerate(
            CANDIDATES
        )

        if item[
            "id"
        ]
        ==
        expected_secret[
            "id"
        ]
    )


    prediction_index = int(
        torch.argmax(
            candidate_logits
        )
        .item()
    )


    prediction = (
        CANDIDATES[
            prediction_index
        ]
    )


    p_correct = float(
        candidate_probs[
            expected_index
        ]
        .item()
    )


    best_distractor = max(

        float(
            candidate_logits[
                index
            ]
            .item()
        )

        for index in range(
            len(
                CANDIDATES
            )
        )

        if index
        !=
        expected_index
    )


    correct_logit = float(
        candidate_logits[
            expected_index
        ]
        .item()
    )


    margin = (
        correct_logit
        -
        best_distractor
    )


    greedy_token_id = int(
        torch.argmax(
            logits
        )
        .item()
    )


    greedy_text = tokenizer.decode(
        [
            greedy_token_id
        ],
        clean_up_tokenization_spaces=False,
    ).strip()


    # Full-vocabulary rank.
    rank = int(
        (
            logits
            >
            logits[
                expected_secret[
                    "id"
                ]
            ]
        )
        .sum()
        .item()
    ) + 1


    return {
        "prediction":
            prediction[
                "text"
            ],

        "candidate_correct":
            prediction[
                "id"
            ]
            ==
            expected_secret[
                "id"
            ],

        "p_correct":
            p_correct,

        "margin":
            margin,

        "greedy_token_id":
            greedy_token_id,

        "greedy_text":
            greedy_text,

        "greedy_correct":
            greedy_token_id
            ==
            expected_secret[
                "id"
            ],

        "full_vocab_rank":
            rank,

        "expected_logit":
            float(
                logits[
                    expected_secret[
                        "id"
                    ]
                ]
                .item()
            ),
    }


def evaluate_all(
    expected_secret,
    full_context=None,
):

    rows = [
        eval_prompt(
            question,
            expected_secret,
            full_context=
                full_context,
        )

        for question in TEST_QUESTIONS
    ]


    return {
        "candidate_accuracy":
            statistics.mean(
                row[
                    "candidate_correct"
                ]
                for row in rows
            ),

        "greedy_accuracy":
            statistics.mean(
                row[
                    "greedy_correct"
                ]
                for row in rows
            ),

        "mean_p_correct":
            statistics.mean(
                row[
                    "p_correct"
                ]
                for row in rows
            ),

        "mean_margin":
            statistics.mean(
                row[
                    "margin"
                ]
                for row in rows
            ),

        "mean_full_vocab_rank":
            statistics.mean(
                row[
                    "full_vocab_rank"
                ]
                for row in rows
            ),

        "rows":
            rows,
    }


# =============================================================================
# W0 BASELINES
# =============================================================================

print()
print("=" * 120)
print("W0 BASELINES")
print("=" * 120)


reset_states()


W0_NONE = evaluate_all(
    SECRET_A,
    full_context=None,
)


W0_CONTEXT = evaluate_all(
    SECRET_A,
    full_context=
        FULL_CONTEXT_A,
)


print(
    "W0 no-context:"
)

print(
    "  candidate accuracy :",
    f"{W0_NONE['candidate_accuracy'] * 100:.2f}%",
)

print(
    "  greedy accuracy    :",
    f"{W0_NONE['greedy_accuracy'] * 100:.2f}%",
)

print(
    "  P(correct)         :",
    f"{W0_NONE['mean_p_correct']:.6f}",
)

print(
    "  margin             :",
    f"{W0_NONE['mean_margin']:.6f}",
)

print(
    "  vocab rank         :",
    f"{W0_NONE['mean_full_vocab_rank']:.1f}",
)


print()


print(
    "W0 full-context:"
)

print(
    "  candidate accuracy :",
    f"{W0_CONTEXT['candidate_accuracy'] * 100:.2f}%",
)

print(
    "  greedy accuracy    :",
    f"{W0_CONTEXT['greedy_accuracy'] * 100:.2f}%",
)

print(
    "  P(correct)         :",
    f"{W0_CONTEXT['mean_p_correct']:.6f}",
)

print(
    "  margin             :",
    f"{W0_CONTEXT['mean_margin']:.6f}",
)

print(
    "  vocab rank         :",
    f"{W0_CONTEXT['mean_full_vocab_rank']:.1f}",
)


# =============================================================================
# JIT WARMUP
# =============================================================================

print()
print("=" * 120)
print("REAL-GRADIENT JIT WARMUP")
print("=" * 120)


reset_states()


(
    warm_gradients,
    warm_loss,
    warm_capture_ms,
) = capture_write_gradient(

    INSERT_WRITE_QUESTIONS,

    SECRET_A,
)


warm_stats = apply_update(
    warm_gradients
)


print(
    "Loss                :",
    warm_loss,
)

print(
    "Gradient capture    :",
    f"{warm_capture_ms:.3f} ms",
)

print(
    "Kernel              :",
    f"{warm_stats['kernel_ms']:.3f} ms",
)

print(
    "Publish             :",
    f"{warm_stats['publish_ms']:.3f} ms",
)

print(
    "Ready               :",
    f"{warm_stats['total_ms']:.3f} ms",
)


del warm_gradients


# Remove warmup update.
reset_states()


print(
    "WARMUP COMPLETE."
)


# =============================================================================
# ONLINE INSERT
# =============================================================================

print()
print("=" * 145)
print("ONLINE INSERT — UNKNOWN ASSOCIATION")
print("=" * 145)


print(
    f"{'step':>5}"
    f"{'loss':>11}"
    f"{'fullCE':>11}"
    f"{'candCE':>11}"
    f"{'grad ms':>11}"
    f"{'kernel':>10}"
    f"{'publish':>10}"
    f"{'ready':>10}"
    f"{'cand acc':>11}"
    f"{'greedy':>10}"
    f"{'P*':>10}"
    f"{'rank':>10}"
)


print(
    "-" * 145
)


INSERT_HISTORY = []


for step in range(
    1,
    INSERT_STEPS + 1,
):

    (
        gradients,
        loss_info,
        capture_ms,
    ) = capture_write_gradient(

        INSERT_WRITE_QUESTIONS,

        SECRET_A,
    )


    update_stats = apply_update(
        gradients
    )


    retrieval = evaluate_all(
        SECRET_A,
        full_context=None,
    )


    INSERT_HISTORY.append(
        {
            "step":
                step,

            "loss":
                loss_info,

            "gradient_capture_ms":
                capture_ms,

            "update":
                update_stats,

            "retrieval":
                retrieval,
        }
    )


    print(
        f"{step:>5}"
        f"{loss_info['total_loss']:>11.4f}"
        f"{loss_info['full_ce']:>11.4f}"
        f"{loss_info['candidate_ce']:>11.4f}"
        f"{capture_ms:>11.2f}"
        f"{update_stats['kernel_ms']:>10.3f}"
        f"{update_stats['publish_ms']:>10.3f}"
        f"{update_stats['total_ms']:>10.3f}"
        f"{retrieval['candidate_accuracy'] * 100:>10.1f}%"
        f"{retrieval['greedy_accuracy'] * 100:>9.1f}%"
        f"{retrieval['mean_p_correct']:>10.4f}"
        f"{retrieval['mean_full_vocab_rank']:>10.1f}"
    )


    del gradients


# =============================================================================
# FINAL INSERT RESULT
# =============================================================================

AFTER_INSERT = evaluate_all(
    SECRET_A,
    full_context=None,
)


print()
print("=" * 120)
print("UNKNOWN ASSOCIATION — FINAL INSERT RESULT")
print("=" * 120)


print(
    "Candidate accuracy  :",
    f"{AFTER_INSERT['candidate_accuracy'] * 100:.2f}%",
)

print(
    "Greedy accuracy     :",
    f"{AFTER_INSERT['greedy_accuracy'] * 100:.2f}%",
)

print(
    "P(correct)          :",
    f"{AFTER_INSERT['mean_p_correct']:.6f}",
)

print(
    "Mean margin         :",
    f"{AFTER_INSERT['mean_margin']:.6f}",
)

print(
    "Mean vocab rank     :",
    f"{AFTER_INSERT['mean_full_vocab_rank']:.2f}",
)


for index, row in enumerate(
    AFTER_INSERT[
        "rows"
    ]
):

    print(
        f"Q{index + 1} | "
        f"candidate={row['prediction']:<12} | "
        f"greedy={row['greedy_text']!r:<14} | "
        f"P*={row['p_correct']:.4f} | "
        f"margin={row['margin']:+.3f} | "
        f"rank={row['full_vocab_rank']}"
    )


# =============================================================================
# CONTEXT-ADVANTAGE RECOVERY
# =============================================================================

denominator = (
    W0_CONTEXT[
        "mean_p_correct"
    ]
    -
    W0_NONE[
        "mean_p_correct"
    ]
)


if abs(
    denominator
) > 1e-12:

    transfer_fraction = (
        (
            AFTER_INSERT[
                "mean_p_correct"
            ]
            -
            W0_NONE[
                "mean_p_correct"
            ]
        )
        /
        denominator
    )

else:

    transfer_fraction = None


print()


if transfer_fraction is not None:

    print(
        "Recovered context advantage:",
        f"{transfer_fraction * 100:.2f}%",
    )


# =============================================================================
# ONLINE OVERWRITE
#
# NO RESET.
#
# Current memory:
#
#       QEVAX-731 -> SECRET_A
#
# Write:
#
#       QEVAX-731 -> SECRET_B
#
# with explicit old-secret suppression.
# =============================================================================

print()
print("=" * 145)
print("ONLINE OVERWRITE — A -> B")
print("=" * 145)


A_BEFORE_OVERWRITE = evaluate_all(
    SECRET_A,
    full_context=None,
)


B_BEFORE_OVERWRITE = evaluate_all(
    SECRET_B,
    full_context=None,
)


print(
    "Before overwrite:"
)

print(
    "  A P(correct)       :",
    f"{A_BEFORE_OVERWRITE['mean_p_correct']:.6f}",
)

print(
    "  B P(correct)       :",
    f"{B_BEFORE_OVERWRITE['mean_p_correct']:.6f}",
)

print(
    "  A greedy accuracy  :",
    f"{A_BEFORE_OVERWRITE['greedy_accuracy'] * 100:.2f}%",
)

print(
    "  B greedy accuracy  :",
    f"{B_BEFORE_OVERWRITE['greedy_accuracy'] * 100:.2f}%",
)


print()


print(
    f"{'step':>5}"
    f"{'loss':>11}"
    f"{'fullCE':>11}"
    f"{'candCE':>11}"
    f"{'marginL':>11}"
    f"{'grad ms':>11}"
    f"{'ready':>10}"
    f"{'B cand':>10}"
    f"{'B greedy':>11}"
    f"{'P(B)':>10}"
    f"{'P(A)':>10}"
)


print(
    "-" * 145
)


OVERWRITE_HISTORY = []


for step in range(
    1,
    OVERWRITE_STEPS + 1,
):

    (
        gradients,
        loss_info,
        capture_ms,
    ) = capture_write_gradient(

        OVERWRITE_WRITE_QUESTIONS,

        SECRET_B,

        old_secret=
            SECRET_A,
    )


    update_stats = apply_update(
        gradients
    )


    eval_b = evaluate_all(
        SECRET_B,
        full_context=None,
    )


    eval_a = evaluate_all(
        SECRET_A,
        full_context=None,
    )


    OVERWRITE_HISTORY.append(
        {
            "step":
                step,

            "loss":
                loss_info,

            "capture_ms":
                capture_ms,

            "update":
                update_stats,

            "A":
                eval_a,

            "B":
                eval_b,
        }
    )


    print(
        f"{step:>5}"
        f"{loss_info['total_loss']:>11.4f}"
        f"{loss_info['full_ce']:>11.4f}"
        f"{loss_info['candidate_ce']:>11.4f}"
        f"{loss_info['margin_loss']:>11.4f}"
        f"{capture_ms:>11.2f}"
        f"{update_stats['total_ms']:>10.3f}"
        f"{eval_b['candidate_accuracy'] * 100:>9.1f}%"
        f"{eval_b['greedy_accuracy'] * 100:>10.1f}%"
        f"{eval_b['mean_p_correct']:>10.4f}"
        f"{eval_a['mean_p_correct']:>10.4f}"
    )


    del gradients


# =============================================================================
# FINAL OVERWRITE
# =============================================================================

AFTER_OVERWRITE_B = evaluate_all(
    SECRET_B,
    full_context=None,
)


AFTER_OVERWRITE_A = evaluate_all(
    SECRET_A,
    full_context=None,
)


print()
print("=" * 120)
print("FINAL OVERWRITE RESULT")
print("=" * 120)


print(
    "OLD SECRET A:"
)

print(
    "  candidate accuracy :",
    f"{AFTER_OVERWRITE_A['candidate_accuracy'] * 100:.2f}%",
)

print(
    "  greedy accuracy    :",
    f"{AFTER_OVERWRITE_A['greedy_accuracy'] * 100:.2f}%",
)

print(
    "  P(A)               :",
    f"{AFTER_OVERWRITE_A['mean_p_correct']:.6f}",
)


print()


print(
    "NEW SECRET B:"
)

print(
    "  candidate accuracy :",
    f"{AFTER_OVERWRITE_B['candidate_accuracy'] * 100:.2f}%",
)

print(
    "  greedy accuracy    :",
    f"{AFTER_OVERWRITE_B['greedy_accuracy'] * 100:.2f}%",
)

print(
    "  P(B)               :",
    f"{AFTER_OVERWRITE_B['mean_p_correct']:.6f}",
)

print(
    "  margin             :",
    f"{AFTER_OVERWRITE_B['mean_margin']:.6f}",
)

print(
    "  vocab rank         :",
    f"{AFTER_OVERWRITE_B['mean_full_vocab_rank']:.2f}",
)


# =============================================================================
# STATE MEMORY
# =============================================================================

bytes_per_fast_layer = (
    M
    *
    N
    *
    12
)


prototype_state_bytes = (
    bytes_per_fast_layer
    *
    len(
        FAST_LAYERS
    )
)


kv_bytes_per_token = (
    NUM_LAYERS
    *
    2
    *
    NUM_KV_HEADS
    *
    HEAD_DIM
    *
    2
)


kv_break_even = (
    prototype_state_bytes
    /
    kv_bytes_per_token
)


print()
print("=" * 120)
print("STATE SIZE")
print("=" * 120)


print(
    "Prototype fast state:",
    f"{prototype_state_bytes / 1024**2:.2f} MiB",
)

print(
    "KV bytes / token    :",
    f"{kv_bytes_per_token / 1024:.2f} KiB",
)

print(
    "KV break-even       :",
    f"{kv_break_even:,.0f} tokens",
)


# =============================================================================
# TIMING SUMMARY
# =============================================================================

insert_ready = [
    row[
        "update"
    ][
        "total_ms"
    ]
    for row in INSERT_HISTORY
]


insert_gradient = [
    row[
        "gradient_capture_ms"
    ]
    for row in INSERT_HISTORY
]


overwrite_ready = [
    row[
        "update"
    ][
        "total_ms"
    ]
    for row in OVERWRITE_HISTORY
]


overwrite_gradient = [
    row[
        "capture_ms"
    ]
    for row in OVERWRITE_HISTORY
]


print()
print("=" * 120)
print("TIMING")
print("=" * 120)


print(
    "Insert gradient p50 :",
    f"{statistics.median(insert_gradient):.3f} ms",
)

print(
    "Insert ready p50    :",
    f"{statistics.median(insert_ready):.3f} ms",
)

print(
    "Overwrite grad p50  :",
    f"{statistics.median(overwrite_gradient):.3f} ms",
)

print(
    "Overwrite ready p50 :",
    f"{statistics.median(overwrite_ready):.3f} ms",
)

print(
    "Full dW HBM tensor  :",
    "NO",
)


# =============================================================================
# FINAL DIAGNOSTIC
# =============================================================================

print()
print("=" * 120)
print("FINAL DIAGNOSTIC")
print("=" * 120)


insert_success = (
    AFTER_INSERT[
        "candidate_accuracy"
    ]
    >=
    0.75
)


generation_success = (
    AFTER_INSERT[
        "greedy_accuracy"
    ]
    >=
    0.50
)


overwrite_success = (
    AFTER_OVERWRITE_B[
        "candidate_accuracy"
    ]
    >=
    0.75

    and

    AFTER_OVERWRITE_B[
        "mean_p_correct"
    ]
    >
    AFTER_OVERWRITE_A[
        "mean_p_correct"
    ]
)


open_vocab_overwrite = (
    AFTER_OVERWRITE_B[
        "greedy_accuracy"
    ]
    >=
    0.50
)


print(
    "Association retrieval:",
    "PASS"
    if insert_success
    else
    "FAIL",
)

print(
    "Open-vocab generation:",
    "PASS"
    if generation_success
    else
    "FAIL",
)

print(
    "Online overwrite     :",
    "PASS"
    if overwrite_success
    else
    "FAIL",
)

print(
    "Overwrite generation :",
    "PASS"
    if open_vocab_overwrite
    else
    "FAIL",
)


if (
    insert_success
    and
    generation_success
    and
    overwrite_success
):

    print()
    print(
        "RESULT: the fast state demonstrates online insertion, "
        "no-context retrieval and correction of previously learned "
        "unknown information."
    )


elif insert_success:

    print()
    print(
        "RESULT: persistent fast-weight memory is confirmed, but "
        "generation and/or online correction still requires improvement."
    )


else:

    print()
    print(
        "RESULT: this ordinary W_down retrofit is not sufficiently "
        "reliable. Move to a dedicated compact TTT memory state."
    )


# =============================================================================
# SAVE
# =============================================================================

RESULT_PATH = Path(
    "async_ttt_qwen7b_single_token_overwrite.json"
)


with open(
    RESULT_PATH,
    "w",
) as file:

    json.dump(
        {
            "environment": {
                "model":
                    MODEL_ID,

                "pytorch":
                    torch.__version__,

                "cuda":
                    torch.version.cuda,

                "triton":
                    triton.__version__,

                "gpu":
                    torch.cuda.get_device_name(0),
            },

            "association": {
                "entity":
                    ENTITY,

                "secret_a":
                    SECRET_A,

                "secret_b":
                    SECRET_B,

                "candidates":
                    CANDIDATES,
            },

            "config": {
                "fast_layers":
                    FAST_LAYERS,

                "total_delta_norm":
                    TOTAL_DELTA_NORM,

                "insert_steps":
                    INSERT_STEPS,

                "overwrite_steps":
                    OVERWRITE_STEPS,

                "write_batch":
                    WRITE_BATCH,

                "prompt_k":
                    PROMPT_K,

                "flat_k":
                    FLAT_K,

                "overwrite_margin":
                    OVERWRITE_MARGIN,
            },

            "parity": {
                "relative_l2":
                    parity_rel,

                "max_abs":
                    parity_max,
            },

            "triton": {
                "norm_relative_error":
                    relative_error,

                "norm_workspace_bytes":
                    NUM_NORM_TILES
                    *
                    4,

                "full_dw_materialized":
                    False,
            },

            "baseline": {
                "no_context":
                    W0_NONE,

                "full_context":
                    W0_CONTEXT,
            },

            "insert_history":
                INSERT_HISTORY,

            "after_insert":
                AFTER_INSERT,

            "transfer_fraction":
                transfer_fraction,

            "overwrite": {
                "before_a":
                    A_BEFORE_OVERWRITE,

                "before_b":
                    B_BEFORE_OVERWRITE,

                "history":
                    OVERWRITE_HISTORY,

                "after_a":
                    AFTER_OVERWRITE_A,

                "after_b":
                    AFTER_OVERWRITE_B,
            },

            "memory": {
                "prototype_state_bytes":
                    prototype_state_bytes,

                "kv_bytes_per_token":
                    kv_bytes_per_token,

                "kv_break_even_tokens":
                    kv_break_even,
            },

            "timing": {
                "insert_gradient_p50_ms":
                    statistics.median(
                        insert_gradient
                    ),

                "insert_ready_p50_ms":
                    statistics.median(
                        insert_ready
                    ),

                "overwrite_gradient_p50_ms":
                    statistics.median(
                        overwrite_gradient
                    ),

                "overwrite_ready_p50_ms":
                    statistics.median(
                        overwrite_ready
                    ),
            },
        },

        file,

        indent=2,
    )


print()
print("=" * 120)
print("DONE")
print("=" * 120)


print(
    "Saved:",
    RESULT_PATH.resolve(),
)


# =============================================================================
# RESTORE QWEN
# =============================================================================

for layer in FAST_LAYERS:

    FAST[
        layer
    ].module.forward = (
        FAST[
            layer
        ].original_forward
    )


torch.cuda.synchronize()


free_bytes, _ = (
    torch.cuda.mem_get_info()
)


print(
    "Original Qwen forwards restored."
)

print(
    "Free VRAM:",
    f"{free_bytes / 1024**3:.2f} GiB",
)

FLUSHING PREVIOUS GPU STATE


Free VRAM : 39.08 GiB
Total VRAM: 39.49 GiB

QWEN2.5-7B — SINGLE-TOKEN ONLINE MEMORY
Model               : Qwen/Qwen2.5-7B-Instruct
Python              : 3.12.11
PyTorch             : 2.8.0+cu128
CUDA                : 12.8
Triton              : 3.4.0
GPU                 : NVIDIA A100-SXM4-40GB
Fast layers         : [12, 13, 14, 15]
Prompt K            : 96
Write batch         : 8
Flattened Triton K  : 768

LOADING QWEN


Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

M                   : 3584
N                   : 18944
Layers              : 28
KV heads            : 4
Head dim            : 128
Free after load     : 24.89 GiB

SEARCHING FOR SINGLE-TOKEN SECRETS
Safe token pool      : 1024
Prior logit span     : 0.000000
id=4933    | text='shows'        | neutral=+3.0469
id=3550    | text='space'        | neutral=+3.0469
id=4822    | text='callback'     | neutral=+3.0469
id=1161    | text='char'         | neutral=+3.0469
id=3082    | text='area'         | neutral=+3.0469
id=3592    | text='React'        | neutral=+3.0469
id=3880    | text='needs'        | neutral=+3.0469
id=1449    | text='every'        | neutral=+3.0469

SECRET_A             : shows | token 4933
SECRET_B             : space | token 3550

UNKNOWN ASSOCIATION
QEVAX-731 -> shows

ALLOCATING 4-LAYER FAST STATE
Free after fast state: 21.74 GiB

STATIC BF16 PARITY
Relative L2         : 0.00000000e+00
Max abs             : 0.00000000e+00

TRITON WORKSPACE
Flattened K          : 768
Norm w

In [1]:
# =============================================================================
# ASYNC-TTT POC FINAL STRESS SUITE
# =============================================================================
#
# Model:
#   Qwen/Qwen2.5-7B-Instruct
#
# Proven POC configuration:
#   Fast layers        = 12, 13, 14, 15
#   Update norm        = 0.10 total across layers
#   Write steps/fact   = 8
#   FP32 master A/B
#   BF16 serving A/B
#   Triton G^T Z rematerialization
#   No full dW HBM tensor
#
#
# THIS SCRIPT CLOSES THE POC BY TESTING:
#
#  1. Kernel correctness / parity
#  2. Unknown-fact online insertion
#  3. 1 -> 4 -> 16 -> 64 -> 128 fact capacity
#  4. No replay of old facts
#  5. Held-out paraphrase retrieval
#  6. Candidate-set retrieval
#  7. Full-vocabulary greedy retrieval
#  8. Full-vocabulary rank
#  9. Old-vs-new fact interference
# 10. Retention by memory age
# 11. Catastrophic forgetting
# 12. Immediate-learning vs final-retention gap
# 13. Stock W0 no-context baseline
# 14. Stock W0 full-context reference
# 15. Stock truncated-context baseline
# 16. Random correction of 25% of learned facts
# 17. New-value retrieval after correction
# 18. Old-value suppression
# 19. Untouched-fact retention after corrections
# 20. Gradient-generation latency
# 21. Triton state-transition latency
# 22. BF16 publication latency
# 23. p50 / p95 / p99 timings
# 24. Fixed fast-state memory vs growing KV memory
# 25. Context-token growth at each capacity
#
#
# IMPORTANT:
#
# This is the final POC stress benchmark.
#
# It is NOT the final compact architecture.
#
# Current 4x W_down fast state is expected to be ~3.1 GiB/session.
# That is a proof-of-mechanism implementation.
#
# The paper can use this to establish:
#
#   online learned-state semantics + systems feasibility
#
# before replacing W_down with a compact dedicated TTT memory module.
#
#
# RESTART THE NOTEBOOK KERNEL BEFORE RUNNING.
# =============================================================================


# =============================================================================
# IMPORTS
# =============================================================================

import gc
import json
import math
import random
import re
import statistics
import sys
import time

from collections import defaultdict
from pathlib import Path

import torch
import torch.nn.functional as F

import triton
import triton.language as tl

from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
)


# =============================================================================
# CONFIG
# =============================================================================

MODEL_ID = "Qwen/Qwen2.5-7B-Instruct"

DEVICE = "cuda"

SEED = 928731


# -----------------------------------------------------------------------------
# Proven POC configuration
# -----------------------------------------------------------------------------

FAST_LAYERS = [
    12,
    13,
    14,
    15,
]


TOTAL_DELTA_NORM = 0.10


INSERT_STEPS = 8


OVERWRITE_STEPS = 8


WRITE_BATCH = 8


PROMPT_K = 96


FLAT_K = (
    WRITE_BATCH
    *
    PROMPT_K
)


# -----------------------------------------------------------------------------
# Main stress capacities.
# -----------------------------------------------------------------------------

CAPACITY_CHECKPOINTS = [
    1,
    4,
    16,
    64,
    128,
]


MAX_FACTS = max(
    CAPACITY_CHECKPOINTS
)


# -----------------------------------------------------------------------------
# At the end, overwrite 25% of all facts.
# -----------------------------------------------------------------------------

CORRECTION_FRACTION = 0.25


NUM_CORRECTIONS = int(
    MAX_FACTS
    *
    CORRECTION_FRACTION
)


# -----------------------------------------------------------------------------
# Evaluation
# -----------------------------------------------------------------------------

CHECKPOINT_PARAPHRASES = 2


FINAL_PARAPHRASES = 4


EVAL_BATCH_SIZE = 16


FULL_CONTEXT_SAMPLE_SIZE = 8


FULL_CONTEXT_BATCH_SIZE = 1


TRUNCATED_CONTEXT_WINDOW = 16


# -----------------------------------------------------------------------------
# Candidate token pool.
#
# Need:
#
#   MAX_FACTS initial secrets
# + MAX_FACTS replacement secrets
# + extra distractors
# -----------------------------------------------------------------------------

EXTRA_DISTRACTORS = 32


NUM_GLOBAL_CANDIDATES = (
    MAX_FACTS
    *
    2
    +
    EXTRA_DISTRACTORS
)


SAFE_TOKEN_POOL_TARGET = 2048


# -----------------------------------------------------------------------------
# Write objective
# -----------------------------------------------------------------------------

CANDIDATE_LOSS_WEIGHT = 0.50


OVERWRITE_MARGIN = 4.0


OVERWRITE_MARGIN_WEIGHT = 1.0


# -----------------------------------------------------------------------------
# Files
# -----------------------------------------------------------------------------

RESULT_PATH = Path(
    "async_ttt_final_poc_stress_suite.json"
)


# =============================================================================
# REPRODUCIBILITY
# =============================================================================

torch.manual_seed(
    SEED
)


random.seed(
    SEED
)


rng = random.Random(
    SEED
)


torch.backends.cuda.matmul.allow_tf32 = False

torch.backends.cudnn.allow_tf32 = False


# =============================================================================
# VRAM CLEANUP
# =============================================================================

print("=" * 130)
print("FLUSHING PREVIOUS GPU STATE")
print("=" * 130)


for name in [
    "model",
    "tokenizer",
    "FAST",
    "CAPTURE_DATA",
    "output",
]:

    if name in globals():

        try:
            del globals()[name]
        except Exception:
            pass


gc.collect()


if torch.cuda.is_available():

    torch.cuda.synchronize()

    torch.cuda.empty_cache()

    try:
        torch.cuda.ipc_collect()
    except Exception:
        pass

    gc.collect()

    torch.cuda.empty_cache()


assert torch.cuda.is_available()


free_bytes, total_bytes = (
    torch.cuda.mem_get_info()
)


print(
    "Free VRAM :",
    f"{free_bytes / 1024**3:.2f} GiB",
)


print(
    "Total VRAM:",
    f"{total_bytes / 1024**3:.2f} GiB",
)


if (
    free_bytes
    /
    1024**3
    <
    30
):

    raise RuntimeError(
        "Less than 30 GiB free. "
        "Restart the kernel before running."
    )


# =============================================================================
# ENVIRONMENT
# =============================================================================

props = (
    torch.cuda.get_device_properties(
        0
    )
)


print()
print("=" * 130)
print("ASYNC-TTT FINAL POC STRESS SUITE")
print("=" * 130)


print(
    "Model                    :",
    MODEL_ID,
)


print(
    "Python                   :",
    sys.version.split()[0],
)


print(
    "PyTorch                  :",
    torch.__version__,
)


print(
    "CUDA                     :",
    torch.version.cuda,
)


print(
    "Triton                   :",
    triton.__version__,
)


print(
    "GPU                      :",
    torch.cuda.get_device_name(0),
)


print(
    "VRAM                     :",
    f"{props.total_memory / 1024**3:.2f} GiB",
)


print(
    "Fast layers              :",
    FAST_LAYERS,
)


print(
    "Capacity checkpoints     :",
    CAPACITY_CHECKPOINTS,
)


print(
    "Final corrections        :",
    NUM_CORRECTIONS,
)


# =============================================================================
# LOAD MODEL
# =============================================================================

print()
print("=" * 130)
print("LOADING QWEN")
print("=" * 130)


tokenizer = (
    AutoTokenizer.from_pretrained(
        MODEL_ID,
        trust_remote_code=True,
    )
)


if tokenizer.pad_token_id is None:

    tokenizer.pad_token = (
        tokenizer.eos_token
    )


model = (
    AutoModelForCausalLM.from_pretrained(

        MODEL_ID,

        dtype=
            torch.bfloat16,

        device_map={
            "": 0
        },

        low_cpu_mem_usage=True,

        trust_remote_code=True,

        attn_implementation=
            "sdpa",
    )
)


model.eval()


for parameter in model.parameters():

    parameter.requires_grad_(
        False
    )


M = int(
    model.config.hidden_size
)


N = int(
    model.config.intermediate_size
)


NUM_LAYERS = int(
    model.config.num_hidden_layers
)


NUM_ATTENTION_HEADS = int(
    model.config.num_attention_heads
)


NUM_KV_HEADS = int(
    getattr(
        model.config,
        "num_key_value_heads",
        NUM_ATTENTION_HEADS,
    )
)


HEAD_DIM = (
    M
    //
    NUM_ATTENTION_HEADS
)


print(
    "Hidden size              :",
    M,
)


print(
    "Intermediate size        :",
    N,
)


print(
    "Layers                   :",
    NUM_LAYERS,
)


print(
    "KV heads                 :",
    NUM_KV_HEADS,
)


print(
    "Head dim                 :",
    HEAD_DIM,
)


print(
    "Write batch              :",
    WRITE_BATCH,
)


print(
    "Prompt K                 :",
    PROMPT_K,
)


print(
    "Flattened Triton K       :",
    FLAT_K,
)


free_bytes, _ = (
    torch.cuda.mem_get_info()
)


print(
    "Free after model load    :",
    f"{free_bytes / 1024**3:.2f} GiB",
)


# =============================================================================
# BUILD MATCHED SINGLE-TOKEN SECRET POOL
# =============================================================================

print()
print("=" * 130)
print("BUILDING GLOBAL SINGLE-TOKEN SECRET POOL")
print("=" * 130)


safe_tokens = []


special_ids = set(
    tokenizer.all_special_ids
)


for token_id in range(
    tokenizer.vocab_size
):

    if token_id in special_ids:
        continue


    decoded = tokenizer.decode(
        [
            token_id
        ],
        clean_up_tokenization_spaces=False,
    )


    # Human-readable single token only.
    if not re.fullmatch(
        r" [A-Za-z]{4,12}",
        decoded,
    ):

        continue


    text = decoded.strip()


    roundtrip = tokenizer(
        " " + text,
        add_special_tokens=False,
    )[
        "input_ids"
    ]


    if roundtrip != [
        token_id
    ]:

        continue


    safe_tokens.append(
        {
            "id":
                token_id,

            "text":
                text,
        }
    )


    if (
        len(
            safe_tokens
        )
        >=
        SAFE_TOKEN_POOL_TARGET
    ):

        break


if (
    len(
        safe_tokens
    )
    <
    NUM_GLOBAL_CANDIDATES
):

    raise RuntimeError(
        "Not enough safe single-token candidates."
    )


print(
    "Safe token pool          :",
    len(
        safe_tokens
    ),
)


# =============================================================================
# MATCH STOCK-MODEL PRIORS
# =============================================================================

neutral_prompt = (
    "Return the exact private registry value.\n"
    "Answer:"
)


neutral_ids = tokenizer(
    neutral_prompt,
    return_tensors=
        "pt",
)[
    "input_ids"
].to(
    DEVICE
)


with torch.no_grad():

    neutral_logits = (
        model(
            input_ids=
                neutral_ids,

            use_cache=
                False,
        )
        .logits[
            0,
            -1,
            :
        ]
        .float()
    )


scored_safe_tokens = [
    (
        item[
            "id"
        ],

        item[
            "text"
        ],

        float(
            neutral_logits[
                item[
                    "id"
                ]
            ]
            .item()
        ),
    )

    for item in safe_tokens
]


scored_safe_tokens.sort(
    key=lambda x:
        x[
            2
        ]
)


best_window = None

best_span = float(
    "inf"
)


for start in range(
    len(
        scored_safe_tokens
    )
    -
    NUM_GLOBAL_CANDIDATES
    +
    1
):

    window = (
        scored_safe_tokens[
            start:
            start
            +
            NUM_GLOBAL_CANDIDATES
        ]
    )


    span = (
        window[
            -1
        ][
            2
        ]
        -
        window[
            0
        ][
            2
        ]
    )


    if span < best_span:

        best_span = span

        best_window = window


GLOBAL_CANDIDATES = [
    {
        "id":
            token_id,

        "text":
            text,

        "neutral_logit":
            logit,
    }

    for token_id, text, logit
    in best_window
]


rng.shuffle(
    GLOBAL_CANDIDATES
)


INITIAL_SECRETS = (
    GLOBAL_CANDIDATES[
        :
        MAX_FACTS
    ]
)


CORRECTION_SECRETS = (
    GLOBAL_CANDIDATES[
        MAX_FACTS:
        MAX_FACTS
        *
        2
    ]
)


DISTRACTOR_SECRETS = (
    GLOBAL_CANDIDATES[
        MAX_FACTS
        *
        2:
    ]
)


GLOBAL_CANDIDATE_IDS = torch.tensor(
    [
        item[
            "id"
        ]
        for item in GLOBAL_CANDIDATES
    ],

    dtype=
        torch.long,

    device=
        DEVICE,
)


TOKEN_TO_CANDIDATE_INDEX = {
    item[
        "id"
    ]:
        index

    for index, item in enumerate(
        GLOBAL_CANDIDATES
    )
}


print(
    "Global candidates        :",
    len(
        GLOBAL_CANDIDATES
    ),
)


print(
    "Initial secrets          :",
    len(
        INITIAL_SECRETS
    ),
)


print(
    "Correction secrets       :",
    len(
        CORRECTION_SECRETS
    ),
)


print(
    "Prior-logit span         :",
    f"{best_span:.6f}",
)


print(
    "Chance candidate accuracy:",
    f"{100 / len(GLOBAL_CANDIDATES):.4f}%",
)


del neutral_logits
del neutral_ids


# =============================================================================
# BUILD UNKNOWN WORLD
# =============================================================================

WORLD = []


for index in range(
    MAX_FACTS
):

    entity = (
        f"QEVAX-{index + 1:04d}"
    )


    WORLD.append(
        {
            "index":
                index,

            "entity":
                entity,

            "secret":
                INITIAL_SECRETS[
                    index
                ],

            "replacement":
                CORRECTION_SECRETS[
                    index
                ],
        }
    )


print()
print("=" * 130)
print("WORLD")
print("=" * 130)


print(
    "Unknown facts            :",
    len(
        WORLD
    ),
)


print(
    "Example                  :",
    WORLD[
        0
    ][
        "entity"
    ],
    "->",
    WORLD[
        0
    ][
        "secret"
    ][
        "text"
    ],
)


# =============================================================================
# PROMPT TEMPLATES
# =============================================================================

WRITE_TEMPLATES = [

    "What private registry value is assigned to identity {entity}?",

    "State the exact stored private value belonging to {entity}.",

    "Which secret value maps to identifier {entity}?",

    "Give the confidential lookup result for {entity}.",

    "What exact value should the registry return for {entity}?",

    "Recall the private value associated with {entity}.",

    "Resolve {entity} to its stored confidential value.",

    "Which private token belongs to registry key {entity}?",
]


CORRECTION_TEMPLATES = [

    "The old value for {entity} is obsolete. What is its new value?",

    "After the update, what value now belongs to {entity}?",

    "Which replacement secret currently maps to {entity}?",

    "What new confidential value supersedes the old record for {entity}?",

    "Return the latest private value registered for {entity}.",

    "What is the current replacement value of {entity}?",

    "Resolve {entity} using the newly updated registry value.",

    "Which current private token replaces the old value for {entity}?",
]


TEST_TEMPLATES = [

    "An operator queries {entity}. What exact private value should be returned?",

    "What value is stored under lookup identity {entity}?",

    "Resolve identifier {entity} to its secret registry value.",

    "If a request references {entity}, what confidential value belongs to it?",
]


# =============================================================================
# FAST-WEIGHT STATE
# =============================================================================

class LayerState:

    def __init__(
        self,
        layer_index,
    ):

        self.layer_index = (
            layer_index
        )


        self.module = (
            model
            .model
            .layers[
                layer_index
            ]
            .mlp
            .down_proj
        )


        self.original_forward = (
            self.module.forward
        )


        self.original_weight = (
            self.module
            .weight
            .detach()
        )


        # FP32 master double buffer.
        self.master_a = (
            self.original_weight
            .float()
            .contiguous()
        )


        self.master_b = (
            torch.empty_like(
                self.master_a
            )
        )


        # BF16 serving double buffer.
        self.serve_a = (
            self.original_weight
            .clone()
            .contiguous()
        )


        self.serve_b = (
            torch.empty_like(
                self.serve_a
            )
        )


        self.master_active = (
            self.master_a
        )


        self.master_staging = (
            self.master_b
        )


        self.serve_active = (
            self.serve_a
        )


        self.serve_staging = (
            self.serve_b
        )


        self.version = 0


    def reset(
        self,
    ):

        self.master_a.copy_(
            self.original_weight
        )

        self.master_b.copy_(
            self.original_weight
        )


        self.serve_a.copy_(
            self.original_weight
        )

        self.serve_b.copy_(
            self.original_weight
        )


        self.master_active = (
            self.master_a
        )

        self.master_staging = (
            self.master_b
        )


        self.serve_active = (
            self.serve_a
        )

        self.serve_staging = (
            self.serve_b
        )


        self.version = 0


    def commit(
        self,
    ):

        self.master_active, self.master_staging = (
            self.master_staging,
            self.master_active,
        )


        self.serve_active, self.serve_staging = (
            self.serve_staging,
            self.serve_active,
        )


        self.version += 1


print()
print("=" * 130)
print("ALLOCATING FAST STATE")
print("=" * 130)


FAST = {
    layer:
        LayerState(
            layer
        )

    for layer in FAST_LAYERS
}


torch.cuda.synchronize()


free_bytes, _ = (
    torch.cuda.mem_get_info()
)


print(
    "Free after fast state    :",
    f"{free_bytes / 1024**3:.2f} GiB",
)


# =============================================================================
# GLOBAL FAST-WEIGHT FLAGS
# =============================================================================

CAPTURE_MODE = False

CAPTURE_DATA = {}

SERVE_FAST = True


def reset_states():

    for state in FAST.values():

        state.reset()


    torch.cuda.synchronize()


def commit_states():

    for layer in FAST_LAYERS:

        FAST[
            layer
        ].commit()


# =============================================================================
# PATCH DOWN_PROJ FORWARDS
# =============================================================================

def make_fast_forward(
    layer_index,
):

    fast = (
        FAST[
            layer_index
        ]
    )


    def forward(
        x,
    ):

        global CAPTURE_DATA


        # Stock model baseline.
        if not SERVE_FAST:

            return fast.original_forward(
                x
            )


        y = F.linear(
            x,
            fast.serve_active,
            bias=None,
        )


        if not CAPTURE_MODE:

            return y


        z = (
            x
            .detach()
            .float()
            .reshape(
                -1,
                N,
            )
            .contiguous()
        )


        if tuple(
            z.shape
        ) != (
            FLAT_K,
            N,
        ):

            raise RuntimeError(
                f"Layer {layer_index}: "
                f"Z={tuple(z.shape)}, "
                f"expected={(FLAT_K, N)}"
            )


        CAPTURE_DATA[
            layer_index
        ] = {
            "z":
                z
        }


        if (
            layer_index
            ==
            FAST_LAYERS[
                0
            ]
        ):

            y = (
                y
                .detach()
                .requires_grad_(
                    True
                )
            )


        if not y.requires_grad:

            raise RuntimeError(
                f"Gradient graph broken at layer {layer_index}."
            )


        y.retain_grad()


        CAPTURE_DATA[
            layer_index
        ][
            "y"
        ] = y


        return y


    return forward


for layer in FAST_LAYERS:

    FAST[
        layer
    ].module.forward = (
        make_fast_forward(
            layer
        )
    )


# =============================================================================
# STATIC PARITY
# =============================================================================

print()
print("=" * 130)
print("STATIC BF16 PARITY")
print("=" * 130)


parity_ids = tokenizer(
    "The quick brown fox jumps over the lazy dog.",
    return_tensors=
        "pt",
)[
    "input_ids"
].to(
    DEVICE
)


SERVE_FAST = False


with torch.no_grad():

    stock_logits = (
        model(
            input_ids=
                parity_ids,

            use_cache=
                False,
        )
        .logits[
            :,
            -1,
            :
        ]
        .float()
        .clone()
    )


reset_states()


SERVE_FAST = True


with torch.no_grad():

    fast_logits = (
        model(
            input_ids=
                parity_ids,

            use_cache=
                False,
        )
        .logits[
            :,
            -1,
            :
        ]
        .float()
        .clone()
    )


diff = (
    fast_logits
    -
    stock_logits
)


PARITY_REL_L2 = float(
    (
        torch.linalg.vector_norm(
            diff
        )
        /
        torch.linalg.vector_norm(
            stock_logits
        )
    )
    .item()
)


PARITY_MAX_ABS = float(
    diff
    .abs()
    .max()
    .item()
)


print(
    "Relative L2             :",
    f"{PARITY_REL_L2:.8e}",
)


print(
    "Max abs                 :",
    f"{PARITY_MAX_ABS:.8e}",
)


del stock_logits
del fast_logits
del diff


# =============================================================================
# TRITON KERNELS
# =============================================================================

NBM = 32
NBN = 64
NBK = 64


UBM = 64
UBN = 64
UBK = 64


NUM_M_TILES = (
    triton.cdiv(
        M,
        NBM,
    )
)


NUM_N_TILES = (
    triton.cdiv(
        N,
        NBN,
    )
)


NUM_NORM_TILES = (
    NUM_M_TILES
    *
    NUM_N_TILES
)


norm_grid = (
    NUM_M_TILES,
    NUM_N_TILES,
)


update_grid = (

    triton.cdiv(
        M,
        UBM,
    ),

    triton.cdiv(
        N,
        UBN,
    ),
)


norm_tiles = torch.empty(

    NUM_NORM_TILES,

    dtype=
        torch.float32,

    device=
        DEVICE,
)


print()
print("=" * 130)
print("TRITON CONFIG")
print("=" * 130)


print(
    "Norm workspace           :",
    f"{NUM_NORM_TILES * 4 / 1024:.2f} KiB",
)


print(
    "Full dW materialization  :",
    "NO",
)


@triton.jit
def norm_kernel(
    gt_ptr,
    z_ptr,

    norm_ptr,

    stride_gm,
    stride_gk,

    stride_zk,
    stride_zn,

    M: tl.constexpr,
    N: tl.constexpr,
    K: tl.constexpr,

    NUM_N_TILES: tl.constexpr,

    BM: tl.constexpr,
    BN: tl.constexpr,
    BK: tl.constexpr,
):

    pid_m = tl.program_id(
        0
    )

    pid_n = tl.program_id(
        1
    )


    offs_m = (
        pid_m
        *
        BM
        +
        tl.arange(
            0,
            BM,
        )
    )


    offs_n = (
        pid_n
        *
        BN
        +
        tl.arange(
            0,
            BN,
        )
    )


    acc = tl.zeros(
        (
            BM,
            BN,
        ),

        dtype=
            tl.float32,
    )


    for k_start in range(
        0,
        K,
        BK,
    ):

        offs_k = (
            k_start
            +
            tl.arange(
                0,
                BK,
            )
        )


        g = tl.load(

            gt_ptr
            +
            offs_m[:, None]
            *
            stride_gm
            +
            offs_k[None, :]
            *
            stride_gk,

            mask=(
                (
                    offs_m[:, None]
                    <
                    M
                )
                &
                (
                    offs_k[None, :]
                    <
                    K
                )
            ),

            other=
                0.0,
        )


        z = tl.load(

            z_ptr
            +
            offs_k[:, None]
            *
            stride_zk
            +
            offs_n[None, :]
            *
            stride_zn,

            mask=(
                (
                    offs_k[:, None]
                    <
                    K
                )
                &
                (
                    offs_n[None, :]
                    <
                    N
                )
            ),

            other=
                0.0,
        )


        acc = tl.dot(
            g,
            z,
            acc,

            input_precision=
                "ieee",
        )


    valid = (
        (
            offs_m[:, None]
            <
            M
        )
        &
        (
            offs_n[None, :]
            <
            N
        )
    )


    acc = tl.where(
        valid,
        acc,
        0.0,
    )


    row_sum = tl.sum(
        acc
        *
        acc,
        axis=1,
    )


    tile_sum = tl.sum(
        row_sum,
        axis=0,
    )


    linear_pid = (
        pid_m
        *
        NUM_N_TILES
        +
        pid_n
    )


    tl.store(
        norm_ptr
        +
        linear_pid,

        tile_sum,
    )


@triton.jit
def update_kernel(
    gt_ptr,
    z_ptr,

    src_ptr,
    dst_ptr,

    stride_gm,
    stride_gk,

    stride_zk,
    stride_zn,

    stride_sm,
    stride_sn,

    stride_dm,
    stride_dn,

    step_scale,

    M: tl.constexpr,
    N: tl.constexpr,
    K: tl.constexpr,

    BM: tl.constexpr,
    BN: tl.constexpr,
    BK: tl.constexpr,
):

    pid_m = tl.program_id(
        0
    )

    pid_n = tl.program_id(
        1
    )


    offs_m = (
        pid_m
        *
        BM
        +
        tl.arange(
            0,
            BM,
        )
    )


    offs_n = (
        pid_n
        *
        BN
        +
        tl.arange(
            0,
            BN,
        )
    )


    acc = tl.zeros(
        (
            BM,
            BN,
        ),

        dtype=
            tl.float32,
    )


    for k_start in range(
        0,
        K,
        BK,
    ):

        offs_k = (
            k_start
            +
            tl.arange(
                0,
                BK,
            )
        )


        g = tl.load(

            gt_ptr
            +
            offs_m[:, None]
            *
            stride_gm
            +
            offs_k[None, :]
            *
            stride_gk,

            mask=(
                (
                    offs_m[:, None]
                    <
                    M
                )
                &
                (
                    offs_k[None, :]
                    <
                    K
                )
            ),

            other=
                0.0,
        )


        z = tl.load(

            z_ptr
            +
            offs_k[:, None]
            *
            stride_zk
            +
            offs_n[None, :]
            *
            stride_zn,

            mask=(
                (
                    offs_k[:, None]
                    <
                    K
                )
                &
                (
                    offs_n[None, :]
                    <
                    N
                )
            ),

            other=
                0.0,
        )


        acc = tl.dot(
            g,
            z,
            acc,

            input_precision=
                "ieee",
        )


    valid = (
        (
            offs_m[:, None]
            <
            M
        )
        &
        (
            offs_n[None, :]
            <
            N
        )
    )


    old_weight = tl.load(

        src_ptr
        +
        offs_m[:, None]
        *
        stride_sm
        +
        offs_n[None, :]
        *
        stride_sn,

        mask=
            valid,

        other=
            0.0,
    )


    updated = (
        old_weight
        -
        step_scale
        *
        acc
    )


    tl.store(

        dst_ptr
        +
        offs_m[:, None]
        *
        stride_dm
        +
        offs_n[None, :]
        *
        stride_dn,

        updated,

        mask=
            valid,
    )


# =============================================================================
# GRADIENT NORM
# =============================================================================

def gradient_norm(
    gt,
    z,
):

    if tuple(
        gt.shape
    ) != (
        M,
        FLAT_K,
    ):

        raise RuntimeError(
            f"GT mismatch: {tuple(gt.shape)}"
        )


    if tuple(
        z.shape
    ) != (
        FLAT_K,
        N,
    ):

        raise RuntimeError(
            f"Z mismatch: {tuple(z.shape)}"
        )


    torch.cuda.synchronize()


    norm_kernel[
        norm_grid
    ](
        gt,
        z,

        norm_tiles,

        gt.stride(0),
        gt.stride(1),

        z.stride(0),
        z.stride(1),

        M=M,
        N=N,
        K=FLAT_K,

        NUM_N_TILES=
            NUM_N_TILES,

        BM=
            NBM,

        BN=
            NBN,

        BK=
            NBK,

        num_warps=
            4,
    )


    torch.cuda.synchronize()


    norm_sq = (
        norm_tiles
        .sum(
            dtype=
                torch.float64
        )
    )


    value = float(
        torch.sqrt(
            norm_sq
        )
        .item()
    )


    if not math.isfinite(
        value
    ):

        raise RuntimeError(
            "Non-finite gradient norm."
        )


    return value


# =============================================================================
# KERNEL SMOKE TEST
# =============================================================================

print()
print("=" * 130)
print("TRITON CORRECTNESS SMOKE TEST")
print("=" * 130)


test_gt = (
    torch.randn(
        (
            M,
            FLAT_K,
        ),

        dtype=
            torch.float32,

        device=
            DEVICE,
    )
    *
    1e-3
)


test_z = (
    torch.randn(
        (
            FLAT_K,
            N,
        ),

        dtype=
            torch.float32,

        device=
            DEVICE,
    )
    *
    1e-3
)


triton_test_norm = gradient_norm(
    test_gt,
    test_z,
)


reference_dw = (
    test_gt
    @
    test_z
)


reference_norm = float(
    torch.linalg.vector_norm(
        reference_dw
    )
    .item()
)


NORM_RELATIVE_ERROR = (
    abs(
        triton_test_norm
        -
        reference_norm
    )
    /
    max(
        reference_norm,
        1e-30,
    )
)


print(
    "PyTorch norm             :",
    f"{reference_norm:.8e}",
)


print(
    "Triton norm              :",
    f"{triton_test_norm:.8e}",
)


print(
    "Relative error           :",
    f"{NORM_RELATIVE_ERROR:.8e}",
)


if (
    NORM_RELATIVE_ERROR
    >
    1e-4
):

    raise RuntimeError(
        "Triton norm validation failed."
    )


del test_gt
del test_z
del reference_dw


gc.collect()
torch.cuda.empty_cache()


# =============================================================================
# WRITE PROMPT BATCH
# =============================================================================

def build_write_batch(
    entity,
    correction=False,
):

    templates = (
        CORRECTION_TEMPLATES
        if correction
        else
        WRITE_TEMPLATES
    )


    questions = [
        template.format(
            entity=
                entity
        )

        for template in templates
    ]


    if (
        len(
            questions
        )
        !=
        WRITE_BATCH
    ):

        raise RuntimeError(
            "WRITE_BATCH/template mismatch."
        )


    rows = []

    masks = []

    last_positions = []


    for question in questions:

        prompt = (
            "Delta Registry query.\n"
            "Return only the exact private value.\n\n"
            f"Question: {question}\n"
            "Answer:"
        )


        ids = tokenizer(
            prompt,
            add_special_tokens=
                True,
        )[
            "input_ids"
        ]


        if (
            len(
                ids
            )
            >
            PROMPT_K
        ):

            raise RuntimeError(
                f"Prompt too long: {len(ids)}"
            )


        last_positions.append(
            len(
                ids
            )
            -
            1
        )


        padding = (
            PROMPT_K
            -
            len(
                ids
            )
        )


        rows.append(
            ids
            +
            [
                tokenizer.pad_token_id
            ]
            *
            padding
        )


        masks.append(
            [1]
            *
            len(
                ids
            )
            +
            [0]
            *
            padding
        )


    return (

        torch.tensor(
            rows,
            dtype=
                torch.long,
            device=
                DEVICE,
        ),

        torch.tensor(
            masks,
            dtype=
                torch.long,
            device=
                DEVICE,
        ),

        torch.tensor(
            last_positions,
            dtype=
                torch.long,
            device=
                DEVICE,
        ),
    )


# =============================================================================
# REAL WRITE GRADIENT
# =============================================================================

def capture_write_gradient(
    entity,
    target_secret,
    old_secret=None,
):

    global CAPTURE_MODE
    global CAPTURE_DATA
    global SERVE_FAST


    SERVE_FAST = True


    correction = (
        old_secret
        is not None
    )


    (
        input_ids,
        attention_mask,
        last_positions,
    ) = build_write_batch(
        entity,
        correction=
            correction,
    )


    CAPTURE_DATA = {}


    CAPTURE_MODE = True


    model.zero_grad(
        set_to_none=True
    )


    torch.cuda.synchronize()


    start = time.perf_counter()


    try:

        output = model(

            input_ids=
                input_ids,

            attention_mask=
                attention_mask,

            use_cache=
                False,
        )


        batch_indices = torch.arange(
            WRITE_BATCH,
            device=
                DEVICE,
        )


        answer_logits = (
            output.logits[
                batch_indices,
                last_positions,
                :
            ]
            .float()
        )


        target_ids = torch.full(

            (
                WRITE_BATCH,
            ),

            target_secret[
                "id"
            ],

            dtype=
                torch.long,

            device=
                DEVICE,
        )


        full_ce = F.cross_entropy(
            answer_logits,
            target_ids,
        )


        candidate_logits = (
            answer_logits[
                :,
                GLOBAL_CANDIDATE_IDS
            ]
        )


        target_candidate_index = (
            TOKEN_TO_CANDIDATE_INDEX[
                target_secret[
                    "id"
                ]
            ]
        )


        candidate_targets = torch.full(

            (
                WRITE_BATCH,
            ),

            target_candidate_index,

            dtype=
                torch.long,

            device=
                DEVICE,
        )


        candidate_ce = F.cross_entropy(
            candidate_logits,
            candidate_targets,
        )


        loss = (
            full_ce
            +
            CANDIDATE_LOSS_WEIGHT
            *
            candidate_ce
        )


        margin_loss = torch.tensor(
            0.0,
            device=
                DEVICE,
        )


        if old_secret is not None:

            new_logits = (
                answer_logits[
                    :,
                    target_secret[
                        "id"
                    ]
                ]
            )


            old_logits = (
                answer_logits[
                    :,
                    old_secret[
                        "id"
                    ]
                ]
            )


            margin_loss = (
                F.softplus(
                    old_logits
                    -
                    new_logits
                    +
                    OVERWRITE_MARGIN
                )
                .mean()
            )


            loss = (
                loss
                +
                OVERWRITE_MARGIN_WEIGHT
                *
                margin_loss
            )


        loss_info = {
            "total":
                float(
                    loss
                    .detach()
                    .item()
                ),

            "full_ce":
                float(
                    full_ce
                    .detach()
                    .item()
                ),

            "candidate_ce":
                float(
                    candidate_ce
                    .detach()
                    .item()
                ),

            "margin":
                float(
                    margin_loss
                    .detach()
                    .item()
                ),
        }


        loss.backward()


        torch.cuda.synchronize()


    finally:

        CAPTURE_MODE = False


    gradient_capture_ms = (
        time.perf_counter()
        -
        start
    ) * 1000.0


    gradients = {}


    for layer in FAST_LAYERS:

        data = (
            CAPTURE_DATA[
                layer
            ]
        )


        y = (
            data[
                "y"
            ]
        )


        if y.grad is None:

            raise RuntimeError(
                f"No grad at layer {layer}."
            )


        g = (
            y.grad
            .float()
            .reshape(
                -1,
                M,
            )
            .contiguous()
        )


        gt = (
            g
            .T
            .contiguous()
        )


        gradients[
            layer
        ] = {
            "gt":
                gt,

            "z":
                data[
                    "z"
                ],
        }


    del output
    del input_ids
    del attention_mask
    del last_positions


    CAPTURE_DATA = {}


    gc.collect()

    torch.cuda.empty_cache()


    return (
        gradients,
        loss_info,
        gradient_capture_ms,
    )


# =============================================================================
# APPLY FAST-WEIGHT UPDATE
# =============================================================================

def apply_update(
    gradients,
):

    per_layer_target = (
        TOTAL_DELTA_NORM
        /
        math.sqrt(
            len(
                FAST_LAYERS
            )
        )
    )


    scales = {}

    norms = {}


    for layer in FAST_LAYERS:

        norm = gradient_norm(

            gradients[
                layer
            ][
                "gt"
            ],

            gradients[
                layer
            ][
                "z"
            ],
        )


        norms[
            layer
        ] = norm


        scales[
            layer
        ] = (
            per_layer_target
            /
            max(
                norm,
                1e-30,
            )
        )


    start_event = torch.cuda.Event(
        enable_timing=
            True
    )


    kernel_event = torch.cuda.Event(
        enable_timing=
            True
    )


    publish_event = torch.cuda.Event(
        enable_timing=
            True
    )


    start_event.record()


    for layer in FAST_LAYERS:

        fast = (
            FAST[
                layer
            ]
        )


        gt = (
            gradients[
                layer
            ][
                "gt"
            ]
        )


        z = (
            gradients[
                layer
            ][
                "z"
            ]
        )


        update_kernel[
            update_grid
        ](
            gt,
            z,

            fast.master_active,
            fast.master_staging,

            gt.stride(0),
            gt.stride(1),

            z.stride(0),
            z.stride(1),

            fast.master_active.stride(0),
            fast.master_active.stride(1),

            fast.master_staging.stride(0),
            fast.master_staging.stride(1),

            scales[
                layer
            ],

            M=M,
            N=N,
            K=FLAT_K,

            BM=
                UBM,

            BN=
                UBN,

            BK=
                UBK,

            num_warps=
                8,
        )


    kernel_event.record()


    for layer in FAST_LAYERS:

        fast = (
            FAST[
                layer
            ]
        )


        fast.serve_staging.copy_(
            fast.master_staging
        )


    publish_event.record()


    publish_event.synchronize()


    kernel_ms = (
        start_event.elapsed_time(
            kernel_event
        )
    )


    publish_ms = (
        kernel_event.elapsed_time(
            publish_event
        )
    )


    ready_ms = (
        start_event.elapsed_time(
            publish_event
        )
    )


    commit_states()


    return {
        "kernel_ms":
            kernel_ms,

        "publish_ms":
            publish_ms,

        "ready_ms":
            ready_ms,

        "gradient_norms":
            norms,
    }


# =============================================================================
# EVALUATION PROMPTS
# =============================================================================

def make_query(
    entity,
    paraphrase_index,
):

    return (
        TEST_TEMPLATES[
            paraphrase_index
            %
            len(
                TEST_TEMPLATES
            )
        ]
        .format(
            entity=
                entity
        )
    )


def make_no_context_prompt(
    entity,
    paraphrase_index,
):

    question = make_query(
        entity,
        paraphrase_index,
    )


    return (
        "Delta Registry query.\n"
        "Return only the exact private value.\n\n"
        f"Question: {question}\n"
        "Answer:"
    )


def build_context_text(
    mappings,
):

    lines = [
        "Authoritative Delta Registry:"
    ]


    for record in mappings:

        lines.append(
            f"{record['entity']} = "
            f"{record['secret']['text']}."
        )


    return "\n".join(
        lines
    )


def make_context_prompt(
    context_text,
    entity,
    paraphrase_index,
):

    question = make_query(
        entity,
        paraphrase_index,
    )


    return (
        context_text
        +
        "\n\n"
        "Return only the exact private value.\n\n"
        f"Question: {question}\n"
        "Answer:"
    )


# =============================================================================
# BATCH EVALUATION
# =============================================================================

def evaluate_prompt_records(
    prompt_records,
    use_fast,
    batch_size,
):

    global SERVE_FAST
    global CAPTURE_MODE


    CAPTURE_MODE = False


    old_serve_fast = SERVE_FAST


    SERVE_FAST = bool(
        use_fast
    )


    rows = []


    try:

        for start in range(
            0,
            len(
                prompt_records
            ),
            batch_size,
        ):

            batch_records = (
                prompt_records[
                    start:
                    start
                    +
                    batch_size
                ]
            )


            prompts = [
                record[
                    "prompt"
                ]
                for record in batch_records
            ]


            encoded_rows = [
                tokenizer(
                    prompt,
                    add_special_tokens=
                        True,
                )[
                    "input_ids"
                ]
                for prompt in prompts
            ]


            max_len = max(
                len(
                    ids
                )
                for ids in encoded_rows
            )


            padded = []

            masks = []

            last_positions = []


            for ids in encoded_rows:

                last_positions.append(
                    len(
                        ids
                    )
                    -
                    1
                )


                pad = (
                    max_len
                    -
                    len(
                        ids
                    )
                )


                padded.append(
                    ids
                    +
                    [
                        tokenizer.pad_token_id
                    ]
                    *
                    pad
                )


                masks.append(
                    [1]
                    *
                    len(
                        ids
                    )
                    +
                    [0]
                    *
                    pad
                )


            input_ids = torch.tensor(
                padded,
                dtype=
                    torch.long,
                device=
                    DEVICE,
            )


            attention_mask = torch.tensor(
                masks,
                dtype=
                    torch.long,
                device=
                    DEVICE,
            )


            last_positions_tensor = torch.tensor(
                last_positions,
                dtype=
                    torch.long,
                device=
                    DEVICE,
            )


            with torch.no_grad():

                output = model(

                    input_ids=
                        input_ids,

                    attention_mask=
                        attention_mask,

                    use_cache=
                        False,
                )


                batch_indices = torch.arange(
                    len(
                        batch_records
                    ),
                    device=
                        DEVICE,
                )


                answer_logits = (
                    output.logits[
                        batch_indices,
                        last_positions_tensor,
                        :
                    ]
                    .float()
                )


            candidate_logits = (
                answer_logits[
                    :,
                    GLOBAL_CANDIDATE_IDS
                ]
            )


            candidate_probs = torch.softmax(
                candidate_logits,
                dim=1,
            )


            greedy_ids = (
                torch.argmax(
                    answer_logits,
                    dim=1,
                )
            )


            candidate_pred_indices = (
                torch.argmax(
                    candidate_logits,
                    dim=1,
                )
            )


            candidate_pred_ids = (
                GLOBAL_CANDIDATE_IDS[
                    candidate_pred_indices
                ]
            )


            expected_ids = torch.tensor(
                [
                    record[
                        "expected_secret"
                    ][
                        "id"
                    ]
                    for record in batch_records
                ],

                dtype=
                    torch.long,

                device=
                    DEVICE,
            )


            expected_candidate_indices = torch.tensor(
                [
                    TOKEN_TO_CANDIDATE_INDEX[
                        record[
                            "expected_secret"
                        ][
                            "id"
                        ]
                    ]
                    for record in batch_records
                ],

                dtype=
                    torch.long,

                device=
                    DEVICE,
            )


            row_indices = torch.arange(
                len(
                    batch_records
                ),
                device=
                    DEVICE,
            )


            p_correct = (
                candidate_probs[
                    row_indices,
                    expected_candidate_indices
                ]
            )


            expected_candidate_logits = (
                candidate_logits[
                    row_indices,
                    expected_candidate_indices
                ]
            )


            distractor_logits = (
                candidate_logits
                .clone()
            )


            distractor_logits[
                row_indices,
                expected_candidate_indices
            ] = -float(
                "inf"
            )


            best_distractor = (
                distractor_logits
                .max(
                    dim=1
                )
                .values
            )


            margins = (
                expected_candidate_logits
                -
                best_distractor
            )


            expected_vocab_logits = (
                answer_logits[
                    row_indices,
                    expected_ids
                ]
            )


            ranks = (
                (
                    answer_logits
                    >
                    expected_vocab_logits[
                        :,
                        None
                    ]
                )
                .sum(
                    dim=1
                )
                +
                1
            )


            for offset, record in enumerate(
                batch_records
            ):

                rows.append(
                    {
                        "fact_index":
                            record[
                                "fact_index"
                            ],

                        "entity":
                            record[
                                "entity"
                            ],

                        "expected":
                            record[
                                "expected_secret"
                            ][
                                "text"
                            ],

                        "expected_id":
                            record[
                                "expected_secret"
                            ][
                                "id"
                            ],

                        "paraphrase_index":
                            record[
                                "paraphrase_index"
                            ],

                        "candidate_correct":
                            bool(
                                candidate_pred_ids[
                                    offset
                                ]
                                .item()
                                ==
                                expected_ids[
                                    offset
                                ]
                                .item()
                            ),

                        "greedy_correct":
                            bool(
                                greedy_ids[
                                    offset
                                ]
                                .item()
                                ==
                                expected_ids[
                                    offset
                                ]
                                .item()
                            ),

                        "candidate_prediction_id":
                            int(
                                candidate_pred_ids[
                                    offset
                                ]
                                .item()
                            ),

                        "greedy_id":
                            int(
                                greedy_ids[
                                    offset
                                ]
                                .item()
                            ),

                        "p_correct":
                            float(
                                p_correct[
                                    offset
                                ]
                                .item()
                            ),

                        "margin":
                            float(
                                margins[
                                    offset
                                ]
                                .item()
                            ),

                        "rank":
                            int(
                                ranks[
                                    offset
                                ]
                                .item()
                            ),
                    }
                )


            del output
            del input_ids
            del attention_mask
            del answer_logits
            del candidate_logits
            del candidate_probs


    finally:

        SERVE_FAST = (
            old_serve_fast
        )


    return rows


# =============================================================================
# EVALUATION AGGREGATION
# =============================================================================

def aggregate_rows(
    rows,
):

    if not rows:

        return {
            "candidate_accuracy":
                0.0,

            "greedy_accuracy":
                0.0,

            "mean_p_correct":
                0.0,

            "mean_margin":
                0.0,

            "mean_rank":
                float(
                    "nan"
                ),
        }


    return {
        "candidate_accuracy":
            statistics.mean(
                row[
                    "candidate_correct"
                ]
                for row in rows
            ),

        "greedy_accuracy":
            statistics.mean(
                row[
                    "greedy_correct"
                ]
                for row in rows
            ),

        "mean_p_correct":
            statistics.mean(
                row[
                    "p_correct"
                ]
                for row in rows
            ),

        "mean_margin":
            statistics.mean(
                row[
                    "margin"
                ]
                for row in rows
            ),

        "mean_rank":
            statistics.mean(
                row[
                    "rank"
                ]
                for row in rows
            ),
    }


def aggregate_by_entity(
    rows,
):

    grouped = defaultdict(
        list
    )


    for row in rows:

        grouped[
            row[
                "entity"
            ]
        ].append(
            row
        )


    result = {}


    for entity, entity_rows in grouped.items():

        result[
            entity
        ] = aggregate_rows(
            entity_rows
        )


    return result


def filter_rows_by_entities(
    rows,
    entities,
):

    entity_set = set(
        entities
    )


    return [
        row
        for row in rows
        if (
            row[
                "entity"
            ]
            in entity_set
        )
    ]


# =============================================================================
# MAPPING EVALUATORS
# =============================================================================

def evaluate_no_context(
    mappings,
    paraphrase_count,
    use_fast,
):

    prompt_records = []


    for mapping in mappings:

        for p in range(
            paraphrase_count
        ):

            prompt_records.append(
                {
                    "fact_index":
                        mapping[
                            "index"
                        ],

                    "entity":
                        mapping[
                            "entity"
                        ],

                    "expected_secret":
                        mapping[
                            "secret"
                        ],

                    "paraphrase_index":
                        p,

                    "prompt":
                        make_no_context_prompt(
                            mapping[
                                "entity"
                            ],
                            p,
                        ),
                }
            )


    rows = evaluate_prompt_records(

        prompt_records,

        use_fast=
            use_fast,

        batch_size=
            EVAL_BATCH_SIZE,
    )


    return {
        "summary":
            aggregate_rows(
                rows
            ),

        "rows":
            rows,

        "by_entity":
            aggregate_by_entity(
                rows
            ),
    }


def evenly_sample_mappings(
    mappings,
    count,
):

    if (
        len(
            mappings
        )
        <=
        count
    ):

        return list(
            mappings
        )


    indices = [
        round(
            i
            *
            (
                len(
                    mappings
                )
                -
                1
            )
            /
            (
                count
                -
                1
            )
        )

        for i in range(
            count
        )
    ]


    return [
        mappings[
            index
        ]
        for index in indices
    ]


def evaluate_context_reference(
    all_context_mappings,
    query_mappings,
    paraphrase_count,
):

    context_text = (
        build_context_text(
            all_context_mappings
        )
    )


    prompt_records = []


    for mapping in query_mappings:

        for p in range(
            paraphrase_count
        ):

            prompt_records.append(
                {
                    "fact_index":
                        mapping[
                            "index"
                        ],

                    "entity":
                        mapping[
                            "entity"
                        ],

                    "expected_secret":
                        mapping[
                            "secret"
                        ],

                    "paraphrase_index":
                        p,

                    "prompt":
                        make_context_prompt(
                            context_text,
                            mapping[
                                "entity"
                            ],
                            p,
                        ),
                }
            )


    rows = evaluate_prompt_records(

        prompt_records,

        use_fast=
            False,

        batch_size=
            FULL_CONTEXT_BATCH_SIZE,
    )


    return {
        "summary":
            aggregate_rows(
                rows
            ),

        "rows":
            rows,

        "context_tokens":
            len(
                tokenizer(
                    context_text,
                    add_special_tokens=
                        True,
                )[
                    "input_ids"
                ]
            ),
    }


# =============================================================================
# BASELINE W0 — ALL UNKNOWN FACTS, NO CONTEXT
# =============================================================================

print()
print("=" * 130)
print("W0 NO-CONTEXT BASELINE")
print("=" * 130)


reset_states()


SERVE_FAST = False


W0_ALL = evaluate_no_context(

    WORLD,

    paraphrase_count=
        CHECKPOINT_PARAPHRASES,

    use_fast=
        False,
)


print(
    "Candidate accuracy       :",
    f"{W0_ALL['summary']['candidate_accuracy'] * 100:.2f}%",
)


print(
    "Greedy accuracy          :",
    f"{W0_ALL['summary']['greedy_accuracy'] * 100:.2f}%",
)


print(
    "P(correct)               :",
    f"{W0_ALL['summary']['mean_p_correct']:.6f}",
)


print(
    "Mean full-vocab rank     :",
    f"{W0_ALL['summary']['mean_rank']:.1f}",
)


SERVE_FAST = True


# =============================================================================
# REAL-GRADIENT / TRITON JIT WARMUP
# =============================================================================

print()
print("=" * 130)
print("JIT WARMUP")
print("=" * 130)


reset_states()


first_fact = (
    WORLD[
        0
    ]
)


(
    warm_gradients,
    warm_loss,
    warm_gradient_ms,
) = capture_write_gradient(

    first_fact[
        "entity"
    ],

    first_fact[
        "secret"
    ],
)


warm_update = apply_update(
    warm_gradients
)


print(
    "Gradient capture         :",
    f"{warm_gradient_ms:.3f} ms",
)


print(
    "Kernel                   :",
    f"{warm_update['kernel_ms']:.3f} ms",
)


print(
    "Publish                  :",
    f"{warm_update['publish_ms']:.3f} ms",
)


print(
    "Ready                    :",
    f"{warm_update['ready_ms']:.3f} ms",
)


del warm_gradients


# Completely remove warmup knowledge.
reset_states()


# =============================================================================
# ONLINE STREAMING INSERTION
#
# NO OLD FACT REPLAY.
# =============================================================================

print()
print("=" * 170)
print("ONLINE CAPACITY / RETENTION STRESS")
print("=" * 170)


print(
    f"{'facts':>6}"
    f"{'cand acc':>11}"
    f"{'greedy':>11}"
    f"{'P(correct)':>13}"
    f"{'rank':>11}"
    f"{'oldest':>11}"
    f"{'newest':>11}"
    f"{'full ctx':>11}"
    f"{'trunc ctx':>11}"
    f"{'ctx tok':>10}"
    f"{'KV MiB':>10}"
)


print(
    "-" * 170
)


STREAM_WRITE_HISTORY = []


IMMEDIATE_MEMORY = {}


CAPACITY_RESULTS = []


seen_mappings = []


for fact_number, fact in enumerate(
    WORLD,
    start=1,
):

    # -------------------------------------------------------------------------
    # Online insert.
    #
    # This fact is learned without replaying any previous facts.
    # -------------------------------------------------------------------------

    fact_write_steps = []


    for step in range(
        1,
        INSERT_STEPS
        +
        1
    ):

        (
            gradients,
            loss_info,
            gradient_ms,
        ) = capture_write_gradient(

            fact[
                "entity"
            ],

            fact[
                "secret"
            ],
        )


        update_info = apply_update(
            gradients
        )


        fact_write_steps.append(
            {
                "step":
                    step,

                "loss":
                    loss_info,

                "gradient_ms":
                    gradient_ms,

                "kernel_ms":
                    update_info[
                        "kernel_ms"
                    ],

                "publish_ms":
                    update_info[
                        "publish_ms"
                    ],

                "ready_ms":
                    update_info[
                        "ready_ms"
                    ],
            }
        )


        STREAM_WRITE_HISTORY.append(
            {
                "fact_number":
                    fact_number,

                "fact_index":
                    fact[
                        "index"
                    ],

                "entity":
                    fact[
                        "entity"
                    ],

                "step":
                    step,

                "loss":
                    loss_info,

                "gradient_ms":
                    gradient_ms,

                "kernel_ms":
                    update_info[
                        "kernel_ms"
                    ],

                "publish_ms":
                    update_info[
                        "publish_ms"
                    ],

                "ready_ms":
                    update_info[
                        "ready_ms"
                    ],
            }
        )


        del gradients


    seen_mappings.append(
        {
            "index":
                fact[
                    "index"
                ],

            "entity":
                fact[
                    "entity"
                ],

            "secret":
                fact[
                    "secret"
                ],
        }
    )


    # -------------------------------------------------------------------------
    # Immediate plasticity measurement.
    # -------------------------------------------------------------------------

    immediate_eval = evaluate_no_context(

        [
            seen_mappings[
                -1
            ]
        ],

        paraphrase_count=
            1,

        use_fast=
            True,
    )


    IMMEDIATE_MEMORY[
        fact[
            "entity"
        ]
    ] = (
        immediate_eval[
            "summary"
        ]
    )


    # -------------------------------------------------------------------------
    # Capacity checkpoint.
    # -------------------------------------------------------------------------

    if (
        fact_number
        in
        CAPACITY_CHECKPOINTS
    ):

        fast_eval = evaluate_no_context(

            seen_mappings,

            paraphrase_count=
                CHECKPOINT_PARAPHRASES,

            use_fast=
                True,
        )


        num_seen = len(
            seen_mappings
        )


        quartile = max(
            1,
            num_seen
            //
            4
        )


        oldest_entities = [
            mapping[
                "entity"
            ]
            for mapping in (
                seen_mappings[
                    :
                    quartile
                ]
            )
        ]


        newest_entities = [
            mapping[
                "entity"
            ]
            for mapping in (
                seen_mappings[
                    -quartile:
                ]
            )
        ]


        oldest_rows = filter_rows_by_entities(
            fast_eval[
                "rows"
            ],
            oldest_entities,
        )


        newest_rows = filter_rows_by_entities(
            fast_eval[
                "rows"
            ],
            newest_entities,
        )


        oldest_summary = aggregate_rows(
            oldest_rows
        )


        newest_summary = aggregate_rows(
            newest_rows
        )


        # ---------------------------------------------------------------------
        # Full-context stock reference.
        # ---------------------------------------------------------------------

        reference_queries = evenly_sample_mappings(

            seen_mappings,

            FULL_CONTEXT_SAMPLE_SIZE,
        )


        full_context_eval = (
            evaluate_context_reference(

                seen_mappings,

                reference_queries,

                paraphrase_count=
                    1,
            )
        )


        # ---------------------------------------------------------------------
        # Truncated stock context.
        #
        # Only most recent N facts remain in textual context.
        # ---------------------------------------------------------------------

        truncated_context = (
            seen_mappings[
                -TRUNCATED_CONTEXT_WINDOW:
            ]
        )


        truncated_eval = (
            evaluate_context_reference(

                truncated_context,

                reference_queries,

                paraphrase_count=
                    1,
            )
        )


        context_tokens = (
            full_context_eval[
                "context_tokens"
            ]
        )


        kv_bytes_per_token = (
            NUM_LAYERS
            *
            2
            *
            NUM_KV_HEADS
            *
            HEAD_DIM
            *
            2
        )


        context_kv_mib = (
            context_tokens
            *
            kv_bytes_per_token
            /
            1024**2
        )


        checkpoint_record = {
            "facts":
                fact_number,

            "fast":
                fast_eval[
                    "summary"
                ],

            "oldest_quartile":
                oldest_summary,

            "newest_quartile":
                newest_summary,

            "full_context":
                full_context_eval[
                    "summary"
                ],

            "truncated_context":
                truncated_eval[
                    "summary"
                ],

            "context_tokens":
                context_tokens,

            "context_kv_mib":
                context_kv_mib,
        }


        CAPACITY_RESULTS.append(
            checkpoint_record
        )


        print(
            f"{fact_number:>6}"
            f"{fast_eval['summary']['candidate_accuracy'] * 100:>10.1f}%"
            f"{fast_eval['summary']['greedy_accuracy'] * 100:>10.1f}%"
            f"{fast_eval['summary']['mean_p_correct']:>13.4f}"
            f"{fast_eval['summary']['mean_rank']:>11.1f}"
            f"{oldest_summary['greedy_accuracy'] * 100:>10.1f}%"
            f"{newest_summary['greedy_accuracy'] * 100:>10.1f}%"
            f"{full_context_eval['summary']['greedy_accuracy'] * 100:>10.1f}%"
            f"{truncated_eval['summary']['greedy_accuracy'] * 100:>10.1f}%"
            f"{context_tokens:>10}"
            f"{context_kv_mib:>10.2f}"
        )


# =============================================================================
# FINAL PRE-CORRECTION EVALUATION — 4 HELD-OUT PARAPHRASES
# =============================================================================

print()
print("=" * 130)
print("FINAL PRE-CORRECTION RETENTION")
print("=" * 130)


PRE_CORRECTION_FINAL = evaluate_no_context(

    seen_mappings,

    paraphrase_count=
        FINAL_PARAPHRASES,

    use_fast=
        True,
)


print(
    "Facts                    :",
    len(
        seen_mappings
    ),
)


print(
    "Candidate accuracy       :",
    f"{PRE_CORRECTION_FINAL['summary']['candidate_accuracy'] * 100:.2f}%",
)


print(
    "Greedy accuracy          :",
    f"{PRE_CORRECTION_FINAL['summary']['greedy_accuracy'] * 100:.2f}%",
)


print(
    "P(correct)               :",
    f"{PRE_CORRECTION_FINAL['summary']['mean_p_correct']:.6f}",
)


print(
    "Mean margin              :",
    f"{PRE_CORRECTION_FINAL['summary']['mean_margin']:.6f}",
)


print(
    "Mean rank                :",
    f"{PRE_CORRECTION_FINAL['summary']['mean_rank']:.2f}",
)


# =============================================================================
# IMMEDIATE -> FINAL FORGETTING
# =============================================================================

final_by_entity = (
    PRE_CORRECTION_FINAL[
        "by_entity"
    ]
)


forgetting_values = []


for mapping in seen_mappings:

    entity = (
        mapping[
            "entity"
        ]
    )


    immediate = (
        IMMEDIATE_MEMORY[
            entity
        ]
    )


    final = (
        final_by_entity[
            entity
        ]
    )


    forgetting_values.append(
        immediate[
            "mean_p_correct"
        ]
        -
        final[
            "mean_p_correct"
        ]
    )


MEAN_PROBABILITY_FORGETTING = (
    statistics.mean(
        forgetting_values
    )
)


print()
print(
    "Mean immediate→final P* loss:",
    f"{MEAN_PROBABILITY_FORGETTING:+.6f}",
)


# =============================================================================
# AGE-BUCKET RETENTION
# =============================================================================

print()
print("=" * 130)
print("MEMORY AGE RETENTION")
print("=" * 130)


bucket_size = max(
    1,
    MAX_FACTS
    //
    4
)


AGE_BUCKET_RESULTS = []


for bucket_index in range(
    4
):

    start = (
        bucket_index
        *
        bucket_size
    )


    end = (
        MAX_FACTS
        if bucket_index == 3
        else
        (
            bucket_index
            +
            1
        )
        *
        bucket_size
    )


    bucket_entities = [
        mapping[
            "entity"
        ]
        for mapping in (
            seen_mappings[
                start:
                end
            ]
        )
    ]


    bucket_rows = (
        filter_rows_by_entities(
            PRE_CORRECTION_FINAL[
                "rows"
            ],
            bucket_entities,
        )
    )


    bucket_summary = (
        aggregate_rows(
            bucket_rows
        )
    )


    AGE_BUCKET_RESULTS.append(
        {
            "bucket":
                bucket_index + 1,

            "start_fact":
                start + 1,

            "end_fact":
                end,

            "summary":
                bucket_summary,
        }
    )


    print(
        f"Facts {start + 1:>3}-{end:<3} | "
        f"candidate={bucket_summary['candidate_accuracy'] * 100:>6.2f}% | "
        f"greedy={bucket_summary['greedy_accuracy'] * 100:>6.2f}% | "
        f"P*={bucket_summary['mean_p_correct']:.4f} | "
        f"rank={bucket_summary['mean_rank']:.1f}"
    )


# =============================================================================
# RANDOM ONLINE CORRECTION STRESS
# =============================================================================

print()
print("=" * 170)
print("ONLINE CORRECTION STRESS — 25% OF FACTS")
print("=" * 170)


correction_rng = random.Random(
    SEED + 999
)


CORRECTED_INDICES = sorted(
    correction_rng.sample(
        range(
            MAX_FACTS
        ),
        NUM_CORRECTIONS,
    )
)


CORRECTED_ENTITY_SET = {
    WORLD[
        index
    ][
        "entity"
    ]
    for index in CORRECTED_INDICES
}


UNTOUCHED_ENTITY_SET = {
    mapping[
        "entity"
    ]
    for mapping in seen_mappings
} - CORRECTED_ENTITY_SET


print(
    "Corrections              :",
    len(
        CORRECTED_INDICES
    ),
)


print(
    f"{'corr':>6}"
    f"{'entity':>14}"
    f"{'old P':>11}"
    f"{'new P':>11}"
    f"{'new greedy':>13}"
    f"{'grad ms':>11}"
    f"{'ready':>10}"
)


print(
    "-" * 100
)


CORRECTION_HISTORY = []


# Current expected mapping.
FINAL_MAPPING_BY_INDEX = {
    mapping[
        "index"
    ]:
        {
            "index":
                mapping[
                    "index"
                ],

            "entity":
                mapping[
                    "entity"
                ],

            "secret":
                mapping[
                    "secret"
                ],
        }

    for mapping in seen_mappings
}


for correction_number, world_index in enumerate(
    CORRECTED_INDICES,
    start=1,
):

    fact = (
        WORLD[
            world_index
        ]
    )


    old_secret = (
        fact[
            "secret"
        ]
    )


    new_secret = (
        fact[
            "replacement"
        ]
    )


    correction_step_history = []


    for step in range(
        1,
        OVERWRITE_STEPS
        +
        1
    ):

        (
            gradients,
            loss_info,
            gradient_ms,
        ) = capture_write_gradient(

            fact[
                "entity"
            ],

            new_secret,

            old_secret=
                old_secret,
        )


        update_info = apply_update(
            gradients
        )


        correction_step_history.append(
            {
                "step":
                    step,

                "loss":
                    loss_info,

                "gradient_ms":
                    gradient_ms,

                "kernel_ms":
                    update_info[
                        "kernel_ms"
                    ],

                "publish_ms":
                    update_info[
                        "publish_ms"
                    ],

                "ready_ms":
                    update_info[
                        "ready_ms"
                    ],
            }
        )


        del gradients


    # New-secret immediate evaluation.
    new_mapping = {
        "index":
            fact[
                "index"
            ],

        "entity":
            fact[
                "entity"
            ],

        "secret":
            new_secret,
    }


    old_mapping = {
        "index":
            fact[
                "index"
            ],

        "entity":
            fact[
                "entity"
            ],

        "secret":
            old_secret,
    }


    new_eval = evaluate_no_context(

        [
            new_mapping
        ],

        paraphrase_count=
            1,

        use_fast=
            True,
    )


    old_eval = evaluate_no_context(

        [
            old_mapping
        ],

        paraphrase_count=
            1,

        use_fast=
            True,
    )


    FINAL_MAPPING_BY_INDEX[
        world_index
    ] = new_mapping


    median_grad = statistics.median(
        [
            row[
                "gradient_ms"
            ]
            for row in correction_step_history
        ]
    )


    median_ready = statistics.median(
        [
            row[
                "ready_ms"
            ]
            for row in correction_step_history
        ]
    )


    print(
        f"{correction_number:>6}"
        f"{fact['entity']:>14}"
        f"{old_eval['summary']['mean_p_correct']:>11.4f}"
        f"{new_eval['summary']['mean_p_correct']:>11.4f}"
        f"{new_eval['summary']['greedy_accuracy'] * 100:>12.1f}%"
        f"{median_grad:>11.2f}"
        f"{median_ready:>10.3f}"
    )


    CORRECTION_HISTORY.append(
        {
            "correction_number":
                correction_number,

            "fact_index":
                world_index,

            "entity":
                fact[
                    "entity"
                ],

            "old_secret":
                old_secret,

            "new_secret":
                new_secret,

            "steps":
                correction_step_history,

            "immediate_new":
                new_eval[
                    "summary"
                ],

            "immediate_old":
                old_eval[
                    "summary"
                ],
        }
    )


# =============================================================================
# FINAL POST-CORRECTION WORLD
# =============================================================================

FINAL_MAPPINGS = [
    FINAL_MAPPING_BY_INDEX[
        index
    ]
    for index in range(
        MAX_FACTS
    )
]


POST_CORRECTION_FINAL = evaluate_no_context(

    FINAL_MAPPINGS,

    paraphrase_count=
        FINAL_PARAPHRASES,

    use_fast=
        True,
)


# =============================================================================
# CORRECTED SUBSET
# =============================================================================

corrected_rows = (
    filter_rows_by_entities(
        POST_CORRECTION_FINAL[
            "rows"
        ],
        CORRECTED_ENTITY_SET,
    )
)


CORRECTED_SUMMARY = (
    aggregate_rows(
        corrected_rows
    )
)


# =============================================================================
# UNTOUCHED SUBSET AFTER CORRECTION
# =============================================================================

untouched_rows_after = (
    filter_rows_by_entities(
        POST_CORRECTION_FINAL[
            "rows"
        ],
        UNTOUCHED_ENTITY_SET,
    )
)


UNTOUCHED_AFTER_SUMMARY = (
    aggregate_rows(
        untouched_rows_after
    )
)


untouched_rows_before = (
    filter_rows_by_entities(
        PRE_CORRECTION_FINAL[
            "rows"
        ],
        UNTOUCHED_ENTITY_SET,
    )
)


UNTOUCHED_BEFORE_SUMMARY = (
    aggregate_rows(
        untouched_rows_before
    )
)


# =============================================================================
# OLD-VALUE SUPPRESSION
# =============================================================================

OLD_SECRET_MAPPINGS = [
    {
        "index":
            WORLD[
                index
            ][
                "index"
            ],

        "entity":
            WORLD[
                index
            ][
                "entity"
            ],

        "secret":
            WORLD[
                index
            ][
                "secret"
            ],
    }

    for index in CORRECTED_INDICES
]


OLD_SECRET_AFTER = evaluate_no_context(

    OLD_SECRET_MAPPINGS,

    paraphrase_count=
        FINAL_PARAPHRASES,

    use_fast=
        True,
)


print()
print("=" * 130)
print("POST-CORRECTION SUMMARY")
print("=" * 130)


print(
    "Overall candidate acc    :",
    f"{POST_CORRECTION_FINAL['summary']['candidate_accuracy'] * 100:.2f}%",
)


print(
    "Overall greedy acc       :",
    f"{POST_CORRECTION_FINAL['summary']['greedy_accuracy'] * 100:.2f}%",
)


print(
    "Corrected greedy acc     :",
    f"{CORRECTED_SUMMARY['greedy_accuracy'] * 100:.2f}%",
)


print(
    "Corrected P(new)         :",
    f"{CORRECTED_SUMMARY['mean_p_correct']:.6f}",
)


print(
    "Residual P(old)          :",
    f"{OLD_SECRET_AFTER['summary']['mean_p_correct']:.6f}",
)


print(
    "Untouched acc BEFORE corr:",
    f"{UNTOUCHED_BEFORE_SUMMARY['greedy_accuracy'] * 100:.2f}%",
)


print(
    "Untouched acc AFTER corr :",
    f"{UNTOUCHED_AFTER_SUMMARY['greedy_accuracy'] * 100:.2f}%",
)


print(
    "Untouched retention Δ    :",
    f"{(UNTOUCHED_AFTER_SUMMARY['greedy_accuracy'] - UNTOUCHED_BEFORE_SUMMARY['greedy_accuracy']) * 100:+.2f} pp",
)


# =============================================================================
# TIMING STRESS SUMMARY
# =============================================================================

def percentile(
    values,
    q,
):

    tensor = torch.tensor(
        values,
        dtype=
            torch.float64,
    )


    return float(
        torch.quantile(
            tensor,
            q,
        )
        .item()
    )


insert_gradient_times = [
    row[
        "gradient_ms"
    ]
    for row in STREAM_WRITE_HISTORY
]


insert_kernel_times = [
    row[
        "kernel_ms"
    ]
    for row in STREAM_WRITE_HISTORY
]


insert_publish_times = [
    row[
        "publish_ms"
    ]
    for row in STREAM_WRITE_HISTORY
]


insert_ready_times = [
    row[
        "ready_ms"
    ]
    for row in STREAM_WRITE_HISTORY
]


correction_gradient_times = [
    step[
        "gradient_ms"
    ]
    for correction in CORRECTION_HISTORY
    for step in correction[
        "steps"
    ]
]


correction_ready_times = [
    step[
        "ready_ms"
    ]
    for correction in CORRECTION_HISTORY
    for step in correction[
        "steps"
    ]
]


TIMING_SUMMARY = {

    "insert_gradient": {
        "p50":
            percentile(
                insert_gradient_times,
                0.50,
            ),

        "p95":
            percentile(
                insert_gradient_times,
                0.95,
            ),

        "p99":
            percentile(
                insert_gradient_times,
                0.99,
            ),
    },

    "insert_kernel": {
        "p50":
            percentile(
                insert_kernel_times,
                0.50,
            ),

        "p95":
            percentile(
                insert_kernel_times,
                0.95,
            ),

        "p99":
            percentile(
                insert_kernel_times,
                0.99,
            ),
    },

    "insert_publish": {
        "p50":
            percentile(
                insert_publish_times,
                0.50,
            ),

        "p95":
            percentile(
                insert_publish_times,
                0.95,
            ),

        "p99":
            percentile(
                insert_publish_times,
                0.99,
            ),
    },

    "insert_ready": {
        "p50":
            percentile(
                insert_ready_times,
                0.50,
            ),

        "p95":
            percentile(
                insert_ready_times,
                0.95,
            ),

        "p99":
            percentile(
                insert_ready_times,
                0.99,
            ),
    },

    "correction_gradient": {
        "p50":
            percentile(
                correction_gradient_times,
                0.50,
            ),

        "p95":
            percentile(
                correction_gradient_times,
                0.95,
            ),

        "p99":
            percentile(
                correction_gradient_times,
                0.99,
            ),
    },

    "correction_ready": {
        "p50":
            percentile(
                correction_ready_times,
                0.50,
            ),

        "p95":
            percentile(
                correction_ready_times,
                0.95,
            ),

        "p99":
            percentile(
                correction_ready_times,
                0.99,
            ),
    },
}


print()
print("=" * 130)
print("TIMING SUMMARY")
print("=" * 130)


for name, values in TIMING_SUMMARY.items():

    print(
        f"{name:<22} | "
        f"p50={values['p50']:>8.3f} ms | "
        f"p95={values['p95']:>8.3f} ms | "
        f"p99={values['p99']:>8.3f} ms"
    )


# =============================================================================
# MEMORY / KV GROWTH
# =============================================================================

bytes_per_fast_layer = (
    M
    *
    N
    *
    12
)


FAST_STATE_BYTES = (
    bytes_per_fast_layer
    *
    len(
        FAST_LAYERS
    )
)


KV_BYTES_PER_TOKEN = (
    NUM_LAYERS
    *
    2
    *
    NUM_KV_HEADS
    *
    HEAD_DIM
    *
    2
)


KV_BREAK_EVEN_TOKENS = (
    FAST_STATE_BYTES
    /
    KV_BYTES_PER_TOKEN
)


print()
print("=" * 130)
print("FIXED FAST STATE VS GROWING CONTEXT/KV")
print("=" * 130)


print(
    "Prototype fast state     :",
    f"{FAST_STATE_BYTES / 1024**2:.2f} MiB",
)


print(
    "KV bytes/token           :",
    f"{KV_BYTES_PER_TOKEN / 1024:.2f} KiB",
)


print(
    "KV break-even            :",
    f"{KV_BREAK_EVEN_TOKENS:,.0f} tokens",
)


print()


print(
    f"{'facts':>7}"
    f"{'context tokens':>18}"
    f"{'context KV MiB':>17}"
    f"{'fast state MiB':>18}"
)


print(
    "-" * 65
)


for checkpoint in CAPACITY_RESULTS:

    print(
        f"{checkpoint['facts']:>7}"
        f"{checkpoint['context_tokens']:>18}"
        f"{checkpoint['context_kv_mib']:>17.2f}"
        f"{FAST_STATE_BYTES / 1024**2:>18.2f}"
    )


# =============================================================================
# POC DECISION METRICS
# =============================================================================

capacity_1 = next(
    x
    for x in CAPACITY_RESULTS
    if x[
        "facts"
    ] == 1
)


capacity_max = next(
    x
    for x in CAPACITY_RESULTS
    if x[
        "facts"
    ] == MAX_FACTS
)


CAPACITY_RETENTION_RATIO = (

    capacity_max[
        "fast"
    ][
        "greedy_accuracy"
    ]
    /
    max(
        capacity_1[
            "fast"
        ][
            "greedy_accuracy"
        ],
        1e-12,
    )
)


OLDEST_NEWEST_GAP = (

    capacity_max[
        "newest_quartile"
    ][
        "greedy_accuracy"
    ]
    -
    capacity_max[
        "oldest_quartile"
    ][
        "greedy_accuracy"
    ]
)


print()
print("=" * 130)
print("POC STRESS VERDICT")
print("=" * 130)


print(
    "1-fact greedy accuracy   :",
    f"{capacity_1['fast']['greedy_accuracy'] * 100:.2f}%",
)


print(
    f"{MAX_FACTS}-fact greedy accuracy:",
    f"{capacity_max['fast']['greedy_accuracy'] * 100:.2f}%",
)


print(
    "Capacity retention ratio :",
    f"{CAPACITY_RETENTION_RATIO:.4f}",
)


print(
    "Oldest quartile greedy   :",
    f"{capacity_max['oldest_quartile']['greedy_accuracy'] * 100:.2f}%",
)


print(
    "Newest quartile greedy   :",
    f"{capacity_max['newest_quartile']['greedy_accuracy'] * 100:.2f}%",
)


print(
    "Newest-oldest gap        :",
    f"{OLDEST_NEWEST_GAP * 100:+.2f} pp",
)


print(
    "Mean probability forgetting:",
    f"{MEAN_PROBABILITY_FORGETTING:+.6f}",
)


print(
    "Correction greedy acc    :",
    f"{CORRECTED_SUMMARY['greedy_accuracy'] * 100:.2f}%",
)


print(
    "Residual old-value P     :",
    f"{OLD_SECRET_AFTER['summary']['mean_p_correct']:.6f}",
)


print(
    "Untouched retention Δ    :",
    f"{(UNTOUCHED_AFTER_SUMMARY['greedy_accuracy'] - UNTOUCHED_BEFORE_SUMMARY['greedy_accuracy']) * 100:+.2f} pp",
)


print(
    "Full dW materialized     :",
    "NO",
)


print(
    "Norm scratch             :",
    f"{NUM_NORM_TILES * 4 / 1024:.2f} KiB",
)


# =============================================================================
# SAVE FULL RESULT
# =============================================================================

RESULT = {

    "environment": {
        "model":
            MODEL_ID,

        "python":
            sys.version.split()[0],

        "pytorch":
            torch.__version__,

        "cuda":
            torch.version.cuda,

        "triton":
            triton.__version__,

        "gpu":
            torch.cuda.get_device_name(0),
    },

    "config": {
        "seed":
            SEED,

        "fast_layers":
            FAST_LAYERS,

        "total_delta_norm":
            TOTAL_DELTA_NORM,

        "insert_steps":
            INSERT_STEPS,

        "overwrite_steps":
            OVERWRITE_STEPS,

        "capacity_checkpoints":
            CAPACITY_CHECKPOINTS,

        "max_facts":
            MAX_FACTS,

        "correction_fraction":
            CORRECTION_FRACTION,

        "write_batch":
            WRITE_BATCH,

        "prompt_k":
            PROMPT_K,

        "flat_k":
            FLAT_K,

        "global_candidate_count":
            len(
                GLOBAL_CANDIDATES
            ),
    },

    "kernel": {
        "parity_relative_l2":
            PARITY_REL_L2,

        "parity_max_abs":
            PARITY_MAX_ABS,

        "norm_relative_error":
            NORM_RELATIVE_ERROR,

        "norm_workspace_bytes":
            NUM_NORM_TILES
            *
            4,

        "full_dw_materialized":
            False,
    },

    "baseline": {
        "w0_no_context":
            W0_ALL,
    },

    "capacity": {
        "checkpoints":
            CAPACITY_RESULTS,

        "immediate_memory":
            IMMEDIATE_MEMORY,

        "final_pre_correction":
            PRE_CORRECTION_FINAL,

        "age_buckets":
            AGE_BUCKET_RESULTS,

        "mean_probability_forgetting":
            MEAN_PROBABILITY_FORGETTING,
    },

    "corrections": {
        "corrected_indices":
            CORRECTED_INDICES,

        "history":
            CORRECTION_HISTORY,

        "post_correction_final":
            POST_CORRECTION_FINAL,

        "corrected_summary":
            CORRECTED_SUMMARY,

        "old_secret_after":
            OLD_SECRET_AFTER,

        "untouched_before":
            UNTOUCHED_BEFORE_SUMMARY,

        "untouched_after":
            UNTOUCHED_AFTER_SUMMARY,
    },

    "timing": {
        "summary":
            TIMING_SUMMARY,

        "write_history":
            STREAM_WRITE_HISTORY,
    },

    "memory": {
        "fast_state_bytes":
            FAST_STATE_BYTES,

        "kv_bytes_per_token":
            KV_BYTES_PER_TOKEN,

        "kv_break_even_tokens":
            KV_BREAK_EVEN_TOKENS,

        "capacity_context_growth":
            [
                {
                    "facts":
                        checkpoint[
                            "facts"
                        ],

                    "context_tokens":
                        checkpoint[
                            "context_tokens"
                        ],

                    "context_kv_mib":
                        checkpoint[
                            "context_kv_mib"
                        ],
                }

                for checkpoint in CAPACITY_RESULTS
            ],
    },

    "poc_summary": {
        "capacity_retention_ratio":
            CAPACITY_RETENTION_RATIO,

        "oldest_newest_gap":
            OLDEST_NEWEST_GAP,

        "mean_probability_forgetting":
            MEAN_PROBABILITY_FORGETTING,

        "correction_greedy_accuracy":
            CORRECTED_SUMMARY[
                "greedy_accuracy"
            ],

        "old_value_residual_probability":
            OLD_SECRET_AFTER[
                "summary"
            ][
                "mean_p_correct"
            ],

        "untouched_retention_delta":
            (
                UNTOUCHED_AFTER_SUMMARY[
                    "greedy_accuracy"
                ]
                -
                UNTOUCHED_BEFORE_SUMMARY[
                    "greedy_accuracy"
                ]
            ),
    },
}


with open(
    RESULT_PATH,
    "w",
) as file:

    json.dump(
        RESULT,
        file,
        indent=2,
    )


print()
print("=" * 130)
print("DONE")
print("=" * 130)


print(
    "Saved                   :",
    RESULT_PATH.resolve(),
)


# =============================================================================
# RESTORE ORIGINAL QWEN
# =============================================================================

for layer in FAST_LAYERS:

    FAST[
        layer
    ].module.forward = (
        FAST[
            layer
        ].original_forward
    )


torch.cuda.synchronize()


free_bytes, _ = (
    torch.cuda.mem_get_info()
)


print(
    "Original Qwen restored."
)


print(
    "Free VRAM               :",
    f"{free_bytes / 1024**3:.2f} GiB",
)

FLUSHING PREVIOUS GPU STATE


Free VRAM : 39.08 GiB
Total VRAM: 39.49 GiB

ASYNC-TTT FINAL POC STRESS SUITE
Model                    : Qwen/Qwen2.5-7B-Instruct
Python                   : 3.12.11
PyTorch                  : 2.8.0+cu128
CUDA                     : 12.8
Triton                   : 3.4.0
GPU                      : NVIDIA A100-SXM4-40GB
VRAM                     : 39.49 GiB
Fast layers              : [12, 13, 14, 15]
Capacity checkpoints     : [1, 4, 16, 64, 128]
Final corrections        : 32

LOADING QWEN


Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

Hidden size              : 3584
Intermediate size        : 18944
Layers                   : 28
KV heads                 : 4
Head dim                 : 128
Write batch              : 8
Prompt K                 : 96
Flattened Triton K       : 768
Free after model load    : 24.89 GiB

BUILDING GLOBAL SINGLE-TOKEN SECRET POOL
Safe token pool          : 2048
Global candidates        : 288
Initial secrets          : 128
Correction secrets       : 128
Prior-logit span         : 0.890625
Chance candidate accuracy: 0.3472%

WORLD
Unknown facts            : 128
Example                  : QEVAX-0001 -> avoid

ALLOCATING FAST STATE
Free after fast state    : 21.74 GiB

STATIC BF16 PARITY
Relative L2             : 0.00000000e+00
Max abs                 : 0.00000000e+00

TRITON CONFIG
Norm workspace           : 129.50 KiB
Full dW materialization  : NO

TRITON CORRECTNESS SMOKE TEST
PyTorch norm             : 2.28269294e-01
Triton norm              : 2.28269300e-01
Relative error           : 2.868828